
## UW-Whitewater Scouting Parser (portable version)

Parses UW-Whitewater's own schedule/box-score, each scouted opponent's schedule and FastScout "ScoutBuilder" game-plan PDF, and (further down) play-by-play + video-tagging exports -- turning them into player/lineup-level scouting data, cross-checked against actual results. Ported from a Databricks notebook to run locally via Databricks Connect (Playwright/asyncio/subprocess-based scraping runs on the local machine, not the remote cluster); Databricks-only APIs (`dbutils`, `display()`, widgets, Spark table writes) were swapped for plain Python equivalents everywhere.

### Configuration

Edit `INPUT_DIR`/`OUTPUT_DIR`/`USE_LLM`/`reference_date_str` below for your local setup -- everything downstream reads from these instead of hardcoded paths or Databricks widgets.

In [2]:
# UW-Whitewater Schedule and Scout Report Parser -- Portable Version
#
# Converted from the Databricks notebook. All Databricks-specific dependencies replaced:
# - /Volumes/... and /Workspace/... paths -> configurable INPUT_DIR / OUTPUT_DIR below
# - dbutils.widgets -> plain variables below (edit these directly -- no widget UI outside Databricks)
# - spark.createDataFrame / saveAsTable -> removed (CSV export only)
# - display() -> print()
# - dbutils -> removed
# - ai_query() -> OpenAI client (via player_comparison module, when USE_LLM is enabled)

import email
import glob
import json
import logging
import math
import os
import re
import sys
from collections import Counter
from datetime import datetime
from email import policy
from io import StringIO

import pandas as pd
from bs4 import BeautifulSoup

# --- Configuration -- edit these to match your local setup ---
INPUT_DIR = "./inputs"    # directory containing MHTML/PDF input files
OUTPUT_DIR = "../data"    # directory to write CSV output files
USE_LLM = False            # set True to enable LLM-based player comparisons (requires OPENAI_API_KEY)
reference_date_str = "2026-03-06"   # games on/after this date are treated as not yet played; the first
                                     # scouted UWW game on/after it is flagged as the upcoming game
reference_date = datetime.strptime(reference_date_str, "%Y-%m-%d")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"LLM comparisons: {'enabled' if USE_LLM else 'disabled'}")
print(f"Reference date: {reference_date_str}")

Input directory: ./inputs
Output directory: ../data
LLM comparisons: disabled
Reference date: 2026-03-06



### Legacy install/restart cells (disabled)

These two cells mirror the original Databricks notebook's `%pip install lxml` + `dbutils.library.restartPython()` steps, kept only for parity -- both are commented out here since the portable setup installs dependencies via `requirements.txt` ahead of time, and there's no Databricks kernel to restart outside the web UI.

In [4]:
# %pip install -q lxml

In [5]:
# dbutils.library.restartPython()


### Parse every team's schedule; scrape live from FastScout with a local backup fallback

The largest cell in this notebook -- kept as one block rather than split further, since it's a single, carefully-debugged live-scraping engine and separating it risks pulling apart tightly-coupled state (like the shared Playwright session) from the functions that manage it. It:

* Defines the MHTML/HTML loading helpers (`load_html_snapshot`, `_save_scraped_html`) shared by every other scraping step in this notebook (schedules, scouting-report PDFs, play-by-play, video tagging).
* Defines the one shared, lazily-opened, self-healing FastScout Playwright session (`run_in_fastscout_session`) -- reused everywhere else a live scrape is needed instead of opening a new browser per call.
* Scrapes UW-Whitewater's own schedule and every opponent's schedule live (skipping the live scrape when a local backup file already exists), falling back to a local `"<Team> - Schedule.mhtml"`/`".html"` snapshot when live scraping is unavailable or fails.
* Downloads any missing scouting-report PDF straight from FastScout.
* Produces `schedule`, `team_schedules`, and `scouted_opponents`, used throughout the rest of the notebook.

In [7]:
# Every team's own FastScout schedule snapshot (MHTML) lives in INPUT_DIR alongside per-game pbp/video/box
# MHTML files and scout-report PDFs -- process every "<Team> - Schedule.mhtml" file found there (not just
# UWW's). Filtering strictly on the "- Schedule.mhtml" suffix (not just "*.mhtml") matters here: INPUT_DIR is
# a single flat portable folder (unlike the original Databricks setup, which kept per-game MHTMLs in a
# separate volume from the team-schedule snapshots), so a bare "*.mhtml" glob would also match per-game files
# like "11_14_25 UW-Whitewater @ St. Thomas (TX)_box.mhtml" -- which also contains "whitewater" in its name
# and would otherwise be mistaken for UWW's own schedule snapshot below. Each real schedule file has the same
# 3 <table> structure: [0] game-by-game schedule/results, [1] season player box-score stats, [2] empty
# (template for future games).
import asyncio
import concurrent.futures
import subprocess
import time
import traceback
from urllib.parse import urljoin, urlparse, parse_qs

schedules_dir = INPUT_DIR
# Glob both ".mhtml" (a manually-exported/uploaded snapshot) and ".html" (this notebook's own live-scrape
# cache -- see _save_scraped_html) -- the same *_scout.pdf/*_scout.html duality already used for reports.
schedule_mhtml_paths = sorted(
    p for p in glob.glob(f"{schedules_dir}/*.mhtml") + glob.glob(f"{schedules_dir}/*.html")
    if re.search(r"-\s*Schedule\.(mhtml|html)$", os.path.basename(p), re.IGNORECASE)
)
print(f"Found {len(schedule_mhtml_paths)} schedule MHTML file(s) in {schedules_dir}:")
for p in schedule_mhtml_paths:
    print(" -", os.path.basename(p))


def load_mhtml_html(path):
    """Extract the embedded text/html part from a saved MHTML web page archive."""
    with open(path, "rb") as f:
        raw = f.read()
    msg = email.message_from_bytes(raw, policy=policy.default)
    for part in msg.walk():
        if part.get_content_type() == "text/html":
            charset = part.get_content_charset() or "utf-8"
            return part.get_payload(decode=True).decode(charset, errors="replace")
    return None


def load_html_snapshot(path):
    """Load a saved HTML snapshot from disk, handling both a genuine MHTML web-page archive (from a
    browser's "Save as Webpage, Single File", parsed via load_mhtml_html) AND a plain rendered-HTML file (as
    saved by this notebook's own live-scrape caching -- see _save_scraped_html below) -- mirrors the same
    duality already established for scouting reports ("*_scout.pdf" vs "*_scout.html")."""
    if path.lower().endswith(".mhtml"):
        return load_mhtml_html(path)
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()


def _save_scraped_html(html, dest_path, label):
    """Save raw scraped HTML to disk, the same way scouting reports are already cached as "*_scout.html" --
    confirmed by the user: they want everything this notebook scrapes persisted this way, not just held in
    memory for the current run, so a future run has a local backup if live scraping is unavailable or fails.
    Read back via load_html_snapshot (NOT load_mhtml_html directly -- this is plain HTML, not a real
    multipart MHTML archive)."""
    try:
        with open(dest_path, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"    [save] {label} -> {os.path.basename(dest_path)}")
    except Exception as save_error:
        print(f"    [save] Could not save {label} to {dest_path}: {type(save_error).__name__}: {save_error}")


def split_opponent(text):
    m = re.match(r"^(.*?)(\d+-\d+)$", str(text).strip())
    return (m.group(1).strip(), m.group(2)) if m else (text, None)


def split_result(text):
    m = re.match(r"^([WL])(\d+)-(\d+)$", str(text).strip())
    if not m:
        return (None, None, None)
    return (m.group(1), int(m.group(2)), int(m.group(3)))


FASTSCOUT_ORIGIN = "https://fastscout.fastmodelsports.com"


def _resolve_team_link(href):
    """Normalize a team link href to an absolute fastscout.fastmodelsports.com URL. Live-rendered SPA pages
    often use RELATIVE hrefs (e.g. "/teams/<id>") for internal client-side-routed links, unlike an exported
    MHTML snapshot's browser-resolved absolute hrefs -- urljoin makes both cases resolve the same way."""
    return href if href.startswith("http") else urljoin(FASTSCOUT_ORIGIN, href)


# Games that have a scouting report -- one "*_scout.pdf" file per scouted matchup, named
# "<date> <Team A> @ <Team B>_scout.pdf", sitting alongside the schedule MHTMLs in INPUT_DIR. Used below
# to filter each team's own schedule down to only the games that have a matching scout report.
volume_dir = INPUT_DIR
# Confirmed by the user: no PDF is needed at all -- scouting reports auto-downloaded from FastScout are now
# saved as "*_scout.html" (the live report page's own rendered DOM) instead of trying to reproduce a PDF via
# Chromium's print pipeline, which never worked reliably across headless/headed and every print-media
# variation tried. Manually-uploaded reports stay as "*_scout.pdf" (from before this change) -- glob both so
# either format counts as "this opponent already has a report".
scout_pdf_files = sorted(glob.glob(f"{volume_dir}/*_scout.pdf") + glob.glob(f"{volume_dir}/*_scout.html"))


def scouted_opponents_for(team_name, scout_files):
    team_key = team_name.split()[0].lower()
    opponents = []
    for p in scout_files:
        # Confirmed by a live run: this only stripped "_scout.pdf" -- never updated when "*_scout.html"
        # downloads were added above, so any HTML-sourced report kept its "_scout.html" suffix baked into
        # the parsed opponent name (e.g. "Ripon Red Hawks_scout.html"). That garbled name then never matches
        # the real schedule's plain "Ripon Red Hawks" opponent text below, silently dropping that game from
        # the scouted-opponents filter -- confirmed by a live run where exactly the HTML-sourced AWAY-game
        # opponents (whose name ends up on the right side of " @ ", where the suffix lands) vanished from
        # UWW's own scouted schedule. Strip either extension.
        name = re.sub(r"_scout\.(pdf|html)$", "", os.path.basename(p), flags=re.IGNORECASE)
        name = re.sub(r"^\d+_\d+_\d+\s+", "", name)
        if " @ " not in name:
            continue
        left, right = [side.strip() for side in name.split(" @ ", 1)]
        if team_key in left.lower():
            opponents.append(right)
        elif team_key in right.lower():
            opponents.append(left)
    return opponents


# Parse the schedule dates (format from FastScout is like "Sat, Nov 16") into comparable datetimes.
# Since the schedule spans a single academic year (2025-26), infer the year: months Aug-Dec -> 2025, Jan-Jul -> 2026.
def parse_schedule_date(date_str):
    try:
        parsed = datetime.strptime(date_str.strip(), "%a, %b %d")
        year = 2025 if parsed.month >= 8 else 2026
        return parsed.replace(year=year)
    except (ValueError, AttributeError):
        return None


def build_team_schedule_from_html(html, source_label):
    """Parse one team's own FastScout schedule page HTML (from an MHTML snapshot OR a live scrape) into a
    cleaned schedule DataFrame."""
    page_soup = BeautifulSoup(html, "lxml")
    page_tables = page_soup.find_all("table")

    team_name_raw = page_soup.find("h1").get_text(strip=True)
    team_name = re.match(r"^([^\d]+)", team_name_raw).group(1).strip()

    sched_raw = pd.read_html(StringIO(str(page_tables[0])))[0]

    row_els = [r for r in page_tables[0].find_all("tr") if r.find_all("td")]
    opponent_urls, game_urls, video_urls = [], [], []
    for row_el in row_els:
        hrefs = [a["href"] for a in row_el.find_all("a", href=True)]
        # Match on the "/teams/" path alone (not requiring the full domain) so RELATIVE hrefs from a live
        # SPA render match too -- see _resolve_team_link.
        fastscout_team_links = [h for h in hrefs if "/teams/" in h and "identity.hudl.com" not in h]
        opponent_links = [h for h in fastscout_team_links if "/games/" not in h]
        boxscore_links = [h for h in fastscout_team_links if "/boxscore" in h]
        video_links = [h for h in hrefs if "synergysports.com/video" in h]
        opponent_urls.append(_resolve_team_link(opponent_links[0]) if opponent_links else None)
        game_urls.append(_resolve_team_link(boxscore_links[0]) if boxscore_links else None)
        video_urls.append(video_links[0] if video_links else None)
    sched_raw["opponent_url"] = opponent_urls
    sched_raw["game_url"] = game_urls
    sched_raw["video_url"] = video_urls

    team_schedule = pd.DataFrame()
    team_schedule["date"] = sched_raw["Date"]
    opp_split = sched_raw["Opponent"].apply(split_opponent)
    team_schedule["opponent"] = opp_split.apply(lambda x: x[0])
    team_schedule["team"] = team_name
    team_schedule["location"] = sched_raw["Location"]
    team_schedule["opponent_url"] = sched_raw["opponent_url"]
    team_schedule["game_url"] = sched_raw["game_url"]
    team_schedule["video_url"] = sched_raw["video_url"]
    res_split = sched_raw["Result"].apply(split_result)
    team_schedule["outcome"] = res_split.apply(lambda x: x[0])
    team_schedule["team_score"] = res_split.apply(lambda x: x[1])
    team_schedule["opponent_score"] = res_split.apply(lambda x: x[2])
    team_schedule["point_margin"] = team_schedule["team_score"] - team_schedule["opponent_score"]

    team_schedule["_parsed_date"] = team_schedule["date"].apply(parse_schedule_date)
    is_primary_team = "whitewater" in team_name.lower()

    if is_primary_team:
        scouted_opponents = scouted_opponents_for(team_name, scout_pdf_files)
        is_scouted = team_schedule["opponent"].apply(
            lambda opp: any(re.search(re.escape(short), opp, re.IGNORECASE) for short in scouted_opponents)
        )

        # Determine upcoming from the FULL schedule (not filtered to scouted-only) so that a new
        # opponent whose scout report hasn't been downloaded yet still gets flagged as upcoming and
        # triggers the live-scrape download below.
        team_schedule["Upcoming"] = "No"
        upcoming_idx = None
        upcoming_candidates = team_schedule[team_schedule["_parsed_date"] >= reference_date]
        if not upcoming_candidates.empty:
            upcoming_idx = upcoming_candidates["_parsed_date"].idxmin()
        if upcoming_idx is not None:
            team_schedule.loc[upcoming_idx, "Upcoming"] = "Yes"

        # Keep scouted opponents (played games) + the upcoming game (even if not yet scouted)
        keep_mask = (is_scouted & (team_schedule["_parsed_date"] < reference_date)) | (team_schedule.index == upcoming_idx)
        team_schedule = team_schedule[keep_mask].reset_index(drop=True)
        summary_note = f"scouted opponents: {scouted_opponents}"
    else:
        team_schedule = team_schedule[team_schedule["_parsed_date"] < reference_date].reset_index(drop=True)
        team_schedule["Upcoming"] = "No"
        summary_note = f"all games before {reference_date_str}"

    team_schedule = team_schedule.drop(columns=["_parsed_date"])

    team_schedule.loc[
        team_schedule["Upcoming"] == "Yes",
        team_schedule.columns.difference(["date", "opponent", "Upcoming", "team", "location", "opponent_url", "game_url", "video_url"]),
    ] = None

    print(f"  {team_name} ({source_label}): {len(team_schedule)} game(s) -- {summary_note}")
    return team_schedule, page_soup, page_tables


def build_team_schedule(schedule_path):
    """Parse one team's own saved FastScout schedule snapshot (a genuine ".mhtml" export OR this notebook's
    own live-scrape ".html" cache -- see load_html_snapshot) into a cleaned schedule DataFrame."""
    html = load_html_snapshot(schedule_path)
    return build_team_schedule_from_html(html, source_label=os.path.basename(schedule_path))


def login_to_fastscout(page, username, password, timeout_ms=20000):
    """Log into FastScout via Hudl's Auth0 Universal Login flow. Selectors avoid the dynamic
    React-generated ids/names seen in a saved snapshot of this flow (e.g. "uniId_:r0:") since those
    regenerate every session -- input[type=...] plus button role/name are stable across sessions instead.

    Each step is wrapped separately and re-raises with the URL at that point PLUS any visible on-page error
    text (Auth0's Universal Login shows invalid-credential/MFA/CAPTCHA errors as text on the page itself,
    not as an HTTP error or a distinct exception type) -- otherwise every failure mode collapses into the
    same generic timeout with no way to tell WHY the login didn't go through.
    """

    def _page_error_text():
        for selector in ('[role="alert"]', ".error-message", "#error-element-password", "#error-element-username"):
            try:
                text = page.locator(selector).first.inner_text(timeout=1000)
                if text and text.strip():
                    return text.strip()
            except Exception:
                continue
        return None

    try:
        email_input = page.locator('input[type="email"]')
        email_input.wait_for(timeout=timeout_ms)
        email_input.fill(username)
        # An UNANCHORED "continue" regex also matches Hudl's "Continue with Google/Facebook/Apple" social
        # login buttons on this page, which Playwright's strict mode rejects as an ambiguous match (4
        # elements). Anchoring to the exact button text disambiguates it from those.
        page.get_by_role("button", name=re.compile(r"^continue$", re.IGNORECASE)).click()
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"FastScout login failed at the EMAIL step (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        password_input = page.locator('input[type="password"]')
        password_input.wait_for(timeout=timeout_ms)
        password_input.fill(password)
        page.get_by_role("button", name=re.compile(r"^(continue|log ?in)$", re.IGNORECASE)).click()
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"FastScout login failed at the PASSWORD step (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        page.wait_for_url(re.compile(r"fastscout\.fastmodelsports\.com"), timeout=timeout_ms)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            "FastScout login did not redirect back to fastscout.fastmodelsports.com after submitting "
            f"credentials (still at url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
            + " -- this usually means the username/password was rejected (wrong credentials, an MFA prompt, "
            "or a CAPTCHA), not a code bug."
        ) from e


def _goto_with_auth_retry(page, url, wait_selector, timeout_ms=30000):
    """Navigate to url and wait for wait_selector to appear. FastScout's auth-guard redirect to
    identity.hudl.com is ASYNC client-side JS -- observed to fire well AFTER "domcontentloaded" (and even
    "networkidle") have already been reached, so checking page.url immediately after goto() returns is
    unreliable and can miss it entirely (page.url still showed the ORIGINAL url right after goto(), yet
    ended up on identity.hudl.com by the time wait_for_selector's own timeout had elapsed). Instead, let
    wait_for_selector run its full course; if it fails AND we're on identity.hudl.com by then (checked AFTER
    that wait, when the async redirect has had time to actually happen), log in and retry once."""
    page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
    try:
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
        return
    except Exception:
        if "access_token=" in page.url:
            # A SILENT SSO re-auth (valid Hudl session cookies already present, so no interactive
            # email/password form was ever shown) can complete its ENTIRE identity.hudl.com -> callback ->
            # "#access_token=..." redirect dance WHILE this wait_for_selector call was already polling --
            # confirmed by a live run's own Playwright action log showing exactly that sequence happen
            # mid-wait. By the time it lands back on fastscout.fastmodelsports.com with the token in the URL
            # hash, most/all of the original timeout budget is already spent, leaving the SPA no time to
            # actually consume that token and render the page. This is NOT the "stuck needing interactive
            # login" case below (we're not on identity.hudl.com) -- just give it one more full timeout
            # window to finish rendering on its own instead of failing immediately.
            print(f"    [auth-retry] landed on an in-progress SSO callback ({page.url}) while waiting on {url} -- giving it a fresh wait instead of failing")
            page.wait_for_selector(wait_selector, timeout=timeout_ms)
            return
        if "identity.hudl.com" not in page.url:
            raise

    print(f"    [auth-retry] got redirected to identity.hudl.com while waiting on {url} -- logging in and retrying")
    login_to_fastscout(page, fastscout_username, fastscout_password, timeout_ms=timeout_ms)
    print(f"    [auth-retry] login_to_fastscout returned, now at {page.url} -- re-navigating to {url}")
    page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
    print(f"    [auth-retry] re-navigated, now at {page.url} -- waiting for {wait_selector!r}")
    try:
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
    except Exception:
        # This is the SECOND failure (after an already-completed login) -- dump whatever is actually
        # visible on screen, since a bare timeout gives no clue whether this is a stuck spinner, a repeat
        # MFA/consent prompt, or something else entirely looping back through the auth provider.
        try:
            visible_text = page.locator("body").inner_text(timeout=2000)[:800]
        except Exception as text_error:
            visible_text = f"<could not read body text: {type(text_error).__name__}: {text_error}>"
        print(
            f"    [auth-retry] STILL stuck after retry -- url={page.url}, title={page.title()!r}\n"
            f"    [auth-retry] visible body text (first 800 chars): {visible_text!r}"
        )
        raise


def _click_tab_by_text(page, label, timeout_ms=30000):
    """Click a nav tab/link by its exact visible text (case-insensitive) within the already-loaded FastScout
    SPA. Direct page.goto() to a sub-route URL (e.g. '/games', '/documents') gets silently redirected back to
    '/analytics/dashboard' -- FastScout only renders those views via client-side navigation (an in-app click
    on the corresponding nav tab), not a fresh full-page load. Confirmed by observation: navigating straight
    to '.../games?...' consistently lands on '.../analytics/dashboard' instead, showing the SAME efficiency
    panel regardless of which team's page was requested."""
    page.get_by_text(re.compile(rf"^{re.escape(label)}$", re.IGNORECASE)).first.click(timeout=timeout_ms)


def scrape_rendered_html(page, url, wait_selector="#myTeamSchedule", timeout_ms=30000, save_path=None):
    """Navigate to a FastScout team's base page, then click the 'SCHEDULE' nav tab to reach its schedule
    table client-side -- see _click_tab_by_text for why a direct URL to the schedule sub-route doesn't work.
    If save_path is given, caches the scraped HTML there (see _save_scraped_html) the same way scouting
    reports are cached."""
    print(f"    [scrape] navigating to {url}")
    try:
        _goto_with_auth_retry(page, url, "text=SCHEDULE", timeout_ms)
        print(f"    [scrape] landed on {page.url} -- clicking 'SCHEDULE' tab")
        _click_tab_by_text(page, "SCHEDULE", timeout_ms)
        page.wait_for_selector(wait_selector, timeout=timeout_ms)
        # The container can become visible before its rows finish an async data fetch triggered by the tab
        # click -- wait for an actual <tr> inside it too, not just the container itself.
        page.wait_for_selector(f"{wait_selector} tr", timeout=timeout_ms)
        # This page ALSO renders a second table just below the schedule -- the season player box-score
        # stats table (tables[1] in the next cell) -- which loads via its OWN separate async fetch that the
        # waits above don't cover at all (they only target #myTeamSchedule, i.e. tables[0]). Confirmed by a
        # live run: tables[0] came back fully populated but tables[1] had zero data rows -- same "captured
        # before the async widget finished loading" issue already seen with the ScoutBuilder boxscore Tile.
        # Each populated player row renders its name in a "<div data-id=\"...\">" cell (header cells use
        # col/label/statkey attributes instead, never data-id), so wait on that as a stable marker that this
        # second table has actually loaded before capturing the page.
        #
        # Confirmed by a live run: for 3 opponents this STILL timed out even though the selector resolved to
        # 30+ matching elements on every single poll (e.g. 32 for Aurora) -- Playwright just never considered
        # the first one "visible". That's the signature of a virtualized/lazy-rendered table (react-window
        # style): rows already exist in the DOM with real data (a data-id already set), but with a
        # collapsed/zero-size bounding box until actually scrolled into view -- the exact same lazy-render
        # behavior already confirmed for the ScoutBuilder boxscore Tile, which needed an explicit
        # scroll_into_view_if_needed() to force it to render. Scroll the stats table's own header row into
        # view first (its rows share the same "stat-table-row" class already seen on the header) to trigger
        # that, before waiting on its data cells.
        try:
            page.locator("tr.stat-table-row").first.scroll_into_view_if_needed(timeout=5000)
        except Exception:
            pass
    except Exception as e:
        raise RuntimeError(
            f"Could not reach the schedule view from {url} (actual url={page.url}, title={page.title()!r}): "
            f"{type(e).__name__}: {e}"
        ) from e

    # Confirmed by a live run: for EVERY opponent, this wait timed out even though the schedule table
    # ({wait_selector} and its <tr> rows, both awaited above) had already loaded successfully -- the
    # elements it found (e.g. 30 for Ripon, showing a player name like "Olin Zellmer") belong to a
    # DIFFERENT widget entirely (a roster/top-players panel elsewhere on the team page), not the schedule
    # table's own rows. This step only exists for the season player box-score STATS table (tables[1]),
    # which build_team_schedule_from_html (the only consumer of this function's result) never reads --
    # it only ever uses tables[0], the schedule table already confirmed loaded above. Making a failure here
    # non-fatal instead of raising means an opponent's schedule (already successfully captured) no longer
    # gets thrown away and replaced by a usually-nonexistent local backup file just because this unrelated,
    # unused-by-this-caller widget didn't finish loading.
    try:
        page.wait_for_selector("td div[data-id]", timeout=timeout_ms)
    except Exception as e:
        print(f"    [scrape] (non-fatal) second stats table never loaded, proceeding with schedule table only: {type(e).__name__}: {e}")
    print(f"    [scrape] landed on {page.url}")
    html = page.content()
    if save_path:
        _save_scraped_html(html, save_path, "scraped opponent schedule")
    return html


def _ensure_playwright_ready():
    """Import playwright.sync_api, installing the pip package and/or the Chromium browser binary first if
    either isn't already present. Both the scouting-report-PDF download step and the opponent-schedule
    scraping step below need this, so it's centralized here instead of duplicated in each."""
    try:
        from playwright.sync_api import sync_playwright
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "playwright"])
        from playwright.sync_api import sync_playwright

    # The "playwright" PIP PACKAGE and the actual Chromium BROWSER BINARY install separately, and can drift
    # out of sync after a fresh pip install, an environment rebuild, or a Playwright version bump -- that's
    # exactly what the "Looks like Playwright was just installed or updated -- please run playwright install"
    # message means. Running this every time is a fast no-op once the browser is already downloaded, so
    # there's no need to run it manually in a separate terminal.
    subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
    return sync_playwright


FASTSCOUT_DOCS_LEAGUE = "NCAAB-III"
FASTSCOUT_DOCS_SEASON = "25"


def _ensure_fastscout_login(page, timeout_ms=30000):
    """Log into FastScout if not already authenticated. Uses a KNOWN, always-valid URL ("myTeam" is a
    literal alias FastScout resolves to whichever team the logged-in account belongs to) to reliably trigger
    Hudl's auth redirect, rather than navigating to an arbitrary opponent's page first and hoping the
    redirect behaves consistently.

    IMPORTANT: after a fresh login, FastScout always lands on
    ".../teams/myTeam/analytics/dashboard?league=...&season=..." regardless of what URL originally triggered
    it -- so callers must always explicitly (re-)navigate to wherever they actually want afterward. This
    function only guarantees the session is authenticated, not that the page is showing anything useful.

    Confirmed by the user hitting a "season=25 -> season=26" redirect on this exact "myTeam" URL: FastScout's
    "myTeam" alias appears to resolve to the ACCOUNT'S currently-active season, silently overriding whatever
    `season=` this URL requested once real-world time moves past that season (e.g. the 2025-26 season this
    notebook analyzes was already over by the time this ran). Not fatal by itself (login still succeeds),
    but risky: any view reached via "myTeam" AFTER this redirect -- e.g. scrape_uww_live_schedule's SCHEDULE
    tab click below -- could then silently show the WRONG season's data instead of raising an error. Log a
    loud, explicit warning whenever the landed season doesn't match FASTSCOUT_DOCS_SEASON, rather than
    letting that redirect pass by unnoticed in the ordinary "landed on {page.url}" print below.
    """
    known_url = f"https://fastscout.fastmodelsports.com/teams/myTeam/analytics/dashboard?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}"
    print(f"    [login] navigating to {known_url}")
    _goto_with_auth_retry(page, known_url, "text=SCHEDULE", timeout_ms)
    print(f"    [login] confirmed logged in, landed on {page.url}")

    landed_season = parse_qs(urlparse(page.url).query).get("season", [None])[0]
    if landed_season is not None and landed_season != FASTSCOUT_DOCS_SEASON:
        print(
            f"    [login] WARNING: requested season={FASTSCOUT_DOCS_SEASON!r} but FastScout's \"myTeam\" "
            f"redirect landed on season={landed_season!r} instead -- this account's \"current\" season has "
            f"moved on. Any view reached via \"myTeam\" from here (e.g. the live schedule scrape below) may "
            f"now be showing season {landed_season} data instead of season {FASTSCOUT_DOCS_SEASON}. Verify "
            "the scraped schedule's games actually belong to the intended season before trusting this run's output."
        )

    # Diagnostic: persist this login's own LANDING page HTML too -- the "myTeam" analytics/dashboard view,
    # captured BEFORE any caller clicks away to another tab (e.g. scrape_uww_live_schedule's SCHEDULE click).
    # Confirmed by inspecting the already-saved SCHEDULE tab snapshot: the "Top Rebounders/Scorers/3PT/FT"
    # leaderboard tiles the user asked about do NOT appear there -- so if they exist anywhere on this account,
    # this dashboard landing page (nothing currently captures it) is the next most likely place. One-time save
    # per session bootstrap (this function only runs once per lazily-opened session -- see
    # run_in_fastscout_session) -- cheap, and gives a real snapshot to search instead of guessing selectors.
    try:
        _save_scraped_html(
            page.content(), os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html"),
            "myTeam analytics dashboard (diagnostic)",
        )
    except Exception as dashboard_save_error:
        print(f"    [login] (non-fatal) could not save analytics/dashboard diagnostic snapshot: {type(dashboard_save_error).__name__}: {dashboard_save_error}")


def scrape_uww_live_schedule(page, timeout_ms=30000, save_path=None):
    """Scrape UW-Whitewater's own schedule table live from FastScout, instead of relying on a manually
    exported/uploaded schedule MHTML snapshot. Assumes the page is already on a loaded, authenticated team
    page (e.g. via _ensure_fastscout_login) -- just clicks the 'SCHEDULE' nav tab from there (see
    _click_tab_by_text for why a direct URL to this sub-route doesn't work). If save_path is given, caches
    the scraped HTML there (see _save_scraped_html) the same way scouting reports are cached."""
    print("    [scrape] clicking 'SCHEDULE' tab")
    _click_tab_by_text(page, "SCHEDULE", timeout_ms)
    page.wait_for_selector("#myTeamSchedule", timeout=timeout_ms)
    # The container can become visible before its rows finish an async data fetch triggered by the tab
    # click -- wait for an actual <tr> inside it too, not just the container itself.
    try:
        page.wait_for_selector("#myTeamSchedule tr", timeout=timeout_ms)
    except Exception as e:
        raise RuntimeError(
            f"'#myTeamSchedule' appeared but never got any <tr> rows within {timeout_ms}ms (url={page.url}): "
            f"{type(e).__name__}: {e}"
        ) from e
    print(f"    [scrape] landed on {page.url}")
    html = page.content()
    if save_path:
        _save_scraped_html(html, save_path, "UW-Whitewater's own scraped schedule")
    return html


# FastScout login lives behind Hudl's identity provider. Credentials are resolved from
# FASTSCOUT_USERNAME / FASTSCOUT_PASSWORD environment variables, optionally loaded from a local ".env"
# file (via python-dotenv) sitting next to this notebook. A ".env" file (git-ignored!) would look like:
#   FASTSCOUT_USERNAME=you@example.com
#   FASTSCOUT_PASSWORD=your-password
try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    pass

fastscout_username = os.environ.get("FASTSCOUT_USERNAME")
fastscout_password = os.environ.get("FASTSCOUT_PASSWORD")

if not (fastscout_username and fastscout_password):
    print(
        "No FastScout credentials found (checked FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD env vars / .env) -- "
        "live scraping will be skipped and every opponent will fall back to its local backup MHTML."
    )

# Synergy Sports Tech (auth.synergysportstech.com) is a SEPARATE identity provider from FastScout/Hudl --
# confirmed by a live run that it does NOT share FastScout's username/password (login_to_synergy in the
# video-tagging helpers cell below submitted the FastScout credentials and got a real "Invalid username or
# password" back from Synergy's own server). Resolved the same way, from its own SYNERGY_USERNAME /
# SYNERGY_PASSWORD env vars / ".env" entries:
#   SYNERGY_USERNAME=you@example.com
#   SYNERGY_PASSWORD=your-synergy-password
synergy_username = os.environ.get("SYNERGY_USERNAME")
synergy_password = os.environ.get("SYNERGY_PASSWORD")

if not (synergy_username and synergy_password):
    print(
        "No Synergy credentials found (checked SYNERGY_USERNAME/SYNERGY_PASSWORD env vars / .env) -- "
        "live video-clip scraping will fail for any game not already cached locally."
    )


def find_backup_mhtml(opponent_name, schedules_dir):
    """Find a saved schedule snapshot for this opponent in schedules_dir, matched loosely by name -- either a
    genuine ".mhtml" export or this notebook's own live-scrape ".html" cache (see _save_scraped_html)."""
    for path in sorted(glob.glob(f"{schedules_dir}/*.mhtml") + glob.glob(f"{schedules_dir}/*.html")):
        base = re.sub(r"(_schedule\.(mhtml|html)$|\s*-\s*Schedule\.(mhtml|html)$)", "", os.path.basename(path), flags=re.IGNORECASE)
        if base.lower() in opponent_name.lower():
            return path
    return None


# --- One shared FastScout Playwright session, reused by every scraping step below AND by the live-scrape
# fallbacks in the opponent-pbp/video cells further down -- instead of each opening and closing its own
# browser+thread. Confirmed by the user hitting a Windows greenlet "MemoryError" inside Playwright's own
# dispatch loop after many independent open/close cycles accumulated across repeated cell re-runs in one
# long-lived kernel: this notebook previously opened up to 5 separate sessions per full run (UWW schedule,
# scout PDFs, opponent schedules, plus the 2 pbp/video live-scrape fallbacks) -- now just 1, opened lazily on
# first use and left open/reused for the rest of this kernel session (call close_fastscout_session() to
# explicitly tear it down early if needed; otherwise a kernel restart cleans it up).
_fastscout_session = {"executor": None, "playwright": None, "browser": None, "page": None}


def run_in_fastscout_session(fn):
    """Run fn(page) against the single shared, lazily-opened FastScout session. All Playwright sync-API calls
    for a given browser/page must run on the SAME thread that created them (a brand-new thread has no
    asyncio event loop of its own, sidestepping Playwright's sync API refusing to start on a thread that
    already has one -- see the historical comment this replaced, preserved in git history), so this always
    dispatches through one persistent ThreadPoolExecutor(max_workers=1) worker thread -- created once and
    reused for every call -- rather than a fresh executor + browser + login per call. The Windows
    WindowsProactorEventLoopPolicy swap (needed because Jupyter/ipykernel forces WindowsSelectorEventLoopPolicy
    process-wide, which breaks Playwright's Node-driver asyncio subprocess) only needs to happen once, at
    first-use bootstrap, and is restored immediately after -- not on every call."""
    def _bootstrap():
        original_policy = asyncio.get_event_loop_policy() if sys.platform == "win32" else None
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            sync_playwright = _ensure_playwright_ready()
            playwright = sync_playwright().start()
            browser = playwright.chromium.launch(headless=True)
            page = browser.new_page()
            _ensure_fastscout_login(page)
            _fastscout_session.update(playwright=playwright, browser=browser, page=page)
        finally:
            if sys.platform == "win32":
                asyncio.set_event_loop_policy(original_policy)

    def _discard_stale_session():
        for key, closer in (("browser", "close"), ("playwright", "stop")):
            obj = _fastscout_session.get(key)
            if obj is not None:
                try:
                    getattr(obj, closer)()
                except Exception:
                    pass
        _fastscout_session.update(playwright=None, browser=None, page=None)

    def _job():
        if _fastscout_session["page"] is None:
            _bootstrap()
        # A single automatic retry wasn't always enough -- confirmed by the user hitting the SAME
        # "Page.goto: Connection closed while reading from the driver" error again right after a first
        # reopen-and-retry (a flaky Windows-side Playwright driver subprocess can take more than one restart
        # to recover). Retry up to MAX_DEAD_SESSION_RETRIES times, with a short pause before each fresh
        # bootstrap to let the OS fully release the previous browser/driver process first, instead of giving
        # up (or leaving the shared session permanently broken) after only one attempt.
        MAX_DEAD_SESSION_RETRIES = 2
        for attempt in range(MAX_DEAD_SESSION_RETRIES + 1):
            try:
                return fn(_fastscout_session["page"])
            except Exception as e:
                # The shared browser/driver can die BETWEEN calls (a crash, a timeout, or a Windows-side
                # Playwright driver subprocess issue) even though it was fine at bootstrap. There was
                # previously no liveness check at all, so once the underlying browser died, EVERY subsequent
                # call in this kernel session would keep failing the same way. Treat any exception
                # mentioning a dead connection/driver/target as that signal: discard the stale session and
                # retry against a freshly-bootstrapped one.
                msg = str(e).lower()
                is_dead_session = any(s in msg for s in ("connection closed", "target closed", "browser has been closed", "driver"))
                if is_dead_session and attempt < MAX_DEAD_SESSION_RETRIES:
                    print(f"    [session] Detected a dead FastScout browser session (attempt {attempt + 1}/{MAX_DEAD_SESSION_RETRIES}) -- reopening and retrying.")
                    _discard_stale_session()
                    time.sleep(2)
                    _bootstrap()
                    continue
                raise

    if _fastscout_session["executor"] is None:
        _fastscout_session["executor"] = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    return _fastscout_session["executor"].submit(_job).result()


def close_fastscout_session():
    """Explicitly tear down the shared session (browser + driver + worker thread) to reclaim resources
    without a full kernel restart. Not required between runs -- the session is left open and reused by
    default."""
    def _job():
        if _fastscout_session["browser"] is not None:
            _fastscout_session["browser"].close()
        if _fastscout_session["playwright"] is not None:
            _fastscout_session["playwright"].stop()
        _fastscout_session.update(playwright=None, browser=None, page=None)

    executor = _fastscout_session["executor"]
    if executor is not None:
        executor.submit(_job).result()
        executor.shutdown(wait=False)
        _fastscout_session["executor"] = None


# 1) UW-Whitewater's own schedule: try scraping it LIVE from FastScout first (via the "myTeam" alias, which
#    resolves to whichever team the logged-in account belongs to -- no need to know UWW's own team id), the
#    same live-scrape-with-MHTML-backup pattern already used for every opponent below. Falls back to the
#    local MHTML snapshot if no credentials are set or the live scrape fails for any reason.
uww_mhtml_path = next((p for p in schedule_mhtml_paths if "whitewater" in os.path.basename(p).lower()), None)
uww_html = None
if fastscout_username and fastscout_password:
    # Skip the live scrape entirely when a local schedule file already exists for UWW itself -- same
    # skip-check applied to every opponent below (see _run_fastscout_scrape_session's find_backup_mhtml
    # check), just using uww_mhtml_path (already resolved above via the same "whitewater" filename match)
    # instead of re-deriving it. Confirmed by the user: once a schedule has been captured once, there's no
    # need to pay for a fresh (slow, resource-heavy) live scrape on every subsequent run.
    if uww_mhtml_path is not None:
        print(f"Skipping live scrape for UW-Whitewater's own schedule -- a local schedule file already exists: {os.path.basename(uww_mhtml_path)}")
    else:
        try:
            _uww_save_path = os.path.join(schedules_dir, "UW-Whitewater - Schedule.html")
            uww_html = run_in_fastscout_session(lambda page: scrape_uww_live_schedule(page, save_path=_uww_save_path))
            print("Scraped UW-Whitewater's own schedule live from FastScout (teams/myTeam/games).")
        except Exception as e:
            print(f"Could not scrape UWW's own live schedule, falling back to local MHTML: {type(e).__name__}: {e}")

if uww_html is None and uww_mhtml_path is None:
    raise FileNotFoundError(
        f"No UW-Whitewater schedule MHTML found in {schedules_dir}, and live scraping was unavailable or failed."
    )

# 1b) Before building UWW's (scouted-games-only) schedule, proactively download any MISSING scouting-report
#     PDF straight from each opponent's own FastScout "documents" page, using the same authenticated session
#     -- rather than requiring these to be manually collected ahead of time. This has to run on the RAW
#     (unfiltered) schedule, since the whole point is to discover games that don't have a scout PDF locally
#     YET; build_team_schedule_from_html's own scouted-games filter can't include a game until AFTER this
#     step has filled in its missing report.
def parse_raw_schedule_rows(html):
    """Parse a team's own FastScout schedule page HTML into (date, parsed_date, opponent, location,
    opponent_url) rows, with NO scouted-games filtering applied -- unlike build_team_schedule_from_html,
    which only returns games that ALREADY have a matching scout PDF. Used solely to discover which games are
    missing a report so one can be downloaded before that filter runs for real."""
    page_soup = BeautifulSoup(html, "lxml")
    page_tables = page_soup.find_all("table")
    sched_raw = pd.read_html(StringIO(str(page_tables[0])))[0]

    row_els = [r for r in page_tables[0].find_all("tr") if r.find_all("td")]
    opponent_urls = []
    for row_el in row_els:
        hrefs = [a["href"] for a in row_el.find_all("a", href=True)]
        fastscout_team_links = [h for h in hrefs if "/teams/" in h and "identity.hudl.com" not in h]
        opponent_links = [h for h in fastscout_team_links if "/games/" not in h]
        opponent_urls.append(_resolve_team_link(opponent_links[0]) if opponent_links else None)
    sched_raw["opponent_url"] = opponent_urls

    rows = []
    for _, r in sched_raw.iterrows():
        opponent, _ = split_opponent(r["Opponent"])
        rows.append({
            "date": r["Date"],
            "parsed_date": parse_schedule_date(r["Date"]),
            "opponent": opponent,
            "location": r["Location"],
            "opponent_url": r["opponent_url"],
        })
    return pd.DataFrame(rows)


def _scout_pdf_already_exists(opponent, scout_files):
    opponent_key = opponent.split()[0].lower()
    return any(opponent_key in os.path.basename(p).lower() for p in scout_files)


def _find_matching_scout_document_row(docs_soup, game_date):
    """Within an opponent's FastScout '/documents' page HTML, find the <tr> whose 'Game Date' column matches
    game_date (comparing month/day only, since the visible text format varies). Returns (row_element,
    row_index_within_body_rows) on a match, or (None, seen_dates) with whatever date text WAS found, so a
    non-match can still be explained rather than just failing silently."""
    docs_table = docs_soup.find("table")
    if docs_table is None:
        raise RuntimeError("No <table> found on the documents page.")

    headers = [th.get_text(strip=True) for th in docs_table.find_all("th")]
    try:
        date_col_idx = next(i for i, h in enumerate(headers) if "game date" in h.lower())
    except StopIteration:
        raise RuntimeError(f"No 'Game Date' column found in the documents table headers: {headers}")

    body_rows = [r for r in docs_table.find_all("tr") if r.find_all("td")]
    seen_dates = []
    for row_idx, row_el in enumerate(body_rows):
        cells = row_el.find_all("td")
        if date_col_idx >= len(cells):
            continue
        cell_text = cells[date_col_idx].get_text(strip=True)
        seen_dates.append(cell_text)
        for fmt in ("%m/%d/%Y", "%m/%d/%y", "%b %d, %Y", "%B %d, %Y", "%a, %b %d, %Y", "%a, %b %d"):
            try:
                parsed = datetime.strptime(cell_text, fmt)
            except ValueError:
                continue
            if parsed.month == game_date.month and parsed.day == game_date.day:
                return row_el, row_idx
    return None, seen_dates


def _download_matching_scout_pdf(page, docs_url, opponent, game_date, location, timeout_ms=30000):
    # docs_url's "/documents?..." deep link doesn't render directly via page.goto() (see _click_tab_by_text)
    # -- land on the opponent's own base team page first, then click through to their documents/scouts tab.
    opponent_base_url = docs_url.split("/documents")[0]
    try:
        _goto_with_auth_retry(page, opponent_base_url, "text=SCHEDULE", timeout_ms)
    except Exception as e:
        raise RuntimeError(
            f"Could not load {opponent}'s team page ({opponent_base_url}) (actual url={page.url}, "
            f"title={page.title()!r}): {type(e).__name__}: {e}"
        ) from e

    # Confirmed from saved snapshots: an opponent's tab pointing at this "/documents?..." URL is labeled
    # "SCOUTS" (MY OWN team's equivalent is "SELF SCOUTS") -- but the app ALSO has a global top-nav link
    # also labeled "SCOUTS" (pointing at a different library page), so a plain text match risks clicking the
    # wrong one. Click by href instead of by text to avoid that ambiguity -- but confirmed by an actual run
    # that an EXACT match against the absolute docs_url silently never matched (it landed on the global
    # "/library/opponents" page instead, meaning it fell through to the text-based fallback below and hit
    # the wrong "SCOUTS" link): a live SPA render authors this anchor's href ATTRIBUTE as a relative path
    # (e.g. "/teams/<id>/documents?..."), and browsers don't rewrite the raw attribute value to absolute --
    # only the resolved ".href" PROPERTY -- so an exact match against our absolute docs_url can never hit it.
    # Match on a *suffix* instead, which works whether this specific anchor's href happens to be authored as
    # relative or absolute.
    docs_path = docs_url.replace(FASTSCOUT_ORIGIN, "")
    try:
        page.locator(f'a[href$="{docs_path}"]').first.click(timeout=5000)
        clicked = True
    except Exception:
        clicked = False

    if not clicked:
        for label in ("SCOUTS", "SELF SCOUTS", "DOCUMENTS", "SCOUTING REPORTS"):
            try:
                _click_tab_by_text(page, label, timeout_ms=5000)
                break
            except Exception:
                continue
        else:
            try:
                visible_text = page.locator("body").inner_text(timeout=2000)[:800]
            except Exception:
                visible_text = "<could not read body text>"
            raise RuntimeError(
                f"Could not find a Documents/Scouts tab on {opponent}'s page (url={page.url}, "
                f"title={page.title()!r}) -- visible body text (first 800 chars): {visible_text!r}"
            )

    try:
        page.wait_for_selector("table", timeout=timeout_ms)
    except Exception as e:
        raise RuntimeError(
            f"No <table> appeared after clicking a Documents/Scouts tab for {opponent} (url={page.url}, "
            f"title={page.title()!r}): {type(e).__name__}: {e} -- this may mean no scouting reports exist "
            f"for {opponent} under league={FASTSCOUT_DOCS_LEAGUE!r} season={FASTSCOUT_DOCS_SEASON!r}."
        ) from e

    docs_soup = BeautifulSoup(page.content(), "lxml")
    try:
        matched_row_el, row_idx_or_seen_dates = _find_matching_scout_document_row(docs_soup, game_date)
    except Exception as e:
        # Surface the ACTUAL url/title Playwright ended up on, not just docs_url we asked for -- a table
        # with the wrong headers (e.g. a stats/efficiency panel instead of a documents list) usually means
        # the SPA redirected/rendered a different view than the one we navigated to, and the requested vs.
        # actual url diverging is the key signal for that.
        raise RuntimeError(
            f"Could not find a 'Game Date' column on {docs_url} (actual url={page.url}, title={page.title()!r}): "
            f"{type(e).__name__}: {e}"
        ) from e
    if matched_row_el is None:
        raise RuntimeError(
            f"No row matched game date {game_date.strftime('%b %d')} on {docs_url} -- 'Game Date' values "
            f"seen on the page: {row_idx_or_seen_dates}"
        )
    matched_row_idx = row_idx_or_seen_dates

    # Confirmed by the user: no PDF is needed at all -- auto-downloaded reports are saved as the live report
    # page's own rendered HTML instead (see below), so this filename ends in "_scout.html", not "_scout.pdf".
    matchup = f"{opponent} @ UW-Whitewater" if str(location).strip().lower() == "home" else f"UW-Whitewater @ {opponent}"
    filename = f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')} {matchup}_scout.html"
    dest_path = os.path.join(schedules_dir, filename)

    # games_needing_scout_pdf was already filtered against scout_pdf_files (which now globs both "*_scout.pdf"
    # and "*_scout.html") up front, but that check is a fuzzy match on the OPPONENT'S first name-word against
    # ANY existing report filename -- it can't know the EXACT filename this specific game would produce until
    # game_date/matchup are resolved (both only available here, mid-function). Re-check the precise dest_path
    # too, as a second, exact line of defense, before doing any browser work at all.
    if os.path.exists(dest_path):
        print(f"  [{opponent}] scout report already exists in {schedules_dir} -- skipping download: {filename}")
        return dest_path

    # Prefer a direct <a href="...pdf"> link in the matched row -- fetch it with the browser context's own
    # authenticated cookies (page.context.request) rather than clicking through the UI.
    pdf_hrefs = [a["href"] for a in matched_row_el.find_all("a", href=True) if ".pdf" in a["href"].lower()]
    if pdf_hrefs:
        response = page.context.request.get(pdf_hrefs[0])
        if not response.ok:
            raise RuntimeError(f"GET {pdf_hrefs[0]} returned HTTP {response.status}")
        with open(dest_path, "wb") as f:
            f.write(response.body())
        return dest_path

    # No download button here after all (confirmed by the user) -- the team-name cell's edit icon carries
    # the report's numeric id directly, e.g. id="pencil-729471" -> https://fastscout.fastmodelsports.com/
    # report/729471. Extracting it from the already-parsed row avoids any ambiguous click target entirely.
    pencil_icon = matched_row_el.find(id=re.compile(r"^pencil-\d+$"))
    if pencil_icon is None:
        raise RuntimeError(
            f"Could not find a report id (an element with id='pencil-<id>') in the matched row for "
            f"{opponent} (url={page.url})."
        )
    report_id = pencil_icon["id"].split("-", 1)[1]
    # Confirmed the boxscore widget still stays empty no matter how long we wait, scroll it into view, or add
    # the SAME "?league=...&season=..." query params every other FastScout URL in this notebook carries --
    # none of that fixed it when reaching the report via a fresh page.goto() (a hard full-page load). That
    # matches the EXACT pattern _click_tab_by_text already documented for other sub-routes in this app: a
    # hard page load doesn't carry over whatever client-side app/router state a normal in-app navigation
    # would leave in place, and this SPA only fully renders some views when reached via an actual in-app
    # click. report_url is kept only for the direct-PDF-response check right below (a raw HTTP fetch, not a
    # page navigation, so it's unaffected either way) -- the actual navigation into the report happens further
    # down by CLICKING the pencil icon in the still-live documents-list page instead of goto()'ing this URL.
    report_url = f"{FASTSCOUT_ORIGIN}/report/{report_id}?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}"

    response = page.context.request.get(report_url)
    if not response.ok:
        raise RuntimeError(f"GET {report_url} returned HTTP {response.status}")
    content_type = response.headers.get("content-type", "")
    if "pdf" in content_type.lower():
        with open(dest_path, "wb") as f:
            f.write(response.body())
        return dest_path

    # Not a direct PDF response -- the report renders as an HTML page instead, with the SAME information a
    # manually-exported PDF would have. Rather than writing a parallel HTML-parsing path, render the
    # currently-loaded page straight to a real PDF file via headless Chromium's native print engine
    # (page.pdf() always uses "print" media -- and the report's own CSS already has "hidden-print" classes
    # on UI chrome like the edit icon, so this should closely match what a real "Print"/"Export" action
    # would produce). The saved file then flows through the EXACT SAME PDF-parsing pipeline
    # (read_boxscore_table, extract_pdf_season_stats, etc.) as any manually-provided scout PDF.
    try:
        # CLICK into the report from the live documents-list page (still open in `page` from the matching
        # step above) instead of page.goto()'ing report_url -- see the comment where report_url is built for
        # why a hard full-page load leaves the boxscore widget permanently empty. This is an in-app
        # client-side navigation, exactly like a real user opening the report would trigger.
        #
        # Confirmed from a saved snapshot's own markup: the pencil <i> icon itself has NO click handler --
        # its ancestor <tr class="... scout-row cursor-pointer ..."> is the real click target (the whole row
        # is clickable; "cursor-pointer" on the row, not the icon, is the tell). A prior attempt clicked the
        # icon directly (even with force=True) and it silently did nothing -- no exception, but page.url
        # never changed, so the code went on to save the STILL-open documents-list page as the "report" HTML
        # (confirmed by the user opening the saved file and finding the reports LIST, not an actual report).
        # Click the containing row instead, and explicitly VERIFY the navigation actually happened afterward
        # -- silently saving the wrong page if it doesn't is exactly the bug just described, so this must
        # raise loudly rather than let that repeat.
        row_locator = page.locator(f"tr:has(#pencil-{report_id})").first
        try:
            row_locator.scroll_into_view_if_needed(timeout=5000)
        except Exception:
            pass
        row_locator.click(timeout=timeout_ms)
        try:
            page.wait_for_url(re.compile(r"/report/\d+"), timeout=timeout_ms)
        except Exception:
            pass
        try:
            page.wait_for_load_state("networkidle", timeout=5000)
        except Exception:
            pass
        if "/report/" not in page.url:
            raise RuntimeError(
                f"Clicking the documents-list row for report {report_id} did not navigate into the report "
                f"(still at url={page.url}, title={page.title()!r}) -- refusing to save this page, since it "
                f"would silently be the wrong content (the documents list itself, not the report)."
            )

        # Confirmed by the user: no PDF is needed at all -- every attempt at reproducing a clean PDF via
        # Chromium's print pipeline (headless page.pdf() under default/print media, forcing window.print(),
        # manual CSS/JS hacks under screen media, clicking the real "#print" icon in a headed browser) failed
        # in a different way each time (blank body, leftover chrome frame, broken layout, or a blocking native
        # OS dialog Playwright can't dismiss). Save the live report's own rendered HTML directly instead --
        # parse_scout_html_elements() (Cell 8) reconstructs the same element schema straight from this DOM,
        # which is actually MORE reliable than pdfplumber's text-clustering heuristics on a printed PDF.
        #
        # The season boxscore ("Tile boxscore") specifically is populated by an async API call after the
        # initial page shell loads -- confirmed empty in a "Save Page As" MHTML snapshot for exactly that
        # reason. Wait for its actual text content to appear (there's no <table> tag anywhere in this app at
        # all, so the earlier "wait_for_selector('table')" calls above never actually caught anything real)
        # before saving.
        # Confirmed by a real live download: waiting (even 30s) without ever SCROLLING to the boxscore's
        # page did NOT populate it -- it stayed completely empty. This report renders across 5 separate
        # print-style "pages" (react-grid-layout items), with the boxscore on the LAST one -- a very common
        # pattern for this kind of layout is to lazy-load/virtualize off-screen pages and only fetch a
        # widget's data once it's actually scrolled into view (e.g. via IntersectionObserver). Explicitly
        # scroll the boxscore tile into view first to trigger that, before waiting for its content.
        try:
            page.locator(".Tile.boxscore").scroll_into_view_if_needed(timeout=5000)
        except Exception:
            pass

        # Confirmed fixed once (a real downloaded report showed real boxscore content) but NOT reliable --
        # a later 6-opponent run had this populate for only 1 of 6. Root cause: waiting for ".fa-spin" to
        # clear (below) is an ABSENCE-of-loading inference, not a positive content check -- if the scroll
        # above fires before whatever actually triggers the boxscore's async fetch (a race condition, since
        # this is a one-shot scroll immediately followed by polling), there's simply no spinner to see in the
        # first place, so that loop finds 0 spinners on its very FIRST check and declares victory even though
        # the boxscore table is still completely empty -- confirmed exactly by that run (5 of 6 opponents
        # saved with an empty boxscore, no exception, no spinner ever observed). Wait for an ACTUAL populated
        # row in the boxscore table itself instead, retrying the scroll a few times in case an earlier
        # attempt didn't land while the fetch was still in flight.
        boxscore_loaded = False
        for _scroll_attempt in range(5):
            try:
                page.wait_for_selector(".Tile.boxscore table tbody tr", timeout=6000)
                boxscore_loaded = True
                break
            except Exception:
                try:
                    page.locator(".Tile.boxscore").scroll_into_view_if_needed(timeout=5000)
                except Exception:
                    pass
        if not boxscore_loaded:
            print(f"    [{opponent}] boxscore table never populated after retrying the scroll {5} times -- saving anyway (season-stat cells relying on it will be incomplete).")

        # The 4 playerStatTable tiles (Top Rebounders/Scorers/3PT/FT Shooters) each load via their OWN
        # independent async call too -- confirmed one ("Top Rebounders") was still showing its loading
        # spinner (a "<i class=\"fa fa-refresh fa-spin\">" icon, with an empty <tbody>) even after the
        # boxscore had already finished. Poll for ANY ".fa-spin" element left anywhere on the page as a
        # secondary check covering all of them at once (this absence-of-loading signal is fine here since the
        # boxscore's OWN readiness is now verified by positive content above, not inferred from this alone).
        for _attempt in range(10):
            _spinner_count = page.evaluate("document.querySelectorAll('.fa-spin').length")
            if _spinner_count == 0:
                break
            page.wait_for_timeout(3000)
        else:
            print(f"    [{opponent}] {_spinner_count} loading spinner(s) still visible after 30s -- saving anyway (some stat tiles may be incomplete).")

        with open(dest_path, "w", encoding="utf-8") as f:
            f.write(page.content())
    except Exception as e:
        raise RuntimeError(
            f"GET {report_url} was not a PDF (content-type={content_type!r}), and saving its rendered HTML "
            f"also failed (url={page.url}, title={page.title()!r}): {type(e).__name__}: {e}"
        ) from e
    return dest_path


raw_uww_games = parse_raw_schedule_rows(uww_html if uww_html is not None else load_html_snapshot(uww_mhtml_path))

# Only games already played (parsed_date < reference_date) need a report for retrospective scouting, PLUS
# the single NEXT upcoming game (closest parsed_date >= reference_date) for pre-game prep -- not every game
# still remaining on the schedule after that. Mirrors the same "Upcoming" selection build_team_schedule_from_html
# uses for the primary team, computed independently here since this runs on the unfiltered raw schedule.
valid_dates = raw_uww_games.loc[raw_uww_games["parsed_date"].notna(), "parsed_date"]
upcoming_dates = valid_dates[valid_dates >= reference_date]
next_upcoming_date = upcoming_dates.min() if not upcoming_dates.empty else None


def _is_past_or_next_upcoming(d):
    if d is None:
        return False
    if d < reference_date:
        return True
    return next_upcoming_date is not None and d == next_upcoming_date


in_scope_mask = raw_uww_games["parsed_date"].apply(_is_past_or_next_upcoming)
n_future_excluded = (raw_uww_games["parsed_date"].notna() & ~in_scope_mask & (raw_uww_games["parsed_date"] >= reference_date)).sum()
if n_future_excluded:
    print(f"Excluding {n_future_excluded} game(s) scheduled beyond the next upcoming game -- not downloading those reports yet.")

games_needing_scout_pdf = raw_uww_games[
    raw_uww_games["opponent_url"].notna()
    & raw_uww_games["parsed_date"].notna()
    & in_scope_mask
    & ~raw_uww_games["opponent"].apply(lambda o: _scout_pdf_already_exists(o, scout_pdf_files))
]

if not fastscout_username or not fastscout_password:
    if not games_needing_scout_pdf.empty:
        print(
            f"\n{len(games_needing_scout_pdf)} game(s) are missing a local '*_scout.pdf' report and could be "
            "downloaded automatically from FastScout, but no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were "
            "found -- skipping."
        )
elif raw_uww_games.empty:
    print("\nNo games were found in UWW's raw schedule at all (live scrape or MHTML) -- nothing to download.")
elif games_needing_scout_pdf.empty:
    n_missing_url = raw_uww_games["opponent_url"].isna().sum()
    n_missing_date = raw_uww_games["parsed_date"].isna().sum()
    print(
        f"\nEvery game already has a local scouting-report PDF, or is missing required data -- nothing to "
        f"download from FastScout. ({len(raw_uww_games)} raw game(s) found; {n_missing_url} missing "
        f"opponent_url, {n_missing_date} missing a parseable date.)"
    )
else:
    print(f"\nDownloading {len(games_needing_scout_pdf)} missing scouting-report PDF(s) from FastScout:")

    def _download_scout_pdfs(login_page):
        downloaded = {}
        for _, game_row in games_needing_scout_pdf.iterrows():
            opponent, opponent_url = game_row["opponent"], game_row["opponent_url"]
            game_date, location = game_row["parsed_date"], game_row["location"]
            # opponent_url already carries its own "?league=...&season=..." query string (e.g.
            # ".../teams/<id>?league=NCAAB-III&season=25"), so naively appending
            # "/documents?league=...&season=..." after it lands "/documents" INSIDE that first
            # query string instead of as a real path segment, producing a malformed URL. Strip
            # any existing query string before appending the documents path.
            opponent_base_url = opponent_url.split("?")[0].rstrip("/")
            docs_url = f"{opponent_base_url}/documents?league={FASTSCOUT_DOCS_LEAGUE}&season={FASTSCOUT_DOCS_SEASON}"
            try:
                saved_path = _download_matching_scout_pdf(login_page, docs_url, opponent, game_date, location)
                downloaded[opponent] = saved_path
                print(f"  [{opponent}] saved {os.path.basename(saved_path)}")
            except Exception as doc_error:
                print(f"  [{opponent}] could not download scouting report: {type(doc_error).__name__}: {doc_error}")
        return downloaded

    try:
        run_in_fastscout_session(_download_scout_pdfs)
    except Exception as download_session_error:
        print(
            "  Could not start an authenticated FastScout session for PDF downloads: "
            f"{type(download_session_error).__name__}: {download_session_error}"
        )
        traceback.print_exc()

    # Re-glob so build_team_schedule_from_html's scouted-games filter (right below) picks up whatever PDF(s)
    # were just downloaded.
    # Confirmed by the user: no PDF is needed at all -- scouting reports auto-downloaded from FastScout are now
# saved as "*_scout.html" (the live report page's own rendered DOM) instead of trying to reproduce a PDF via
# Chromium's print pipeline, which never worked reliably across headless/headed and every print-media
# variation tried. Manually-uploaded reports stay as "*_scout.pdf" (from before this change) -- glob both so
# either format counts as "this opponent already has a report".
scout_pdf_files = sorted(glob.glob(f"{volume_dir}/*_scout.pdf") + glob.glob(f"{volume_dir}/*_scout.html"))

if uww_html is not None:
    uww_team_schedule, soup, tables = build_team_schedule_from_html(uww_html, source_label="scraped live (myTeam)")
else:
    uww_team_schedule, soup, tables = build_team_schedule(uww_mhtml_path)
# The next cell (season player box-score stats) expects "soup"/"tables" for UWW specifically.
team_schedules = [uww_team_schedule]
# Exposed at module level (not just inside build_team_schedule_from_html's local scope) since a later cell
# (identifying the upcoming opponent's own prior-game tendencies) also needs UWW's scouted-opponent list. The
# original notebook cell references a bare "scouted_opponents" name that was never actually assigned at module
# level either -- another pre-existing gap in the source notebook, filled in here the same way as
# opponent_from_scout_filename() above.
scouted_opponents = scouted_opponents_for(uww_team_schedule["team"].iloc[0], scout_pdf_files) if not uww_team_schedule.empty else []

# 2) For every opponent that shows up in UWW's own (scouted, date-capped) schedule, scrape their live
#    FastScout team page via opponent_url -- falling back to a local " - Schedule.mhtml" backup on failure.
opponents = (
    uww_team_schedule[["opponent", "opponent_url"]]
    .dropna(subset=["opponent_url"])
    .drop_duplicates(subset=["opponent_url"])
)
print(f"\nFetching {len(opponents)} opponent schedule(s) (scrape opponent_url, MHTML backup on failure):")

scraped_html_by_opponent = {}
if fastscout_username and fastscout_password and opponents.empty:
    print(
        f"  No opponents with a resolvable opponent_url were found in {os.path.abspath(schedules_dir)} -- "
        "skipping the authenticated FastScout session entirely (every opponent will fall back to its local "
        "backup MHTML instead, if one exists). uww_team_schedule likely ended up empty because it's filtered "
        "down to only SCOUTED games, and that filter comes from '*_scout.pdf' files found in INPUT_DIR -- if "
        "none are found there, every game gets filtered out. Double check that INPUT_DIR (the absolute path "
        "above) actually contains all the '*_scout.pdf' scouting-report files alongside the schedule "
        "MHTMLs, not just the schedule MHTMLs themselves."
    )

if fastscout_username and fastscout_password and not opponents.empty:
    def _run_fastscout_scrape_session(login_page):
        # See run_in_fastscout_session above for why this needs to run inside a dedicated worker thread
        # with a temporarily-swapped WindowsProactorEventLoopPolicy on Windows.
        session_results = {}
        for _, opp_row in opponents.iterrows():
            opponent_name, opponent_url = opp_row["opponent"], opp_row["opponent_url"]
            # Skip the live scrape entirely when a local schedule file already exists for this opponent --
            # either a genuine manually-uploaded ".mhtml" export OR this notebook's own live-scrape ".html"
            # cache from a PRIOR run (find_backup_mhtml matches both). Confirmed by the user: once a
            # schedule has been captured once, there's no need to pay for a fresh (slow, resource-heavy)
            # live scrape on every subsequent run. This only skips the ATTEMPT -- the fallback loop below
            # still loads that same file via find_backup_mhtml, so the resulting team_schedules entry is
            # unchanged either way.
            if find_backup_mhtml(opponent_name, schedules_dir):
                print(f"  Skipping live scrape for {opponent_name} -- a local schedule file already exists.")
                continue
            try:
                opp_save_path = os.path.join(schedules_dir, f"{opponent_name} - Schedule.html")
                session_results[opponent_name] = scrape_rendered_html(login_page, opponent_url, save_path=opp_save_path)
            except Exception as scrape_error:
                print(f"  Could not scrape {opponent_name} ({opponent_url}): {type(scrape_error).__name__}: {scrape_error}")
        return session_results

    try:
        scraped_html_by_opponent = run_in_fastscout_session(_run_fastscout_scrape_session)
    except Exception as session_error:
        print(f"  Could not start an authenticated FastScout browser session: {type(session_error).__name__}: {session_error}")
        traceback.print_exc()

for _, opp_row in opponents.iterrows():
    opponent_name, opponent_url = opp_row["opponent"], opp_row["opponent_url"]
    opponent_html = scraped_html_by_opponent.get(opponent_name)
    if opponent_html:
        opp_schedule, _, _ = build_team_schedule_from_html(opponent_html, source_label="scraped live")
        team_schedules.append(opp_schedule)
        continue

    backup_path = find_backup_mhtml(opponent_name, schedules_dir)
    if backup_path:
        print(f"  Falling back to local backup MHTML for {opponent_name}: {os.path.basename(backup_path)}")
        opp_schedule, _, _ = build_team_schedule(backup_path)
        team_schedules.append(opp_schedule)
    else:
        print(f"  No backup MHTML found for {opponent_name} either -- skipping this opponent.")

schedule = pd.concat(team_schedules, ignore_index=True) if team_schedules else pd.DataFrame()
print(schedule)

Found 20 schedule MHTML file(s) in ./inputs:
 - Alma Scots - Schedule.html
 - Aurora Spartans - Schedule.html
 - Carroll (WI) Pioneers - Schedule.html
 - Coe Kohawks - Schedule.html
 - Elmhurst Bluejays - Schedule.html
 - Eureka Red Devils - Schedule.html
 - Hope Flying Dutchmen - Schedule.html
 - Lawrence Vikings - Schedule.html
 - Loras Duhawks - Schedule.html
 - Ripon Red Hawks - Schedule.html
 - Simpson Storm - Schedule.html
 - St. Thomas (TX) Celts - Schedule.html
 - UW-Eau Claire Blugolds - Schedule.html
 - UW-La Crosse Eagles - Schedule.html
 - UW-Oshkosh Titans - Schedule.html
 - UW-Platteville Pioneers - Schedule.html
 - UW-River Falls Falcons - Schedule.html
 - UW-Stevens Point Pointers - Schedule.html
 - UW-Stout Blue Devils - Schedule.html
 - UW-Whitewater - Schedule.mhtml
Skipping live scrape for UW-Whitewater's own schedule -- a local schedule file already exists: UW-Whitewater - Schedule.mhtml
Excluding 1 game(s) scheduled beyond the next upcoming game -- not downloading


### Preview `schedule`

In [9]:
schedule

,date,opponent,team,location,opponent_url,game_url,video_url,outcome,team_score,opponent_score,point_margin,Upcoming
0,"Fri, Nov 7",Ripon Red Hawks,UW-Whitewater Warhawks,Away,https://fastscout.fastmodelsports.com/teams/NY...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,W,76.0,58.0,18.0,No
1,"Fri, Nov 14",St. Thomas (TX) Celts,UW-Whitewater Warhawks,Neutral Court,https://fastscout.fastmodelsports.com/teams/an...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,L,73.0,81.0,-8.0,No
2,"Sat, Nov 15",Eureka Red Devils,UW-Whitewater Warhawks,Neutral Court,https://fastscout.fastmodelsports.com/teams/jV...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,W,116.0,73.0,43.0,No
3,"Wed, Nov 19",Aurora Spartans,UW-Whitewater Warhawks,Home,https://fastscout.fastmodelsports.com/teams/Ux...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,W,82.0,73.0,9.0,No
4,"Tue, Nov 25",Simpson Storm,UW-Whitewater Warhawks,Home,https://fastscout.fastmodelsports.com/teams/gJ...,https://fastscout.fastmodelsports.com/teams/YN...,https://editor-web.synergysports.com/video?pla...,W,100.0,78.0,22.0,No
...,...,...,...,...,...,...,...,...,...,...,...,...
536,"Wed, Feb 11",Luther Norse,Loras Duhawks,Home,https://fastscout.fastmodelsports.com/teams/Z-...,https://fastscout.fastmodelsports.com/teams/aj...,https://editor-web.synergysports.com/video?pla...,W,63.0,58.0,5.0,No
537,"Sat, Feb 14",Nebraska Wesleyan Prairie Wolves,Loras Duhawks,Home,https://fastscout.fastmodelsports.com/teams/g7...,https://fastscout.fastmodelsports.com/teams/aj...,https://editor-web.synergysports.com/video?pla...,W,72.0,66.0,6.0,No
538,"Sat, Feb 21",Buena Vista Beavers,Loras Duhawks,Away,https://fastscout.fastmodelsports.com/teams/Tf...,https://fastscout.fastmodelsports.com/teams/aj...,https://editor-web.synergysports.com/video?pla...,L,62.0,65.0,-3.0,No
539,"Thu, Feb 26",Coe Kohawks,Loras Duhawks,Home,https://fastscout.fastmodelsports.com/teams/jX...,https://fastscout.fastmodelsports.com/teams/aj...,https://editor-web.synergysports.com/video?pla...,W,72.0,71.0,1.0,No



### Extract UW-Whitewater's season player box-score stats

Reads the second `<table>` on the schedule page (`tables[1]`, captured while parsing UWW's own schedule in the previous cell) -- season-long per-player averages, not per-game data.

In [11]:
stats_raw = pd.read_html(StringIO(str(tables[1])))[0]

# This table comes from FastScout's live-rendered "season player box-score" widget (see the extensive
# lazy-load/virtualization workarounds for it elsewhere in this notebook -- it's a genuinely quirky piece of
# markup to scrape). Confirmed downstream (the Streamlit app crashing with "truth value of a Series is
# ambiguous" the moment it read a "PTS" cell out of this table): pd.read_html can come back with two columns
# sharing the exact same header text -- e.g. a sortable-column control or a hidden group label that doesn't
# survive read_html's flattening, so what LOOKS like one header cell in the browser produces two identically-
# named columns here. Any df[col] or row[col] access on a duplicated name then silently returns a Series
# instead of a scalar instead of raising, so this is worth catching loudly right at the source rather than
# letting it surface downstream as a cryptic app crash.
_dupe_cols = stats_raw.columns[stats_raw.columns.duplicated()].unique().tolist()
if _dupe_cols:
    print(f"WARNING: stats_raw has duplicate column name(s) {_dupe_cols} -- keeping the FIRST occurrence of "
          f"each and dropping the rest. If the dropped column actually held different data (e.g. a genuine "
          f"second stat under the same header text) this silently loses it -- inspect stats_raw.columns and "
          f"the raw `tables[1]` HTML directly if that seems possible for this table.")
    stats_raw = stats_raw.loc[:, ~stats_raw.columns.duplicated()]

# Drop the leading unnamed/blank columns (they held player headshot/jersey-icon cells with no text)
stats = stats_raw.drop(columns=[c for c in stats_raw.columns if c.startswith("Unnamed")])
stats = stats.rename(columns={"#": "jersey_number"})

print(stats)

    jersey_number                  PLAYER  GP-GS   MIN      FGM-A    FG%  \
0            14.0            Mikey Wildes    4-0     2    0.0-0.2     0%   
1            24.0           Richie Warren  29-24    22    3.5-8.1  43.2%   
2             0.0            Isaac Verges  27-26    23    3.5-6.7  52.5%   
3            23.0          Mauryon Turner      -     -          -      -   
4            23.0         Maurquis Turner    3-0     3    0.7-1.0  66.7%   
5            40.0  Tyshawn Teague-Johnson   18-1    11    1.7-3.9  42.9%   
6            35.0           Rashad Rogers   19-2     9    0.8-1.7  48.5%   
7             5.0         Isaiah Robinson   20-0     8    0.4-1.0  33.3%   
8            12.0              Jake Quast   28-0    16    2.0-4.0  51.4%   
9            30.0           Matthew McKay    3-0     2    0.7-0.7   100%   
10            2.0           Kelton McEwen   28-0    11    1.1-2.6  41.7%   
11           21.0            Brock Marino  28-27    21    3.2-6.1  52.3%   
12          


### Investigate `playerStatTable` tile widgets (Top Rebounders/Scorers/3PT/FT)

The season stats table above (`tables[1]`) is FastScout's plain per-player-averages `<table>` -- it does NOT cover the separate "Top Rebounders" / "Top Scorers" / "Top 3PT" / "Top FT" leaderboard TILES that also appear somewhere on this schedule page (a different UI widget, presumably div/card-based rather than a `<table>`, so `pd.read_html` over `page_tables` would never see it). This diagnostic cell re-parses the same saved schedule snapshot (`uww_mhtml_path`) with a fresh, full-page `BeautifulSoup` parse and searches for any element that looks like those tiles -- by class name and by known label text -- so their real markup can be confirmed before writing extraction logic for them. Run this cell (it only reads the already-saved local backup file, no live scrape needed) and share the printed output.

In [13]:
# --- Diagnostic: locate the "Top Rebounders/Scorers/3PT/FT" tile widgets on the schedule page -------------
# These are presumed to be a SEPARATE UI widget from the plain season stats table (tables[1], extracted
# above) -- likely rendered as div/card elements, not a <table>, so pd.read_html over page_tables never sees
# them. Re-parse the FULL page (not just the <table> elements already captured in `tables`) and search two
# ways since the exact markup isn't confirmed yet: (1) any element whose class attribute mentions "stat"
# (case-insensitive) that ISN'T a <table> itself, and (2) any element whose own text contains one of the
# known tile labels. Prints just enough of each match (tag, classes, a short text preview) to identify the
# real selector to parse against, without assuming a specific structure upfront.
tile_investigation_html = load_html_snapshot(uww_mhtml_path)
tile_investigation_soup = BeautifulSoup(tile_investigation_html, "lxml")

TILE_LABELS = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
stat_class_matches = [
    el for el in tile_investigation_soup.find_all(True, class_=True)
    if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
]
print(f"  found {len(stat_class_matches)}")
for el in stat_class_matches[:15]:
    preview = el.get_text(" ", strip=True)[:80]
    print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

print(f"\nElements whose text contains a known tile label {TILE_LABELS}:")
found_any = False
for label in TILE_LABELS:
    matches = tile_investigation_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
    if matches:
        found_any = True
    for txt in matches[:5]:
        ancestor = txt.parent
        for _ in range(3):
            if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                break
            ancestor = ancestor.parent
        if ancestor is not None:
            print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
if not found_any:
    print("  (none found in this saved snapshot -- these tiles may not exist on this page, may use "
          "different wording, or may not have been present/loaded when this snapshot was captured)")

print(f"\nTotal <table> elements on this page: {len(tile_investigation_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 27
  <tr class=['stat-table-row', 'stat-table-row']> -- text preview: '# PLAYER GP-GS MIN FGM-A FG% 3PM-A 3P% 3P-R FTM-A FT% PTS ORB DRB REB AST TO STL'
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '14 Mikey Wildes 4 - 0 2 0.0 - 0.2 0% 0.0 - 0.0 - 0% 0.0 - 0.0 - 0.0 0.0 0.2 0.2 '
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '24 Richie Warren 29 - 24 22 3.5 - 8.1 43.2% 0.3 - 1.2 25% 15.3% 1.3 - 2.1 65% 8.'
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '0 Isaac Verges 27 - 26 23 3.5 - 6.7 52.5% 0.4 - 1.2 37.5% 17.7% 2.3 - 3.3 71.6% '
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '23 Mauryon Turner - - - - - - - - - - - - - - - - - -'
  <tr class=['stat-table-row', 'stat-table-row', 'no-border-row']> -- text preview: '23 Maurquis Turner 3 - 0 3 0.7 - 1.0 66.7


### Investigate the `myTeam` analytics/dashboard landing page for the same tile widgets

The diagnostic above found nothing on the SCHEDULE tab -- no `playerStatTable`-style tile widget, and no "Top Rebounders/Scorers/3PT/FT" text anywhere on that page. `_ensure_fastscout_login` (Cell 4) now also saves a diagnostic snapshot of the login's actual LANDING page -- `.../teams/myTeam/analytics/dashboard?...` -- to `myTeam - Analytics Dashboard.html` every time a live FastScout session bootstraps, since that's a different page from SCHEDULE that nothing has captured or searched yet. This cell re-runs the same tile-detection search against that file, once it exists locally (requires a live session with credentials to have run at least once).

In [15]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet -- run a live FastScout session with credentials first, then re-run this cell.")
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0


In [16]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet -- run a live FastScout session with credentials first, then re-run this cell.")
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0


In [17]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(
        f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet in {schedules_dir} -- it's only saved "
        "the first time a live FastScout session actually bootstraps (Cell 4's _ensure_fastscout_login), which "
        "requires FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD to be set. Run any cell that live-scrapes at least once "
        "with credentials, then re-run this cell."
    )
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0


In [18]:
dashboard_snapshot_path = os.path.join(schedules_dir, "myTeam - Analytics Dashboard.html")

if not os.path.exists(dashboard_snapshot_path):
    print(
        f"'{os.path.basename(dashboard_snapshot_path)}' doesn't exist yet in {schedules_dir} -- it's only saved "
        "the first time a live FastScout session actually bootstraps (Cell 4's _ensure_fastscout_login), which "
        "requires FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD to be set. Run any cell that live-scrapes at least once "
        "with credentials, then re-run this cell."
    )
else:
    dashboard_html = load_html_snapshot(dashboard_snapshot_path)
    dashboard_soup = BeautifulSoup(dashboard_html, "lxml")
    dash_tile_labels = ["Top Rebounders", "Top Scorers", "Top 3PT", "Top FT", "Top Assists", "Leaders"]

    print("Elements with a 'stat'-like class attribute (excluding <table> itself):")
    dash_stat_matches = [
        el for el in dashboard_soup.find_all(True, class_=True)
        if el.name != "table" and any("stat" in c.lower() for c in el.get("class", []))
    ]
    print(f"  found {len(dash_stat_matches)}")
    for el in dash_stat_matches[:15]:
        preview = el.get_text(" ", strip=True)[:80]
        print(f"  <{el.name} class={el.get('class')}> -- text preview: {preview!r}")

    print(f"\nElements whose text contains a known tile label {dash_tile_labels}:")
    dash_found_any = False
    for label in dash_tile_labels:
        matches = dashboard_soup.find_all(string=re.compile(re.escape(label), re.IGNORECASE))
        if matches:
            dash_found_any = True
        for txt in matches[:5]:
            ancestor = txt.parent
            for _ in range(3):
                if ancestor is None or ancestor.get("class") or ancestor.parent is None:
                    break
                ancestor = ancestor.parent
            if ancestor is not None:
                print(f"  [{label}] <{ancestor.name} class={ancestor.get('class')}> -- text preview: {ancestor.get_text(' ', strip=True)[:120]!r}")
    if not dash_found_any:
        print("  (none found on the analytics/dashboard page either)")

    print(f"\nTotal <table> elements on this page: {len(dashboard_soup.find_all('table'))}")

Elements with a 'stat'-like class attribute (excluding <table> itself):
  found 4
  <div class=['Tile', 'teamStatsLineup', 'top-left-tile']> -- text preview: 'LINEUP STATS Loading...'
  <div class=['teamStatsLineup']> -- text preview: 'Loading...'
  <div class=['Tile', 'teamAdvancedStatsTable', 'top-left-tile']> -- text preview: 'ADVANCED STATS Loading...'
  <div class=['teamAdvancedStatsTable']> -- text preview: 'Loading...'

Elements whose text contains a known tile label ['Top Rebounders', 'Top Scorers', 'Top 3PT', 'Top FT', 'Top Assists', 'Leaders']:
  [Leaders] <a class=['helvetica-neue-bold', 'uppercase']> -- text preview: 'Leaders'

Total <table> elements on this page: 0



### Summarize UW-Whitewater's season record and home/away/neutral splits

`schedule` spans every team found in `schedules_dir` (UWW's own games plus each opponent's) -- filtered down to just UWW's own rows before aggregating the win/loss record and average point-margin splits by location.

In [20]:
# "schedule" now spans every team found in schedules_dir (UWW's own games plus any opponent's, e.g.
# Elmhurst) -- this record/splits summary is specifically about UW-Whitewater, so filter down to just its
# rows before aggregating; otherwise the win/loss counts and splits below would blend in other teams' games too.
uww_schedule = schedule[schedule["team"] == "UW-Whitewater Warhawks"]

record_counts = uww_schedule["outcome"].value_counts()
print("Overall record (W-L):", f"{record_counts.get('W', 0)}-{record_counts.get('L', 0)}")

split_summary = (
    uww_schedule.groupby(["location", "outcome"])
    .size()
    .unstack(fill_value=0)
    .assign(games=lambda d: d.sum(axis=1))
)
print(split_summary.reset_index())

avg_margin_by_location = uww_schedule.groupby("location")["point_margin"].mean().round(1)
print(avg_margin_by_location.reset_index().rename(columns={"point_margin": "avg_point_margin"}))

Overall record (W-L): 20-7
outcome       location  L   W  games
0                 Away  2   9     11
1                 Home  4  10     14
2        Neutral Court  1   1      2
        location  avg_point_margin
0           Away               7.9
1           Home               6.6
2  Neutral Court              17.5



### Parse each opponent's FastScout "ScoutBuilder" game-plan report

Scouting reports are sourced exclusively from PDF exports (MHTML snapshots leave live-data stat widgets empty). Since `ai_parse_document()` only runs on serverless AI Functions compute, this falls back to `pdfplumber` and reconstructs the same `section_header`/`text`/`table` element schema used by every downstream cell, looping over every `"*_scout.pdf"` file found in `INPUT_DIR`.

In [22]:
# FastScout "ScoutBuilder" game-plan reports, now sourced EXCLUSIVELY from PDF exports -- MHTML is retired.
# PDF exports render everything server-side before printing, so their tables are fully populated (unlike MHTML
# "Save Page As" snapshots, whose live-data stat widgets are empty placeholders). The original ai_parse_document()
# approach fails on this classic cluster because that routine only runs on serverless AI Functions compute, so this
# cell falls back to pdfplumber and reconstructs the same element schema used downstream: section_header / text / table.
# Loop through every "*_scout.pdf" file in the inputs volume so newly added reports are picked up automatically
# without touching this code. 
try:
    import pdfplumber
except ModuleNotFoundError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pdfplumber"])
    import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)

GAME_PLAN_HEADERS = {
    "TEAM STRENGTHS", "KEYS TO VICTORY",
    "Overall Defensive Scheme", "Attacking their man defense", "Ball Screen & DHO Defense",
    "Ball Screen Actions & Reads", "Speciality Defensive Notes", "Potential Adjustments",
    "Defending Their Action", "Overall OFFENSIVE SCHEME", "Ball Screen Actions & Personnel",
    "Ball Screen Coverage(s)", "ELOB & SLOB",
}
PLAYER_LINE_RE = re.compile(r"^#\d+\s*•")
PAGE_NUM_RE = re.compile(r"^\d+\s+of\s+\d+$")

volume_dir = INPUT_DIR
# Confirmed by the user: no PDF is ever needed -- reports auto-downloaded going forward are saved as the live
# report page's own rendered HTML ("*_scout.html", see Cell 4) instead of a Chromium-printed PDF, which never
# rendered cleanly no matter the approach tried. Manually-uploaded reports from before this change stay as
# "*_scout.pdf". Both are parsed below (by their own dedicated parser) into the identical element schema, so
# every downstream cell keeps working unchanged regardless of which format a given opponent's report is in.
scout_pdf_files = sorted(glob.glob(f"{volume_dir}/*_scout.pdf"))
scout_html_files = sorted(glob.glob(f"{volume_dir}/*_scout.html"))
scout_report_files = sorted(scout_pdf_files + scout_html_files)
print(f"Found {len(scout_pdf_files)} PDF scout report(s) and {len(scout_html_files)} HTML scout report(s):")
for f in scout_report_files:
    print(" -", os.path.basename(f))


def normalize_text(text):
    text = text.replace("• ", "•").replace(" •", "•")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def grouped_lines(page):
    # Bucketing words by a bare round(word["top"]) is fragile: the LEFT half's baseline ("Player Notes:")
    # and the RIGHT half's baseline ("Keys to Defending:") of the SAME visual header line can land on
    # different sub-pixel tops that round to ADJACENT integers (e.g. 556 vs 557) rather than the same one.
    # When that happens, the two halves get split into two separate one-sided rows, the "Player Notes" +
    # "Keys to Defending" combined-header detection below never fires on either row, and that whole player's
    # notes/keys bullets get silently dropped (confirmed for Michael Asman and Kolby Williams on the Ripon
    # PDF's STARTERS page -- their header row split 556/557 and 684/685). Cluster words within a small
    # vertical tolerance into the same row instead of relying on exact/rounded top equality.
    split_x = page.width / 2
    words = page.extract_words()
    ordered = sorted(words, key=lambda w: (w["top"], w["x0"]))
    clusters = []
    row_top_tolerance = 2
    for word in ordered:
        if clusters and abs(word["top"] - clusters[-1]["top"]) <= row_top_tolerance:
            clusters[-1]["words"].append(word)
        else:
            clusters.append({"top": word["top"], "words": [word]})

    rows = []
    for cluster in clusters:
        line_words = sorted(cluster["words"], key=lambda w: w["x0"])
        rows.append({
            "top": cluster["top"],
            "left": normalize_text(" ".join(w["text"] for w in line_words if w["x0"] < split_x)),
            "right": normalize_text(" ".join(w["text"] for w in line_words if w["x0"] >= split_x)),
            "full": normalize_text(" ".join(w["text"] for w in line_words)),
        })
    return rows


def parse_game_plan_page(page, elements):
    segments = []
    for side_name in ["left", "right"]:
        current = None
        for row in grouped_lines(page):
            if row["top"] < 80:
                continue
            text = row[side_name]
            if not text or PAGE_NUM_RE.fullmatch(text) or text in {"STARTERS", "BENCH"}:
                continue
            header_candidate = text.rstrip(" -•")
            if header_candidate in GAME_PLAN_HEADERS:
                current = {
                    "top": row["top"],
                    "side": 0 if side_name == "left" else 1,
                    "header": header_candidate,
                    "texts": [],
                }
                segments.append(current)
            elif text.upper() == text and len(text) > 8 and not re.match(r"^\d+\.", text):
                continue
            elif current is not None:
                current["texts"].append(text)

    for segment in sorted(segments, key=lambda s: (s["top"], s["side"])):
        elements.append(("section_header", segment["header"]))
        for text in segment["texts"]:
            elements.append(("text", text))


def parse_roster_page(page, elements):
    in_notes_block = False
    for row in grouped_lines(page):
        if row["top"] < 80:
            continue
        left, right, full = row["left"], row["right"], row["full"]
        if PAGE_NUM_RE.fullmatch(full):
            continue
        if full in {"STARTERS", "BENCH"}:
            elements.append(("section_header", full))
            in_notes_block = False
            continue
        if PLAYER_LINE_RE.match(left or full):
            elements.append(("text", left or full))
            in_notes_block = False
            continue
        if (
            left.startswith("GP-GS")
            or full.startswith("GP-GS")
            or left.startswith("Last Season")
            or full.startswith("Last Season")
            or re.match(r"^\d{2}-\d{2}\s*\(", left or full)
            or re.match(r"^\d{2}-\d{2}\s*\(", full)
        ):
            continue
        if "Player Notes" in full or "Keys to Defending" in full:
            in_notes_block = True
            continue
        if in_notes_block:
            if left:
                elements.append(("section_header", "Player Notes"))
                elements.append(("text", left))
            if right:
                elements.append(("section_header", "Keys to Defending"))
                elements.append(("text", right))


def parse_boxscore_page(page, elements):
    lines = [
        row["full"].replace("", "").strip()
        for row in grouped_lines(page)
        if row["top"] >= 80 and row["full"].strip()
    ]
    box_idx = next((i for i, line in enumerate(lines) if "BOXSCORE" in line.upper()), None)
    header_idx = next((i for i in range(box_idx + 1, len(lines)) if lines[i].startswith("# PLAYER ")), None) if box_idx is not None else None
    if box_idx is None or header_idx is None:
        return

    stop_idx = next(
        (
            i for i in range(header_idx + 1, len(lines))
            if lines[i].startswith("Top Scorers")
            or lines[i].startswith("3PT Shooters")
            or PAGE_NUM_RE.fullmatch(lines[i])
        ),
        len(lines),
    )
    header_tokens = lines[header_idx].split()
    stat_cols = header_tokens[2:]
    rows = []
    for line in lines[header_idx + 1:stop_idx]:
        tokens = line.split()
        if len(tokens) < 2 + len(stat_cols):
            continue
        name = " ".join(tokens[1:-len(stat_cols)])
        if name == "TeamTotal":
            name = "Team Total"
        rows.append([tokens[0], name] + tokens[-len(stat_cols):])

    if rows:
        box_df = pd.DataFrame(rows, columns=["#", "PLAYER"] + stat_cols)
        elements.append(("section_header", lines[box_idx]))
        elements.append(("table", box_df.to_html(index=False)))


def parse_scout_pdf_elements(path):
    elements = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            if "BOXSCORE" in page_text:
                parse_boxscore_page(page, elements)
            elif "STARTERS" in page_text or "BENCH" in page_text or re.search(r"#\d+•", page_text):
                parse_roster_page(page, elements)
            else:
                parse_game_plan_page(page, elements)

    # pd.DataFrame([]) on an empty list produces a DataFrame with NO COLUMNS at all (not just 0 rows) --
    # explicitly pin the expected columns so a PDF that yields zero elements (e.g. a page layout these
    # heuristics don't recognize) still reports "0 elements, 0 tables" downstream instead of crashing with
    # a KeyError on element_type.
    return pd.DataFrame(
        [
            {"element_index": idx, "element_type": element_type, "element_content": element_content}
            for idx, (element_type, element_content) in enumerate(elements)
        ],
        columns=["element_index", "element_type", "element_content"],
    )


# NOTE: home/away-aware opponent-name extraction for scout report filenames -- ported as-is from the original
# notebook cell, which called an "opponent_from_scout_filename()" that was never actually defined there
# either (a pre-existing bug in the source notebook, not introduced by this portable conversion). This local
# helper fills that gap using the same "<date> <A> @ <B>_scout.<ext>" filename convention documented below --
# handles both the legacy "_scout.pdf" and the new "_scout.html" extension.
def opponent_from_scout_filename(path):
    name = re.sub(r"_scout\.(pdf|html)$", "", os.path.basename(path))
    name = re.sub(r"^\d+_\d+_\d+\s+", "", name)
    if " @ " not in name:
        return name
    left, right = [side.strip() for side in name.split(" @ ", 1)]
    return right if "whitewater" in left.lower() else left


def normalize_html_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _has_exact_class(tag, cls):
    # BeautifulSoup's class_=lambda predicate is invoked once PER CLASS TOKEN (as a bare string), not once
    # per tag with the full class list -- so a naive "cls in tag['class']" substring/membership check done
    # the wrong way (e.g. "Tile" in "EditableTile") can silently match an unrelated wrapper class too. Do the
    # membership check explicitly against the tag's own parsed class LIST instead, so only an exact token
    # match counts (confirmed: without this, "Tile" incorrectly matched "EditableTile" and duplicated every
    # game-plan/roster element).
    classes = tag.get("class")
    return bool(classes) and cls in classes


def _draft_editor_blocks(tile):
    """Every paragraph/bullet line inside a Tile's rich-text (Draft.js) content shares the class
    'public-DraftStyleDefault-block', whether it's wrapped in an <li> (bulleted/numbered list) or a plain
    <div> (unformatted lines, e.g. KEYS TO VICTORY's numbered lines) -- selecting on that class uniformly
    covers both cases in visual top-to-bottom order, instead of special-casing <li> vs plain paragraphs."""
    draft = tile.find(class_="public-DraftEditor-content")
    if draft is None:
        return []
    blocks = [b for b in draft.find_all(True) if _has_exact_class(b, "public-DraftStyleDefault-block")]
    return [normalize_html_text(b.get_text(" ", strip=True)) for b in blocks]


def parse_scout_html_elements(path):
    """HTML counterpart to parse_scout_pdf_elements() -- reconstructs the identical element schema
    (element_index, element_type in {section_header, text, table}, element_content) directly from the
    ScoutBuilder report's own live DOM (saved by Cell 4's download step), instead of from a printed PDF.
    Confirmed structurally MORE reliable than pdfplumber's text-clustering heuristics: game-plan bullets,
    player notes, and keys-to-defending are each their own clean DOM node here, with no PDF column-merging
    to work around (the Eureka-style merged notes/keys blob that split_combined_notes_keys() exists for in
    Cell 9 simply can't happen with this source).
    """
    with open(path, "r", encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html, "html.parser")
    printable = soup.find(class_="PrintableNode")
    elements = []
    if printable is None:
        return pd.DataFrame([], columns=["element_index", "element_type", "element_content"])

    # ---- game-plan tiles: any exact-class "Tile" whose title span matches a known header, in document order.
    # Confirmed the site does NOT reliably split these across pages the same way every time (unlike the PDF
    # export) -- parsing the whole PrintableNode in one pass sidesteps relying on any particular page boundary.
    for tile in printable.find_all("div"):
        if not _has_exact_class(tile, "Tile"):
            continue
        title_el = tile.find("span", class_="scout-tile-title")
        if title_el is None:
            continue
        header = normalize_html_text(title_el.get_text(strip=True))
        if header not in GAME_PLAN_HEADERS:
            continue
        elements.append(("section_header", header))
        for block in _draft_editor_blocks(tile):
            elements.append(("text", block))

    # ---- roster tiles: STARTERS/BENCH section headers interleaved with each player's own "playerGroup" tile,
    # in document order (this interleaving matters -- Cell 9 infers Starter vs Bench role from each player's
    # position relative to the BENCH header).
    for tile in printable.find_all("div"):
        if not _has_exact_class(tile, "Tile"):
            continue
        classes = tile.get("class")
        if "section" in classes:
            header_text = normalize_html_text(tile.get_text(" ", strip=True))
            if header_text in ("STARTERS", "BENCH"):
                elements.append(("section_header", header_text))
        elif "playerGroup" in classes:
            info_span = tile.find("span", class_="player-info-line")
            if info_span is None:
                continue
            info_div = info_span.find("div", class_=lambda c: c and "display-flex" in c)
            # Direct-child <span>s only (jersey/name/pos/height/weight/class in order) -- reading them
            # positionally (rather than via .stripped_strings, which silently DROPS an empty span, e.g. a
            # player missing a weight) keeps all 6 fields and 5 "•" separators intact even when one field is
            # blank, matching Cell 9's PLAYER_LINE_RE exactly (it already tolerates an empty weight group).
            field_spans = info_div.find_all("span", recursive=False) if info_div else []
            fields = [normalize_html_text(s.get_text(" ", strip=True)) for s in field_spans]
            elements.append(("text", " • ".join(fields)))

            for text_tile in tile.find_all("div"):
                if not (_has_exact_class(text_tile, "Tile") and _has_exact_class(text_tile, "text")):
                    continue
                blocks = [b for b in _draft_editor_blocks(text_tile) if b]
                if not blocks:
                    continue
                label = blocks[0].lower()
                if label.startswith("player notes"):
                    section_name = "Player Notes"
                elif label.startswith("keys to defending"):
                    section_name = "Keys to Defending"
                else:
                    continue
                elements.append(("section_header", section_name))
                for block in blocks:
                    if block.lower().startswith(section_name.lower()):
                        continue
                    elements.append(("text", block))

    # ---- season boxscore: confirmed against a real live-downloaded report -- once loaded (Cell 4's download
    # step explicitly waits for it), it's a genuine <table> (not a div-grid as originally guessed), with the
    # same PLAYER/Team Total/Opponent row shape as the PDF version. Emit the same (section_header, "table")
    # element pair the PDF path produces (a "...BOXSCORE" header immediately followed by a table element) so
    # extract_team_totals_from_pdf() downstream picks it up unchanged regardless of source format.
    boxscore_tile = next(
        (t for t in printable.find_all("div") if _has_exact_class(t, "Tile") and _has_exact_class(t, "boxscore")),
        None,
    )
    if boxscore_tile is not None:
        boxscore_table = boxscore_tile.find("table")
        boxscore_tbody = boxscore_table.find("tbody") if boxscore_table is not None else None
        if boxscore_tbody is not None and boxscore_tbody.find("tr") is not None:
            title_el = boxscore_tile.find("span", class_="scout-tile-title")
            header_text = normalize_html_text(title_el.get_text(strip=True)) if title_el else "BOXSCORE"
            elements.append(("section_header", header_text))
            elements.append(("table", str(boxscore_table)))
        else:
            print(
                f"    NOTE: '{os.path.basename(path)}' boxscore Tile has no populated table yet (still "
                f"loading, or the wait in Cell 4 didn't catch it this time) -- season-stat cells relying on "
                f"it will be incomplete for this opponent."
            )

    return pd.DataFrame(
        [
            {"element_index": idx, "element_type": element_type, "element_content": element_content}
            for idx, (element_type, element_content) in enumerate(elements)
        ],
        columns=["element_index", "element_type", "element_content"],
    )


scout_reports = {}  # opponent short name (from filename) -> parsed element table (element_index, element_type, element_content)
for path in scout_report_files:
    # Filenames are either "<date> UW-Whitewater @ <Opponent>_scout.<ext>" (away game) or
    # "<date> <Opponent> @ UW-Whitewater_scout.<ext>" (home game, e.g. the Aurora report) -- reuse the
    # same home/away-aware extraction defined above instead of always taking the text after "@", which
    # would mislabel every home game's report as "UW-Whitewater". Dispatch to the parser matching this
    # specific file's format -- older manually-uploaded reports are PDFs, auto-downloaded ones are HTML.
    opponent_short = opponent_from_scout_filename(path)
    parser_fn = parse_scout_html_elements if path.lower().endswith(".html") else parse_scout_pdf_elements
    elements_df = parser_fn(path)
    scout_reports[opponent_short] = elements_df
    n_tables = (elements_df["element_type"] == "table").sum()
    print(f"  Parsed '{opponent_short}' ({os.path.splitext(path)[1]}): {len(elements_df)} elements, {n_tables} tables")

# Flag any opponent that only has an MHTML report -- since MHTML is no longer used as a source at all, they're
# excluded from every downstream cell until a PDF version is added for them too.
mhtml_only_files = sorted(glob.glob(f"{volume_dir}/*_scout.mhtml"))
mhtml_opponents = {re.search(r"@ (.+)_scout\.mhtml$", os.path.basename(f)).group(1) for f in mhtml_only_files}
dropped_opponents = mhtml_opponents - set(scout_reports)
if dropped_opponents:
    print(f"\nWARNING: no PDF scout report exists yet for {sorted(dropped_opponents)} -- "
          f"MHTML is retired as a source, so these opponents are excluded from the analysis below.")

Found 0 PDF scout report(s) and 24 HTML scout report(s):
 - 11_14_25 UW-Whitewater @ St. Thomas (TX) Celts_scout.html
 - 11_15_25 UW-Whitewater @ Eureka Red Devils_scout.html
 - 11_19_25 Aurora Spartans @ UW-Whitewater_scout.html
 - 11_25_25 Simpson Storm @ UW-Whitewater_scout.html
 - 11_7_25 UW-Whitewater @ Ripon Red Hawks_scout.html
 - 12_10_25 UW-Whitewater @ Lawrence Vikings_scout.html
 - 12_13_25 Carroll (WI) Pioneers @ UW-Whitewater_scout.html
 - 12_19_25 UW-Whitewater @ Hope Flying Dutchmen_scout.html
 - 12_20_25 UW-Whitewater @ Alma Scots_scout.html
 - 12_2_25 Elmhurst Bluejays @ UW-Whitewater_scout.html
 - 12_30_25 UW-Whitewater @ Coe Kohawks_scout.html
 - 1_10_26 UW-Whitewater @ UW-River Falls Falcons_scout.html
 - 1_14_26 UW-Whitewater @ UW-La Crosse Eagles_scout.html
 - 1_17_26 UW-Stout Blue Devils @ UW-Whitewater_scout.html
 - 1_21_26 UW-Whitewater @ UW-Platteville Pioneers_scout.html
 - 1_24_26 UW-Eau Claire Blugolds @ UW-Whitewater_scout.html
 - 1_3_26 UW-Oshkosh Titans 


### Extract each opponent's roster, player notes, and keys to defending

Parses the `"#<jersey> • <name> • <pos> • <height> • <weight> • <class>"` identity lines out of each scouting report's parsed elements, along with the "Player Notes" and "Keys to Defending" text that follows each player. Starter/bench role is read from the report's own "BENCH" section header when present, falling back to roster order (1st-5th = Starter) for reports that omit it.

In [24]:
# Player identity lines in the PDF's parsed text look like "#<jersey> \u2022 <name> \u2022 <pos> \u2022 <height> \u2022 <weight> \u2022
# <class>", each followed by a "Player Notes:" section_header + text and a "Keys to Defending" section_header +
# text. Starter/bench split (and roster size) varies by opponent, so it's derived from the "BENCH" section
# header's position when the PDF actually prints one. Some PDFs (e.g. Eureka) never print a "BENCH" header at
# all even though later players still have Player Notes/Keys to Defending filled in (Damuzha Moore, Jacob
# Gonzalez) -- for those, fall back to roster order: the 1st-5th players listed are Starters, 6th onward Bench.
PLAYER_LINE_RE = re.compile(r"^#(\d+)\s*\u2022\s*(.+?)\s*\u2022\s*([A-Z/]+)\s*\u2022\s*(\d+'\d+\")\s*\u2022\s*(.*?)\s*\u2022\s*([A-Z]+)$")
STARTERS_BY_ORDER_CUTOFF = 5

import json
from datetime import date

# The schedule's "date" strings (e.g. "Fri, Nov 14") carry NO year at all -- a men's basketball season spans
# two calendar years, so the year has to be inferred from the season boundary rather than read off the string.
# All the scouted games so far fall in the 2025-26 season: Aug-Dec dates are 2025, Jan-Jul dates are 2026.
SEASON_START_YEAR = 2025

def parse_schedule_date(date_str):
    """'Fri, Nov 14' -> date(2025, 11, 14)."""
    if pd.isna(date_str):
        return None
    parsed = datetime.strptime(str(date_str).strip(), "%a, %b %d")
    year = SEASON_START_YEAR if parsed.month >= 8 else SEASON_START_YEAR + 1
    return date(year, parsed.month, parsed.day)

# A few entries (e.g. Eureka's Damuzha Moore, Jacob Gonzalez) don't render "Player Notes"/"Keys to Defending" as
# separate labeled sections at all -- both labels AND both fields' text come through as ONE run-on text element,
# because the PDF's two side-by-side columns got merged with their lines interleaved by the text extractor.
# Splitting that reliably with regex isn't feasible (the interleaving order isn't consistent), so an LLM call
# separates the blob back into the two original fields based on their content -- notes describe playing style,
# keys are short defensive coaching instructions. (Portable: uses a plain OpenAI-compatible client instead of
# Databricks' ai_query() SQL function -- see USE_LLM / OPENAI_API_KEY / AI_MODEL / OPENAI_BASE_URL.)
def split_combined_notes_keys(blob):
    if not USE_LLM:
        return "", ""
    prompt = (
        "The following text is a run-on merge of two DIFFERENT scouting-report fields for one basketball "
        "player, with their lines interleaved: 'Player Notes' (describes how the player plays -- shooting, "
        "driving, position role, etc.) and 'Keys to Defending' (short defensive coaching instructions on how "
        "to guard him -- e.g. 'keep in front', 'box out', 'be a helper', 'anticipate drive', 'get off into "
        "gaps'). Split the text below back into its two original fields, preserving the original wording "
        "exactly (do not paraphrase, do not add words), and drop the literal labels 'Player Notes:' / "
        "'Keys to Defending' themselves from the output. Respond with ONLY a JSON object of the exact shape "
        '{"player_notes": "...", "keys_to_defending": "..."}, no other text.'
        f"\n\nTEXT: {blob}"
    )
    try:
        from openai import OpenAI

        client = OpenAI(base_url=os.environ.get("OPENAI_BASE_URL") or None)
        response = client.chat.completions.create(
            model=os.environ.get("AI_MODEL", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        parsed = json.loads(response.choices[0].message.content)
    except Exception as llm_error:
        print(f"  LLM split failed for a combined notes/keys blob: {type(llm_error).__name__}: {llm_error} -- leaving both fields blank.")
        return "", ""
    return parsed.get("player_notes", "").strip(), parsed.get("keys_to_defending", "").strip()

# Reuse the schedule as the source of truth for each game's date (same fuzzy opponent-name match used later to
# resolve each scouted opponent's game number), rather than re-parsing it out of the PDF filename. Returns a
# real datetime.date (via parse_schedule_date) rather than the schedule's raw display string, since every
# "game_date" column built from this function (player_profiles, pbp_events, and everything grouped from pbp_events
# -- pbp_box_score, lineup_box_score) is meant for date arithmetic/sorting, not just display.
def game_date_for(opponent_short):
    matches = schedule[schedule["opponent"].str.contains(re.escape(opponent_short), case=False)]
    return parse_schedule_date(matches.iloc[0]["date"]) if not matches.empty else None

def extract_roster_from_pdf(elements_df, opponent):
    elements = elements_df.to_dict("records")
    n = len(elements)
    bench_idx = next(
        (e["element_index"] for e in elements if e["element_type"] == "section_header" and e["element_content"].strip() == "BENCH"),
        None,
    )
    game_date = game_date_for(opponent)

    rows = []
    player_seq = 0
    i = 0
    while i < n:
        el = elements[i]
        if el["element_type"] == "text":
            m = PLAYER_LINE_RE.match(el["element_content"].strip())
            if m:
                jersey, name, pos, height, weight, cls = m.groups()
                player_seq += 1
                notes, keys, combined_blob = [], [], None
                j = i + 1
                while j < n:
                    nel = elements[j]
                    if nel["element_type"] == "text" and PLAYER_LINE_RE.match(nel["element_content"].strip()):
                        break
                    if nel["element_type"] == "section_header" and nel["element_content"].strip() in ("STARTERS", "BENCH"):
                        break
                    if nel["element_type"] == "text":
                        content = nel["element_content"].strip()
                        prev = elements[j - 1]
                        if prev["element_type"] == "section_header" and prev["element_content"].startswith("Player Notes"):
                            notes.append(content)
                        elif prev["element_type"] == "section_header" and prev["element_content"].startswith("Keys to Defending"):
                            keys.append(content)
                        elif "player notes" in content.lower() and "keys to defending" in content.lower():
                            combined_blob = content
                    j += 1
                if not notes and not keys and combined_blob:
                    notes_text, keys_text = split_combined_notes_keys(combined_blob)
                else:
                    notes_text, keys_text = " ".join(notes), " ".join(keys)
                if bench_idx is not None:
                    role = "Starter" if el["element_index"] < bench_idx else "Bench"
                else:
                    role = "Starter" if player_seq <= STARTERS_BY_ORDER_CUTOFF else "Bench"
                rows.append({
                    "opponent": opponent,
                    "game_date": game_date,
                    "jersey_number": f"#{jersey}",
                    "name": name.strip(),
                    "position": pos,
                    "height": height,
                    "weight": (weight.strip() or None),
                    "class_year": cls,
                    "role": role,
                    "player_notes": notes_text,
                    "keys_to_defending": keys_text,
                    # Every row here came from parsing an actual scouting-report player entry (jersey/name/pos
                    # line + its Player Notes / Keys to Defending text) -- as opposed to a player later recovered
                    # ONLY from the season boxscore table (see player_profiles), who was never individually
                    # scouted. Carried forward through player_profiles so downstream cells that judge free-text
                    # scouting language (e.g. the LLM comparison) can restrict themselves to real scouted players.
                    "has_scouting_report": True,
                })
                i = j
                continue
        i += 1
    return pd.DataFrame(rows)

roster_cols = ["opponent", "game_date", "jersey_number", "name", "position", "height", "weight", "class_year", "role", "player_notes", "keys_to_defending", "has_scouting_report"]
all_rosters = pd.concat(
    [extract_roster_from_pdf(df, opponent) for opponent, df in scout_reports.items()],
    ignore_index=True,
) if scout_reports else pd.DataFrame(columns=roster_cols)
print(all_rosters)

                       opponent   game_date jersey_number               name  \
0         St. Thomas (TX) Celts  2025-11-14           #10      Angel Johnson   
1         St. Thomas (TX) Celts  2025-11-14            #1     Corey Thompson   
2         St. Thomas (TX) Celts  2025-11-14            #0     Nathan Kongolo   
3         St. Thomas (TX) Celts  2025-11-14           #12   Nicholas Buffalo   
4         St. Thomas (TX) Celts  2025-11-14           #25    Charles Gitonga   
..                          ...         ...           ...                ...   
157  Washington-St. Louis Bears  2025-11-15           #34      Calvin Kapral   
158  Washington-St. Louis Bears  2025-11-15           #23  Anthony Przybilla   
159  Washington-St. Louis Bears  2025-11-15           #32    Will Grudzinski   
160  Washington-St. Louis Bears  2025-11-15           #30         Theo Rocca   
161  Washington-St. Louis Bears  2025-11-15           #33        George Gale   

    position height   weight class_year


### Extract UW-Whitewater's offensive and defensive game plan for each opponent

Pulls the "TEAM STRENGTHS" / "KEYS TO VICTORY" / defensive-scheme / offensive-scheme sections out of each scouting report's parsed elements, grouped under a consistent `section_group` label so they can be compared across opponents.

In [26]:
headers_in_order = [
    "TEAM STRENGTHS", "KEYS TO VICTORY",
    "Overall Defensive Scheme", "Attacking their man defense", "Ball Screen & DHO Defense",
    "Ball Screen Actions & Reads", "Speciality Defensive Notes", "Potential Adjustments",
    "Defending Their Action", "Overall OFFENSIVE SCHEME", "Ball Screen Actions & Personnel",
    "Ball Screen Coverage(s)", "Potential Adjustments", "ELOB & SLOB",
]
section_group = {
    "TEAM STRENGTHS": "Game Plan Overview", "KEYS TO VICTORY": "Game Plan Overview",
    "Overall Defensive Scheme": "Offensive Game Plan (vs. Opponent Defense)",
    "Attacking their man defense": "Offensive Game Plan (vs. Opponent Defense)",
    "Ball Screen & DHO Defense": "Offensive Game Plan (vs. Opponent Defense)",
    "Ball Screen Actions & Reads": "Offensive Game Plan (vs. Opponent Defense)",
    "Speciality Defensive Notes": "Offensive Game Plan (vs. Opponent Defense)",
    "Defending Their Action": "Defensive Game Plan (vs. Opponent Offense)",
    "Overall OFFENSIVE SCHEME": "Defensive Game Plan (vs. Opponent Offense)",
    "Ball Screen Actions & Personnel": "Defensive Game Plan (vs. Opponent Offense)",
    "Ball Screen Coverage(s)": "Defensive Game Plan (vs. Opponent Offense)",
    "ELOB & SLOB": "Defensive Game Plan (vs. Opponent Offense)",
}
# PDF elements already separate section_header from text cleanly (no banner/page-chrome noise mixed in, unlike
# flattened MHTML text) -- reconstruct the game plan by bucketing consecutive "text" elements under the most
# recent matching "section_header", stopping once the roster ("STARTERS") begins.
def extract_game_plan_from_pdf(elements_df, opponent):
    buckets, order = {}, []
    current_label = None
    adj_seen = 0
    for el in elements_df.to_dict("records"):
        if el["element_type"] == "section_header":
            content = el["element_content"].strip()
            if content == "STARTERS":
                break
            if content in headers_in_order:
                if content == "Potential Adjustments":
                    adj_seen += 1
                    label = f"Potential Adjustments ({'Defense' if adj_seen == 1 else 'Offense'})"
                    category = "Offensive Game Plan (vs. Opponent Defense)" if adj_seen == 1 else "Defensive Game Plan (vs. Opponent Offense)"
                else:
                    label, category = content, section_group[content]
                current_label = label
                buckets[label] = []
                order.append((category, label))
            else:
                current_label = None  # banner/divider header (e.g. "RIPON DEFENSE") we don't care about
        elif el["element_type"] == "text" and current_label is not None:
            txt = el["element_content"].strip()
            if txt:
                buckets[current_label].append(txt)
    return pd.DataFrame([
        {"opponent": opponent, "category": category, "topic": label, "notes": " | ".join(buckets[label])}
        for category, label in order
    ])

game_plan_cols = ["opponent", "category", "topic", "notes"]
all_game_plans = pd.concat(
    [extract_game_plan_from_pdf(df, opponent) for opponent, df in scout_reports.items()],
    ignore_index=True,
) if scout_reports else pd.DataFrame(columns=game_plan_cols)
pd.set_option("display.max_colwidth", 150)
print(all_game_plans)

                       opponent                                    category  \
0         St. Thomas (TX) Celts                          Game Plan Overview   
1         St. Thomas (TX) Celts                          Game Plan Overview   
2         St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
3         St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
4         St. Thomas (TX) Celts  Offensive Game Plan (vs. Opponent Defense)   
..                          ...                                         ...   
256  Washington-St. Louis Bears  Defensive Game Plan (vs. Opponent Offense)   
257  Washington-St. Louis Bears  Defensive Game Plan (vs. Opponent Offense)   
258  Washington-St. Louis Bears  Defensive Game Plan (vs. Opponent Offense)   
259  Washington-St. Louis Bears  Defensive Game Plan (vs. Opponent Offense)   
260  Washington-St. Louis Bears  Defensive Game Plan (vs. Opponent Offense)   

                               topic  \
0          


### Team efficiency narrative (deprecated -- PDF exports don't carry it)

The FASTINTELLIGENCE panel (ORtg/DRtg/Pace + percentile narrative) is a ScoutBuilder UI-only widget that never appears in the printable/PDF report. Kept as an empty placeholder with the same schema as before so nothing downstream breaks, now that scouting reports are sourced exclusively from PDFs.

In [28]:
# The "FASTINTELLIGENCE" panel (ORtg/DRtg/Pace + national-percentile narrative) is ScoutBuilder's UI-only
# "Scout Assistant" widget -- it is NOT part of the printable report, so it never appears in the PDF export
# (confirmed: 0 of the Ripon PDF's 133 parsed elements mention ORtg/DRtg/FASTINTELLIGENCE). This metric is no
# longer available now that scouting reports are sourced exclusively from PDFs. Kept as an empty placeholder
# (same schema as before) so nothing downstream breaks.

# all_team_stats = pd.DataFrame(columns=["opponent", "ORtg", "DRtg", "Pace", "offense_narrative", "defense_narrative"])

print(
    "Team efficiency narrative (ORtg/DRtg/Pace/FASTINTELLIGENCE) is not captured in the PDF export -- it's a\n"
    "UI-only panel that isn't part of the printable ScoutBuilder report, and is no longer available now that\n"
    "scouting reports are sourced exclusively from PDFs."
)

Team efficiency narrative (ORtg/DRtg/Pace/FASTINTELLIGENCE) is not captured in the PDF export -- it's a
UI-only panel that isn't part of the printable ScoutBuilder report, and is no longer available now that
scouting reports are sourced exclusively from PDFs.



### Cross-reference each scouted opponent's report against the actual result

Since the FASTINTELLIGENCE narrative is gone (see previous cell), this instead reads each opponent's season-average points scored/allowed straight from their PDF's own boxscore table ("Team Total" and "Opponent" rows) to sanity-check the scouting report against what actually happened.

In [30]:
# The FASTINTELLIGENCE narrative used previously is gone (see prior cell), so this now cross-references each
# opponent's season-average PPG scored/allowed straight from their PDF's "...BOXSCORE" table's "Team Total"
# and "Opponent" rows -- a different underlying data source than the old narrative, so exact figures may differ
# from what MHTML-based reports previously showed.
def _dedupe_boxscore_columns(df):
    """read_boxscore_table below builds its DataFrame from a plain Python `headers` list via
    `pd.DataFrame(records, columns=headers)` -- unlike pd.read_html elsewhere in this notebook, this gets NO
    automatic duplicate-column mangling from pandas at all, so two header cells that happen to render the
    same text (confirmed to happen on the live-rendered ScoutBuilder boxscore widget -- see the app-side fix
    this mirrors) silently produce a DataFrame with two identically-named columns. Any later df[col] or
    row[col] access on that name then returns a Series instead of a scalar without raising, which is exactly
    the "truth value of a Series is ambiguous" crash this was tracked down from. Dedupe once here, right at
    construction, rather than downstream wherever it happens to first get accessed."""
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].unique().tolist()
        print(f"WARNING: boxscore table has duplicate column name(s) {dupes} -- keeping the first occurrence "
              f"of each and dropping the rest.")
        df = df.loc[:, ~df.columns.duplicated()]
    return df


def read_boxscore_table(html_str):
    """Parse a FastScout PDF boxscore <table> into a DataFrame. On some (not all) opponents' PDFs, a player's
    full name is split across two separate <td> cells (first name, last name) while the header row only
    allocates a single <th>PLAYER</th> slot for it -- silently shifting every later column (FG%, 3P%, etc.) one
    position to the right of its real header when read naively via pd.read_html(). Detect that per-row and
    merge the split name cells back into one before assigning headers, so every stat column lines up correctly
    regardless of which layout a given PDF used."""
    table_soup = BeautifulSoup(html_str, "html.parser")
    header_row = table_soup.find("tr")
    header_cells = header_row.find_all("th")

    # The LIVE ScoutBuilder HTML boxscore table (Cell 8's HTML path, as opposed to a pdfplumber-extracted PDF
    # table) wraps every REAL header/data cell in its own "cell-content" div, and ALSO carries decorative
    # leading cells (a blank "table-gutter" cell + a sort-handle cell) that never get one -- confirmed against
    # a real live-downloaded report: naive positional header/cell alignment (the PDF-oriented path below)
    # silently misaligns every column by those decorative cells, since headers ends up 2 shorter than the
    # actual <td> count per row for a totally unrelated reason than the split-name case it was built for.
    # Detect that shape via "cell-content" and filter on it directly instead of position.
    if any(th.find(class_="cell-content") is not None for th in header_cells):
        headers = []
        for th in header_cells:
            content_div = th.find(class_="cell-content")
            if content_div is not None:
                headers.append(normalize_html_text(content_div.get_text(" ", strip=True)))
        body = table_soup.find("tbody")
        body_rows = body.find_all("tr") if body else table_soup.find_all("tr")[1:]
        records = []
        for tr in body_rows:
            cells = []
            for td in tr.find_all("td"):
                content_div = td.find(class_="cell-content")
                cells.append(normalize_html_text(content_div.get_text(" ", strip=True)) if content_div is not None else None)
            cells = [c for c in cells if c is not None]
            records.append(cells[: len(headers)])
        _boxscore_df = pd.DataFrame(records, columns=headers)
        return _dedupe_boxscore_columns(_boxscore_df)

    # --- PDF-oriented path (pdfplumber's plain-text cells have no "cell-content" wrapper at all) ---
    header_cells_text = [th.get_text(strip=True) for th in header_cells]
    # Drop trailing decorative header cell(s) with no real stat name -- this is sometimes a blank "" and
    # sometimes a private-use-area icon glyph (e.g. "\uf107", presumably a sort-arrow icon that pdfplumber
    # extracted as text) depending on the PDF, but either way it never has any alphanumeric content.
    headers = header_cells_text[:]
    while headers and not any(c.isalnum() for c in headers[-1]):
        headers = headers[:-1]
    name_idx = headers.index("PLAYER")
    body = table_soup.find("tbody")
    body_rows = body.find_all("tr") if body else table_soup.find_all("tr")[1:]
    records = []
    for tr in body_rows:
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) == len(headers) + 1:
            cells = cells[:name_idx] + [" ".join(c for c in cells[name_idx:name_idx + 2] if c)] + cells[name_idx + 2:]
        records.append(cells[: len(headers)])
    _boxscore_df = pd.DataFrame(records, columns=headers)
    return _dedupe_boxscore_columns(_boxscore_df)


def extract_team_totals_from_pdf(elements_df, opponent):
    rows = elements_df.to_dict("records")
    box_idx = next(
        (i for i, r in enumerate(rows) if r["element_type"] == "section_header" and "BOXSCORE" in r["element_content"].upper()),
        None,
    )
    if box_idx is None or rows[box_idx + 1]["element_type"] != "table":
        return None
    stats_df = read_boxscore_table(rows[box_idx + 1]["element_content"]).rename(columns={"PLAYER": "name"})
    # See the analogous fix in the player-tagging cell above -- pdfplumber can truncate "Team Total" down to just
    # "Team", so also fall back to the jersey-number column ("#" == "-" for the aggregate row) to find it.
    team_row = stats_df[(stats_df["name"] == "Team Total") | (stats_df.get("#") == "-")]
    opp_row = stats_df[stats_df["name"] == "Opponent"]
    if team_row.empty:
        print(f"  Skipping '{opponent}': no 'Team Total' row in its boxscore table.")
        return None
    if opp_row.empty:
        print(f"  Note: '{opponent}' boxscore has no 'Opponent' row (points allowed) -- reporting team_ppg only.")
    return {
        "opponent": opponent,
        "team_ppg": float(team_row["PTS"].iloc[0]),
        "opp_ppg_allowed": float(opp_row["PTS"].iloc[0]) if not opp_row.empty else None,
    }

team_totals = pd.DataFrame([
    r for r in (extract_team_totals_from_pdf(df, opponent) for opponent, df in scout_reports.items()) if r
])

# "Keys to Victory" notes use basketball terminology rather than stat names directly ("Ball Security" ==
# turnovers, "Own the Paint"/"Bully them on the glass" == rebounding, etc). Map that terminology to the UWW
# season stat columns it corresponds to, then surface UWW's season-long "Team Total" (their own average) vs.
# "Opponent" (what teams average against UWW) rows from `stats` for whichever columns a given note touches on --
# there's no per-game UWW box score in this pipeline (the PDF boxscore only covers the OPPONENT's roster), so the
# season averages are the best available context for whether that emphasis is one UWW has actually executed on.
KEYS_TO_VICTORY_STAT_MAP = {
    # --- Ball Security / Turnovers (TO) ---
    "ball security": ["TO"], "turnover": ["TO"], "protect the ball": ["TO"], "take care of the ball": ["TO"],
    "limit turnovers": ["TO"], "careless": ["TO"],
    # --- Rebounding (REB, ORB, DRB) ---
    "own the paint": ["REB", "ORB", "DRB"], "bully": ["REB", "ORB", "DRB"], "glass": ["REB", "ORB", "DRB"],
    "rebound": ["REB", "ORB", "DRB"], "board": ["REB", "ORB", "DRB"], "second chance": ["ORB"],
    "crash": ["REB", "ORB", "DRB"],
    # --- Three-Point Shooting (3PM-A, 3P%) ---
    "three": ["3PM-A", "3P%"], "3 pt": ["3PM-A", "3P%"], "3pt": ["3PM-A", "3P%"],
    "perimeter shooting": ["3PM-A", "3P%"], "spacing": ["3PM-A", "3P%"],
    "shooting ability": ["3PM-A", "3P%"], "shooting team": ["3PM-A", "3P%"],
    "sniper": ["3PM-A", "3P%"], "will shoot": ["3PM-A", "3P%"],
    # --- Free Throws (FTM-A, FT%) ---
    "free throw": ["FTM-A", "FT%"], "ft line": ["FTM-A", "FT%"], "getting to ft": ["FTM-A", "FT%"],
    # --- Fouls / Discipline (PF) ---
    "foul": ["PF"], "wall up": ["PF"], "drawing fouls": ["PF"],
    # --- Ball Movement / Assists (AST) ---
    "assist": ["AST"], "ball movement": ["AST"], "share the ball": ["AST"],
    "playmaking": ["AST"], "playmaker": ["AST"], "create": ["AST"],
    # --- Perimeter Defense / Ball Pressure (STL) ---
    "steal": ["STL"], "press capable": ["STL", "TO"], "full court press": ["STL", "TO"],
    "force turnovers": ["STL"], "force to's": ["STL"],
    "guard your yard": ["STL"], "keep the ball in front": ["STL"], "guard 1 on 1": ["STL"],
    "early gap": ["STL"], "help side": ["STL"], "active hands": ["STL"],
    "physical & aggressive on ball": ["STL"], "on ball defensively": ["STL"],
    "pressure": ["STL"],
    # --- Paint Protection / Blocks (BLK) ---
    "block": ["BLK"], "protect the rim": ["BLK"], "paint protection": ["BLK"],
    # --- Scoring Inside (FG2M, FG2A, FG2% — derived as FGM-FG3M, FGA-FG3A) ---
    "dominate the paint": ["FG2M", "FG2A", "FG2%"], "attack the paint": ["FG2M", "FG2A", "FG2%"],
    "live in the paint": ["FG2M", "FG2A", "FG2%"], "attack the basket": ["FG2M", "FG2A", "FG2%"],
    "scoring at the rim": ["FG2M", "FG2A", "FG2%"], "get to rim": ["FG2M", "FG2A", "FG2%"],
    "attack the rim": ["FG2M", "FG2A", "FG2%"], "get to the rim": ["FG2M", "FG2A", "FG2%"],
    # --- Field Goal Efficiency (FGM-A, FG%) — overall ---
    "limit their scoring": ["FGM-A", "FG%"],
}

# --- Side attribution: Is this phrase about what UWW does (offense/proactive) or containing the opponent? ---
PHRASE_SIDE = {
    "ball security": "UWW", "turnover": "UWW", "protect the ball": "UWW",
    "take care of the ball": "UWW", "limit turnovers": "UWW", "careless": "UWW",
    "own the paint": "UWW", "bully": "UWW", "glass": "UWW",
    "rebound": "UWW", "board": "UWW", "second chance": "UWW", "crash": "UWW",
    "box out": "OPP", "keep off glass": "OPP",
    "three": "UWW", "3 pt": "OPP", "3pt": "OPP", "perimeter shooting": "UWW", "spacing": "UWW",
    "shooting ability": "OPP", "shooting team": "OPP", "sniper": "OPP", "will shoot": "OPP",
    "close out": "OPP", "closeout": "OPP", "run off the line": "OPP",
    "free throw": "UWW", "ft line": "UWW", "getting to ft": "UWW",
    "don't foul": "OPP", "keep them off": "OPP",
    "foul": "OPP", "wall up": "OPP", "drawing fouls": "OPP",
    "assist": "UWW", "ball movement": "UWW", "share the ball": "UWW",
    "playmaking": "UWW", "playmaker": "UWW", "create": "UWW",
    "steal": "OPP", "press capable": "OPP", "full court press": "OPP",
    "force turnovers": "OPP", "force to's": "OPP",
    "guard your yard": "OPP", "keep the ball in front": "OPP", "guard 1 on 1": "OPP",
    "early gap": "OPP", "help side": "OPP", "active hands": "OPP",
    "physical & aggressive on ball": "OPP", "on ball defensively": "OPP",
    "pressure": "OPP",
    "block": "OPP", "protect the rim": "OPP", "paint protection": "OPP",
    "attack the paint": "UWW", "live in the paint": "UWW",
    "dominate the paint": "UWW", "attack the basket": "UWW",
    "scoring at the rim": "UWW", "get to rim": "UWW",
    "attack the rim": "UWW", "get to the rim": "UWW",
    "limit their scoring": "OPP",
    "take away": "OPP", "funnel": "OPP", "deny": "OPP", "contain": "OPP",
    "limit": "OPP", "contest": "OPP", "make them": "OPP", "load up": "OPP",
    "transition defense": "OPP", "fight over": "OPP", "switch": "OPP",
    "trap": "OPP", "double team": "OPP", "coverage": "OPP",
    "don't help off": "OPP", "take away personnel": "OPP",
    "run the floor": "UWW", "push tempo": "UWW", "fast break": "UWW",
    "score in transition": "UWW", "finish": "UWW", "execute": "UWW",
    "dominate": "UWW", "impose": "UWW", "push the pace": "UWW",
}
STAT_LABELS = {
    "TO": "Turnovers/gm", "REB": "Rebounds/gm", "ORB": "Off. rebounds/gm", "DRB": "Def. rebounds/gm",
    "AST": "Assists/gm", "STL": "Steals/gm", "BLK": "Blocks/gm", "3PM-A": "3PM-A/gm", "3P%": "3P%",
    "FGM-A": "FGM-A/gm", "FG%": "FG%", "FTM-A": "FTM-A/gm", "FT%": "FT%", "PF": "Fouls/gm",
}
# uww_team_totals = stats[stats["PLAYER"] == "Team Total"].iloc[0] if (stats["PLAYER"] == "Team Total").any() else None
uww_team_totals = None
# uww_opp_totals = stats[stats["PLAYER"] == "Opponent"].iloc[0] if (stats["PLAYER"] == "Opponent").any() else None
uww_opp_totals = None

def stat_cols_for_note(note_text):
    text = str(note_text).lower()
    cols = []
    for phrase, stat_cols in KEYS_TO_VICTORY_STAT_MAP.items():
        if phrase in text:
            for c in stat_cols:
                if c not in cols:
                    cols.append(c)
    return cols

def print_keys_to_victory_stats(note_text):
    cols = stat_cols_for_note(note_text)
    if not cols or uww_team_totals is None:
        return
    print("    Relevant UWW season averages (Team Total = UWW's own avg, Opponent = what teams average against UWW):")
    for c in cols:
        print(f"      {STAT_LABELS.get(c, c)}: UWW {uww_team_totals[c]}  |  Opponent avg {uww_opp_totals[c]}")

comparison_rows = []
for _, tt in team_totals.iterrows():
    opponent = tt["opponent"]
    matches = schedule[schedule["opponent"].str.contains(re.escape(opponent), case=False)]
    if matches.empty:
        print(f"No schedule match found for scouted opponent '{opponent}' -- skipping.")
        continue
    game_result = matches.iloc[0]

    comparison_rows.append({
        "opponent": opponent,
        "date": game_result["date"],
        "location": game_result["location"],
        "outcome": game_result["outcome"],
        "opp_season_avg_ppg": tt["team_ppg"],
        "opp_actual_ppg": game_result["opponent_score"],
        "opp_ppg_vs_average": round(game_result["opponent_score"] - tt["team_ppg"], 1),
        "uww_points": game_result["team_score"],
        "opp_season_avg_ppg_allowed": tt["opp_ppg_allowed"],
        "uww_ppg_vs_opp_avg_allowed": (
            round(game_result["team_score"] - tt["opp_ppg_allowed"], 1) if pd.notna(tt["opp_ppg_allowed"]) else None
        ),
        "point_margin": game_result["point_margin"],
    })

scouted_game_comparison = pd.DataFrame(comparison_rows)
if scouted_game_comparison.empty:
    print("No opponents with both a PDF scout report and a schedule match -- nothing to cross-reference.")
else:
    print(scouted_game_comparison)
    for _, row in scouted_game_comparison.iterrows():
        keys_to_victory = all_game_plans.loc[
            (all_game_plans["opponent"] == row["opponent"]) & (all_game_plans["topic"] == "KEYS TO VICTORY"), "notes"
        ]
        print(f"\n{row['opponent']} ({row['date']}, {row['location']}): UWW {row['uww_points']} - "
              f"{row['opponent']} {row['opp_actual_ppg']} ({row['outcome']}, margin {row['point_margin']:+.0f})")
        if not keys_to_victory.empty:
            print("  Pre-game keys to victory:", keys_to_victory.iloc[0])
            print_keys_to_victory_stats(keys_to_victory.iloc[0])
        print(f"  Opponent scored {row['opp_ppg_vs_average']:+.1f} vs their season-average PPG (from the PDF box score).")
        if pd.notna(row["uww_ppg_vs_opp_avg_allowed"]):
            print(f"  UWW scored {row['uww_ppg_vs_opp_avg_allowed']:+.1f} vs the opponent's season-average points allowed.")
        else:
            print(f"  No 'points allowed' figure available for {row['opponent']} (their boxscore table has no 'Opponent' row).")

                      opponent         date       location outcome  \
0        St. Thomas (TX) Celts  Fri, Nov 14  Neutral Court       L   
1            Eureka Red Devils  Sat, Nov 15  Neutral Court       W   
2              Aurora Spartans  Wed, Nov 19           Home       W   
3                Simpson Storm  Tue, Nov 25           Home       W   
4              Ripon Red Hawks   Fri, Nov 7           Away       W   
5             Lawrence Vikings  Wed, Dec 10           Away       W   
6        Carroll (WI) Pioneers  Sat, Dec 13           Home       W   
7                   Alma Scots  Sat, Dec 20           Away       W   
8            Elmhurst Bluejays   Tue, Dec 2           Home       W   
9                  Coe Kohawks  Tue, Dec 30           Away       L   
10      UW-River Falls Falcons  Sat, Jan 10           Away       W   
11         UW-La Crosse Eagles  Wed, Jan 14           Away       W   
12        UW-Stout Blue Devils  Sat, Jan 17           Home       W   
13     UW-Plattevill


### Win/loss splits by Keys-to-Victory category, with side attribution

Tags each matched "Keys to Victory" phrase with a side -- UWW (what Whitewater does proactively: attack, score, rebound, share) or OPP (what Whitewater does to contain the opponent: guard, force turnovers, limit, pressure) -- then breaks down win/loss outcomes by category and side.

In [32]:
# Win/loss splits by "Keys to Victory" category WITH SIDE ATTRIBUTION (UWW vs OPP).
# Each matched phrase now carries a "side" from PHRASE_SIDE: UWW = what Whitewater proactively does (offense/
# hustle), OPP = what Whitewater does to CONTAIN the opponent (defense/discipline). This answers the coaching
# question: "When we emphasize attacking the rim ourselves (UWW) vs limiting THEIR rim attacks (OPP), which
# approach correlates with winning?"
# The stat-category grouping is unchanged (rebounding columns still roll up to "Rebounding"), but now each
# game-category row also carries its side, so splits can be cut both ways.
STAT_COL_CATEGORY = {
    "TO": "Ball Security / Turnovers", "STL": "Perimeter Defense / Ball Pressure",
    "REB": "Rebounding", "ORB": "Rebounding", "DRB": "Rebounding",
    "3PM-A": "Three-Point Shooting", "3P%": "Three-Point Shooting",
    "FTM-A": "Free Throws", "FT%": "Free Throws",
    "PF": "Fouls / Discipline",
    "AST": "Ball Movement / Assists",
    "BLK": "Paint Protection / Blocks",
    "FG2M": "Scoring Inside", "FG2A": "Scoring Inside", "FG2%": "Scoring Inside",
    "FGM-A": "Field Goal Efficiency", "FG%": "Field Goal Efficiency",
}

SIDE_DISPLAY_LABELS = {
    ("Ball Security / Turnovers", "UWW"): "UWW: Protect the Ball",
    ("Ball Security / Turnovers", "OPP"): "OPP: Force Turnovers",
    ("Rebounding", "UWW"): "UWW: Crash the Boards",
    ("Rebounding", "OPP"): "OPP: Limit Their Rebounding",
    ("Three-Point Shooting", "UWW"): "UWW: Hit Our Threes",
    ("Three-Point Shooting", "OPP"): "OPP: Contest Their Shooting",
    ("Free Throws", "UWW"): "UWW: Get to the FT Line",
    ("Free Throws", "OPP"): "OPP: Keep Them Off the Line",
    ("Fouls / Discipline", "UWW"): "UWW: Stay Disciplined",
    ("Fouls / Discipline", "OPP"): "OPP: They Draw Fouls",
    ("Ball Movement / Assists", "UWW"): "UWW: Share the Ball",
    ("Ball Movement / Assists", "OPP"): "OPP: Disrupt Their Ball Movement",
    ("Perimeter Defense / Ball Pressure", "UWW"): "UWW: Create Pressure",
    ("Perimeter Defense / Ball Pressure", "OPP"): "OPP: On-Ball Defense",
    ("Paint Protection / Blocks", "UWW"): "UWW: Protect Our Rim",
    ("Paint Protection / Blocks", "OPP"): "OPP: Limit Their Interior",
    ("Scoring Inside", "UWW"): "UWW: Attack the Paint",
    ("Scoring Inside", "OPP"): "OPP: Limit Their Inside Scoring",
    ("Field Goal Efficiency", "UWW"): "UWW: Efficient Shooting",
    ("Field Goal Efficiency", "OPP"): "OPP: Limit Their FG Efficiency",
}

def categories_for_note(note_text):
    """Return list of (category, side) tuples matched from note text."""
    text = str(note_text).lower()
    seen = set()
    results = []
    for phrase, stat_cols in KEYS_TO_VICTORY_STAT_MAP.items():
        if phrase in text:
            side = PHRASE_SIDE.get(phrase, "UWW")  # default to UWW if not explicitly listed
            for c in stat_cols:
                cat = STAT_COL_CATEGORY.get(c)
                if cat and (cat, side) not in seen:
                    seen.add((cat, side))
                    results.append((cat, side))
    return sorted(results)

def stat_cols_for_note(note_text):
    """Legacy helper: just the stat columns (no side), used by downstream How We Stack Up."""
    text = str(note_text).lower()
    cols = []
    for phrase, stat_cols in KEYS_TO_VICTORY_STAT_MAP.items():
        if phrase in text:
            for c in stat_cols:
                if c not in cols:
                    cols.append(c)
    return cols

game_category_rows = []
for _, row in scouted_game_comparison.iterrows():
    keys_to_victory = all_game_plans.loc[
        (all_game_plans["opponent"] == row["opponent"]) & (all_game_plans["topic"] == "KEYS TO VICTORY"), "notes"
    ]
    if keys_to_victory.empty:
        continue
    cat_sides = categories_for_note(keys_to_victory.iloc[0])
    if not cat_sides:
        game_category_rows.append({"opponent": row["opponent"], "outcome": row["outcome"], "category": "(no matched category)", "side": ""})
    for cat, side in cat_sides:
        game_category_rows.append({"opponent": row["opponent"], "outcome": row["outcome"], "category": cat, "side": side})

game_categories = pd.DataFrame(game_category_rows)
print("Per-game 'Keys to Victory' categories detected WITH SIDE ATTRIBUTION:")
print("  UWW = what Whitewater does proactively (attack, score, rebound, share)")
print("  OPP = what Whitewater does to contain the opponent (guard, force TOs, limit, pressure)")
print(game_categories)

if not game_categories.empty:
    # Granular splits: by category + side
    game_categories["display_label"] = game_categories.apply(
        lambda r: SIDE_DISPLAY_LABELS.get((r["category"], r["side"]), f"{r['side']}: {r['category']}"), axis=1
    )
    splits = (
        game_categories.groupby(["category", "side", "display_label"])["outcome"]
        .agg(games="count", wins=lambda s: (s == "W").sum(), losses=lambda s: (s == "L").sum())
        .reset_index()
        .sort_values(["category", "side"], ascending=[True, True])
    )
    splits["win_pct"] = (splits["wins"] / splits["games"]).round(3)
    print("\nWin/loss splits by category + side (UWW vs OPP emphasis):")
    print(splits[["display_label", "side", "category", "games", "wins", "losses", "win_pct"]])
else:
    print("No 'Keys to Victory' notes with a matched category yet.")

Per-game 'Keys to Victory' categories detected WITH SIDE ATTRIBUTION:
  UWW = what Whitewater does proactively (attack, score, rebound, share)
  OPP = what Whitewater does to contain the opponent (guard, force TOs, limit, pressure)
                      opponent outcome                           category side
0        St. Thomas (TX) Celts       L          Ball Security / Turnovers  UWW
1        St. Thomas (TX) Celts       L              Field Goal Efficiency  OPP
2        St. Thomas (TX) Celts       L                         Rebounding  UWW
3            Eureka Red Devils       W  Perimeter Defense / Ball Pressure  OPP
4            Eureka Red Devils       W                         Rebounding  UWW
5              Aurora Spartans       W                 Fouls / Discipline  OPP
6              Aurora Spartans       W  Perimeter Defense / Ball Pressure  OPP
7              Aurora Spartans       W                     Scoring Inside  UWW
8                Simpson Storm       W                   


### Keys-to-Victory category reference

Documents WHY each category exists: which Keys-to-Victory phrases trigger it, which stat column(s) it's graded on, and cases where a phrase is deliberately left unmapped because no stat in a season box score can measure it. Update this table any time a new KTV phrase pattern shows up.

In [34]:
# --- Keys-to-Victory category reference -----------------------------------------------------------------------
# Documents WHY each category exists: which Keys-to-Victory phrases trigger it, which stat column(s) it's graded
# on, and the reasoning -- including cases where a phrase is deliberately left unmapped because no stat in a
# season box score can measure it. Update this table any time KEYS_TO_VICTORY_STAT_MAP / STAT_COL_CATEGORY change.
# Last updated: expanded keywords after reviewing all 6 opponents' KTV + Team Strengths notes for unmapped phrases.
KTV_CATEGORY_REFERENCE = [
    {
        "category": "Ball Security / Turnovers", "stat_cols": "TO",
        "example_phrases": "ball security, turnover, protect/take care of the ball, limit turnovers, careless",
        "why_chosen": "Directly named -- \"ball security\"/\"turnovers\" have a 1:1 stat column (TO), no proxy needed.",
    },
    {
        "category": "Rebounding", "stat_cols": "REB, ORB, DRB",
        "example_phrases": "own the paint, bully/dominate the glass, rebound, board, second chance, crash (the glass)",
        "why_chosen": "\"Glass\"/\"board\"/\"crash\"/\"own the paint\" possession language in this scouting vocabulary is "
                      "always about winning the rebounding battle, not shot-making -- REB/ORB/DRB are the direct stats.",
    },
    {
        "category": "Three-Point Shooting", "stat_cols": "3PM-A, 3P%",
        "example_phrases": "three, 3 pt, 3pt, perimeter shooting, spacing, shooting ability, shooting team, "
                           "will shoot, sniper",
        "why_chosen": "Direct stat match for perimeter shot volume/efficiency. Expanded with \"3 pt\"/\"shooting "
                      "ability\"/\"shooting team\" from Elmhurst notes (\"High level 3 pt shooting team\") and "
                      "\"will shoot\" from Eureka (\"All 5 Will Shoot\").",
    },
    {
        "category": "Free Throws", "stat_cols": "FTM-A, FT%",
        "example_phrases": "free throw, ft line, getting to ft",
        "why_chosen": "Direct stat match. Added \"ft line\" / \"getting to ft\" from Aurora (\"getting to FT line\").",
    },
    {
        "category": "Fouls / Discipline", "stat_cols": "PF",
        "example_phrases": "foul, wall up, drawing fouls",
        "why_chosen": "Direct stat match for foul-discipline emphasis. Added \"wall up\" (Aurora) and \"drawing "
                      "fouls\" (Aurora \"Great at drawing fouls\").",
    },
    {
        "category": "Ball Movement / Assists", "stat_cols": "AST",
        "example_phrases": "assist, ball movement, share the ball, playmaking, playmaker, create",
        "why_chosen": "Direct stat match for offensive ball-sharing emphasis. Added \"playmaking\"/\"playmaker\"/ "
                      "\"create\" from Simpson/Ripon notes (\"2 playmaking guards\", \"Multiple guys that can create\").",
    },
    {
        "category": "Paint Protection / Blocks", "stat_cols": "BLK",
        "example_phrases": "block, protect the rim, paint protection",
        "why_chosen": "Direct stat match for interior shot-blocking emphasis.",
    },
    {
        "category": "Perimeter Defense / Ball Pressure", "stat_cols": "STL",
        "example_phrases": "steal, press capable, full court press, force turnovers, force to's, guard your yard, "
                           "keep the ball in front, guard 1 on 1, early gap, help side, active hands, pressure, "
                           "physical & aggressive on ball, on ball defensively",
        "why_chosen": "Renamed from \"Ball Pressure / Steals\" once Aurora/Eureka/Elmhurst introduced CONTAINMENT "
                      "phrasing (\"guard your yard\", \"keep the ball in front\", \"guard 1 on 1\") alongside the "
                      "original steal-gambling phrasing (\"press\", \"force turnovers\"). Both are point-of-attack, "
                      "on-ball defensive emphases. STL is the only stat in the season box score that reflects "
                      "defensive activity at all. Added \"pressure\" (data-driven keys), \"physical & aggressive on "
                      "ball\" / \"on ball defensively\" (Elmhurst \"Physical & Aggressive on ball defensively\"), "
                      "and \"force to's\" (Ripon). NOTE: \"press\" changed to \"press capable\"/\"full court press\" to "
                      "avoid false-positive substring matches (e.g. \"pressure\" contains \"press\").",
    },
    {
        "category": "Scoring Inside", "stat_cols": "FG2M, FG2A, FG2%",
        "example_phrases": "dominate the paint, attack the paint, live in the paint, attack the basket, "
                           "scoring at the rim, get to rim, attack the rim, get to the rim",
        "why_chosen": "Interior-scoring emphasis -- uses derived 2PT FG stats (FGM-FG3M, FGA-FG3A) as a "
                      "direct measure of inside scoring rather than overall FG efficiency. Does NOT include "
                      "free throws. Added from Aurora/Simpson/Ripon/Elmhurst/St. Thomas phrasing.",
    },
    {
        "category": "Field Goal Efficiency", "stat_cols": "FGM-A, FG%",
        "example_phrases": "limit their scoring",
        "why_chosen": "Overall shooting efficiency emphasis -- used when the scouting note is about limiting "
                      "opponent scoring broadly rather than specifically inside or from 3PT range.",
    },
    {
        "category": "(intentionally unmapped)", "stat_cols": "-",
        "example_phrases": "communication screening action / communicate screens & actions (Ripon, Elmhurst), "
                           "take away personnel tendencies (Simpson), heavy ball screen usage (Ripon), "
                           "will be their 4th game (Eureka)",
        "why_chosen": "No corresponding stat exists in a season box score for screen-navigation communication, "
                      "opponent-personnel-specific keys, scheme-specific ball screen usage, or schedule context. "
                      "These stay qualitative-only in the game-plan notes and don't produce a category row.",
    },
]

pd.set_option("display.max_colwidth", 300)
print(pd.DataFrame(KTV_CATEGORY_REFERENCE))

                             category         stat_cols  \
0           Ball Security / Turnovers                TO   
1                          Rebounding     REB, ORB, DRB   
2                Three-Point Shooting        3PM-A, 3P%   
3                         Free Throws        FTM-A, FT%   
4                  Fouls / Discipline                PF   
5             Ball Movement / Assists               AST   
6           Paint Protection / Blocks               BLK   
7   Perimeter Defense / Ball Pressure               STL   
8                      Scoring Inside  FG2M, FG2A, FG2%   
9               Field Goal Efficiency        FGM-A, FG%   
10           (intentionally unmapped)                 -   

                                                                                                                                                                                                                 example_phrases  \
0                                                              


### Build per-player scouting profiles: position, height, and playing-style tags

Builds a comparable `player_profiles` entry for every scouted player -- normalized position group, height in inches, and a set of playing-style tags mined from their free-text scouting notes (player notes + keys to defending). Box-score-only players with no scouting writeup are added with a default Bench role so they aren't silently dropped from later lineup analysis.

In [36]:
# Build a comparable "player profile" for every scouted player: normalized position group, height in inches,
# and a set of playing-style tags mined from their free-text scouting notes (player_notes + keys_to_defending).
def parse_height_inches(h):
    m = re.match(r"(\d+)'(\d+)\"?", str(h))
    return int(m.group(1)) * 12 + int(m.group(2)) if m else None

def normalize_position(pos):
    pos = str(pos).upper()
    if "G" in pos and "F" in pos:
        return "Wing"
    if "G" in pos:
        return "Guard"
    if "F" in pos or "C" in pos:
        return "Forward/Post"
    return "Unknown"

NOTES_TAG_KEYWORDS = {
    "catch_and_shoot": ["c&s", "c & s", "catch & shoot", "catch and shoot", "quick release", "spot-up", "spot up"],
    "pull_up_shooter": ["pull up", "pull-up", "mid range", "mid-range", "go to=mid", "step back"],
    "three_point_shooter": ["3's", "3pt", "three", "sniper", "shooter", "shoot"],
    "slasher_driver": ["driver", "drive", "gets to rim", "attacks the rim", "rhd", "lhd", "finish", "attack"],
    "post_scorer": ["post game", "back to the basket", "ls/rh", "post", "power post"],
    "rebounder": ["rebound", "board"],
    "playmaker": ["playmaker", "assist", "creator", "create", "distributor", "main creator", "ball mover", "ball move"],
    "physical_finisher": ["physical", "strong", "bully", "through his defender"],
    "high_usage": ["main creator", "go to"],
    "cutter": ["back cut", "curl"],
}

KEYS_TAG_KEYWORDS = {
    "deny_catch_and_shoot": [
        "c&s", "c & s", "closeout", "close out", "chase", "high hand", "early hand", "arrive on the catch",
        "stunt", "pick and pop", "pick & pop",
    ],
    "anticipate_move": ["anticipate", "antcipate", "antipipate"],
    "keep_in_front": ["keep in front", "keep him in front", "contest", "active hands", "vision off ball",
                       "stay in front", "step up", "spin back"],
    "help_defense": ["help", "gap", "wedge", "talk switches"],
    "box_out_priority": ["box out", "boxout", "keep off glass", "off the glass"],
    "post_defense": ["nls", "no ls/rh", "no rs/lh", "post defense", "front the post", "limit post touches", "post touches"],
    "physical_discipline": ["be physical", "stay down", "do not foul", "no fouls", "no foul", "wall up"],
    "pressure_disrupt": ["pressure", "presure", "disrupt", "speed up"],
    "transition_defense": ["locate in transition", "transition"],
    "deny_cuts": ["back cut"],
    "screen_navigation": ["fight over screens", "over screens", "under screens", "pops after screens"],
}

def tag_player(text, keywords):
    text = text.lower()
    text = re.sub(r"\bnon[- ]shooter\b", "", text)
    return {tag for tag, kws in keywords.items() if any(kw in text for kw in kws)}

player_profiles = all_rosters.copy()
player_profiles["height_inches"] = player_profiles["height"].apply(parse_height_inches)
player_profiles["position_group"] = player_profiles["position"].apply(normalize_position)
player_profiles["notes_tags"] = player_profiles["player_notes"].apply(lambda t: tag_player(t, NOTES_TAG_KEYWORDS))
player_profiles["keys_tags"] = player_profiles["keys_to_defending"].apply(lambda t: tag_player(t, KEYS_TAG_KEYWORDS))
player_profiles["notes_tags_display"] = player_profiles["notes_tags"].apply(lambda s: ", ".join(sorted(s)) if s else "")
player_profiles["keys_tags_display"] = player_profiles["keys_tags"].apply(lambda s: ", ".join(sorted(s)) if s else "")

# The season box score is the "table" element immediately following the "...BOXSCORE" section header. It covers
# the WHOLE roster (including deep bench players never mentioned in the scouting notes) plus a team-total row.
def extract_pdf_season_stats(elements_df, opponent):
    rows = elements_df.to_dict("records")
    box_idx = next(
        (i for i, r in enumerate(rows) if r["element_type"] == "section_header" and "BOXSCORE" in r["element_content"].upper()),
        None,
    )
    if box_idx is None or rows[box_idx + 1]["element_type"] != "table":
        print(f"No season boxscore table found for '{opponent}'.")
        return pd.DataFrame()

    stats_df = read_boxscore_table(rows[box_idx + 1]["element_content"])
    stats_df = stats_df.rename(columns={"#": "jersey_number", "PLAYER": "name"})
    # pdfplumber's table extraction sometimes truncates a multi-word name cell down to its first word (e.g. this
    # PDF's own "Team Total" row comes through as just "Team") -- so filtering on the exact string "Team Total"
    # alone can silently let that aggregate row through into player_profiles (and downstream FG%/3P% consumers
    # like player_comparison.py's parse_pct(), which then chokes on a "made-attempted" string like "617-1511").
    # Its jersey number is reliably "-" regardless of the name-cell truncation, so filter on that instead.
    stats_df = stats_df[stats_df["jersey_number"].astype(str).str.strip() != "-"]
    stats_df = stats_df[stats_df["name"] != "Opponent"]
    stats_df["jersey_number"] = "#" + stats_df["jersey_number"].astype(str)
    stats_df.insert(0, "opponent", opponent)
    return stats_df

pdf_season_stats = pd.concat(
    [extract_pdf_season_stats(df, opponent) for opponent, df in scout_reports.items()],
    ignore_index=True,
) if scout_reports else pd.DataFrame()

missing = pdf_season_stats.merge(
    player_profiles[["opponent", "jersey_number"]], on=["opponent", "jersey_number"], how="left", indicator=True
)
box_only = missing[missing["_merge"] == "left_only"][["opponent", "jersey_number", "name"]].drop_duplicates()

if not box_only.empty:
    box_only = box_only.copy()
    box_only["game_date"] = box_only["opponent"].apply(game_date_for)
    box_only["position"] = None
    box_only["height"] = None
    box_only["weight"] = None
    box_only["class_year"] = None
    box_only["role"] = "Bench"
    box_only["player_notes"] = ""
    box_only["keys_to_defending"] = ""
    box_only["height_inches"] = None
    box_only["position_group"] = "Unknown"
    box_only["notes_tags"] = [set() for _ in range(len(box_only))]
    box_only["keys_tags"] = [set() for _ in range(len(box_only))]
    box_only["notes_tags_display"] = ""
    box_only["keys_tags_display"] = ""
    box_only["has_scouting_report"] = False
    print(f"Adding {len(box_only)} box-score-only player(s) with no scouting writeup (defaulted to role=Bench):")
    print(box_only[["opponent", "jersey_number", "name"]].to_string(index=False))
    player_profiles = pd.concat([player_profiles, box_only[player_profiles.columns.tolist()]], ignore_index=True)
else:
    print("No box-score-only players found -- every player in the season boxscore already has a roster/notes entry.")

# Left-join season stats onto player_profiles by opponent + jersey number. Only opponents with a PDF report
# gain real numbers; opponents still on MHTML-only stay null -- this scales automatically as more PDF scout
# reports are added to INPUT_DIR.
stat_cols = ["MIN", "FG%", "3PM-A", "3P%", "FTM-A", "FT%", "REB", "AST", "TO", "STL", "BLK", "PTS"]
player_profiles = player_profiles.merge(
    pdf_season_stats[["opponent", "jersey_number"] + stat_cols],
    on=["opponent", "jersey_number"],
    how="left",
)

coverage = player_profiles.groupby("opponent")["PTS"].apply(lambda s: s.notna().sum())
print("\nPlayers with real season stats attached (from a PDF scout report), by opponent:")
print(coverage.to_string())

print(player_profiles[[
    "opponent", "jersey_number", "name", "role", "notes_tags_display", "keys_tags_display", "PTS", "REB", "AST", "FG%", "3P%",
]])

No season boxscore table found for 'Hope Flying Dutchmen'.
Adding 64 box-score-only player(s) with no scouting writeup (defaulted to role=Bench):
                  opponent jersey_number                name
     St. Thomas (TX) Celts           #23          Jaden Ross
     St. Thomas (TX) Celts           #11     Tamarcus Butler
     St. Thomas (TX) Celts           #13       Legborsi Mato
         Eureka Red Devils           #35      Max Richardson
         Eureka Red Devils           #20          Nolan Kerr
         Eureka Red Devils           #22          Tony Mabon
         Eureka Red Devils           #15     Amare Brokemond
         Eureka Red Devils            #3       Blake Logsdon
         Eureka Red Devils            #0       Colin DeLaere
         Eureka Red Devils            #1       Dylan Logsdon
           Aurora Spartans            #0        Cullen Rauls
           Aurora Spartans           #30           Alex Ross
           Aurora Spartans           #21     Isaiah Thompson



### Break out Keys-to-Victory categories by opponent starter vs. bench role

For each per-game Keys-to-Victory category (from the win/loss-splits cell above), breaks out the scouted opponent's own production in that category by role -- Starter vs. Bench -- to show whether an emphasis like "own the glass" was really driven by starters or bench depth.

In [38]:
# For each per-game "Keys to Victory" category (from the win/loss-splits cell above), break out the SCOUTED
# OPPONENT's own production in that category by role -- Starter vs. Bench -- using the season stats merged onto
# player_profiles. This shows whether an emphasis like "own the glass" was really about containing the
# opponent's starting five or their bench unit.
CATEGORY_TO_STAT_COLS = {}
for col, cat in STAT_COL_CATEGORY.items():
    CATEGORY_TO_STAT_COLS.setdefault(cat, []).append(col)

COUNT_STATS = {"TO", "REB", "AST", "STL", "BLK", "PTS"}

def parse_numeric_stat(val):
    s = str(val).strip()
    if s in ("", "-", "nan", "None"):
        return None
    if s.endswith("%"):
        return float(s.rstrip("%"))
    if "-" in s and not s.startswith("-"):
        try:
            return float(s.split("-")[0])  # compound "made-attempted" string -- use the made count
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

role_breakdown_rows = []
for _, row in game_categories.iterrows():
    if row["category"] == "(no matched category)":
        continue
    stat_cols = [c for c in CATEGORY_TO_STAT_COLS.get(row["category"], []) if c in player_profiles.columns]
    if not stat_cols:
        continue
    opp_players = player_profiles[player_profiles["opponent"] == row["opponent"]]
    for role in ["Starter", "Bench"]:
        role_players = opp_players[opp_players["role"] == role]
        if role_players.empty:
            continue
        for col in stat_cols:
            vals = role_players[col].apply(parse_numeric_stat)
            if vals.notna().any():
                role_breakdown_rows.append({
                    "opponent": row["opponent"], "outcome": row["outcome"], "category": row["category"],
                    "role": role, "stat": col, "players_with_data": int(vals.notna().sum()),
                    "avg_per_player": round(vals.mean(), 2),
                    "role_total": round(vals.sum(), 2) if col in COUNT_STATS else None,
                })

role_breakdown = pd.DataFrame(
    role_breakdown_rows,
    columns=["opponent", "outcome", "category", "role", "stat", "players_with_data", "avg_per_player", "role_total"],
)
print("Opponent production by role (Starter vs Bench) for the stat(s) behind each matched Keys-to-Victory category:")
print(role_breakdown)

count_stat_rows = role_breakdown[role_breakdown["stat"].isin(COUNT_STATS)]
if not count_stat_rows.empty:
    pivot = count_stat_rows.pivot_table(
        index=["opponent", "category", "stat"], columns="role", values="role_total", aggfunc="first"
    ).reset_index()
    for r in ["Starter", "Bench"]:
        if r not in pivot.columns:
            pivot[r] = 0.0
    pivot["starter_share"] = (pivot["Starter"] / (pivot["Starter"] + pivot["Bench"]).replace(0, pd.NA)).round(3)
    print("\nStarter vs Bench totals for count-type stats, side-by-side (starter_share near 1 = concentrated among "
          "starters; near 0 = bench-driven):")
    print(pivot.sort_values(["opponent", "category", "stat"]))
else:
    print("No count-type stats available yet to compare Starter vs Bench totals for the matched categories.")

Opponent production by role (Starter vs Bench) for the stat(s) behind each matched Keys-to-Victory category:
                      opponent outcome                           category  \
0        St. Thomas (TX) Celts       L          Ball Security / Turnovers   
1        St. Thomas (TX) Celts       L          Ball Security / Turnovers   
2        St. Thomas (TX) Celts       L              Field Goal Efficiency   
3        St. Thomas (TX) Celts       L              Field Goal Efficiency   
4        St. Thomas (TX) Celts       L                         Rebounding   
5        St. Thomas (TX) Celts       L                         Rebounding   
6            Eureka Red Devils       W  Perimeter Defense / Ball Pressure   
7            Eureka Red Devils       W  Perimeter Defense / Ball Pressure   
8            Eureka Red Devils       W                         Rebounding   
9            Eureka Red Devils       W                         Rebounding   
10             Aurora Spartans       W  Peri


### Play-by-play parsing functions

Each game's play-by-play is a separate MHTML snapshot of FastScout's playByPlay page (same "Save as Webpage, Single File" format as the season schedule), uploaded as `"<date> UW-Whitewater @ <Opponent>_pbp.mhtml"` (or `"<date> <Opponent> @ UW-Whitewater_pbp.mhtml"` for home games). `parse_pbp_html`/`parse_pbp_mhtml` parse that table; `scrape_pbp_live` is the live-scrape fallback, reused by every PBP-fetching cell further down.

In [40]:
# --- Play-by-play (PBP) parsing functions -------------------------------------------------------------------
# Each game's play-by-play is a separate MHTML snapshot of FastScout's playByPlay page (same "Save as Webpage,
# Single File" format as the season schedule), uploaded as "<date> UW-Whitewater @ <Opponent>_pbp.mhtml" (or
# "<date> <Opponent> @ UW-Whitewater_pbp.mhtml" for home games).
def parse_pbp_html(html):
    """Core play-by-play table parsing, factored out of parse_pbp_mhtml() so it can run on HTML from either
    source: a manually-exported/uploaded MHTML snapshot, OR a live Playwright page.content() capture -- see
    scrape_pbp_live() below, which the user confirmed is reachable via a small path change to a game's own
    "game_url" (already available per-game via team_schedules/opp_schedule -- see the schedule-scraping cell
    and the opponent-schedule cell)."""
    soup = BeautifulSoup(html, "lxml")
    tables = soup.find_all("table")
    raw_df = pd.read_html(StringIO(str(tables[0])))[0]
    raw_df.columns = ["time_raw", "uww_text", "score_raw", "opp_text"]
    return raw_df


def parse_pbp_mhtml(path):
    return parse_pbp_html(load_html_snapshot(path))


def scrape_pbp_live(page, game_url, timeout_ms=30000, save_path=None):
    """Navigate to a game's own FastScout play-by-play page and capture its rendered event table.

    Confirmed by a live run: the previous approach -- swap the trailing "/boxscore" segment of `game_url`
    for "/playbyplay" and goto() straight there -- does NOT work. FastScout's SPA only renders this kind of
    sub-route via an in-app tab click, exactly like the already-documented behavior in _click_tab_by_text's
    own docstring above ("Direct page.goto() to a sub-route URL ... gets silently redirected back to
    '/analytics/dashboard'"). A direct goto() to ".../playbyplay?..." gets redirected away from the game
    page entirely, so the FIRST "table" wait (8s) never found one -- and then the fallback tab click ALSO
    timed out (30s) because there was no "Play By Play" tab on whatever page it actually landed on. Always
    goto() the game's own ORIGINAL `game_url` (the boxscore URL -- a valid full-page-load entry point, same
    as every other team/opponent page navigation in this notebook) and click the "Play By Play" nav tab
    in-app from there, instead of ever attempting a direct URL swap. If save_path is given, caches the
    scraped HTML there (see _save_scraped_html in Cell 4) -- the same persist-everything-scraped pattern
    already used for scouting reports and team schedules -- so a future run can fall back to it via
    parse_pbp_mhtml (through load_html_snapshot) without needing to live-scrape again.
    """
    _goto_with_auth_retry(page, game_url, "text=Play By Play", timeout_ms)
    _click_tab_by_text(page, "Play By Play", timeout_ms)
    page.wait_for_selector("table", timeout=timeout_ms)
    html = page.content()
    if save_path:
        _save_scraped_html(html, save_path, "scraped play-by-play")
    return parse_pbp_html(html)

### Classify each raw play-by-play text string into an event type

The 4-column layout is "Time | UW-Whitewater | Score | \<Opponent\>" -- almost every row has exactly one of the two team-text columns filled in (one atomic event per row), alongside the running score. A handful of event strings are team-level with no player name at all (a shot-clock/backcourt turnover, or a held-ball jump ball) -- those get matched by exact text instead of a regex.

In [42]:
# The 4-column layout is "Time | UW-Whitewater | Score | <Opponent>" -- almost every row has exactly ONE of the
# two team-text columns filled in (one atomic event per row), alongside the running score. A few event strings
# are TEAM-level with no player name at all (a shot-clock/backcourt turnover, or a held-ball jump ball).
EVENT_PATTERNS = [
    ("jump_ball_won", re.compile(r"^(?P<player>.+?) Wins Jump Ball$")),
    ("jump_ball_lost", re.compile(r"^(?P<player>.+?) Loses Jump Ball$")),
    # A held ball can be logged with any parenthetical reason ("(Held Ball)", "(Block Tie Up)", ...).
    # TEAM_LEVEL_EXACT only ever listed "(Held Ball)", so every other variant fell through to the
    # unclassified fallback and became its own fake player -- "Jump Ball (Block Tie Up)" reached the box
    # score with more total minutes than any real player on the roster. No player group on purpose.
    ("jump_ball_held", re.compile(r"^Jump Ball \(.+\)$")),
    ("made_shot", re.compile(r"^(?P<player>.+?) Makes (?P<shot_type>\d)PT(?: (?P<shot_desc>.+))?$")),
    ("missed_shot", re.compile(r"^(?P<player>.+?) Misses (?P<shot_type>\d)PT(?: (?P<shot_desc>.+))?$")),
    ("rebound_offensive", re.compile(r"^(?P<player>.+?) Offensive Rebound$")),
    ("rebound_defensive", re.compile(r"^(?P<player>.+?) Defensive Rebound$")),
    ("team_deadball_rebound_offensive", re.compile(r"^(?P<player>.+?) Offensive Deadball Rebound$")),
    ("team_deadball_rebound_defensive", re.compile(r"^(?P<player>.+?) Defensive Deadball Rebound$")),
    ("assist", re.compile(r"^(?P<player>.+?) Assists$")),
    ("steal", re.compile(r"^(?P<player>.+?) Steals$")),
    ("block", re.compile(r"^(?P<player>.+?) Blocks$")),
    # The parenthetical turnover-type suffix (e.g. "(Bad Pass)") is present for some individual turnovers but
    # MISSING entirely for others (e.g. "Corey Thompson Turnover") -- making the suffix optional handles both.
    ("turnover", re.compile(r"^(?P<player>.+?) Turnover(?: \((?P<turnover_type>.+)\))?$")),
    # A bare TEAM-level turnover with a parenthetical type but NO player attached (e.g. "Turnover (Offensive
    # Foul)") previously fell all the way through to the "unclassified" catch-all below, whose fallback sets
    # player = the ENTIRE raw text -- so the literal string "Turnover (Offensive Foul)" ended up as its own
    # "player" row in the reconstructed box score (confirmed: exactly this string, in exactly this shape,
    # showed up in a real game's box score). TEAM_LEVEL_EXACT only covered the bare "Turnover" case with no
    # parenthetical at all -- this regex generalizes to ANY bare "Turnover (TYPE)" variant instead of needing
    # every possible type enumerated individually. No `(?P<player>...)` group here on purpose: leaving
    # "player" out of this match's groupdict() means the merged pbp_events row gets player=NaN naturally
    # (pandas fills a missing dict key with NaN when building the DataFrame), which is exactly what makes the
    # box-score builder's `pbp_events["player"].notna()` filter correctly exclude it.
    ("turnover", re.compile(r"^Turnover(?: \((?P<turnover_type>.+)\))?$")),
    # foul_type was required to be "<something> Foul", so a bare "<Player> Commits Foul" -- a real,
    # common line with no foul type recorded -- never matched, and the fallback turned the WHOLE string
    # into a player name ("Damyen Jackson Commits Foul" appeared as a person, with minutes). Making the
    # descriptor optional classifies those correctly AND credits the foul to the right player, rather
    # than just discarding them.
    ("foul", re.compile(r"^(?P<player>.+?) Commits (?P<foul_type>.*?Foul)$")),
    ("free_throw_made", re.compile(r"^(?P<player>.+?) Makes Free Throw \((?P<ft_num>\d+) of (?P<ft_total>\d+)\)$")),
    ("free_throw_missed", re.compile(r"^(?P<player>.+?) Misses Free Throw \((?P<ft_num>\d+) of (?P<ft_total>\d+)\)$")),
    ("sub_in", re.compile(r"^(?P<player>.+?) Subs In$")),
    ("sub_out", re.compile(r"^(?P<player>.+?) Subs Out$")),
    # A timeout is called by a TEAM or an official ("Official TV Timeout"), never by a roster player, so
    # the caller is deliberately not captured as `player` -- it stays in raw_text. Keeping it meant
    # "Official TV" and every team name showed up wherever player names are enumerated.
    ("timeout", re.compile(r"^.+? Timeout$")),
    ("ejected", re.compile(r"^(?P<player>.+?) Ejected$")),
]
TEAM_LEVEL_EXACT = {
    "Turnover": {"event_type": "turnover", "player": None, "turnover_type": "Team"},
    # A few games log a bare team rebound with NO player prefix at all -- classified into the same
    # team_deadball_rebound_* buckets as the other team-level rebound format.
    "Offensive Rebound": {"event_type": "team_deadball_rebound_offensive", "player": None},
    "Defensive Rebound": {"event_type": "team_deadball_rebound_defensive", "player": None},
    "Jump Ball (Held Ball)": {"event_type": "jump_ball_held", "player": None},
    # A foul logged with neither a player nor a type.
    "Commits Foul": {"event_type": "foul", "player": None},
}


def classify_event(text):
    text = text.strip()
    if text in TEAM_LEVEL_EXACT:
        return dict(TEAM_LEVEL_EXACT[text])
    for event_type, pattern in EVENT_PATTERNS:
        m = pattern.match(text)
        if m:
            return {"event_type": event_type, **m.groupdict()}
    # CONFIRMED BUG (fixed here): this used to return `player=text`, so any line the patterns above don't
    # recognise became a PLAYER named after the raw event string -- with its own box-score row, its own
    # minutes from the lineup reconstruction, and a place in the leaderboards. Keep the text in raw_text
    # (build_pbp_events already stores it) and leave `player` empty, so an unrecognised line is counted and
    # reported but can never masquerade as a person. Add a pattern above for anything that shows up here.
    return {"event_type": "unclassified", "player": None}

### Turn the raw 4-column play-by-play table into one row per event

`build_pbp_events` carries the running score forward onto every row and labels each row with which team it belongs to. `self_column` must be resolved per-file (see `resolve_self_column` later on) since FastScout puts the exporting team's own events in a fixed column regardless of home/away, but which raw column that is varies by whose account captured the snapshot.

In [44]:
def parse_time_to_seconds(time_raw):
    """'19:58 (H1)' -> ('H1', 1198)."""
    if pd.isna(time_raw):
        return None, None
    m = re.match(r"(\d+):(\d+)\s*\((\w+)\)", str(time_raw).strip())
    if not m:
        return None, None
    minutes, seconds, period = m.groups()
    return period, int(minutes) * 60 + int(seconds)


def build_pbp_events(raw_df, opponent, game_date, self_team="UW-Whitewater", self_column="uww_text"):
    """One row per event (or per period-start/end marker), carrying the running score after that event.
    `self_team` labels whichever column `self_column` points at. Defaults match UW-Whitewater's own game files,
    where the "uww_text" column is always UWW's own events. For an opponent's OWN schedule snapshot (e.g.
    scouting an opponent's games before they face Whitewater), FastScout puts the exporting team's own events in
    a FIXED column regardless of home/away -- but which raw column that is varies file-to-file depending on
    whose FastScout account captured it, so the caller must resolve `self_column` empirically (e.g. by matching
    known roster player names) rather than assume "uww_text"."""
    rows = []
    current_period = None
    for i, r in raw_df.iterrows():
        period, seconds = parse_time_to_seconds(r["time_raw"])
        if period:
            current_period = period
        score_raw = r["score_raw"]
        score_m = re.match(r"^(\d+)-(\d+)$", str(score_raw).strip()) if pd.notna(score_raw) else None
        if score_m is None:
            # e.g. "Start 1st Half" / "End 2nd Half" -- no team/player/score attached to these marker rows
            rows.append({
                "opponent": opponent, "game_date": game_date, "event_order": i, "period": current_period,
                "time_remaining": None, "time_remaining_seconds": None, "team": None,
                "event_type": "period_marker", "raw_text": str(score_raw).strip(),
                "uww_score": None, "opp_score": None,
            })
            continue
        uww_score, opp_score = int(score_m.group(1)), int(score_m.group(2))
        opp_column = "opp_text" if self_column == "uww_text" else "uww_text"
        for team_label, text in [(self_team, r[self_column]), (opponent, r[opp_column])]:
            if pd.notna(text) and str(text).strip():
                rows.append({
                    "opponent": opponent, "game_date": game_date, "event_order": i, "period": current_period,
                    "time_remaining": r["time_raw"], "time_remaining_seconds": seconds, "team": team_label,
                    "raw_text": str(text).strip(), "uww_score": uww_score, "opp_score": opp_score,
                    **classify_event(str(text).strip()),
                })
    return pd.DataFrame(rows)

### Extract the opponent name from a `_pbp.mhtml` filename

In [46]:
def opponent_from_pbp_filename(path):
    """Home/away-aware opponent extraction -- mirrors opponent_from_scout_filename() used for the scout PDFs.
    Filenames are "<date> <Away> @ <Home>_pbp.mhtml" (or "..._pbp.html" for a live-scraped/cached file -- see
    _save_scraped_html), so the correct opponent is whichever side of "@" ISN'T "UW-Whitewater", not simply
    "everything after @" -- that naive approach is right for away games ("UW-Whitewater @ Ripon" -> "Ripon")
    but wrong for home games ("Aurora @ UW-Whitewater" would otherwise extract "UW-Whitewater" itself as the
    opponent).

    CONFIRMED BUG (fixed here): this only ever stripped the "_pbp.mhtml" suffix, never "_pbp.html" -- so for
    every ".html"-sourced file (any auto-downloaded/live-scraped PBP, which is most of them going forward)
    the extension silently rode along attached to whichever side of "@" this function returns. For an AWAY
    game specifically, that's the side actually returned (left == "UW-Whitewater", so `right` -- e.g. "Ripon
    Red Hawks_pbp.html" -- comes back instead of "Ripon Red Hawks"), corrupting the opponent name used to key
    every downstream table (pbp_events, pbp_box_score, lineup_stints, etc.) for that game. HOME games
    happened to come out clean by accident (the garbage suffix landed on `right`, which isn't the branch
    returned when left != "UW-Whitewater"), which is why this only ever broke away games -- confirmed via the
    box-score reconciliation diagnostic above flagging every single away game and zero home games."""
    name = re.sub(r"_pbp\.(mhtml|html)$", "", os.path.basename(path))
    name = re.sub(r"^\d+_\d+_\d+\s+", "", name)
    left, right = [side.strip() for side in name.split(" @ ", 1)]
    return right if left == "UW-Whitewater" else left


# --- A GAME IS (opponent, game_date), NEVER opponent ALONE ---------------------------------------
# CONFIRMED BUG (fixed here): every table below used the short opponent name as a game's identity.
# That silently merges a home-and-home (or a third conference meeting) into ONE game: box-score
# stats get summed across both meetings, the games-played denominator counts them once, and the
# lineup-stint clock -- diffed within ("opponent", "period") while `event_order` restarts at 0 each
# game -- interleaves the two meetings and re-counts the same seconds, inflating minutes many-fold.
# Reconciling uww_pbp_box_score against uww_schedule showed this exactly: UW-La Crosse came out
# 206-209 (63+65+78 vs 60+68+81), i.e. three real games stacked into one row.
#
# GAME_KEYS is the grouping/merge key every per-game computation must use from here on.
GAME_KEYS = ["opponent", "game_date"]


def game_date_from_pbp_filename(path):
    """The game's own date, read from the '<m>_<d>_<yy> ' prefix these files are named with.

    This is the ONLY per-file source of truth for WHICH meeting a play-by-play file covers.
    game_date_for() cannot answer that -- it fuzzy-matches the schedule on opponent name and takes
    .iloc[0], so for a rematch it always returns the FIRST meeting's date. Returns None when a file
    carries no date prefix, so the caller can fall back (loudly) rather than guessing silently.
    """
    from datetime import date as _date
    m = re.match(r"^(\d+)_(\d+)_(\d+)\s", os.path.basename(path))
    if not m:
        return None
    month, day, yy = (int(g) for g in m.groups())
    return _date(2000 + yy, month, day)


### Identify the upcoming opponent

Read the single "Upcoming" row off `schedule` (built in Cell 4) to get the opponent's full name, then match it against `scouted_opponents` to get the short name used throughout the rest of this notebook (filenames, `player_profiles`, etc).

In [48]:
# Identify the upcoming opponent from the schedule
upcoming_game = schedule[schedule["Upcoming"] == "Yes"].iloc[0]
upcoming_opponent = upcoming_game["opponent"]
upcoming_opponent_short = next(
    s for s in scouted_opponents if re.search(re.escape(s), upcoming_opponent, re.IGNORECASE)
)


### Locate the opponent's own schedule backup file

`schedule` only has ONE row per opponent (their single game vs UWW), so it can't tell us what this opponent's own games looked like before they played Whitewater. `_schedule_file_for` finds their own "\<Team\> - Schedule.mhtml"/".html" backup file on disk -- used only as a fallback (see the next cell, which prefers the schedule Cell 4 already scraped/parsed live).

In [50]:
# `schedule` only has ONE row per opponent (their single game vs UWW), so it can't tell us what THIS
# opponent's own games looked like before they played Whitewater. Load their own FastScout team schedule page
# instead -- saved the same way as UW-Whitewater's own ("<Opponent> - Schedule.mhtml"), parsed identically to
# the schedule-parsing cell above.
def _schedule_file_for(opponent_full_name, opponent_short_hint, volume_dir):
    """Find this opponent's own '<Team> - Schedule.mhtml' backup file, tolerant of short-vs-full naming --
    confirmed by a live run: scouted_opponents (and upcoming_opponent_short derived from it) now holds FULL
    team names (e.g. 'Elmhurst Bluejays') since auto-downloaded HTML scout reports carry the schedule's full
    opponent name in their filename, but the manually-uploaded backup schedule MHTMLs predate that change and
    still use the SHORT team name (e.g. 'Elmhurst - Schedule.mhtml') -- an exact '{short} - Schedule.mhtml'
    match no longer finds them. Match on whichever side is a substring of the other instead."""
    # Glob both ".mhtml" (a manually-exported/uploaded snapshot) and ".html" (this notebook's own
    # live-scrape cache -- see _save_scraped_html in Cell 4).
    for p in glob.glob(f"{volume_dir}/* - Schedule.mhtml") + glob.glob(f"{volume_dir}/* - Schedule.html"):
        file_team = re.sub(r"\s*-\s*Schedule\.(mhtml|html)$", "", os.path.basename(p), flags=re.IGNORECASE)
        if file_team.lower() in opponent_full_name.lower() or opponent_short_hint.lower() in file_team.lower():
            return p
    return f"{volume_dir}/{opponent_short_hint} - Schedule.mhtml"  # fall back to the old guess, for the warning message below


### Resolve the opponent's parsed schedule from Cell 4

Cell 4 already scraped and parsed every opponent's own schedule into `team_schedules` this run (live from FastScout, with a local MHTML backup only as a last resort) -- reuse that in-memory result instead of re-deriving a file path and re-parsing HTML from scratch. A live-scraped opponent schedule IS cached to disk too, via `scrape_rendered_html`'s `save_path` in Cell 4 -- this cell just prefers the in-memory `team_schedules` result already parsed THIS run.

In [52]:
# Live-scraped opponent schedules ARE cached to disk too (scrape_rendered_html's save_path writes
# "<Team> - Schedule.html" via _save_scraped_html). This cell just prefers the in-memory team_schedules
# result already parsed THIS run.
opp_team_schedule = next(
    (
        ts for ts in team_schedules
        if not ts.empty and (
            ts["team"].iloc[0].lower() in upcoming_opponent.lower()
            or upcoming_opponent_short.lower() in ts["team"].iloc[0].lower()
        )
    ),
    None,
)

### Build `opp_schedule`

Prefer the live-scraped/parsed entry already sitting in `team_schedules` (from Cell 4) over a local backup file -- a live scrape never gets written back to disk as "<Team> - Schedule.mhtml", so a file-only lookup would report "not found" even right after a fully successful live scrape.

In [54]:
if opp_team_schedule is not None:
    opp_schedule = pd.DataFrame()
    opp_schedule["date"] = opp_team_schedule["date"]
    opp_schedule["game_date"] = opp_schedule["date"].apply(parse_schedule_date)
    opp_schedule["opponent"] = opp_team_schedule["opponent"]
    opp_schedule["outcome"] = opp_team_schedule["outcome"]
    opp_schedule["team_score"] = opp_team_schedule["team_score"]
    opp_schedule["opponent_score"] = opp_team_schedule["opponent_score"]
    # team_schedules (built in Cell 4 from the live scrape/MHTML) already carries each game's FastScout/Synergy
    # links straight from that row's own <a href> tags -- confirmed by the user: video_url is a direct link to
    # that game's full film on Synergy (e.g. "https://editor-web.synergysports.com/video?playlistUrl=...").
    # Carry it through here (dropped in the manual column selection above) so prev_games below exposes a
    # clickable video link per prior game even when no separately-exported/tagged "_video.mhtml" file exists.
    opp_schedule["video_url"] = opp_team_schedule["video_url"]
    opp_schedule["game_url"] = opp_team_schedule["game_url"]
    # Needed to build an accurate "<Away> @ <Home>" matchup string for live-scrape cache filenames below
    # (mirrors the same location-based matchup naming already used for scout-report downloads in Cell 4).
    opp_schedule["location"] = opp_team_schedule["location"]
else:
    # team_schedules truly has no entry for this opponent (e.g. it wasn't in UWW's own scouted schedule at
    # all this run) -- fall back to the old local "<Team> - Schedule.mhtml" backup file lookup.
    opp_schedule_path = _schedule_file_for(upcoming_opponent, upcoming_opponent_short, volume_dir)
    if not os.path.exists(opp_schedule_path):
        print(f"WARNING: Opponent schedule file not found: {opp_schedule_path}")
        print(f"    Upload '{upcoming_opponent_short} - Schedule.mhtml' to enable full scouting analysis.")
        print(f"    Skipping opponent schedule parsing and prior-game pbp analysis for {upcoming_opponent_short}.")
        opp_schedule = pd.DataFrame(columns=["date", "game_date", "opponent", "outcome", "team_score", "opponent_score"])
    else:
        opp_html = load_html_snapshot(opp_schedule_path)
        opp_schedule_raw = pd.read_html(StringIO(str(BeautifulSoup(opp_html, "lxml").find_all("table")[0])))[0]

        opp_schedule = pd.DataFrame()
        opp_schedule["date"] = opp_schedule_raw["Date"]
        opp_schedule["game_date"] = opp_schedule["date"].apply(parse_schedule_date)
        opp_schedule["opponent"] = opp_schedule_raw["Opponent"].apply(lambda x: split_opponent(x)[0])
        res_split = opp_schedule_raw["Result"].apply(split_result)
        opp_schedule["outcome"] = res_split.apply(lambda x: x[0])
        opp_schedule["team_score"] = res_split.apply(lambda x: x[1])
        opp_schedule["opponent_score"] = res_split.apply(lambda x: x[2])

### Find the opponent's games before they played UW-Whitewater

Also print each game's raw video link and check whether a local `_pbp`/`_video` file already exists for it -- these are the games the next two cells try to fill in via a live scrape when a file is missing.

In [56]:
if opp_schedule.empty:
    prev_games = pd.DataFrame(columns=["date", "game_date", "opponent", "outcome", "team_score", "opponent_score"])
else:
    # Whitewater's own matchup date -- games strictly before that are what to check for existing _pbp/_video
    # files (the opponent's games against teams other than Whitewater). Read this straight from UWW's OWN
    # schedule row (upcoming_game, already resolved above) rather than searching for a "vs Whitewater" row
    # inside the OPPONENT's own schedule -- confirmed by a live run: when opp_schedule comes from
    # team_schedules, that entry was already filtered (in Cell 4) to games strictly before reference_date,
    # which can exclude the Whitewater matchup itself if it falls ON OR AFTER reference_date from the
    # opponent's side (e.g. Elmhurst's Dec 2 game vs a Dec 1 reference_date) -- so searching for it inside
    # opp_schedule can come up empty even though the date is already known independently.
    whitewater_date = parse_schedule_date(upcoming_game["date"])
    if whitewater_date is None:
        raise ValueError(f"Could not parse UW-Whitewater's own matchup date for {upcoming_opponent_short} from {upcoming_game['date']!r}.")

    prev_games = opp_schedule[opp_schedule["game_date"] < whitewater_date].reset_index(drop=True)
    print(f"{upcoming_opponent_short}'s games before facing UW-Whitewater on {whitewater_date}:")
    print(prev_games)
    if "video_url" in prev_games.columns:
        print("\nVideo links for these games (raw full-game film, independent of any manually-tagged _video.mhtml export):")
        for _, _pg_row in prev_games.iterrows():
            # game_date can come through as either datetime.date or datetime.datetime/Timestamp depending on
            # how opp_schedule was sourced -- pd.Timestamp(...) normalizes either into something .date() works
            # on, rather than assuming one specific type (confirmed by a live run: plain .date() raised
            # "'datetime.date' object has no attribute 'date'" here).
            print(f"  {pd.Timestamp(_pg_row['game_date']).date()} vs {_pg_row['opponent']}: {_pg_row['video_url'] or '(no video_url)'}")

    # Check for _pbp and _video files for each previous game. Filenames are "<m>_<d>_<yy> ..." (no leading zeros).
    missing_files = []
    for _, row in prev_games.iterrows():
        # "%-m"/"%-d" (no-leading-zero month/day) are a glibc-only strftime extension -- Windows' CRT
        # strftime rejects them outright with "ValueError: Invalid format string". Since this notebook runs
        # via Databricks Connect from a local Windows machine, build the no-leading-zero month/day manually
        # instead (plain int formatting has no platform-specific behavior), keeping "%y" (a portable, standard
        # strftime directive) for the 2-digit year.
        game_date_str = f"{row['game_date'].month}_{row['game_date'].day}_{row['game_date'].strftime('%y')}"
        # "_pbp.*" (not just "_pbp.mhtml") so this also finds this notebook's own live-scrape ".html" cache
        # (see _save_scraped_html in Cell 4), the same broad wildcard already used for "_video.*" below.
        pbp_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_pbp.*"
        video_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_video.*"
        pbp_files = glob.glob(pbp_pattern)
        video_files = glob.glob(video_pattern)
        if not pbp_files or not video_files:
            missing_files.append({
                "date": row["game_date"], "opponent": row["opponent"],
                "pbp_found": bool(pbp_files), "video_found": bool(video_files)
            })

    if missing_files:
        print("\nMissing _pbp or _video files for the following games before the Whitewater matchup:")
        for mf in missing_files:
            print(f"  {mf['date']} vs {mf['opponent']}: pbp_found={mf['pbp_found']}, video_found={mf['video_found']}")
    else:
        print("\nAll required _pbp and _video files found for the upcoming opponent's previous games before Whitewater.")

Loras Duhawks's games before facing UW-Whitewater on 2026-03-06:
           date   game_date                           opponent outcome  \
0    Sat, Nov 8  2025-11-08                      DePauw Tigers       L   
1    Sun, Nov 9  2025-11-09                  Kalamazoo Hornets       W   
2   Fri, Nov 14  2025-11-14                  Blackburn Beavers       W   
3   Sun, Nov 16  2025-11-16  Southern Indiana Screaming Eagles       L   
4   Sat, Nov 22  2025-11-22       North Central (IL) Cardinals       W   
5   Sat, Nov 29  2025-11-29   Wisconsin-Superior Yellowjackets       W   
6    Wed, Dec 3  2025-12-03                   Dubuque Spartans       W   
7    Sat, Dec 6  2025-12-06                      Simpson Storm       W   
8   Sat, Dec 13  2025-12-13                   Wartburg Knights       W   
9   Mon, Dec 15  2025-12-15         Westminster (MO) Blue Jays       L   
10  Thu, Dec 18  2025-12-18       Monmouth (IL) Fighting Scots       W   
11  Tue, Dec 30  2025-12-30              Lake F

### Live-scrape any missing play-by-play files as a fallback

Reuses the one shared FastScout Playwright session from Cell 4 (`run_in_fastscout_session`) rather than opening its own browser+thread.

In [58]:
if not opp_schedule.empty:
    # Determine which raw column ("uww_text" or "opp_text") actually holds THIS opponent's own events for each pbp
    # file -- FastScout puts the exporting team's events in a fixed column regardless of home/away, but WHICH
    # column that is depends on whose account captured the snapshot. Resolve it per file by matching known
    # roster player names instead of assuming.
    known_names = set(player_profiles.loc[player_profiles["opponent"] == upcoming_opponent_short, "name"].dropna())

    def resolve_self_column(raw_df, known_names):
        uww_matches = sum(any(name in str(t) for name in known_names) for t in raw_df["uww_text"].dropna())
        opp_matches = sum(any(name in str(t) for name in known_names) for t in raw_df["opp_text"].dropna())
        return "uww_text" if uww_matches >= opp_matches else "opp_text"

    # Find which prior games are missing a local "_pbp.mhtml" file AND have a usable "game_url" to live-scrape
    # instead (only present when opp_schedule was sourced from team_schedules -- see above).
    games_needing_live_pbp = []
    for _, row in prev_games.iterrows():
        game_date_str = f"{row['game_date'].month}_{row['game_date'].day}_{row['game_date'].strftime('%y')}"
        pbp_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_pbp.*"
        if not glob.glob(pbp_pattern) and "game_url" in row.index and pd.notna(row.get("game_url")):
            games_needing_live_pbp.append(row)

    # Confirmed by the user: a game's own play-by-play page is reachable via a small path change to its
    # "game_url" (see scrape_pbp_live() in the pbp-parsing cell above) -- live-scrape it for any prior game
    # missing a local "_pbp.mhtml" file, the same live-scrape-with-fallback pattern Cell 4 already uses for
    # schedules and scout reports. One shared session covers every game that needs this, rather than opening
    # a new browser per game.
    live_pbp_by_game = {}
    if games_needing_live_pbp and fastscout_username and fastscout_password:
        # One run_in_fastscout_session call PER GAME (not one call looping over all games) -- run_in_fastscout_
        # session's own retry-on-dead-session logic (Cell 4) can only kick in on a call it directly wraps, so
        # if every game were scraped inside a single call, a mid-batch dead browser/driver would silently fail
        # every remaining game with no chance to self-heal. Reuses the ONE shared FastScout Playwright session
        # (opened lazily on first use) instead of opening its own separate browser+thread per game.
        for g_row in games_needing_live_pbp:
            try:
                # Same location-based "<Away> @ <Home>" matchup naming as scout-report downloads (Cell 4) and
                # the video-clip cache below, so the saved file matches the manually-uploaded naming
                # convention closely enough for the glob patterns above to find it on a future run.
                g_date_str = f"{g_row['game_date'].month}_{g_row['game_date'].day}_{g_row['game_date'].strftime('%y')}"
                if str(g_row.get("location", "")).strip().lower() == "home":
                    matchup = f"{g_row['opponent']} @ {upcoming_opponent_short}"
                else:
                    matchup = f"{upcoming_opponent_short} @ {g_row['opponent']}"
                pbp_save_path = f"{volume_dir}/{g_date_str} {matchup}_pbp.html"
                live_pbp_by_game[g_row["game_date"]] = run_in_fastscout_session(
                    lambda page, url=g_row["game_url"], sp=pbp_save_path: scrape_pbp_live(page, url, save_path=sp)
                )
            except Exception as pbp_scrape_error:
                print(f"  Could not live-scrape pbp for {g_row['opponent']} ({g_row['game_url']}): {type(pbp_scrape_error).__name__}: {pbp_scrape_error}")

### Parse play-by-play events for the opponent's prior games

Builds `pbp_events_upcoming` from every local (or just-cached, live-scraped) `_pbp` file found for these prior games.

In [60]:
if opp_schedule.empty:
    pbp_events_upcoming = pd.DataFrame(columns=[
        "opponent", "game_date", "event_order", "period", "time_remaining",
        "time_remaining_seconds", "team", "event_type", "raw_text",
        "uww_score", "opp_score", "player", "shot_type", "shot_desc",
        "turnover_type", "foul_type", "ft_num", "ft_total"
    ])
else:
    # Run play-by-play parsing for the upcoming opponent's previous games
    pbp_events_list = []
    for _, row in prev_games.iterrows():
        # Same Windows strftime portability fix as above -- "%-m"/"%-d" aren't supported by Windows' CRT.
        game_date_str = f"{row['game_date'].month}_{row['game_date'].day}_{row['game_date'].strftime('%y')}"
        pbp_pattern = f"{volume_dir}/{game_date_str}*{upcoming_opponent_short}*_pbp.*"
        pbp_files = glob.glob(pbp_pattern)
        raw_dfs = [parse_pbp_mhtml(path) for path in pbp_files]
        if not raw_dfs and row["game_date"] in live_pbp_by_game:
            raw_dfs = [live_pbp_by_game[row["game_date"]]]
        for raw_df in raw_dfs:
            self_column = resolve_self_column(raw_df, known_names)
            # `self_team`/`self_column` = the upcoming opponent's own events (whichever raw column actually has
            # their players); `opponent` = row["opponent"] -- whoever they actually played in THIS game, NOT
            # literally "UW-Whitewater", since Whitewater isn't in these games at all.
            events = build_pbp_events(
                raw_df, row["opponent"], row["game_date"], self_team=upcoming_opponent_short, self_column=self_column
            )
            pbp_events_list.append(events)

    # CONFIRMED BUG (fixed here): a bare pd.DataFrame() has ZERO columns, unlike the well-formed empty
    # fallback in the "opp_schedule.empty" branch above -- so if prev_games was non-empty but EVERY one of
    # those games failed to produce PBP data (no local _pbp file, and not in live_pbp_by_game either -- e.g.
    # an opponent whose games are so early in the season that this data simply isn't available yet), the
    # resulting pbp_events_upcoming had no "opponent" column at all, and the very next cell's
    # pbp_events_upcoming["opponent"] lookup crashed with KeyError: 'opponent' instead of just being empty.
    _PBP_EVENTS_UPCOMING_COLS = [
        "opponent", "game_date", "event_order", "period", "time_remaining",
        "time_remaining_seconds", "team", "event_type", "raw_text",
        "uww_score", "opp_score", "player", "shot_type", "shot_desc",
        "turnover_type", "foul_type", "ft_num", "ft_total",
    ]
    pbp_events_upcoming = (
        pd.concat(pbp_events_list, ignore_index=True) if pbp_events_list
        else pd.DataFrame(columns=_PBP_EVENTS_UPCOMING_COLS)
    )
    if not pbp_events_list:
        print(f"  No PBP data found/scraped for any of {upcoming_opponent_short}'s {len(prev_games)} game(s) before UWW -- pbp_events_upcoming is empty but well-formed.")
    print(pbp_events_upcoming.head(20))

         opponent   game_date  event_order period time_remaining  \
0   DePauw Tigers  2025-11-08            0   None           None   
1   DePauw Tigers  2025-11-08            1     H1     19:57 (H1)   
2   DePauw Tigers  2025-11-08            2     H1     19:57 (H1)   
3   DePauw Tigers  2025-11-08            3     H1     19:32 (H1)   
4   DePauw Tigers  2025-11-08            4     H1     19:32 (H1)   
5   DePauw Tigers  2025-11-08            5     H1     19:32 (H1)   
6   DePauw Tigers  2025-11-08            6     H1     19:24 (H1)   
7   DePauw Tigers  2025-11-08            7     H1     19:24 (H1)   
8   DePauw Tigers  2025-11-08            8     H1     19:24 (H1)   
9   DePauw Tigers  2025-11-08            9     H1     19:19 (H1)   
10  DePauw Tigers  2025-11-08           10     H1     19:19 (H1)   
11  DePauw Tigers  2025-11-08           11     H1     19:07 (H1)   
12  DePauw Tigers  2025-11-08           12     H1     19:07 (H1)   
13  DePauw Tigers  2025-11-08           13     H


### Shared video-tagging helper functions

Hoisted here (rather than defined inline where first used) so cell run-order doesn't matter -- `parse_video_mhtml`, `expand_clip_to_subevents`, `global_align`, `RESULT_TO_KEY`, and `FREE_THROW_EVENT_TYPES` are reused both by the opponent's-own-games attachment below and by the later cell that attaches video tags onto UWW's own `pbp_events`.

In [62]:
# --- Shared video-tagging helper functions, hoisted here so cell run-order doesn't matter -------------------
# Self-contained imports -- this cell is meant to work regardless of run order (see title above), so it
# shouldn't rely on another cell (e.g. the Configuration cell) having already run and left these in the
# global namespace. Confirmed by a live run: this cell raised "NameError: name 're' is not defined" during a
# fresh top-to-bottom run of the whole notebook.
import os
import re
from io import StringIO

import pandas as pd
from bs4 import BeautifulSoup

RESULT_TO_KEY = {
    "Make 2 Pts": "made_shot", "Make 3 Pts": "made_shot",
    "Miss 2 Pts": "missed_shot", "Miss 3 Pts": "missed_shot",
    "Turnover": "turnover",
    "Free Throw": "free_throw",
    "Foul": "foul", "Non Shooting Foul": "foul",
}
AND1_SHOT_RE = re.compile(r"Make (2|3) Pts Foul")
FREE_THROW_EVENT_TYPES = {"free_throw_made", "free_throw_missed"}

# Confirmed-by-video-review manual overrides -- keyed by (opponent_short, video clip "No.") -> the pbp_events
# "event_order" it actually corresponds to.
CONFIRMED_CLIP_EVENT_OVERRIDES = {
    ("Ripon", 120): 271,
    ("Ripon", 164): 371,
}


def expand_clip_to_subevents(row):
    """One video clip can represent more than one real pbp event (an \"And-1\" makes a shot + a free throw) --
    return a list of event_type sub-events (made_shot/missed_shot/turnover/free_throw) for this clip, in the
    order they happened."""
    if row["Result"] in RESULT_TO_KEY:
        return [RESULT_TO_KEY[row["Result"]]]
    if row["Result"] in ("1 Pts", "0 Pts"):
        events = ["made_shot"] if AND1_SHOT_RE.search(row["Description"]) else []
        events.append("free_throw")
        return events
    return []


def global_align(n, m, compatible, pos_cost):
    """pbp side has n items (indices 0..n-1, already sorted by event_order), video side has m items (indices
    0..m-1, already sorted by video_clip_number). Returns {pbp_index: video_index} for the order-preserving
    pairing that maximizes total match count (a huge fixed bonus per match dominates the DP), using summed
    `pos_cost` only as a tiebreaker among otherwise-equally-good maximal alignments."""
    BIG_BONUS = 1000.0
    score = [[0.0] * (m + 1) for _ in range(n + 1)]
    choice = [[0] * (m + 1) for _ in range(n + 1)]  # 0=matched (diagonal), 1=skip pbp item, 2=skip video item
    for i in range(1, n + 1):
        choice[i][0] = 1
    for j in range(1, m + 1):
        choice[0][j] = 2
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            best, best_choice = score[i - 1][j], 1
            if score[i][j - 1] > best:
                best, best_choice = score[i][j - 1], 2
            if compatible(i - 1, j - 1):
                diag = score[i - 1][j - 1] + BIG_BONUS - pos_cost(i - 1, j - 1)
                if diag >= best:  # ties prefer matching over skipping
                    best, best_choice = diag, 0
            score[i][j], choice[i][j] = best, best_choice

    pairs = {}
    i, j = n, m
    while i > 0 and j > 0:
        c = choice[i][j]
        if c == 0:
            pairs[i - 1] = j - 1
            i, j = i - 1, j - 1
        elif c == 1:
            i -= 1
        else:
            j -= 1
    return pairs


def parse_video_mhtml_html(html):
    """Core clip-table parsing logic, factored out of parse_video_mhtml() so it can run on HTML from either
    source: a manually-exported/uploaded MHTML snapshot, OR a live Playwright page.content() capture --
    confirmed by the user: a game's own "video_url" (already available per-game via team_schedules/
    opp_schedule -- see the schedule-scraping cell and the opponent-schedule cell above) renders this EXACT
    same clip-tagging table when opened, so a live scrape of that URL is a drop-in replacement for the
    manual "_video.mhtml" export/upload step. See scrape_video_clips_live() below for the live-scrape path.
    """
    soup = BeautifulSoup(html, "lxml")
    parsed_tables = [pd.read_html(StringIO(str(t)))[0] for t in soup.find_all("table")]
    # The file also contains a saved "Edit"/playlist metadata table with no "Description" column -- skip it.
    clips = next(t for t in parsed_tables if "Description" in t.columns)
    clips = clips.copy()
    clips["player"] = clips["Player"].astype(str).str.strip()
    return clips[["No.", "Result", "Description", "player", "Team", "Duration"]]


def parse_video_mhtml(path):
    return parse_video_mhtml_html(load_html_snapshot(path))


def login_to_synergy(page, username, password, timeout_ms=30000):
    """Log into Synergy Sports Tech's own identity provider (auth.synergysportstech.com) -- a separate login
    from FastScout/Hudl with its OWN credentials (see synergy_username/synergy_password in the
    schedule-scraping cell above; confirmed NOT the same as FastScout's). Selectors are best-effort guesses
    at a single-page username+password form; errors include the actual url/visible-error-text to help
    correct them if needed.
    """
    def _page_error_text():
        for sel in ["[class*='error' i]", "[class*='alert' i]", "[role='alert']"]:
            try:
                text = page.locator(sel).first.inner_text(timeout=1000)
                if text and text.strip():
                    return text.strip()
            except Exception:
                continue
        return None

    # Diagnostic: save the pristine login page HTML so its real form markup can be inspected directly if a
    # selector below turns out to be wrong, instead of guessing blindly from a bare timeout.
    try:
        _save_scraped_html(
            page.content(), os.path.join(schedules_dir, "Synergy - Login Page (diagnostic).html"),
            "Synergy login page (diagnostic)",
        )
    except Exception:
        pass

    try:
        username_input = page.locator(
            'input[type="email"], input[name="Username"], input[name="username"], input[id*="username" i], '
            'input[id*="email" i]'
        ).first
        username_input.wait_for(timeout=timeout_ms)
        # Log which field actually got matched (name/id/type only -- never the credential value itself) so a
        # wrong-field match is distinguishable from a genuine server-side credential rejection.
        print(f"    [synergy-login] username field matched: name={username_input.get_attribute('name')!r} id={username_input.get_attribute('id')!r} type={username_input.get_attribute('type')!r}")
        username_input.fill(username)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login failed finding the username field (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        password_input = page.locator('input[type="password"]').first
        password_input.wait_for(timeout=timeout_ms)
        print(f"    [synergy-login] password field matched: name={password_input.get_attribute('name')!r} id={password_input.get_attribute('id')!r}")
        password_input.fill(password)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login failed finding the password field (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        page.get_by_role("button", name=re.compile(r"^(log ?in|sign ?in|continue|submit)$", re.IGNORECASE)).first.click(timeout=timeout_ms)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login failed clicking the submit button (url={page.url}): {type(e).__name__}: {e}"
            + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e

    try:
        page.wait_for_url(lambda url: "/Account/Login" not in url, timeout=timeout_ms)
    except Exception as e:
        error_text = _page_error_text()
        raise RuntimeError(
            f"Synergy login did not leave the login page after submitting credentials (still at "
            f"url={page.url}): {type(e).__name__}: {e}" + (f" -- page showed: {error_text!r}" if error_text else "")
        ) from e


def scrape_video_clips_live(page, video_url, timeout_ms=30000, save_path=None):
    """Navigate to a game's own FastScout "video_url" (the Synergy-embedded clip-tagging player) and parse
    its rendered clip table -- a live-scrape replacement for a manually-exported "_video.mhtml" snapshot.
    Expects an already-authenticated FastScout Playwright `page`. If save_path is given, caches the scraped
    HTML there (see _save_scraped_html) so a future run can fall back to it via parse_video_mhtml instead of
    live-scraping again.

    Navigating here can redirect to Synergy's own login (auth.synergysportstech.com), a separate identity
    provider from FastScout/Hudl -- _attempt() detects that and logs in via login_to_synergy using Synergy's
    own synergy_username/synergy_password before retrying. Media-blocking (_attempt(block_media=True)/False)
    is a separate defensive measure against the video-editor SPA buffering full-game video in the background.
    """
    def _attempt(block_media):
        handler = None
        if block_media:
            def handler(route):
                req = route.request
                if req.resource_type == "media" or re.search(r"\.(mp4|m3u8|ts|webm|mov)(\?|$)", req.url, re.IGNORECASE):
                    route.abort()
                else:
                    route.continue_()
            page.route("**/*", handler)
        try:
            page.goto(video_url, wait_until="domcontentloaded", timeout=timeout_ms)
            try:
                page.wait_for_selector("table", timeout=timeout_ms)
            except Exception:
                if "auth.synergysportstech.com" not in page.url:
                    raise
                print(f"    [video] Redirected to Synergy's own login ({page.url}) -- logging in with Synergy's own credentials and retrying.")
                login_to_synergy(page, synergy_username, synergy_password, timeout_ms=timeout_ms)
                page.goto(video_url, wait_until="domcontentloaded", timeout=timeout_ms)
                page.wait_for_selector("table", timeout=timeout_ms)
            return page.content()
        finally:
            if handler is not None:
                page.unroute("**/*", handler)

    try:
        html = _attempt(block_media=True)
    except Exception as block_error:
        # Only reached if the media-blocked attempt itself failed -- retry once, fully unblocked, rather than
        # let a media-blocking regression silently break every video-clip scrape. If the underlying browser/
        # driver is actually dead (the connection-closed case this function was written to avoid), this
        # unblocked retry will fail the same way and its exception still propagates up to
        # run_in_fastscout_session's own dead-session retry logic, same as before this fallback existed.
        print(f"    [video] Media-blocked scrape failed ({type(block_error).__name__}: {block_error}) -- retrying without blocking media.")
        html = _attempt(block_media=False)

    if save_path:
        _save_scraped_html(html, save_path, "scraped video-tagging clips")
    return parse_video_mhtml_html(html)


### Attach video-tagging clip descriptions onto the opponent's prior-game events

Reuses the helpers from the cell above to align each `_video` export's clip descriptions onto `pbp_events_upcoming`. Only the per-game team resolution differs from the later UWW version: these are the opponent's own games against a varying actual opponent each time, not always UW-Whitewater vs. them.

In [64]:
# --- Attach video-tagging clip descriptions onto pbp_events_upcoming, same method as pbp_events uses --------
# Reuses parse_video_mhtml / expand_clip_to_subevents / global_align / RESULT_TO_KEY / FREE_THROW_EVENT_TYPES
# from the cell above. Only the per-game team resolution differs here: these are the opponent's OWN games
# against a VARYING actual opponent each time, not always UW-Whitewater vs a fixed opponent, and there's no
# lineup reconstruction for these games.
video_desc_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
video_result_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
video_player_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
video_clip_number_col = pd.Series(index=pbp_events_upcoming.index, dtype=float)

# Same Windows strftime portability fix used in the opponent-schedule cell above -- "%-m"/"%-d" (no-leading-
# zero month/day) aren't supported by Windows' CRT strftime.
def _game_date_str(game_date):
    return f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')}"


# Find which prior games are missing a local "_video.*" file AND have a usable "video_url" to live-scrape
# instead (only present when opp_schedule was sourced from team_schedules -- see the opponent-schedule cell).
games_needing_live_video = []
for _, row in prev_games.iterrows():
    video_matches = glob.glob(f"{volume_dir}/{_game_date_str(row['game_date'])}*{upcoming_opponent_short}*_video.*")
    if not video_matches and "video_url" in row.index and pd.notna(row.get("video_url")):
        games_needing_live_video.append(row)

# Confirmed by the user: a game's own video-clip tagging table renders directly at its "video_url" (see
# scrape_video_clips_live() in the video-tagging helpers cell above) -- live-scrape it for any prior game
# missing a local "_video.mhtml" file, the same live-scrape-with-fallback pattern Cell 4 already uses for
# schedules and scout reports. One shared session covers every game that needs this.
live_clips_by_game = {}
if games_needing_live_video and fastscout_username and fastscout_password:
    # One run_in_fastscout_session call PER GAME (not one call looping over all games) -- run_in_fastscout_
    # session's own retry-on-dead-session logic (Cell 4) can only kick in on a call it directly wraps, so if
    # every game were scraped inside a single call, a mid-batch dead browser/driver would silently fail every
    # remaining game with no chance to self-heal. Reuses the ONE shared FastScout Playwright session (opened
    # lazily on first use) instead of opening its own separate browser+thread per game.
    for g_row in games_needing_live_video:
        try:
            # Same location-based "<Away> @ <Home>" matchup naming as scout-report downloads (Cell 4) and
            # the pbp cache above, so the saved file matches the manually-uploaded naming convention closely
            # enough for the glob pattern above to find it on a future run.
            g_date_str = _game_date_str(g_row["game_date"])
            if str(g_row.get("location", "")).strip().lower() == "home":
                matchup = f"{g_row['opponent']} @ {upcoming_opponent_short}"
            else:
                matchup = f"{upcoming_opponent_short} @ {g_row['opponent']}"
            video_save_path = f"{volume_dir}/{g_date_str} {matchup}_video.html"
            live_clips_by_game[g_row["game_date"]] = run_in_fastscout_session(
                lambda page, url=g_row["video_url"], sp=video_save_path: scrape_video_clips_live(page, url, save_path=sp)
            )
        except Exception as video_scrape_error:
            print(f"  Could not live-scrape video clips for {g_row['opponent']} ({g_row['video_url']}): {type(video_scrape_error).__name__}: {video_scrape_error}")

games_with_video = []
for _, row in prev_games.iterrows():
    video_matches = glob.glob(f"{volume_dir}/{_game_date_str(row['game_date'])}*{upcoming_opponent_short}*_video.*")
    actual_opponent = row["opponent"]
    game_mask = (pbp_events_upcoming["opponent"] == actual_opponent) & (pbp_events_upcoming["game_date"] == row["game_date"])
    if video_matches:
        clips = parse_video_mhtml(video_matches[0])
    elif row["game_date"] in live_clips_by_game:
        clips = live_clips_by_game[row["game_date"]]
    else:
        continue
    total_clips_n = len(clips)

    opp_all_events = pbp_events_upcoming[game_mask]
    player_team_lookup = opp_all_events.dropna(subset=["player"]).drop_duplicates("player").set_index("player")["team"]
    known_teams = set(player_team_lookup.unique())
    known_players = set(player_team_lookup.index)

    def normalize_player_name(name):
        if name in known_players:
            return name
        for p in known_players:
            if p.casefold() == str(name).casefold():
                return p
        return name

    def committing_team_for(fouled_player):
        fouled_team = player_team_lookup.get(fouled_player)
        others = known_teams - {fouled_team}
        return next(iter(others)) if len(others) == 1 else None

    subevent_rows = []
    for _, clip_row in clips.iterrows():
        clip_player = normalize_player_name(clip_row["player"])
        for event_type in expand_clip_to_subevents(clip_row):
            match_key = committing_team_for(clip_player) if event_type == "foul" else clip_player
            subevent_rows.append({
                "match_key": match_key, "event_type": event_type,
                "Description": clip_row["Description"], "video_result": clip_row["Result"],
                "video_clip_player": clip_player, "video_clip_number": clip_row["No."],
            })
    matchable_clips = pd.DataFrame(
        subevent_rows, columns=["match_key", "event_type", "Description", "video_result", "video_clip_player", "video_clip_number"]
    ).sort_values("video_clip_number").reset_index(drop=True)

    target_event_types = {"made_shot", "missed_shot", "turnover", "foul"} | FREE_THROW_EVENT_TYPES
    opp_events = pbp_events_upcoming[game_mask & pbp_events_upcoming["event_type"].isin(target_event_types)].sort_values("event_order").copy()
    opp_events["match_event_type"] = opp_events["event_type"].where(~opp_events["event_type"].isin(FREE_THROW_EVENT_TYPES), "free_throw")
    opp_events["match_key"] = opp_events["team"].where(opp_events["event_type"] == "foul", opp_events["player"])
    total_pbp_n = pbp_events_upcoming.loc[game_mask, "event_order"].max()

    pbp_orig_index = opp_events.index.tolist()
    pbp_list = [
        {"event_order": r["event_order"], "match_event_type": r["match_event_type"], "match_key": r["match_key"]}
        for _, r in opp_events.iterrows()
    ]
    video_list = matchable_clips.to_dict("records")
    n, m = len(pbp_list), len(video_list)

    def compatible(i, j):
        p, v = pbp_list[i], video_list[j]
        return p["match_event_type"] == v["event_type"] and p["match_key"] == v["match_key"]

    def pos_cost(i, j):
        return abs(pbp_list[i]["event_order"] / total_pbp_n - video_list[j]["video_clip_number"] / total_clips_n)

    pairs = global_align(n, m, compatible, pos_cost)
    for i, j in pairs.items():
        orig_idx = pbp_orig_index[i]
        v = video_list[j]
        video_desc_col.loc[orig_idx] = v["Description"]
        video_result_col.loc[orig_idx] = v["video_result"]
        video_player_col.loc[orig_idx] = v["video_clip_player"]
        video_clip_number_col.loc[orig_idx] = v["video_clip_number"]

    games_with_video.append(actual_opponent)
    print(f"  {actual_opponent}: matched {len(pairs)}/{len(pbp_list)} eligible pbp events to {len(video_list)} video sub-events ({total_clips_n} clips)")

pbp_events_upcoming["video_description"] = video_desc_col
pbp_events_upcoming["video_result"] = video_result_col
pbp_events_upcoming["video_player"] = video_player_col
pbp_events_upcoming["video_clip_number"] = video_clip_number_col

print(f"\nGames with a video-tagging file: {games_with_video}")
print(f"pbp_events_upcoming rows with a matched video description: {pbp_events_upcoming['video_description'].notna().sum()} of {len(pbp_events_upcoming)}")
if not pbp_events_upcoming.empty:
    print(pbp_events_upcoming[pbp_events_upcoming["video_description"].notna()].head(20))
else:
    print("WARNING: No opponent prior-game events to display (opponent schedule file not loaded).")

  DePauw Tigers: matched 221/235 eligible pbp events to 227 video sub-events (240 clips)
  Kalamazoo Hornets: matched 242/266 eligible pbp events to 247 video sub-events (262 clips)
  Blackburn Beavers: matched 206/232 eligible pbp events to 218 video sub-events (229 clips)
  North Central (IL) Cardinals: matched 198/214 eligible pbp events to 206 video sub-events (210 clips)
  Wisconsin-Superior Yellowjackets: matched 203/228 eligible pbp events to 212 video sub-events (229 clips)
  Dubuque Spartans: matched 213/237 eligible pbp events to 222 video sub-events (233 clips)
  Simpson Storm: matched 201/220 eligible pbp events to 209 video sub-events (219 clips)
  Wartburg Knights: matched 203/245 eligible pbp events to 231 video sub-events (242 clips)
  Westminster (MO) Blue Jays: matched 203/217 eligible pbp events to 209 video sub-events (220 clips)
  Monmouth (IL) Fighting Scots: matched 191/207 eligible pbp events to 199 video sub-events (207 clips)
  Lake Forest Foresters: matched 2


### Scout the upcoming opponent's own tendencies from their prior games

Splits `pbp_events_upcoming` into the opponent's own events vs. their opponents' events, then summarizes shot-type volume/efficiency by play type and contest level -- the opponent's own tendencies heading into the Whitewater game.

In [66]:
# --- Scout the upcoming opponent's own tendencies from their games before facing Whitewater -------------------
_safe_display = lambda df: print(df) if not df.empty else print("  (no data)")

elmhurst_events = pbp_events_upcoming[pbp_events_upcoming["team"] == upcoming_opponent_short].copy()
opponent_events = pbp_events_upcoming[
    pbp_events_upcoming["team"].notna() & (pbp_events_upcoming["team"] != upcoming_opponent_short)
].copy()

print(f"{upcoming_opponent_short}'s own events across their {prev_games.shape[0]} games before Whitewater: {len(elmhurst_events)}")
print(f"Their opponents' events across those same games: {len(opponent_events)}\n")

def normalize_elmhurst_player(name):
    if name in known_names:
        return name
    for p in known_names:
        if p.casefold() == str(name).casefold():
            return p
    return name

def extract_play_segment(description, player):
    if pd.isna(description) or pd.isna(player):
        return None
    segments = [s.strip() for s in description.split(" > ")]
    player_norm = normalize_elmhurst_player(player)
    last_player_idx = None
    for idx, seg in enumerate(segments):
        m = re.match(r"^\d+\s+(.+)$", seg)
        if m and normalize_elmhurst_player(m.group(1)) == player_norm:
            last_player_idx = idx
    if last_player_idx is not None and last_player_idx + 1 < len(segments):
        return segments[last_player_idx + 1]
    return segments[1] if len(segments) > 1 else None

def extract_guarded(description):
    if pd.isna(description):
        return None
    if "Guarded" in description:
        return "Yes"
    if "Open" in description:
        return "No"
    return "N/A"

shots = elmhurst_events[elmhurst_events["event_type"].isin(["made_shot", "missed_shot"])].copy()
shots["made"] = shots["event_type"] == "made_shot"
shots["play_type"] = shots.apply(lambda r: extract_play_segment(r["video_description"], r["player"]), axis=1)
shots["guarded"] = shots["video_description"].apply(extract_guarded)
shot_profile = (
    shots.groupby(["shot_type", "shot_desc", "play_type", "guarded"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
shot_profile["fg_pct"] = (shot_profile["makes"] / shot_profile["attempts"] * 100).round(1)
shot_profile = shot_profile.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"{upcoming_opponent_short}'s shot-type tendencies (volume + efficiency, tagged by play_type/guarded) across their last {prev_games.shape[0]} games:")
_safe_display(shot_profile)

play_type_tendency = shots.groupby("play_type").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
play_type_tendency["fg_pct"] = (play_type_tendency["makes"] / play_type_tendency["attempts"] * 100).round(1)
play_type_tendency = play_type_tendency.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"\n{upcoming_opponent_short}'s shot attempts by play type:")
_safe_display(play_type_tendency)

guarded_tendency = shots.groupby("guarded").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
guarded_tendency["fg_pct"] = (guarded_tendency["makes"] / guarded_tendency["attempts"] * 100).round(1)
guarded_tendency = guarded_tendency.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"\n{upcoming_opponent_short}'s shot attempts by contest level:")
_safe_display(guarded_tendency)

turnovers = elmhurst_events[elmhurst_events["event_type"] == "turnover"].copy()
turnovers["play_type"] = turnovers.apply(lambda r: extract_play_segment(r["video_description"], r["player"]), axis=1)
turnover_profile = (
    turnovers.groupby(["turnover_type", "play_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(f"\n{upcoming_opponent_short}'s turnover types ({len(turnovers)} total, tagged by play_type):")
_safe_display(turnover_profile)

fouls = elmhurst_events[elmhurst_events["event_type"] == "foul"]
foul_profile = (
    fouls.groupby(["foul_type", "video_description"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(f"\n{upcoming_opponent_short}'s foul types ({len(fouls)} total, with video_description):")
_safe_display(foul_profile)

made_shots = elmhurst_events[elmhurst_events["event_type"] == "made_shot"].copy()
made_shots["points"] = made_shots["shot_type"].astype(float)
made_fts = elmhurst_events[elmhurst_events["event_type"] == "free_throw_made"].copy()
made_fts["points"] = 1.0
scoring = pd.concat([made_shots[["player", "points"]], made_fts[["player", "points"]]], ignore_index=True)
top_scorers = scoring.groupby("player")["points"].sum().sort_values(ascending=False).reset_index()
top_scorers.columns = ["player", f"total_points_across_{prev_games.shape[0]}_games"]
print(f"\n{upcoming_opponent_short}'s top scorers across their last {prev_games.shape[0]} games (from play-by-play):")
_safe_display(top_scorers.head(10))

Loras Duhawks's own events across their 28 games before Whitewater: 7744
Their opponents' events across those same games: 6821

Loras Duhawks's shot-type tendencies (volume + efficiency, tagged by play_type/guarded) across their last 28 games:
    shot_type          shot_desc          play_type guarded  attempts  makes  \
0           3          Jump Shot            Spot-Up     Yes       212     79   
1           3          Jump Shot            Spot-Up      No       133     45   
2           2              Layup                Cut     N/A       104     82   
3           2              Layup            Post-Up     N/A        74     42   
4           3          Jump Shot         Off Screen     Yes        71     19   
..        ...                ...                ...     ...       ...    ...   
105         2          (Blocked)                ISO     N/A         1      0   
106         2    Layup (Blocked)       No Play Type     N/A         1      0   
107         2    Layup (Blocked)  Of


### Compare PBP-derived opponent tendencies to UWW's own scouting keys

Cross-checks the tendencies just derived from actual play-by-play against UWW's own scouting-report notes (`TEAM STRENGTHS` / `KEYS TO VICTORY`) for this opponent, to see whether the scouted keys hold up against what they've actually done on the floor.

In [68]:
# --- Compare our PBP-derived findings to UWW's own scouting keys for the upcoming opponent ---------------------
_safe_display = lambda df: print(df) if not df.empty else print("  (no data)")

elmhurst_plan = all_game_plans[
    (all_game_plans["opponent"] == upcoming_opponent_short) & (all_game_plans["topic"].isin(["TEAM STRENGTHS", "KEYS TO VICTORY"]))
]
print(f"UWW's own scouting notes for {upcoming_opponent_short} [SOURCE: SCOUTING REPORT]:")
for _, r in elmhurst_plan.iterrows():
    print(f"  {r['topic']}: {r['notes']}")

# Rebounding split, for the "DOMINATE THE PAINT" key -- covers both individual and team-level rebound events.
rebounds = elmhurst_events[elmhurst_events["event_type"].str.contains("rebound", na=False)]
reb_split = rebounds["event_type"].apply(lambda x: "offensive" if "offensive" in x else "defensive").value_counts().reset_index()
reb_split.columns = ["rebound_type", "count"]

# Blocks recorded AGAINST them by their opponents -- another paint-imposition signal.
blocks_against = opponent_events[opponent_events["event_type"] == "block"]

paint_attempts = shot_profile.loc[shot_profile["shot_desc"].str.contains("Layup", na=False), "attempts"].sum()
three_pt = shot_profile[shot_profile["shot_type"] == "3"]
three_pt_attempts, three_pt_makes = three_pt["attempts"].sum(), three_pt["makes"].sum()
measured_3p_pct = round(three_pt_makes / three_pt_attempts * 100, 1) if three_pt_attempts else None

print("\n--- Comparing PBP evidence to the stated Keys to Victory [SOURCE: SCOUTING REPORT] ---\n")

print("KEY 1 [SCOUTING REPORT] -- COMMUNICATE SCREENS & ACTIONS:")
print("  Not directly measurable from play-by-play event types (no screen-action tagging) -- this is a")
print("  communication/technique key tied to their offensive scheme (per the scout's own Offensive Scheme notes),")
print("  not something the event log alone can confirm or refute.\n")

print("KEY 2 [SCOUTING REPORT] -- DOMINATE THE PAINT:")
print(f"  Their own rebounding split across their {prev_games.shape[0]} games: {reb_split.to_dict('records')}")
print(f"  Their layup-area attempts (Layup + Driving Layup): {paint_attempts} of {shot_profile['attempts'].sum()} total FGA")
print(f"  Blocks recorded against them by opponents: {len(blocks_against)}")
print("  Compare against their own Defensive Scheme notes and layup-area conversion rate to judge whether this key")
print("  is supported by the evidence.\n")

print("KEY 3 [SCOUTING REPORT] -- GUARD 1 ON 1:")
print(f"  Measured 3PT jump-shot rate: {three_pt_attempts} attempts at {measured_3p_pct}%.")
print(f"  Scoring balance (top 5 scorers): {top_scorers.head(5).to_dict('records')}")
print("  Compare against the scout's own TEAM STRENGTHS note on shooting/scoring balance to judge whether")
print("  over-helping creates open catch-and-shoot looks that straight man coverage would limit.")

# --- New Keys to Victory, derived directly from PBP data/tendencies rather than the written scouting report --
print("\n\n--- New Keys to Victory, derived from PBP data/tendencies [SOURCE: PBP-DERIVED] ---\n")

overall_fg_pct = round(shots["made"].mean() * 100, 1)
high_volume_types = play_type_tendency[play_type_tendency["attempts"] >= 15]

most_efficient = high_volume_types.sort_values("fg_pct", ascending=False).head(2)
print("DATA-KEY 1 [PBP-DERIVED] -- TAKE AWAY THEIR MOST EFFICIENT HIGH-VOLUME ACTIONS:")
for _, r in most_efficient.iterrows():
    print(f"  {r['play_type']}: {r['fg_pct']}% on {int(r['attempts'])} attempts")
print(f"  Both well above their {overall_fg_pct}% overall shooting clip (>=15 attempts each) -- load up P&R roll")
print("  coverage and transition defense specifically; live with everything else.\n")

least_efficient = high_volume_types.sort_values("fg_pct").head(2)
print("DATA-KEY 2 [PBP-DERIVED] -- FUNNEL THEM INTO THEIR WORST HIGH-VOLUME LOOKS:")
for _, r in least_efficient.iterrows():
    print(f"  {r['play_type']}: {r['fg_pct']}% on {int(r['attempts'])} attempts")
print(f"  Both well below their {overall_fg_pct}% overall shooting clip -- don't help off these actions; make them")
print("  keep taking them.\n")

turnovers_by_play_type = turnovers.groupby("play_type").size().sort_values(ascending=False)
top_turnover_triggers = turnovers_by_play_type.head(2)
print("DATA-KEY 3 [PBP-DERIVED] -- PRESSURE THEIR BIGGEST TURNOVER TRIGGERS:")
for pt, cnt in top_turnover_triggers.items():
    pct = round(100 * cnt / len(turnovers), 1)
    print(f"  {pt}: {cnt} of {len(turnovers)} turnovers ({pct}%)")
print("  Concentrate ball pressure on whichever action produces the most turnovers, rather than blanket")
print("  full-court pressure.")

# --- Structure the PBP-derived keys into a table-ready DataFrame for later CSV export -- one row per key,
# tagged with an explicit `source` column so downstream consumers can distinguish these from scouting-report
# keys. Keyed by `opponent` so future opponents' PBP-derived keys can append to the same table.
pbp_derived_keys = pd.DataFrame([
    {
        "opponent": upcoming_opponent_short,
        "key_number": 1,
        "title": "Take away their most efficient high-volume actions",
        "source": "PBP-DERIVED",
        "supporting_stats": "; ".join(
            f"{r['play_type']}: {r['fg_pct']}% on {int(r['attempts'])} attempts" for _, r in most_efficient.iterrows()
        ),
        "recommendation": (
            f"Both well above their {overall_fg_pct}% overall shooting clip (>=15 attempts each) -- load up P&R "
            "roll coverage and transition defense specifically; live with everything else."
        ),
    },
    {
        "opponent": upcoming_opponent_short,
        "key_number": 2,
        "title": "Funnel them into their worst high-volume looks",
        "source": "PBP-DERIVED",
        "supporting_stats": "; ".join(
            f"{r['play_type']}: {r['fg_pct']}% on {int(r['attempts'])} attempts" for _, r in least_efficient.iterrows()
        ),
        "recommendation": (
            f"Both well below their {overall_fg_pct}% overall shooting clip -- don't help off these actions; make "
            "them keep taking them."
        ),
    },
    {
        "opponent": upcoming_opponent_short,
        "key_number": 3,
        "title": "Pressure their biggest turnover triggers",
        "source": "PBP-DERIVED",
        "supporting_stats": "; ".join(
            f"{pt}: {cnt} of {len(turnovers)} turnovers ({round(100 * cnt / len(turnovers), 1)}%)"
            for pt, cnt in top_turnover_triggers.items()
        ),
        "recommendation": (
            "Concentrate ball pressure on whichever action produces the most turnovers, rather than blanket "
            "full-court pressure."
        ),
    },
])
print("\nStructured PBP-derived keys, ready for CSV export:")
_safe_display(pbp_derived_keys)

UWW's own scouting notes for Loras Duhawks [SOURCE: SCOUTING REPORT]:
  KEYS TO VICTORY: 1. BALL PRESSURE TO DISRUPT THEIR ACTION(S) | 2. INTENTIONAL ABOUT ATTACKING MISMATCHES | 3. DOMINATE THE GLASS!!!
  TEAM STRENGTHS: 1. Multiple scorers capable of big games | 2. Numerous set plays & counters used to create advantages | 3. Rebounding (12 O-Boards per game)

--- Comparing PBP evidence to the stated Keys to Victory [SOURCE: SCOUTING REPORT] ---

KEY 1 [SCOUTING REPORT] -- COMMUNICATE SCREENS & ACTIONS:
  Not directly measurable from play-by-play event types (no screen-action tagging) -- this is a
  communication/technique key tied to their offensive scheme (per the scout's own Offensive Scheme notes),
  not something the event log alone can confirm or refute.

KEY 2 [SCOUTING REPORT] -- DOMINATE THE PAINT:
  Their own rebounding split across their 28 games: [{'rebound_type': 'defensive', 'count': 694}, {'rebound_type': 'offensive', 'count': 438}]
  Their layup-area attempts (Layup + 


### Diagnose the opponent's weakest video-tagged play type

Same method as UWW's own play-type diagnosis (later in the notebook) -- surfaces which play type the opponent has been least efficient at across their games before facing Whitewater, using video-tagged clip data.

In [70]:
# --- Diagnose the upcoming opponent's weakest video-tagged play type, same method as UWW's own diagnosis -------
_safe_display = lambda df: print(df) if not df.empty else print("  (no data)")
# Play type is whatever chained segment comes right after the LAST "<jersey#> <player name>" token in
# video_description that matches the shooter -- an earlier segment may belong to a DIFFERENT player (e.g. the
# screener/passer who set up the shot), so anchoring on the shooter's own name-token avoids misattributing
# someone else's action. Self-contained name normalization (via `known_names`, their own roster from the
# schedule-loading cell above) rather than reusing the video-attach cell's `normalize_player_name` closure,
# since that one is left pointing at whichever game it last iterated over.
def normalize_elmhurst_name(name, names):
    if name in names:
        return name
    for p in names:
        if p.casefold() == str(name).casefold():
            return p
    return name

def extract_play_type(description, player, names):
    if pd.isna(description) or pd.isna(player):
        return None
    segments = [s.strip() for s in description.split(" > ")]
    player_norm = normalize_elmhurst_name(player, names)
    last_player_idx = None
    for idx, seg in enumerate(segments):
        m = re.match(r"^\d+\s+(.+)$", seg)
        if m and normalize_elmhurst_name(m.group(1), names) == player_norm:
            last_player_idx = idx
    if last_player_idx is not None and last_player_idx + 1 < len(segments):
        return segments[last_player_idx + 1]
    return segments[1] if len(segments) > 1 else None

elmhurst_shot_rows = elmhurst_events[
    elmhurst_events["event_type"].isin(["made_shot", "missed_shot"]) & elmhurst_events["video_description"].notna()
].copy()
elmhurst_shot_rows["play_type"] = elmhurst_shot_rows.apply(
    lambda r: extract_play_type(r["video_description"], r["player"], known_names), axis=1, result_type='reduce'
)
elmhurst_shot_rows["made"] = elmhurst_shot_rows["event_type"] == "made_shot"

play_type_summary = elmhurst_shot_rows.groupby("play_type").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
play_type_summary["fg_pct"] = (100 * play_type_summary["makes"] / play_type_summary["attempts"]).round(1)
play_type_summary = play_type_summary.sort_values("attempts", ascending=False).reset_index(drop=True)
print(f"{upcoming_opponent_short}'s video-tagged play types, across their {prev_games.shape[0]} games before Whitewater "
      f"({len(elmhurst_shot_rows)} video-matched attempts):\n")
_safe_display(play_type_summary)

HIGH_VOLUME_MIN_ATTEMPTS = 10
high_volume = play_type_summary[play_type_summary["attempts"] >= HIGH_VOLUME_MIN_ATTEMPTS]
if not high_volume.empty:
    weakest = high_volume.sort_values("fg_pct").iloc[0]
else:
    weakest = pd.Series({"play_type": "(none)", "fg_pct": 0, "makes": 0, "attempts": 0})
print(f"Weakest high-volume play type (>= {HIGH_VOLUME_MIN_ATTEMPTS} attempts): '{weakest['play_type']}' at "
      f"{weakest['fg_pct']}% ({int(weakest['makes'])}/{int(weakest['attempts'])})\n")

# The tagger's own vocabulary doesn't name every shot; these two labels mark where it runs out. Kept as
# named constants so the "best shot type" logic can recognise a residual bucket instead of presenting it as
# a real shot type.
UNCLASSIFIED_SHOT_MECHANIC = "Unclassified (no mechanic tag)"
NO_CONTEST_TAG = "Not tagged (contest recorded only on catch-and-shoot)"


def extract_shot_mechanic(description):
    """Which kind of shot this was, from the video tagger's own chained description.

    The first three tests are the tagger's SHOT MECHANIC vocabulary. Everything else used to fall through to
    a bucket called "Other" -- which was 19% of all tagged shots and, at 58.9%, the most efficient bucket on
    the board, so it kept winning "best shot type" while telling a coach nothing. Reading the raw tags, it was
    cuts to the rim, putbacks off the offensive glass, and post-ups.

    Those three are tested AFTER the mechanic tests, not before: they describe how a shot was CREATED rather
    than how it was released, and a post-up that finishes as a jumper should still count as a jumper. Checked
    against real tagged data -- "Cut" and "Offensive Rebound" appear in zero already-classified shots, and
    "Post-Up" in 169, all of which keep their existing (more specific) label under this ordering. Adding the
    tier shrinks the residual from 586 shots to 14.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "No Dribble Jumper" in d:
        return "Catch-and-shoot"
    if "Dribble Jumper" in d:
        return "Pull-up off the dribble"
    if "To Basket" in d:
        return "Drive to the basket"
    # --- fallback tier: shot ORIGIN, for tags carrying no mechanic keyword at all ---
    if "Offensive Rebound" in d:
        return "Putback off the offensive glass"
    if "Cut" in d:
        return "Cut to the basket"
    if "Post-Up" in d:
        return "Post-up"
    return UNCLASSIFIED_SHOT_MECHANIC


def extract_contest(description):
    """Defender contest, which the tagger records ONLY on catch-and-shoot jumpers.

    Verified across 3,039 tagged shots: every one of the 1,069 catch-and-shoot attempts carries Guarded or
    Open, and not one of the other 1,970 does. So a missing contest tag does not mean the shot was a drive --
    the previous label said "(drive, no contest tag)", which mislabelled every cut, putback and post-up as a
    drive. It means the contest dimension simply does not apply to this shot type.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "Guarded" in d:
        return "Guarded"
    if "Open" in d:
        return "Open"
    return NO_CONTEST_TAG

def extract_distance(description):
    if pd.isna(description):
        return None
    for tag in ["Long/3pt", "Medium/17' to <3p", "Short to < 17'"]:
        if tag in description:
            return tag
    return "N/A"

weak_type_rows = elmhurst_shot_rows[elmhurst_shot_rows["play_type"] == weakest["play_type"]].copy()
weak_type_rows["shot_mechanic"] = weak_type_rows["video_description"].apply(extract_shot_mechanic)
weak_type_rows["contest"] = weak_type_rows["video_description"].apply(extract_contest)
weak_type_rows["distance"] = weak_type_rows["video_description"].apply(extract_distance)

print(f"'{weakest['play_type']}' shot-quality breakdown ({len(weak_type_rows)} attempts):\n")

mechanic_summary = weak_type_rows.groupby("shot_mechanic").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
mechanic_summary["fg_pct"] = (100 * mechanic_summary["makes"] / mechanic_summary["attempts"]).round(1)
print("By shot mechanic:")
_safe_display(mechanic_summary.sort_values("attempts", ascending=False))

contest_summary = weak_type_rows.groupby("contest").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
contest_summary["fg_pct"] = (100 * contest_summary["makes"] / contest_summary["attempts"]).round(1)
print("\nBy contest level:")
_safe_display(contest_summary.sort_values("attempts", ascending=False))

distance_summary = weak_type_rows.groupby("distance").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
distance_summary["fg_pct"] = (100 * distance_summary["makes"] / distance_summary["attempts"]).round(1)
print("\nBy shot distance:")
_safe_display(distance_summary.sort_values("attempts", ascending=False))

player_weak_type = weak_type_rows.groupby("player").agg(
    attempts=("made", "count"), makes=("made", "sum"),
    pct_catch_and_shoot=("shot_mechanic", lambda s: round(100 * (s == "Catch-and-shoot").mean(), 1)),
    pct_guarded=("contest", lambda s: round(100 * (s == "Guarded").mean(), 1)),
).reset_index()
player_weak_type["fg_pct"] = (100 * player_weak_type["makes"] / player_weak_type["attempts"]).round(1)
player_weak_type = player_weak_type.sort_values("attempts", ascending=False)
print(f"\nPer-player '{weakest['play_type']}' volume/efficiency, with catch-and-shoot% and guarded% of those attempts:")
_safe_display(player_weak_type[["player", "attempts", "makes", "fg_pct", "pct_catch_and_shoot", "pct_guarded"]])

Loras Duhawks's video-tagged play types, across their 28 games before Whitewater (1621 video-matched attempts):

            play_type  attempts  makes  fg_pct
0             Spot-Up       480    176    36.7
1          Transition       242    113    46.7
2    P&R Ball Handler       182     75    41.2
3                 Cut       179    114    63.7
4          Off Screen       139     46    33.1
5             Post-Up       130     67    51.5
6        P&R Roll Man        72     36    50.0
7            Hand Off        67     25    37.3
8                 ISO        60     19    31.7
9   Offensive Rebound        57     50    87.7
10       No Play Type        13      7    53.8
Weakest high-volume play type (>= 10 attempts): 'ISO' at 31.7% (19/60)

'ISO' shot-quality breakdown (60 attempts):

By shot mechanic:
             shot_mechanic  attempts  makes  fg_pct
1      Drive to the basket        30     11    36.7
2  Pull-up off the dribble        27      6    22.2
0          Catch-and-shoot      


### Load player comparison algorithms and compute tag rarity weights

Portable replacement for `%run UW Whitewater Player Comparison Algorithms` -- imports `build_player_comparison_artifacts` from `player_comparison.py` (reloaded each run so edits to that file are picked up), and identifies the target opponent (the most recently scouted game) to compare its players against every other scouted opponent's roster.

In [72]:
# Portable replacement for "%run UW Whitewater Player Comparison Algorithms" -- imports the same helper logic
# from player_comparison.py, sitting alongside this notebook. Force a reload so re-running this cell always
# picks up the latest edits to player_comparison.py, even though the module is already cached in sys.modules
# from an earlier run in this same kernel session.
import importlib
import player_comparison
importlib.reload(player_comparison)
from player_comparison import build_player_comparison_artifacts, PLAYER_COMPARISON_ALGORITHMS_VERSION

# USE_LLM is set in the Configuration cell at the top -- defaults to False so player comparisons run on
# tag-based similarity only, without any LLM calls or cost. Set USE_LLM=True to also run the LLM-based
# comparison (cells below).
use_llm = USE_LLM

comparison_artifacts = build_player_comparison_artifacts(
    schedule=schedule,
    scout_reports=scout_reports,
    player_profiles=player_profiles,
    cache_path=os.path.join(OUTPUT_DIR, "_cache", "llm_player_comparison_cache.jsonl"),
    use_llm=use_llm,
    # Build the comparisons FOR the upcoming opponent (defined in the "Identify the upcoming opponent" cell
    # above). Without this the module defaults to "whichever scouted opponent sits latest on the schedule",
    # which silently produced a uww_player_comparisons full of rows for a DIFFERENT team -- the app's Player
    # Details dialog then matched none of the upcoming opponent's roster players and showed "No comparable
    # player found" against a CSV that was not empty at all.
    upcoming_opponent=upcoming_opponent_short,
)
globals().update(comparison_artifacts)

# Fail loudly rather than quietly shipping comparisons for the wrong team -- this exact mismatch reached the
# app once already and looked like an app bug from there.
if target_opponent != upcoming_opponent_short:
    print(f"WARNING: comparison target is {target_opponent!r}, not the upcoming opponent "
          f"{upcoming_opponent_short!r} -- uww_player_comparisons will not match the app's roster view.")

print(f"Loaded player comparison algorithms (version {PLAYER_COMPARISON_ALGORITHMS_VERSION}). use_llm={use_llm}\n")
print("Scouted opponents and their game number on the schedule:")
for opp, num in sorted(scout_game_numbers.items(), key=lambda x: (x[1] is None, x[1])):
    print(f"  Game #{num}: {opp}")

if target_opponent is None:
    print("\nWARNING: no scouted opponent has a resolvable game number on the schedule, so there's no valid target for a similarity comparison yet.")
else:
    print(f"\nTarget (most recently scouted game): {target_opponent} (game #{target_game_number})")
    print("Previous scouted opponents to compare against:", previous_opponents)

print("\nplayer_notes-derived tag frequency and rarity-based importance weight (rarer tags count more toward similarity):")
print(notes_tag_importance_df)

print("\nkeys_to_defending-derived tag frequency and rarity-based importance weight:")
print(keys_tag_importance_df)

LLM comparison skipped (use_llm=False) -- using tag-based similarity only.
Loaded player comparison algorithms (version 2026-07-29-portable). use_llm=False

Scouted opponents and their game number on the schedule:
  Game #1: Ripon Red Hawks
  Game #2: St. Thomas (TX) Celts
  Game #3: Eureka Red Devils
  Game #4: Aurora Spartans
  Game #5: Simpson Storm
  Game #6: Elmhurst Bluejays
  Game #7: Lawrence Vikings
  Game #8: Carroll (WI) Pioneers
  Game #9: Hope Flying Dutchmen
  Game #10: Alma Scots
  Game #11: Coe Kohawks
  Game #12: UW-Oshkosh Titans
  Game #13: UW-Stevens Point Pointers
  Game #14: UW-River Falls Falcons
  Game #15: UW-La Crosse Eagles
  Game #16: UW-Stout Blue Devils
  Game #17: UW-Platteville Pioneers
  Game #18: UW-Eau Claire Blugolds
  Game #28: Loras Duhawks
  Game #58: Washington-St. Louis Bears

Target (most recently scouted game): Loras Duhawks (game #28)
Previous scouted opponents to compare against: ['St. Thomas (TX) Celts', 'Eureka Red Devils', 'Aurora Spartan


### Surface tag-based best match per target-opponent player

Displays each target player's single most comparable previously-scouted player, ranked by the tag-based similarity score (playing-style tags, notes/keys overlap, and stat similarity) computed in the cell above.

In [74]:
stat_cols_display = [c for c in best_matches.columns if c.startswith("target_") or c.startswith("compared_")]
stat_cols_display = [c for c in stat_cols_display if c.split("_")[-1] in {"PTS", "REB", "AST", "FG%", "3P%"}]
print(best_matches[[
    "target_player", "target_position", "target_opponent", "target_game_date",
    "compared_player", "compared_opponent", "compared_game_date",
    "compared_position", "similarity_score", "stat_similarity", "shared_notes_tags", "shared_keys_tags",
] + stat_cols_display])

print(f"Most comparable previously-scouted player for each {target_opponent} player "
      f"(scouted {scout_game_dates.get(target_opponent)}):\n")
for _, row in best_matches.iterrows():
    notes_note = f" -- shared NOTES tags: {row['shared_notes_tags']}" if row["shared_notes_tags"] else " -- no shared notes tags"
    keys_note = f"; shared KEYS tags: {row['shared_keys_tags']}" if row["shared_keys_tags"] else "; no shared keys tags"
    print(f"{row['target_player']} ({row['target_position']}) -> {row['compared_player']} of "
          f"{row['compared_opponent']} on {row['compared_game_date']} ({row['compared_position']}), "
          f"score={row['similarity_score']}{notes_note}{keys_note}")
    if pd.notna(row.get("stat_similarity")):
        print(f"    Season-stat similarity contribution: {row['stat_similarity']} (0-1 scale, weighted x{stat_weight} into the score above)")
    else:
        print("    No comparable season stats for this pair -- stat similarity contributed nothing to the score.")
    if pd.notna(row.get("compared_PTS")):
        print(f"    {row['compared_player']}'s season averages: {row['compared_PTS']} PTS, {row['compared_REB']} REB, "
              f"{row['compared_AST']} AST, {row['compared_FG%']} FG%, {row['compared_3P%']} 3P%")
    else:
        print(f"    No season stats available for {row['compared_player']} (no PDF scout report for {row['compared_opponent']}).")
    if pd.notna(row.get("target_PTS")):
        print(f"    {row['target_player']}'s season averages: {row['target_PTS']} PTS, {row['target_REB']} REB, "
              f"{row['target_AST']} AST, {row['target_FG%']} FG%, {row['target_3P%']} 3P%")
    else:
        print(f"    No season stats available for {row['target_player']} (no PDF scout report for {target_opponent} yet).")

         target_player target_position target_opponent target_game_date  \
3          Deng Makeer               G   Loras Duhawks       Fri, Mar 6   
8           Kyle Kober               G   Loras Duhawks       Fri, Mar 6   
1        Connor Mosele               G   Loras Duhawks       Fri, Mar 6   
6          Jack Haynes               F   Loras Duhawks       Fri, Mar 6   
9       Nolan Berendes               F   Loras Duhawks       Fri, Mar 6   
2       Damyen Jackson               G   Loras Duhawks       Fri, Mar 6   
7        Johnny Semany               G   Loras Duhawks       Fri, Mar 6   
5         Gavin Sarvis               G   Loras Duhawks       Fri, Mar 6   
11  Patrick Quarnstrom            None   Loras Duhawks       Fri, Mar 6   
10        Patrick Coen            None   Loras Duhawks       Fri, Mar 6   
0         Brock Massey            None   Loras Duhawks       Fri, Mar 6   
4           Dylan Kurt            None   Loras Duhawks       Fri, Mar 6   

       compared_player  


### Compare players using an LLM over their full scouting-note text

An alternative to the tag-based comparison above: sends each target/candidate player's full scouting-note text to an LLM for a similarity judgment. Only runs when `USE_LLM=True` in the Configuration cell (requires `OPENAI_API_KEY`) -- skipped by default.

In [76]:
if not use_llm:
    print("LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.")
else:
    if skipped_targets or skipped_candidates:
        print(f"Skipping {skipped_targets} target player(s) and {skipped_candidates} candidate player(s) with no "
              "scouting report text (box-score-only players) -- the LLM comparison only runs on scouted players.\n")

    print(llm_player_comparison.sort_values(["target_player", "llm_notes_similarity_score"], ascending=[True, False]))

LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.



### Surface the LLM's best match per player, and compare it to the tag-based pick

Shows whether the LLM's top match agrees with the tag-based pick from earlier. Also skipped when `USE_LLM=False`.

In [78]:
if not use_llm:
    print("LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.")
else:
    print(combined[[
        "target_player", "target_position", "compared_player", "compared_opponent",
        "llm_notes_similarity_score", "llm_notes_shared_traits", "llm_keys_similarity_score", "llm_keys_shared_traits",
        "tag_based_pick", "tag_based_score", "picks_agree",
    ]])

    print(f"LLM-based best match per {target_opponent} player -- playing-style (notes) and defensive-approach (keys) "
          "judged as SEPARATE dimensions:\n")
    for _, row in combined.iterrows():
        agree_note = "agrees with" if row["picks_agree"] else "DIFFERS from"
        print(f"{row['target_player']} ({row['target_position']}) -> {row['compared_player']} of "
              f"{row['compared_opponent']} (keyword-tag approach {agree_note} this pick: {row['tag_based_pick']}, score={row['tag_based_score']})")
        print(f"    [Notes] score={row['llm_notes_similarity_score']:.1f}/10 -- shared traits: {row['llm_notes_shared_traits']}")
        print(f"      {row['llm_notes_rationale']}")
        print(f"    [Keys]  score={row['llm_keys_similarity_score']:.1f}/10 -- shared traits: {row['llm_keys_shared_traits']}")
        print(f"      {row['llm_keys_rationale']}\n")

LLM-based comparison was skipped (USE_LLM=False). Set the USE_LLM config variable to True to run this cell.



### Blend the tag-based and LLM similarity scores

Combines both scores into one ranked list per target player. Skipped when `USE_LLM=False`, since there's no LLM score to blend in.

In [80]:
if not use_llm:
    print("LLM-based comparison was skipped (USE_LLM=False), so there is no LLM score to blend with the tag-based "
          "score. Set the USE_LLM config variable to True to run this cell.")
else:
    print(blended_best_matches[[
        "target_player", "target_position", "compared_player", "compared_opponent",
        "blended_score", "tag_similarity_score", "llm_similarity_score",
        "matches_tag_pick", "matches_llm_pick",
        "shared_notes_tags", "shared_keys_tags", "llm_notes_shared_traits", "llm_keys_shared_traits",
    ]])

    print(f"Blended (tag + LLM average) best match per {target_opponent} player, ranked highest to lowest:\n")
    for rank, (_, row) in enumerate(blended_best_matches.iterrows(), start=1):
        tag_note = "same as tag-only pick" if row["matches_tag_pick"] else f"tag-only picked {row['tag_only_pick']}"
        llm_note = "same as LLM-only pick" if row["matches_llm_pick"] else f"LLM-only picked {row['llm_only_pick']}"
        print(f"{rank}. {row['target_player']} ({row['target_position']}) -> {row['compared_player']} of "
              f"{row['compared_opponent']}, blended={row['blended_score']:.2f} "
              f"(tag={row['tag_similarity_score']:.2f}, llm={row['llm_similarity_score']:.1f}/10)")
        print(f"    {tag_note}; {llm_note}")
        print(f"    Tag-based shared NOTES tags: {row['shared_notes_tags'] or '(none)'}")
        print(f"    Tag-based shared KEYS tags: {row['shared_keys_tags'] or '(none)'}")
        print(f"    LLM shared NOTES traits: {row['llm_notes_shared_traits']}")
        print(f"    LLM shared KEYS traits: {row['llm_keys_shared_traits']}\n")

LLM-based comparison was skipped (USE_LLM=False), so there is no LLM score to blend with the tag-based score. Set the USE_LLM config variable to True to run this cell.



### Display best matches

Prints `best_matches` (the tag-based result) on its own -- a plain re-display of the table already shown two cells above, kept as a quick standalone reference.

In [82]:
print(best_matches)

         target_player target_position target_role target_opponent  \
3          Deng Makeer               G       Bench   Loras Duhawks   
8           Kyle Kober               G       Bench   Loras Duhawks   
1        Connor Mosele               G       Bench   Loras Duhawks   
6          Jack Haynes               F     Starter   Loras Duhawks   
9       Nolan Berendes               F     Starter   Loras Duhawks   
2       Damyen Jackson               G     Starter   Loras Duhawks   
7        Johnny Semany               G     Starter   Loras Duhawks   
5         Gavin Sarvis               G     Starter   Loras Duhawks   
11  Patrick Quarnstrom            None       Bench   Loras Duhawks   
10        Patrick Coen            None       Bench   Loras Duhawks   
0         Brock Massey            None       Bench   Loras Duhawks   
4           Dylan Kurt            None       Bench   Loras Duhawks   

   target_game_date        compared_opponent compared_game_date  \
3        Fri, Mar 6   


### Parse play-by-play MHTML files into a unified event log

Runs `parse_pbp_mhtml`/`build_pbp_events`/`opponent_from_pbp_filename` (defined earlier) over every `"*_pbp.mhtml"` file in `INPUT_DIR`, resolving each file's `self_column` per-game, to build `pbp_events` -- UW-Whitewater's own play-by-play across every scouted game so far.

In [84]:
# --- Play-by-play (PBP) data -----------------------------------------------------------------------------
# Parsing functions (parse_pbp_mhtml, classify_event, build_pbp_events, opponent_from_pbp_filename) live in the
# "Play-by-play parsing functions" cell above. This cell just runs them over every "*_pbp.mhtml" file in
# INPUT_DIR (same pattern as the "*_scout.pdf" loop above) so newly added games are picked up automatically.

# Filter to "*UW-Whitewater*_pbp.mhtml" rather than the broader "*_pbp.mhtml" -- INPUT_DIR may also hold pbp
# files for OTHER teams' games that don't involve UW-Whitewater at all (uploaded while cross-scouting an
# opponent's other games). Those have no "UW-Whitewater" column in their 4-column layout at all, so parsing
# them here would silently misread one of the other two teams' event text as if it were UWW's.
pbp_files = sorted(glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.mhtml") + glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.html"))

# Live-scrape+cache any scouted UWW game missing a local pbp file, using its "game_url" from
# uww_team_schedule -- same pattern as the opponent-prior-games pbp cell above.
def _game_date_str(game_date):
    return f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')}"

games_needing_live_uww_pbp = []
if not uww_team_schedule.empty:
    for _, g_row in uww_team_schedule.iterrows():
        g_date = parse_schedule_date(g_row["date"]) if pd.notna(g_row.get("date")) else None
        if g_date is None or pd.isna(g_row.get("game_url")):
            continue
        opp_short = next(
            (s for s in scouted_opponents if re.search(re.escape(s), str(g_row["opponent"]), re.IGNORECASE)), None
        )
        if opp_short is None or glob.glob(f"{volume_dir}/{_game_date_str(g_date)}*{opp_short}*_pbp.*"):
            continue
        games_needing_live_uww_pbp.append((g_row, g_date, opp_short))

if games_needing_live_uww_pbp and fastscout_username and fastscout_password:
    # One session call per game so a dead session can self-heal mid-batch (see the pbp cell above).
    for g_row, g_date, opp_short in games_needing_live_uww_pbp:
        try:
            if str(g_row.get("location", "")).strip().lower() == "home":
                matchup = f"{g_row['opponent']} @ UW-Whitewater"
            else:
                matchup = f"UW-Whitewater @ {g_row['opponent']}"
            pbp_save_path = f"{volume_dir}/{_game_date_str(g_date)} {matchup}_pbp.html"
            run_in_fastscout_session(
                lambda page, url=g_row["game_url"], sp=pbp_save_path: scrape_pbp_live(page, url, save_path=sp)
            )
        except Exception as pbp_scrape_error:
            print(f"  Could not live-scrape UWW's own pbp for {g_row['opponent']} ({g_row['game_url']}): {type(pbp_scrape_error).__name__}: {pbp_scrape_error}")

    # Re-glob so the freshly-cached ".html" file(s) are picked up by the parsing loop below.
    pbp_files = sorted(glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.mhtml") + glob.glob(f"{volume_dir}/*UW-Whitewater*_pbp.html"))
elif games_needing_live_uww_pbp:
    print(
        f"{len(games_needing_live_uww_pbp)} scouted UWW game(s) are missing a local '_pbp' file and could be "
        "live-scraped, but no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were found -- skipping."
    )

print(f"Found {len(pbp_files)} UW-Whitewater play-by-play file(s):")
for f in pbp_files:
    print(" -", os.path.basename(f))

# Build a roster set for auto-detecting column swaps -- UWW season stats' second column is the player name.
_player_col = stats.columns[1]
uww_roster_names = set(stats[_player_col].dropna().tolist()) - {"Team Total", "Opponent"}

pbp_events_list = []
for path in pbp_files:
    opponent_short = opponent_from_pbp_filename(path)
    raw_df = parse_pbp_mhtml(path)

    # Auto-detect if UWW's events ended up in the wrong column. FastScout's 4-column layout is
    # "Time | LeftTeam | Score | RightTeam", and which side UWW is on depends on whether they're home or away
    # in that particular export -- it does NOT always match our assumed "uww_text" = column 2 convention.
    # Cross-reference extracted player names from each text column against the known UWW roster.
    def _extract_player_names(series):
        names = set()
        for text in series.dropna():
            parsed = classify_event(str(text).strip())
            if parsed.get("player"):
                names.add(parsed["player"])
        return names

    uww_col_players = _extract_player_names(raw_df["uww_text"])
    opp_col_players = _extract_player_names(raw_df["opp_text"])
    uww_in_uww_col = len(uww_col_players & uww_roster_names)
    uww_in_opp_col = len(opp_col_players & uww_roster_names)

    if uww_in_opp_col > uww_in_uww_col:
        # Columns are swapped: UWW events are in "opp_text" and opponent events in "uww_text".
        # Swap both text column VALUES and reverse the score format ("OppScore-UWWScore" -> "UWWScore-OppScore")
        # so build_pbp_events' defaults (self_column="uww_text", score group(1)=UWW) work correctly.
        raw_df["uww_text"], raw_df["opp_text"] = raw_df["opp_text"].copy(), raw_df["uww_text"].copy()
        raw_df["score_raw"] = raw_df["score_raw"].apply(
            lambda s: "-".join(reversed(str(s).strip().split("-"))) if pd.notna(s) and re.match(r"^\d+-\d+$", str(s).strip()) else s
        )
        print(f"  WARNING: Detected swapped columns for '{opponent_short}' (UWW roster found in right column) -- auto-corrected")

    # Date this game from its OWN filename, not from game_date_for() -- see
    # game_date_from_pbp_filename(). Without this, a rematch inherits the first meeting's date and
    # collapses into it in every groupby downstream.
    file_game_date = game_date_from_pbp_filename(path)
    if file_game_date is None:
        file_game_date = game_date_for(opponent_short)
        print(f"    WARNING: '{os.path.basename(path)}' has no '<m>_<d>_<yy> ' date prefix -- falling "
              f"back to the schedule's FIRST '{opponent_short}' meeting ({file_game_date}). If UWW "
              f"plays this opponent more than once, rename the file so the two games stay distinct.")
    events = build_pbp_events(raw_df, opponent_short, file_game_date)
    n_unclassified = (events["event_type"] == "unclassified").sum()
    print(f"  Parsed '{opponent_short}': {len(events)} events" + (f" ({n_unclassified} UNCLASSIFIED -- inspect raw_text)" if n_unclassified else ""))
    pbp_events_list.append(events)

pbp_events = pd.concat(pbp_events_list, ignore_index=True) if pbp_events_list else pd.DataFrame()

# CONFIRMED BUG (fixed here): pbp_files (above) is a pure filesystem glob with no reference_date awareness at
# all -- it picks up EVERY local "*_pbp.mhtml"/"*_pbp.html" file regardless of that game's date, so pbp_events
# (and everything built from it downstream: uww_pbp_box_score, uww_lineup_stints, the video-tagging attachment
# in the next section, scoring runs, clutch events...) silently included games on/after reference_date whenever
# local files for them already existed on disk. In normal usage (reference_date left at the real "today") this
# never shows up, since future games' PBP files genuinely don't exist yet -- but when reference_date is set
# earlier than the files actually available (e.g. simulating an earlier point in an already-completed season,
# exactly how this was caught), pbp_events ends up scoped to "every game with a local file" instead of "every
# game played so far," which threw off several season-average stats downstream (confirmed: UWW's own turnovers/
# game on the Upcoming Game page). Filtered here, once, at the source, rather than patching every downstream
# consumer individually. Keeps rows with no resolved game_date rather than dropping them -- an unresolved date
# is a different, separate problem (see game_date_for()) and silently discarding that data isn't the fix for it.
if not pbp_events.empty and "game_date" in pbp_events.columns:
    _pbp_events_before = len(pbp_events)
    pbp_events = pbp_events[pbp_events["game_date"].isna() | (pbp_events["game_date"] < reference_date.date())].reset_index(drop=True)
    _n_dropped = _pbp_events_before - len(pbp_events)
    if _n_dropped:
        print(f"Dropped {_n_dropped} pbp_events row(s) with a game_date on/after reference_date ({reference_date_str}) -- not yet \'played\' in this simulation.")

# Anything still unclassified is invisible in every downstream table now that it carries no player --
# so list the distinct raw strings here, which is what a new EVENT_PATTERNS entry gets written from.
if not pbp_events.empty and (pbp_events["event_type"] == "unclassified").any():
    _unc = pbp_events.loc[pbp_events["event_type"] == "unclassified", "raw_text"].value_counts()
    print(f"\n{int(_unc.sum())} unclassified event(s) across {len(_unc)} distinct string(s) -- "
          f"add a pattern to EVENT_PATTERNS for any of these that should be counted:")
    print(_unc.head(20).to_string())
else:
    print("\nEvery play-by-play line was classified.")

print(pbp_events.head(20))

Found 28 UW-Whitewater play-by-play file(s):
 - 11_14_25 UW-Whitewater @ St. Thomas (TX) Celts_pbp.html
 - 11_15_25 UW-Whitewater @ Eureka Red Devils_pbp.html
 - 11_19_25 Aurora Spartans @ UW-Whitewater_pbp.html
 - 11_25_25 Simpson Storm @ UW-Whitewater_pbp.html
 - 11_7_25 UW-Whitewater @ Ripon Red Hawks_pbp.html
 - 12_10_25 UW-Whitewater @ Lawrence Vikings_pbp.html
 - 12_13_25 Carroll (WI) Pioneers @ UW-Whitewater_pbp.html
 - 12_19_25 UW-Whitewater @ Hope Flying Dutchmen_pbp.html
 - 12_20_25 UW-Whitewater @ Alma Scots_pbp.html
 - 12_2_25 Elmhurst Bluejays @ UW-Whitewater_pbp.html
 - 12_30_25 UW-Whitewater @ Coe Kohawks_pbp.html
 - 1_10_26 UW-Whitewater @ UW-River Falls Falcons_pbp.html
 - 1_14_26 UW-Whitewater @ UW-La Crosse Eagles_pbp.html
 - 1_17_26 UW-Stout Blue Devils @ UW-Whitewater_pbp.html
 - 1_21_26 UW-Whitewater @ UW-Platteville Pioneers_pbp.html
 - 1_24_26 UW-Eau Claire Blugolds @ UW-Whitewater_pbp.html
 - 1_28_26 UW-Whitewater @ UW-Oshkosh Titans_pbp.html
 - 1_3_26 UW-Oshko


### Load coach-tagged "recap" CSVs: per-clip Text Overlay notes (play calls, execution grades)

Some games have a companion `"<matchup>_recap.csv"` export from the video-tagging tool -- one row per
tagged clip, with a `Text Overlay` field holding the coach's own note for that specific play (an offensive
play call and how it was executed, or a defensive breakdown of what went right/wrong). Each row also carries
enough identifying info (period, game clock, team, player) to attach it onto the matching `pbp_events` row
for that same play, so the note shows up in context on the play-by-play rather than as a disconnected list.

Every note is ALSO kept in its own standalone `coach_notes` table regardless of whether it successfully
matched a `pbp_events` row -- a handful of clips have no captured game clock (a between-play/summary note),
which can't be matched to a specific play but shouldn't be silently discarded either; season-wide analytics
(most-called plays, common flagged themes) read from this full table, not just the matched subset.

In [86]:
# --- Parse "*_recap.csv" (single-game coach notes) and "*_plays_*.csv" (season-wide play-call log) exports,
# and attach them onto pbp_events ---------------------------------------------------------------------------
recap_files = sorted(glob.glob(f"{volume_dir}/*_recap.csv") + glob.glob(f"{volume_dir}/*_plays_*.csv"))
print(f"Found {len(recap_files)} coach-note/play-log CSV(s):")
for f in recap_files:
    print(" -", os.path.basename(f))


def _recap_period_label(pd_val):
    """Recap CSVs use a bare half number in "Pd." (1, 2, 3+ for OT) -- pbp_events uses "H1"/"H2"/"OT"/"OT2"
    (whatever token build_pbp_events pulled out of the raw PBP source's own "MM:SS (TOKEN)" format). Map the
    bare number onto that same convention so notes line up with pbp_events' own "period" values."""
    try:
        n = int(float(pd_val))
    except (TypeError, ValueError):
        return None
    if n <= 0:
        return None
    if n <= 2:
        return f"H{n}"
    return "OT" if n == 3 else f"OT{n - 2}"


def _clock_to_seconds(clock_val):
    """"10:46" -> 646. Returns None for a blank/unparseable clock (some clips -- e.g. a between-play summary
    note -- have no captured game clock at all)."""
    m = re.match(r"^(\d+):(\d+)", str(clock_val).strip())
    return int(m.group(1)) * 60 + int(m.group(2)) if m else None


def _normalize_for_match(text):
    """Collapse ALL whitespace and lowercase, so "Ripon Red Hawks" and "Ripon Redhawks" -- a real, confirmed
    spelling inconsistency between one of these CSVs' own "Team" column and the schedule/scout-file spelling
    used to build scouted_opponents -- compare as equal. A plain substring check fails on exactly this kind
    of whitespace difference even though it's obviously the same opponent."""
    return re.sub(r"\s+", "", str(text)).lower()


# Date -> short opponent name, built from UWW's own schedule -- used to resolve each ROW's own opponent in a
# season-wide play-call log (one file covering many games), where a single "whole file belongs to one
# opponent" assumption (used for the single-game recap files below) doesn't hold.
_date_to_opponent = {}
if not uww_team_schedule.empty:
    for _, _sched_row in uww_team_schedule.iterrows():
        _d = parse_schedule_date(_sched_row.get("date"))
        if _d is None:
            continue
        _full_opp = str(_sched_row.get("opponent", ""))
        _short_match = next((s for s in scouted_opponents if re.search(re.escape(s), _full_opp, re.IGNORECASE)), None)
        _date_to_opponent[_d] = _short_match or _full_opp


_recap_rows = []
for path in recap_files:
    try:
        recap_df = pd.read_csv(path)
    except Exception as e:
        print(f"  Could not read {os.path.basename(path)}: {type(e).__name__}: {e}")
        continue

    required_cols = {"Player", "Team", "Pd.", "Clock", "Text Overlay", "Result"}
    missing = required_cols - set(recap_df.columns)
    if missing:
        print(f"  {os.path.basename(path)}: missing expected column(s) {missing} -- skipping.")
        continue

    # Section-header rows ("DEFENSIVE CLIPS" / "OFFENSIVE CLIPS" / a play-family header like "Panther Series"
    # in the season-wide log) and any row with no real note/tag carry no Player at all -- drop those first.
    recap_df = recap_df[recap_df["Player"].notna() & (recap_df["Player"].astype(str).str.strip() != "")].copy()
    recap_df = recap_df[recap_df["Text Overlay"].notna() & (recap_df["Text Overlay"].astype(str).str.strip() != "")]
    if recap_df.empty:
        continue

    _is_season_wide = "Date" in recap_df.columns and pd.to_datetime(recap_df["Date"], errors="coerce").dt.date.nunique() > 1

    if _is_season_wide:
        # Season-wide play-call log: each ROW belongs to a different game, so resolve opponent per-row from
        # that row's own "Date" (matched against UWW's own schedule) rather than assuming one opponent for
        # the whole file. Also: this file's "Text Overlay" is a clean PLAY NAME (e.g. "Panther - Elmhurst",
        # "Twins Right Swirl - Ripon"), not free-text coach commentary like the single-game recap files use
        # -- so it goes into its own "play_call" field instead of "coach_note", rather than force two
        # different kinds of content through a format built for one of them.
        recap_df["_parsed_date"] = pd.to_datetime(recap_df["Date"], errors="coerce").dt.date
        recap_df["opponent"] = recap_df["_parsed_date"].map(_date_to_opponent)
        _unresolved = recap_df["opponent"].isna().sum()
        recap_df = recap_df[recap_df["opponent"].notna()]
        if _unresolved:
            print(f"  {os.path.basename(path)}: {_unresolved} row(s) had a date that didn't match any UWW schedule game -- skipped.")
        if recap_df.empty:
            continue
        recap_df["team"] = recap_df["Team"].apply(lambda t: "UW-Whitewater" if "whitewater" in str(t).lower() else None)
        recap_df = recap_df[recap_df["team"].notna()]  # this log is offense-only (no "Team" = opponent rows observed)
        recap_df["period"] = recap_df["Pd."].apply(_recap_period_label)
        recap_df["time_remaining_seconds"] = recap_df["Clock"].apply(_clock_to_seconds)
        recap_df["player"] = recap_df["Player"].astype(str).str.strip()
        # Play name = the text before the first " - "-style delimiter in Text Overlay (handles the observed
        # inconsistent spacing, e.g. "Panther - Elmhurst" and "Over Action- Pin to Flare..." alike).
        recap_df["play_call"] = recap_df["Text Overlay"].astype(str).apply(lambda t: re.split(r"\s*-\s*", t.strip(), maxsplit=1)[0].strip())
        recap_df["coach_note"] = None
        recap_df["result"] = recap_df["Result"].astype(str).str.strip()
        recap_df["clip_side"] = "Offense"
        _recap_rows.append(recap_df[[
            "opponent", "period", "time_remaining_seconds", "team", "player", "result", "coach_note", "play_call", "clip_side",
        ]])
        print(f"  {os.path.basename(path)}: {len(recap_df)} play-call log row(s) across {recap_df['opponent'].nunique()} opponent(s)")
        continue

    # Single-game recap: whole file belongs to one opponent, resolved from whichever non-UWW "Team" value
    # appears (e.g. "Ripon Redhawks") -- whitespace-normalized match against scouted_opponents.
    _opp_team_vals = recap_df.loc[
        ~recap_df["Team"].astype(str).str.contains("whitewater", case=False, na=False), "Team"
    ].dropna().unique()
    _recap_opponent = None
    for _tv in _opp_team_vals:
        _tv_norm = _normalize_for_match(_tv)
        _match = next(
            (s for s in scouted_opponents if _normalize_for_match(s) in _tv_norm or _tv_norm in _normalize_for_match(s)),
            None,
        )
        if _match:
            _recap_opponent = _match
            break
    if _recap_opponent is None:
        print(f"  Could not resolve which scouted opponent {os.path.basename(path)} belongs to (Team values seen: {list(_opp_team_vals)}) -- skipping.")
        continue

    recap_df["opponent"] = _recap_opponent
    recap_df["team"] = recap_df["Team"].apply(lambda t: "UW-Whitewater" if "whitewater" in str(t).lower() else _recap_opponent)
    recap_df["period"] = recap_df["Pd."].apply(_recap_period_label)
    recap_df["time_remaining_seconds"] = recap_df["Clock"].apply(_clock_to_seconds)
    recap_df["player"] = recap_df["Player"].astype(str).str.strip()
    recap_df["coach_note"] = recap_df["Text Overlay"].astype(str).str.strip()
    recap_df["play_call"] = None
    recap_df["result"] = recap_df["Result"].astype(str).str.strip()
    recap_df["clip_side"] = recap_df["team"].apply(lambda t: "Offense" if t == "UW-Whitewater" else "Defense")

    _recap_rows.append(recap_df[[
        "opponent", "period", "time_remaining_seconds", "team", "player", "result", "coach_note", "play_call", "clip_side",
    ]])
    print(f"  {os.path.basename(path)}: {len(recap_df)} coach note(s) for opponent '{_recap_opponent}'")

coach_notes = pd.concat(_recap_rows, ignore_index=True) if _recap_rows else pd.DataFrame(
    columns=["opponent", "period", "time_remaining_seconds", "team", "player", "result", "coach_note", "play_call", "clip_side"]
)

# Attach coach_note/play_call onto the matching pbp_events row by (opponent, period, time_remaining_seconds,
# team, player) -- pbp_events already carries a clean time_remaining_seconds column (parsed from the raw PBP
# source's own "MM:SS (PERIOD)" text), so match on THAT rather than pbp_events' "time_remaining" column,
# which is that unparsed raw string (e.g. "10:46 (H1)") and would never equal the recap's plain "10:46".
# A small number of clips have no clock at all (time_remaining_seconds is None) and simply won't match any
# row -- their note is still preserved in the standalone coach_notes table above, just not linked in-line.
for _c in ("coach_note", "play_call"):
    if _c in pbp_events.columns:
        pbp_events = pbp_events.drop(columns=[_c])
if not coach_notes.empty and not pbp_events.empty:
    _matchable_notes = coach_notes[coach_notes["time_remaining_seconds"].notna() & coach_notes["period"].notna()]
    _join_keys = ["opponent", "period", "time_remaining_seconds", "team", "player"]
    pbp_events = pbp_events.merge(
        _matchable_notes[_join_keys + ["coach_note", "play_call"]].drop_duplicates(subset=_join_keys),
        on=_join_keys, how="left",
    )
    _n_matched = int((pbp_events["coach_note"].notna() | pbp_events["play_call"].notna()).sum())
    print(
        f"\nAttached {_n_matched} coach note(s)/play-call(s) onto pbp_events out of {len(coach_notes)} parsed "
        f"({len(coach_notes) - len(_matchable_notes)} had no usable clock and can't be linked to a specific "
        "play; the rest may not match if a player name is spelled differently between sources)."
    )
    # If the match rate is suspiciously low, print exactly what didn't line up -- side-by-side against a
    # sample of pbp_events' own keys for the same opponent(s) -- rather than leaving "why" as a guessing game.
    if _n_matched < len(_matchable_notes):
        _unmatched = _matchable_notes.merge(
            pbp_events[_join_keys].drop_duplicates(), on=_join_keys, how="left", indicator=True
        )
        _unmatched = _unmatched[_unmatched["_merge"] == "left_only"]
        if not _unmatched.empty:
            print(f"\n{len(_unmatched)} note(s) did NOT find a matching pbp_events row. First few unmatched note keys:")
            print(_unmatched[_join_keys].head(8).to_string(index=False))
            _sample_opp = _unmatched["opponent"].iloc[0]
            print(f"\nFor comparison, actual pbp_events keys for opponent \'{_sample_opp}\' (first 8 rows with a player):")
            _sample_pbp = pbp_events[(pbp_events["opponent"] == _sample_opp) & pbp_events["player"].notna()]
            print(_sample_pbp[_join_keys].head(8).to_string(index=False))
            print(
                "\nCompare the two tables above column-by-column -- the mismatch (differently-spelled player "
                "name, a period token that doesn\'t match, an opponent string that isn\'t identical) should be "
                "visible directly. A common cause: this game\'s local _pbp file used a different exact player-"
                "name spelling than the recap CSV (e.g. a nickname or suffix like \'Jr\')."
            )
else:
    pbp_events["coach_note"] = None
    pbp_events["play_call"] = None


Found 6 coach-note/play-log CSV(s):
 - 11_19_25 Aurora Spartans @ UW-Whitewater_recap.csv
 - 11_7_25 UW-Whitewater @ Ripon Red Hawks_recap.csv
 - 12_10_25 UW-Whitewater @ Lawrence Vikings_recap.csv
 - 12_13_25 Carroll (WI) Pioneers @ UW-Whitewater_recap.csv
 - 12_2_25 Elmhurst Bluejays @ UW-Whitewater_recap.csv
 - uww_plays_25_26.csv
  Could not resolve which scouted opponent 11_19_25 Aurora Spartans @ UW-Whitewater_recap.csv belongs to (Team values seen: ['Aurora University']) -- skipping.
  11_7_25 UW-Whitewater @ Ripon Red Hawks_recap.csv: 46 coach note(s) for opponent 'Ripon Red Hawks'
  12_10_25 UW-Whitewater @ Lawrence Vikings_recap.csv: 47 coach note(s) for opponent 'Lawrence Vikings'
  12_13_25 Carroll (WI) Pioneers @ UW-Whitewater_recap.csv: 57 coach note(s) for opponent 'Carroll (WI) Pioneers'
  Could not resolve which scouted opponent 12_2_25 Elmhurst Bluejays @ UW-Whitewater_recap.csv belongs to (Team values seen: ['Elmhurst University Bluejays']) -- skipping.
  uww_plays_2


### Reconstruct the on-court 5-man lineup for both teams at every event

Walks each game's `pbp_events` in order, tracking substitutions to derive `uww_lineup`/`opp_lineup` -- the 5 players on the floor for each team at the moment of every event.

In [88]:
# --- On-court 5-man lineups for both teams, at every point in the play-by-play -----------------------------
# Reconstructed purely from substitution events ("Subs In"/"Subs Out") plus a starting lineup inferred from each
# player's FIRST event of the game: if a player's first action is anything other than "Subs In", they were
# already on the floor at tip-off (a starter); if their first action IS "Subs In", they came off the bench.
# Dead-ball substitutions almost always swap multiple players for a team at the EXACT same game-clock time, so
# subs are applied as an atomic batch per (opponent, team, period, time_remaining_seconds) -- applying them one
# row at a time would otherwise show a transient 4-man lineup between an "out" row and its matching "in" row.
# Excludes every TEAM-level event whose "player" field is actually a team name, not a roster player.
TEAM_LEVEL_EVENT_TYPES = {"team_deadball_rebound_offensive", "team_deadball_rebound_defensive", "timeout"}
real_player_events = pbp_events[
    pbp_events["player"].notna() & (~pbp_events["event_type"].isin(TEAM_LEVEL_EVENT_TYPES))
].sort_values("event_order")

def starting_lineup(team_events):
    first_seen = team_events.groupby("player").first()
    return set(first_seen[first_seen["event_type"] != "sub_in"].index)

uww_lineup_col = pd.Series(index=pbp_events.index, dtype=object)
opp_lineup_col = pd.Series(index=pbp_events.index, dtype=object)

for (opponent, game_date), opp_rows in pbp_events.groupby(GAME_KEYS, dropna=False):
    opp_real = real_player_events[
        (real_player_events["opponent"] == opponent) & (real_player_events["game_date"] == game_date)
    ]

    # Period start anchors are shared by BOTH teams: the true boundary for a period is the earliest real event
    # across EITHER team tagged with that period, not just one team's own subset.
    period_starts = opp_real.dropna(subset=["period"]).groupby("period")["event_order"].min().sort_values()
    periods_in_order = period_starts.index.tolist()

    for team_label, target_col in [("UW-Whitewater", uww_lineup_col), (opponent, opp_lineup_col)]:
        team_events = opp_real[opp_real["team"] == team_label]

        # Re-infer the starting five FRESH at the start of EVERY period (H1, H2, OT...), not just tip-off.
        changes = []
        for period in periods_in_order:
            period_events = team_events[team_events["period"] == period].sort_values("event_order")
            starters = starting_lineup(period_events)
            if len(starters) != 5:
                print(f"WARNING: {opponent} {game_date}/{team_label}/{period} starting-lineup detection found {len(starters)} "
                      f"players (expected 5): {sorted(starters)}")

            # Apply substitutions one at a time in chronological order, but only RECORD a new change-point once
            # the running set settles back at exactly 5.
            current = set(starters)
            changes.append((period_starts[period] - 1, frozenset(current)))
            sub_events = period_events[period_events["event_type"].isin(["sub_in", "sub_out"])].sort_values("event_order")
            for _, row in sub_events.iterrows():
                if row["event_type"] == "sub_in":
                    current.add(row["player"])
                else:
                    current.discard(row["player"])
                if len(current) == 5:
                    changes.append((row["event_order"], frozenset(current)))
            if len(current) != 5:
                print(f"WARNING: {opponent} {game_date}/{team_label}/{period} never settled back to a 5-man lineup by the "
                      f"last substitution (ended with {len(current)}): {sorted(current)}")

        changes_df = pd.DataFrame(changes, columns=["event_order", "lineup"]).sort_values("event_order")
        target_rows = opp_rows.sort_values("event_order")
        merged = pd.merge_asof(target_rows[["event_order"]], changes_df, on="event_order", direction="backward")
        lineup_strings = merged["lineup"].apply(lambda s: ", ".join(sorted(s)) if isinstance(s, frozenset) else None)
        lineup_strings.index = target_rows.index
        target_col.loc[target_rows.index] = lineup_strings

pbp_events["uww_lineup"] = uww_lineup_col
pbp_events["opp_lineup"] = opp_lineup_col

print(pbp_events[["opponent", "period", "time_remaining", "team", "event_type", "raw_text", "uww_lineup", "opp_lineup"]].head(30))

print("\nLineup size check (every non-null value should be exactly 5 players):")
print(" uww_lineup sizes:", pbp_events["uww_lineup"].dropna().apply(lambda s: len(s.split(", "))).value_counts().to_dict())
print(" opp_lineup sizes:", pbp_events["opp_lineup"].dropna().apply(lambda s: len(s.split(", "))).value_counts().to_dict())

                 opponent period time_remaining                   team  \
0   St. Thomas (TX) Celts   None           None                   None   
1   St. Thomas (TX) Celts     H1     19:58 (H1)  St. Thomas (TX) Celts   
2   St. Thomas (TX) Celts     H1     19:58 (H1)          UW-Whitewater   
3   St. Thomas (TX) Celts     H1     19:34 (H1)  St. Thomas (TX) Celts   
4   St. Thomas (TX) Celts     H1     19:22 (H1)          UW-Whitewater   
5   St. Thomas (TX) Celts     H1     19:22 (H1)  St. Thomas (TX) Celts   
6   St. Thomas (TX) Celts     H1     19:16 (H1)  St. Thomas (TX) Celts   
7   St. Thomas (TX) Celts     H1     19:14 (H1)          UW-Whitewater   
8   St. Thomas (TX) Celts     H1     19:09 (H1)          UW-Whitewater   
9   St. Thomas (TX) Celts     H1     19:09 (H1)  St. Thomas (TX) Celts   
10  St. Thomas (TX) Celts     H1     19:05 (H1)  St. Thomas (TX) Celts   
11  St. Thomas (TX) Celts     H1     19:05 (H1)  St. Thomas (TX) Celts   
12  St. Thomas (TX) Celts     H1     1


### Classify each on-court opponent lineup using scouting-report tags

Adds `opp_lineup_summary`, describing the specific opponent lineup on the floor at any moment: role composition (Starter/Bench), position composition, and each player's most common notes/keys tags from `player_profiles`. No equivalent exists for `uww_lineup` -- UW-Whitewater isn't a scouted opponent, so there's no scouting-report data to classify our own lineups with.

In [90]:
# --- Classify each on-court OPPONENT 5-man unit using its players' scouting-report data -----------------------
# One new field, opp_lineup_summary, describing the specific opponent lineup on the floor: role composition
# (Starter/Bench), position composition, and each player's most common notes_tags/keys_tags. There's no
# equivalent field for uww_lineup -- UW-Whitewater is "us", not a scouted opponent, so player_profiles has no
# scouting-report rows for our own roster to classify it with.
def summarize_lineup(opponent, lineup_str):
    if pd.isna(lineup_str):
        return None
    names = lineup_str.split(", ")
    rows = player_profiles[(player_profiles["opponent"] == opponent) & (player_profiles["name"].isin(names))]
    if rows.empty:
        return f"No scouting data found for: {', '.join(names)}"
    unmatched = [n for n in names if n not in set(rows["name"])]

    role_summary = " / ".join(f"{count} {role}" for role, count in rows["role"].value_counts().items())
    pos_summary = ", ".join(f"{count} {pos}" for pos, count in rows["position_group"].value_counts().items())

    notes_tag_counts = Counter(tag for tags in rows["notes_tags"] for tag in tags)
    keys_tag_counts = Counter(tag for tags in rows["keys_tags"] for tag in tags)
    top_notes = ", ".join(f"{tag} x{n}" for tag, n in notes_tag_counts.most_common(3))
    top_keys = ", ".join(f"{tag} x{n}" for tag, n in keys_tag_counts.most_common(3))

    parts = [
        role_summary,
        f"Pos: {pos_summary}",
        f"Style: {top_notes}" if top_notes else "Style: (no tagged traits)",
        f"Defend: {top_keys}" if top_keys else "Defend: (no tagged traits)",
    ]
    if unmatched:
        parts.append(f"No scouting match: {', '.join(unmatched)}")
    return " | ".join(parts)

unique_opp_lineups = pbp_events[["opponent", "opp_lineup"]].drop_duplicates().dropna().copy()
unique_opp_lineups["opp_lineup_summary"] = unique_opp_lineups.apply(
    lambda r: summarize_lineup(r["opponent"], r["opp_lineup"]), axis=1
)
# Drop any stale opp_lineup_summary from a previous run before merging, so re-running this cell overwrites
# cleanly instead of colliding into opp_lineup_summary_x/_y.
pbp_events = pbp_events.drop(columns=["opp_lineup_summary"], errors="ignore").merge(unique_opp_lineups, on=["opponent", "opp_lineup"], how="left")

print(
    pbp_events[["opponent", "opp_lineup", "opp_lineup_summary"]]
    .drop_duplicates(subset=["opponent", "opp_lineup"])
    .sort_values("opponent")
)

                   opponent  \
4341             Alma Scots   
4039             Alma Scots   
4097             Alma Scots   
4114             Alma Scots   
4140             Alma Scots   
...                     ...   
6473   UW-Stout Blue Devils   
6448   UW-Stout Blue Devils   
6425   UW-Stout Blue Devils   
6546   UW-Stout Blue Devils   
13659  UW-Stout Blue Devils   

                                                                             opp_lineup  \
4341    Elijah Sykes, Joss Bradley, Korbin Heitzman, Luciano Guerrazzi, Preston Malpass   
4039     Donovan Collins, Josh Elliott, Korbin Heitzman, Logan St. Martin, Raine Rodich   
4097        Elijah Sykes, Josh Elliott, Joss Bradley, Korbin Heitzman, Logan St. Martin   
4114       Elijah Sykes, Josh Elliott, Joss Bradley, Korbin Heitzman, Luciano Guerrazzi   
4140    Donovan Collins, Elijah Sykes, Luciano Guerrazzi, Preston Malpass, Raine Rodich   
...                                                                              


### Attach video-tagging clip descriptions onto `pbp_events`

Same global order-preserving alignment used for the opponent's prior games earlier, applied here to UWW's own `pbp_events` against each `_video` export. The next cell just displays the result.

In [92]:
# --- Attach video-tagging clip descriptions ("*_video.mhtml") onto pbp_events ---------------------------------
# The video-tagging export logs one row per CLIP, with a chained action "Description" (e.g. "3 Seth Bunders >
# Off Screen > ... > Miss 3 Pts") -- one row per POSSESSION-ENDING action, not one row per raw play-by-play
# event, and it carries no game-clock/timestamp. So there's no direct key to join on. Instead, run ONE global
# order-preserving (monotonic) alignment across the whole opponent's clip log against the whole list of
# attempted pbp events at once, using compatible() as a hard gate (event_type + player/team match, plus an
# on-court lineup check for fouls) and maximizing the total number of matches.
# RESULT_TO_KEY, AND1_SHOT_RE, FREE_THROW_EVENT_TYPES, expand_clip_to_subevents, global_align, and
# parse_video_mhtml are reused from the "Shared video-tagging helper functions" cell above.

def _game_date_str(game_date):
    return f"{game_date.month}_{game_date.day}_{game_date.strftime('%y')}"

video_files = sorted(glob.glob(f"{volume_dir}/*_video.mhtml") + glob.glob(f"{volume_dir}/*_video.html"))

# Live-scrape+cache any scouted UWW game missing a local video-tagging file, using its "video_url" from
# uww_team_schedule -- same pattern as the opponent-prior-games video cell above.
games_needing_live_uww_video = []
if not uww_team_schedule.empty:
    for _, g_row in uww_team_schedule.iterrows():
        g_date = parse_schedule_date(g_row["date"]) if pd.notna(g_row.get("date")) else None
        if g_date is None or pd.isna(g_row.get("video_url")):
            continue
        opp_short = next(
            (s for s in scouted_opponents if re.search(re.escape(s), str(g_row["opponent"]), re.IGNORECASE)), None
        )
        if opp_short is None or glob.glob(f"{volume_dir}/{_game_date_str(g_date)}*{opp_short}*_video.*"):
            continue
        games_needing_live_uww_video.append((g_row, g_date, opp_short))

if games_needing_live_uww_video and fastscout_username and fastscout_password:
    for g_row, g_date, opp_short in games_needing_live_uww_video:
        try:
            if str(g_row.get("location", "")).strip().lower() == "home":
                matchup = f"{g_row['opponent']} @ UW-Whitewater"
            else:
                matchup = f"UW-Whitewater @ {g_row['opponent']}"
            video_save_path = f"{volume_dir}/{_game_date_str(g_date)} {matchup}_video.html"
            run_in_fastscout_session(
                lambda page, url=g_row["video_url"], sp=video_save_path: scrape_video_clips_live(page, url, save_path=sp)
            )
        except Exception as video_scrape_error:
            print(f"  Could not live-scrape UWW's own video clips for {g_row['opponent']} ({g_row['video_url']}): {type(video_scrape_error).__name__}: {video_scrape_error}")

    # Re-glob so the freshly-cached ".html" file(s) are picked up by the alignment loop below.
    video_files = sorted(glob.glob(f"{volume_dir}/*_video.mhtml") + glob.glob(f"{volume_dir}/*_video.html"))
elif games_needing_live_uww_video:
    print(
        f"{len(games_needing_live_uww_video)} scouted UWW game(s) are missing a local '_video' file and could "
        "be live-scraped, but no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were found -- skipping."
    )

print(f"Found {len(video_files)} video-tagging file(s):")
for f in video_files:
    print(" -", os.path.basename(f))

video_desc_col = pd.Series(index=pbp_events.index, dtype=object)
video_result_col = pd.Series(index=pbp_events.index, dtype=object)
video_player_col = pd.Series(index=pbp_events.index, dtype=object)
video_clip_number_col = pd.Series(index=pbp_events.index, dtype=float)

# CONFIRMED_CLIP_EVENT_OVERRIDES is reused from the "Shared video-tagging helper functions" cell above.
opponents_with_video = []
for path in video_files:
    # Home/away-aware extraction (mirrors opponent_from_pbp_filename above); matches ".mhtml" or ".html".
    _vname = re.sub(r"_video\.(mhtml|html)$", "", os.path.basename(path), flags=re.IGNORECASE)
    _vname = re.sub(r"^\d+_\d+_\d+\s+", "", _vname)
    if " @ " in _vname:
        _left, _right = [side.strip() for side in _vname.split(" @ ", 1)]
        opponent_short = _right if _left == "UW-Whitewater" else _left
    else:
        opponent_short = _vname
    opponents_with_video.append(opponent_short)
    # A video file covers ONE game. Scope the alignment to that game (by its own filename date), or a
    # rematch's clips get aligned against both meetings' events at once.
    video_game_date = game_date_from_pbp_filename(path.replace("_video.", "_pbp."))
    clips = parse_video_mhtml(path)
    total_clips_n = len(clips)

    game_mask = pbp_events["opponent"] == opponent_short
    if video_game_date is not None and (game_mask & (pbp_events["game_date"] == video_game_date)).any():
        game_mask &= pbp_events["game_date"] == video_game_date
    elif pbp_events.loc[game_mask, "game_date"].nunique() > 1:
        print(f"    WARNING: '{os.path.basename(path)}' has no usable date prefix but "
              f"{opponent_short} has {pbp_events.loc[game_mask, 'game_date'].nunique()} games -- "
              f"clips will be aligned across all of them.")
    opp_all_events = pbp_events[game_mask]
    player_team_lookup = opp_all_events.dropna(subset=["player"]).drop_duplicates("player").set_index("player")["team"]
    known_teams = set(player_team_lookup.unique())
    known_players = set(player_team_lookup.index)

    def normalize_player_name(name):
        if name in known_players:
            return name
        for p in known_players:
            if p.casefold() == str(name).casefold():
                return p
        return name

    def committing_team_for(fouled_player):
        fouled_team = player_team_lookup.get(fouled_player)
        others = known_teams - {fouled_team}
        return next(iter(others)) if len(others) == 1 else None

    subevent_rows = []
    for _, clip_row in clips.iterrows():
        clip_player = normalize_player_name(clip_row["player"])
        for event_type in expand_clip_to_subevents(clip_row):
            match_key = committing_team_for(clip_player) if event_type == "foul" else clip_player
            subevent_rows.append({
                "match_key": match_key, "event_type": event_type,
                "Description": clip_row["Description"], "video_result": clip_row["Result"],
                "video_clip_player": clip_player, "video_clip_number": clip_row["No."],
            })
    matchable_clips = pd.DataFrame(
        subevent_rows, columns=["match_key", "event_type", "Description", "video_result", "video_clip_player", "video_clip_number"]
    )

    target_event_types = {"made_shot", "missed_shot", "turnover", "foul"} | FREE_THROW_EVENT_TYPES
    opp_events = pbp_events[game_mask & pbp_events["event_type"].isin(target_event_types)].sort_values("event_order").copy()
    opp_events["match_event_type"] = opp_events["event_type"].where(
        ~opp_events["event_type"].isin(FREE_THROW_EVENT_TYPES), "free_throw"
    )
    opp_events["match_key"] = opp_events["team"].where(opp_events["event_type"] == "foul", opp_events["player"])
    total_pbp_n = pbp_events.loc[game_mask, "event_order"].max()

    matchable_clips = matchable_clips.sort_values("video_clip_number").reset_index(drop=True)
    opp_events = opp_events.sort_values("event_order")
    pbp_orig_index = opp_events.index.tolist()

    pbp_list = []
    for _, row in opp_events.iterrows():
        uww_set = set(row["uww_lineup"].split(", ")) if pd.notna(row["uww_lineup"]) else None
        opp_set = set(row["opp_lineup"].split(", ")) if pd.notna(row["opp_lineup"]) else None
        pbp_list.append({
            "event_order": row["event_order"], "match_event_type": row["match_event_type"],
            "match_key": row["match_key"], "uww_lineup_set": uww_set, "opp_lineup_set": opp_set,
        })
    video_list = matchable_clips.to_dict("records")
    n, m = len(pbp_list), len(video_list)

    def compatible(i, j):
        p, v = pbp_list[i], video_list[j]
        if p["match_event_type"] != v["event_type"] or p["match_key"] != v["match_key"]:
            return False
        if v["event_type"] == "foul":
            fouled_player = v["video_clip_player"]
            fouled_team = player_team_lookup.get(fouled_player)
            lineup_set = p["uww_lineup_set"] if fouled_team == "UW-Whitewater" else p["opp_lineup_set"]
            if lineup_set is not None and fouled_player not in lineup_set:
                return False
        return True

    def pos_cost(i, j):
        return abs(pbp_list[i]["event_order"] / total_pbp_n - video_list[j]["video_clip_number"] / total_clips_n)

    n_matched = 0
    n_ft_matched = 0
    n_ft_total = int((opp_events["match_event_type"] == "free_throw").sum())
    n_foul_matched = 0
    n_foul_total = int((opp_events["match_event_type"] == "foul").sum())
    pairs = global_align(n, m, compatible, pos_cost)

    pbp_order_to_i = {p["event_order"]: idx for idx, p in enumerate(pbp_list)}
    video_no_to_j = {v["video_clip_number"]: idx for idx, v in enumerate(video_list)}
    for (ov_opponent, ov_clip_no), ov_event_order in CONFIRMED_CLIP_EVENT_OVERRIDES.items():
        if ov_opponent != opponent_short or ov_clip_no not in video_no_to_j or ov_event_order not in pbp_order_to_i:
            continue
        override_i, override_j = pbp_order_to_i[ov_event_order], video_no_to_j[ov_clip_no]
        pairs = {i: j for i, j in pairs.items() if i != override_i and j != override_j}
        pairs[override_i] = override_j

    for i, j in pairs.items():
        pbp_idx = pbp_orig_index[i]
        clip = video_list[j]
        video_desc_col.loc[pbp_idx] = clip["Description"]
        video_result_col.loc[pbp_idx] = clip["video_result"]
        video_player_col.loc[pbp_idx] = clip["video_clip_player"]
        video_clip_number_col.loc[pbp_idx] = clip["video_clip_number"]
        n_matched += 1
        if pbp_list[i]["match_event_type"] == "free_throw":
            n_ft_matched += 1
        if pbp_list[i]["match_event_type"] == "foul":
            n_foul_matched += 1

    print(
        f"  {opponent_short}: matched {n_matched}/{len(opp_events)} made/missed-shot, turnover, free-throw & foul "
        f"pbp_events rows to a video clip description via a single global order-preserving alignment across the "
        f"whole game (of which {n_ft_matched}/{n_ft_total} are free throws, and {n_foul_matched}/{n_foul_total} "
        f"are fouls). video log had {len(matchable_clips)} taggable sub-events of these types."
    )

pbp_events["video_description"] = video_desc_col
pbp_events["video_result"] = video_result_col
pbp_events["video_clip_number"] = video_clip_number_col
pbp_events["video_player"] = video_player_col
print(
    pbp_events[["opponent", "event_order", "team", "player", "event_type", "raw_text","video_clip_number", "video_result", "video_player", "video_description"]]
    .head(20)
)

ATTEMPTED_EVENT_TYPES = {"made_shot", "missed_shot", "turnover", "foul"} | FREE_THROW_EVENT_TYPES
has_video_mask = pbp_events["opponent"].isin(opponents_with_video)
unmatched_mask = has_video_mask & pbp_events["event_type"].isin(ATTEMPTED_EVENT_TYPES) & pbp_events["video_description"].isna()
n_unattributed_turnover = (unmatched_mask & pbp_events["player"].isna()).sum()
unmatched = pbp_events[unmatched_mask & pbp_events["player"].notna()].sort_values(["opponent", "event_order"])
print(
    f"\n{len(unmatched)} still-unmatched row(s) among attempted event types across {len(opponents_with_video)} "
    f"game(s) with a video-tagging file ({n_unattributed_turnover} bare team-level turnover(s) with no player "
    f"excluded -- never matchable):"
)
for opponent_short in opponents_with_video:
    print(f"  {opponent_short}: {(unmatched['opponent'] == opponent_short).sum()} unmatched")
print(unmatched[["opponent", "event_order", "period", "time_remaining", "team", "player", "event_type", "raw_text"]])

matched_clip_numbers = set(video_clip_number_col.dropna().astype(int))
unmatched_clips = pd.DataFrame()
for path in video_files:
    # Home/away-aware extraction (mirrors opponent_from_pbp_filename above); matches ".mhtml" or ".html".
    _vname = re.sub(r"_video\.(mhtml|html)$", "", os.path.basename(path), flags=re.IGNORECASE)
    _vname = re.sub(r"^\d+_\d+_\d+\s+", "", _vname)
    if " @ " in _vname:
        _left, _right = [side.strip() for side in _vname.split(" @ ", 1)]
        opponent_short = _right if _left == "UW-Whitewater" else _left
    else:
        opponent_short = _vname
    clips = parse_video_mhtml(path)
    NEVER_MATCHED_RESULTS = {"Run Offense", "No Violation", "Kicked Ball"}
    unmatched_c = clips[
        ~clips["No."].astype(int).isin(matched_clip_numbers) & ~clips["Result"].isin(NEVER_MATCHED_RESULTS)
    ]
    if not unmatched_c.empty:
        unmatched_c = unmatched_c.copy()
        unmatched_c["opponent"] = opponent_short
        unmatched_clips = pd.concat([unmatched_clips, unmatched_c], ignore_index=True)
if unmatched_clips.empty:
    print("No unmatched clips found in the video-tagging files.")
else:
    print("\nUnmatched clips from the video-tagging file(s):")
    print(unmatched_clips[["opponent", "No.", "player", "Result", "Description", "Team", "Duration"]])

Found 74 video-tagging file(s):
 - 11_11_25 Monmouth (IL) Fighting Scots @ Eureka Red Devils_video.html
 - 11_11_25 St. Thomas (TX) Celts @ East Texas Baptist Tigers_video.html
 - 11_12_25 Hope Flying Dutchmen @ Elmhurst Bluejays_video.html
 - 11_14_25 Eureka Red Devils @ Washington-St. Louis Bears_video.html
 - 11_14_25 Loras Duhawks @ Blackburn Beavers_video.html
 - 11_14_25 UW-Oshkosh Titans @ Maryville (TN) Scots_video.html
 - 11_14_25 UW-Whitewater @ St. Thomas (TX) Celts_video.html
 - 11_15_25 UW-Oshkosh Titans @ Illinois Wesleyan Titans_video.html
 - 11_15_25 UW-Whitewater @ Eureka Red Devils_video.html
 - 11_15_25 Wisconsin Lutheran Warriors @ Elmhurst Bluejays_video.html
 - 11_19_25 Aurora Spartans @ UW-Whitewater_video.html
 - 11_19_25 MSOE Raiders @ Elmhurst Bluejays_video.html
 - 11_22_25 Elmhurst Bluejays @ Illinois College Blueboys_video.html
 - 11_22_25 Loras Duhawks @ North Central (IL) Cardinals_video.html
 - 11_22_25 UW-Oshkosh Titans @ Carroll (WI) Pioneers_video.htm

In [93]:
print(pbp_events[['period','time_remaining','event_order','uww_score','opp_score','team','player','event_type','raw_text','shot_type','shot_desc','video_clip_number','video_description','video_player','video_result']])

      period time_remaining  event_order  uww_score  opp_score  \
0       None           None            0        NaN        NaN   
1         H1     19:58 (H1)            1        0.0        0.0   
2         H1     19:58 (H1)            2        0.0        0.0   
3         H1     19:34 (H1)            3        0.0        2.0   
4         H1     19:22 (H1)            4        0.0        2.0   
...      ...            ...          ...        ...        ...   
13672     H2     00:10 (H2)          483       75.0       77.0   
13673     H2     00:10 (H2)          484       75.0       77.0   
13674     H2     00:00 (H2)          485       78.0       77.0   
13675     H2     00:00 (H2)          486       78.0       77.0   
13676     H2           None          487        NaN        NaN   

                        team           player         event_type  \
0                       None              NaN      period_marker   
1      St. Thomas (TX) Celts  Charles Gitonga      jump_ball_won   
2  


### Reconstruct and validate a per-game box score from play-by-play

Aggregates player-level PBP events into a real box score, then sanity-checks each game's reconstructed final score against the schedule's own recorded result.

In [95]:
# --- Reconstruct a real single-game box score from the play-by-play, and sanity-check it against the schedule's
# final scores ------------------------------------------------------------------------------------------------
TEAM_LEVEL_EVENT_TYPES_BOX = {"team_deadball_rebound_offensive", "team_deadball_rebound_defensive", "timeout"}
player_events = pbp_events[
    pbp_events["player"].notna() & (~pbp_events["event_type"].isin(TEAM_LEVEL_EVENT_TYPES_BOX))
].copy()

player_events["points"] = player_events.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else (1 if row["event_type"] == "free_throw_made" else 0),
    axis=1,
)
player_events["is_fgm"] = player_events["event_type"] == "made_shot"
player_events["is_fga"] = player_events["event_type"].isin(["made_shot", "missed_shot"])
player_events["is_3pm"] = player_events["is_fgm"] & (player_events["shot_type"] == "3")
player_events["is_3pa"] = player_events["is_fga"] & (player_events["shot_type"] == "3")
player_events["is_ftm"] = player_events["event_type"] == "free_throw_made"
player_events["is_fta"] = player_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
player_events["is_oreb"] = player_events["event_type"] == "rebound_offensive"
player_events["is_dreb"] = player_events["event_type"] == "rebound_defensive"
player_events["is_ast"] = player_events["event_type"] == "assist"
player_events["is_stl"] = player_events["event_type"] == "steal"
player_events["is_blk"] = player_events["event_type"] == "block"
player_events["is_to"] = player_events["event_type"] == "turnover"
player_events["is_pf"] = player_events["event_type"] == "foul"

pbp_box_score = player_events.groupby(["opponent", "game_date", "team", "player"]).agg(
    PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
    FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
    OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
    BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
).reset_index()

# CONFIRMED GAP (fixed here): a bare team-level turnover (e.g. "Turnover (Offensive Foul)" with no player
# attached -- see classify_event's dedicated pattern for this) is correctly excluded from player_events above
# via the player.notna() filter, since there's no player to attribute it to. But that also means it never
# showed up ANYWHERE in the box score -- not on a player's line, and not accounted for at all -- so the box
# score's total turnover count silently undercounted the real game total by however many of these occurred.
# Surfaced here as a synthetic "TEAM" row per (opponent, game_date, team) instead, so these are visible rather
# than silently dropped.
_team_level_turnovers = pbp_events[pbp_events["player"].isna() & (pbp_events["event_type"] == "turnover")]
if not _team_level_turnovers.empty:
    _team_to_rows = _team_level_turnovers.groupby(["opponent", "game_date", "team"]).size().reset_index(name="TO")
    _team_to_rows["player"] = "TEAM"
    for _stat_col in ["PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "OREB", "DREB", "AST", "STL", "BLK", "PF"]:
        _team_to_rows[_stat_col] = 0
    pbp_box_score = pd.concat([pbp_box_score, _team_to_rows], ignore_index=True)
    print(f"Added {len(_team_to_rows)} synthetic TEAM row(s) for {int(_team_to_rows['TO'].sum())} bare (unattributed) team-level turnover(s) that would otherwise be missing from the box score entirely.")

pbp_box_score["REB"] = pbp_box_score["OREB"] + pbp_box_score["DREB"]
pbp_box_score["FG%"] = (100 * pbp_box_score["FGM"] / pbp_box_score["FGA"]).round(1)
pbp_box_score["3P%"] = (100 * pbp_box_score["FG3M"] / pbp_box_score["FG3A"]).round(1)
pbp_box_score["FT%"] = (100 * pbp_box_score["FTM"] / pbp_box_score["FTA"]).round(1)

starters_by_game = {}
for (opponent, game_date), group in pbp_events.groupby(GAME_KEYS, dropna=False):
    first_row = group.sort_values("event_order").iloc[0]
    starters_by_game[(opponent, game_date)] = {
        "UW-Whitewater": set(first_row["uww_lineup"].split(", ")) if pd.notna(first_row["uww_lineup"]) else set(),
        opponent: set(first_row["opp_lineup"].split(", ")) if pd.notna(first_row["opp_lineup"]) else set(),
    }
pbp_box_score["started"] = pbp_box_score.apply(
    lambda row: row["player"] in starters_by_game.get((row["opponent"], row["game_date"]), {}).get(row["team"], set()), axis=1
)

pbp_team_totals = pbp_box_score.groupby(["opponent", "game_date", "team"])["PTS"].sum().reset_index()
print("Validating PBP-reconstructed final scores against the schedule:\n")
# Matched on BOTH opponent and date. A name-only match returns the first meeting for every rematch,
# so a merged game would silently "validate" against the wrong row (or appear to be off by exactly
# the other meeting's score) instead of being reported.
_n_ok = _n_bad = 0
for (opponent, game_date) in pbp_events[GAME_KEYS].dropna(subset=["opponent"]).drop_duplicates().itertuples(index=False):
    cand = schedule[schedule["opponent"].str.contains(re.escape(opponent), case=False, na=False)].copy()
    cand = cand[cand["date"].apply(parse_schedule_date) == game_date] if game_date is not None else cand
    if cand.empty:
        print(f"  {opponent} {game_date}: no matching schedule row found -- can't validate.")
        continue
    sched_row = cand.iloc[0]
    sel = (pbp_team_totals["opponent"] == opponent) & (pbp_team_totals["game_date"] == game_date)
    uww_pts = pbp_team_totals[sel & (pbp_team_totals["team"] == "UW-Whitewater")]["PTS"]
    opp_pts = pbp_team_totals[sel & (pbp_team_totals["team"] == opponent)]["PTS"]
    uww_val = int(uww_pts.iloc[0]) if not uww_pts.empty else None
    opp_val = int(opp_pts.iloc[0]) if not opp_pts.empty else None
    ok = (uww_val, opp_val) == (sched_row["team_score"], sched_row["opponent_score"])
    _n_ok, _n_bad = _n_ok + ok, _n_bad + (not ok)
    status = "OK" if ok else "MISMATCH -- check parsing for this game"
    print(f"  {opponent} {game_date}: PBP {uww_val}-{opp_val} vs. schedule "
          f"{sched_row['team_score']}-{sched_row['opponent_score']} [{status}]")
    for team_label in ["UW-Whitewater", opponent]:
        starters_list = sorted(starters_by_game.get((opponent, game_date), {}).get(team_label, set()))
        print(f"    {team_label} starters: {', '.join(starters_list) if starters_list else '(not detected)'}")
print(f"\n{_n_ok} game(s) reconcile exactly, {_n_bad} do not.")
_dupes = pbp_box_score.groupby(["opponent", "game_date", "team", "player"]).size()
_dupes = _dupes[_dupes > 1]
if len(_dupes):
    print(f"WARNING: {len(_dupes)} duplicated (opponent, game_date, team, player) key(s) -- two files for one game?")

print(pbp_box_score.sort_values(["opponent", "PTS"], ascending=[True, False])[[
    "opponent", "game_date", "team", "player", "started", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "FG%", "3P%", "FT%",
]])

# --- Cross-validate the PBP-reconstructed box score against the official per-game box-score snapshot ----------
box_files = sorted(glob.glob(f"{volume_dir}/*_box.mhtml"))
print(f"\nFound {len(box_files)} official box-score file(s) to cross-validate against:")
for f in box_files:
    print(" -", os.path.basename(f))

def parse_box_mhtml(path):
    html = load_mhtml_html(path)
    soup = BeautifulSoup(html, "lxml")
    tables = soup.find_all("table")
    score_df = pd.read_html(StringIO(str(tables[0])))[0]
    team_order = score_df.iloc[:, 2].tolist()
    frames = []
    for team_name, table in zip(team_order, tables[1:]):
        pdf = pd.read_html(StringIO(str(table)))[0]
        pdf["team"] = team_name
        frames.append(pdf)
    box_df = pd.concat(frames, ignore_index=True)
    box_df["started"] = box_df["PLAYER"].astype(str).str.endswith("*")
    box_df["player"] = box_df["PLAYER"].astype(str).str.rstrip("*").str.strip()
    box_df = box_df[~box_df["player"].str.contains("Team Total", case=False)]
    for pair_col, (m_col, a_col) in {"FGM-A": ("FGM", "FGA"), "3PM-A": ("FG3M", "FG3A"), "FTM-A": ("FTM", "FTA")}.items():
        split = box_df[pair_col].astype(str).replace("-", "0-0").str.split("-", n=1, expand=True)
        box_df[m_col] = pd.to_numeric(split[0], errors="coerce").fillna(0)
        box_df[a_col] = pd.to_numeric(split[1], errors="coerce").fillna(0)
    for c in ["PTS", "REB", "AST", "TO", "STL", "BLK", "PF"]:
        box_df[c] = pd.to_numeric(box_df[c], errors="coerce").fillna(0)
    return box_df[["team", "player", "started", "PTS", "REB", "AST", "TO", "STL", "BLK", "PF",
                   "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"]]

box_stat_cols = ["PTS", "REB", "AST", "TO", "STL", "BLK", "PF", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA"]
print("\nValidating PBP-reconstructed per-player box score against the official box-score file, where available:\n")
for path in box_files:
    m = re.search(r"@ (.+)_box\.mhtml$", os.path.basename(path))
    opponent_short = m.group(1) if m else os.path.basename(path)
    official_box = parse_box_mhtml(path)
    pbp_slice = pbp_box_score[pbp_box_score["opponent"] == opponent_short]
    if pbp_slice.empty:
        print(f"{opponent_short}: no PBP-reconstructed box score found for this opponent -- skipping.\n")
        continue

    merged = official_box.merge(pbp_slice, on=["team", "player"], how="outer", suffixes=("_official", "_pbp"), indicator=True)
    only_official = merged[merged["_merge"] == "left_only"]
    only_pbp = merged[merged["_merge"] == "right_only"]
    both = merged[merged["_merge"] == "both"]

    n_stat_mismatch = 0
    n_started_mismatch = 0
    for _, r in both.iterrows():
        stat_mismatches = [c for c in box_stat_cols if r[f"{c}_official"] != r[f"{c}_pbp"]]
        started_mismatch = r["started_official"] != r["started_pbp"]
        if stat_mismatches or started_mismatch:
            detail = []
            if stat_mismatches:
                detail.append("; ".join(f"{c} official={r[f'{c}_official']} vs pbp={r[f'{c}_pbp']}" for c in stat_mismatches))
            if started_mismatch:
                detail.append(f"started official={r['started_official']} vs pbp={r['started_pbp']}")
            print(f"  MISMATCH {opponent_short} {r['team']} {r['player']}: {'; '.join(detail)}")
            n_stat_mismatch += bool(stat_mismatches)
            n_started_mismatch += bool(started_mismatch)
    for _, r in only_official.iterrows():
        print(f"  {opponent_short} {r['team']} {r['player']}: in official box score but missing from PBP reconstruction")
    for _, r in only_pbp.iterrows():
        print(f"  {opponent_short} {r['team']} {r['player']}: in PBP reconstruction but missing from official box score")

    status = (
        "OK" if not (n_stat_mismatch or n_started_mismatch or len(only_official) or len(only_pbp))
        else f"{n_stat_mismatch} stat mismatch(es), {n_started_mismatch} started mismatch(es), "
             f"{len(only_official)} missing-from-PBP, {len(only_pbp)} extra-in-PBP"
    )
    print(f"{opponent_short}: {len(both)} player(s) compared against the official box score -- [{status}]\n")

Added 20 synthetic TEAM row(s) for 26 bare (unattributed) team-level turnover(s) that would otherwise be missing from the box score entirely.
Validating PBP-reconstructed final scores against the schedule:

  St. Thomas (TX) Celts 2025-11-14: PBP 73-81 vs. schedule 73.0-81.0 [OK]
    UW-Whitewater starters: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    St. Thomas (TX) Celts starters: Angel Johnson, Charles Gitonga, Corey Thompson, Nathan Kongolo, Nicholas Buffalo
  Eureka Red Devils 2025-11-15: PBP 116-73 vs. schedule 116.0-73.0 [OK]
    UW-Whitewater starters: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    Eureka Red Devils starters: Andrew Coker, Ben Carter, Colin DeLaere, Jaxson Provost, Micah Bruer
  Aurora Spartans 2025-11-19: PBP 82-73 vs. schedule 82.0-73.0 [OK]
    UW-Whitewater starters: Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara
    Aurora Spartans starters: Cullen Rauls, Devon Richardson, Jeffery Hillme


### Diagnostic: which played/scouted games are missing a box score, and why

Cross-references every game in `uww_team_schedule` (UWW's own scouted + played games -- the set the
Streamlit app expects a box score for) against `pbp_box_score`, and prints a specific reason for any gap:
no local `_pbp` file was found for that game, a live-scrape wasn't attempted (no FastScout credentials),
or the game's `opponent` value in `pbp_events` doesn't match how it's spelled in the schedule (the PBP
builder derives `opponent` from the PBP filename itself, via `opponent_from_pbp_filename()` -- a mismatch
there means the game's events exist in `pbp_events` but never get matched up with its schedule row anywhere
downstream, including in the exported CSVs the app reads from). Run this after the box-score cell above to
catch a silently-missing game before it shows up as a mystery in the app instead of here.

In [97]:
# --- Diagnostic: reconcile uww_team_schedule (what the app expects a box score for) against pbp_box_score
# (what actually got built) -- prints a specific reason for every gap instead of leaving it to be discovered
# later as a blank "Box Score" section in the Streamlit app.
if uww_team_schedule.empty:
    print("uww_team_schedule is empty -- nothing to reconcile.")
else:
    _pbp_opponents_found = set(pbp_events["opponent"].dropna().unique()) if not pbp_events.empty else set()
    _box_opponents_found = set(pbp_box_score["opponent"].dropna().unique()) if not pbp_box_score.empty else set()
    _has_creds = bool(fastscout_username and fastscout_password)

    _gaps = []
    for _, _g in uww_team_schedule.iterrows():
        if _g.get("Upcoming") == "Yes":
            continue  # hasn't been played yet -- not expected to have a box score
        _opp_full = str(_g.get("opponent", ""))
        _opp_short = next((s for s in scouted_opponents if re.search(re.escape(s), _opp_full, re.IGNORECASE)), None)
        if _opp_short is None:
            _gaps.append((_opp_full, "not in scouted_opponents -- no scout report was matched for this game at all"))
            continue
        if _opp_short in _box_opponents_found:
            continue  # has a box score -- nothing to report
        _g_date = parse_schedule_date(_g["date"]) if pd.notna(_g.get("date")) else None
        _local_pbp = glob.glob(f"{volume_dir}/*{_opp_short}*_pbp.*") if _g_date is None else glob.glob(
            f"{volume_dir}/{_g_date.month}_{_g_date.day}_{_g_date.strftime('%y')}*{_opp_short}*_pbp.*"
        )
        if _opp_short in _pbp_opponents_found:
            reason = (
                f"PBP events exist under opponent name mismatch -- check whether any of "
                f"{sorted(_pbp_opponents_found)} was actually meant to be '{_opp_short}' "
                f"(opponent_from_pbp_filename() derives this from the _pbp file's own filename)"
            )
        elif _local_pbp:
            reason = f"a local PBP file exists ({[os.path.basename(p) for p in _local_pbp]}) but produced no box score rows -- check its parsing for errors/UNCLASSIFIED events"
        elif not _has_creds:
            reason = "no local _pbp file found, and no FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD were set to attempt a live scrape"
        else:
            reason = "no local _pbp file found, and the live-scrape fallback either wasn't attempted or failed -- check the earlier cell's output for this opponent's name"
        _gaps.append((_opp_short, reason))

    if not _gaps:
        print("Every played, scouted game in uww_team_schedule has a box score in pbp_box_score. No gaps found.")
    else:
        print(f"{len(_gaps)} played game(s) are missing a box score:\n")
        for opp, reason in _gaps:
            print(f"  {opp}: {reason}")


Every played, scouted game in uww_team_schedule has a box score in pbp_box_score. No gaps found.



### Reconstruct a single-game box score by 5-man lineup

Same reconstruction as the cell above, but aggregated by the 5-man lineup on the floor (`uww_lineup`/`opp_lineup`) instead of by individual player -- validated the same way, against the schedule's final scores.

In [99]:
# --- Single-game box score aggregated by 5-MAN LINEUP instead of by individual player -----------------------
lineup_events = pbp_events[pbp_events["event_type"] != "period_marker"].copy()
lineup_events["lineup"] = lineup_events.apply(
    lambda row: row["uww_lineup"] if row["team"] == "UW-Whitewater" else row["opp_lineup"], axis=1
)

lineup_events["points"] = lineup_events.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else (1 if row["event_type"] == "free_throw_made" else 0),
    axis=1,
)
lineup_events["is_fgm"] = lineup_events["event_type"] == "made_shot"
lineup_events["is_fga"] = lineup_events["event_type"].isin(["made_shot", "missed_shot"])
lineup_events["is_3pm"] = lineup_events["is_fgm"] & (lineup_events["shot_type"] == "3")
lineup_events["is_3pa"] = lineup_events["is_fga"] & (lineup_events["shot_type"] == "3")
lineup_events["is_ftm"] = lineup_events["event_type"] == "free_throw_made"
lineup_events["is_fta"] = lineup_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
lineup_events["is_oreb"] = lineup_events["event_type"].isin(["rebound_offensive", "team_deadball_rebound_offensive"])
lineup_events["is_dreb"] = lineup_events["event_type"].isin(["rebound_defensive", "team_deadball_rebound_defensive"])
lineup_events["is_ast"] = lineup_events["event_type"] == "assist"
lineup_events["is_stl"] = lineup_events["event_type"] == "steal"
lineup_events["is_blk"] = lineup_events["event_type"] == "block"
lineup_events["is_to"] = lineup_events["event_type"] == "turnover"
lineup_events["is_pf"] = lineup_events["event_type"] == "foul"

lineup_box_score = lineup_events.dropna(subset=["lineup"]).groupby(["opponent", "game_date", "team", "lineup"]).agg(
    PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
    FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
    OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
    BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
).reset_index()
lineup_box_score["REB"] = lineup_box_score["OREB"] + lineup_box_score["DREB"]
lineup_box_score["FG%"] = (100 * lineup_box_score["FGM"] / lineup_box_score["FGA"]).round(1)
lineup_box_score["3P%"] = (100 * lineup_box_score["FG3M"] / lineup_box_score["FG3A"]).round(1)
lineup_box_score["FT%"] = (100 * lineup_box_score["FTM"] / lineup_box_score["FTA"]).round(1)

stint_source = pbp_events[pbp_events["event_type"] != "period_marker"].sort_values(GAME_KEYS + ["event_order"]).copy()
prev_uww_lineup = stint_source.groupby(GAME_KEYS, dropna=False)["uww_lineup"].shift(1)
prev_opp_lineup = stint_source.groupby(GAME_KEYS, dropna=False)["opp_lineup"].shift(1)
stint_changed = (stint_source["uww_lineup"] != prev_uww_lineup) | (stint_source["opp_lineup"] != prev_opp_lineup)
stint_source["stint_num"] = stint_changed.fillna(True).groupby([stint_source[k] for k in GAME_KEYS]).cumsum()
stint_source["prev_uww_score"] = stint_source.groupby(GAME_KEYS, dropna=False)["uww_score"].shift(1).fillna(0)
stint_source["prev_opp_score"] = stint_source.groupby(GAME_KEYS, dropna=False)["opp_score"].shift(1).fillna(0)
# The game clock never runs backwards inside a period, so take a running minimum before differencing.
# Without it, ANY out-of-order row makes (prev - now) positive on the way back down and .clip(lower=0)
# keeps that phantom elapsed time while discarding the compensating negative -- silently inventing
# minutes. Grouping on the game (not just the opponent) is what stops two meetings interleaving here.
stint_source["clock"] = stint_source.groupby(GAME_KEYS + ["period"], dropna=False)["time_remaining_seconds"].cummin()
stint_source["prev_time_remaining_seconds"] = stint_source.groupby(GAME_KEYS + ["period"], dropna=False)["clock"].shift(1)
stint_source["seconds_elapsed"] = (stint_source["prev_time_remaining_seconds"] - stint_source["clock"]).clip(lower=0).fillna(0)

stints_for_box = stint_source.groupby(GAME_KEYS + ["stint_num", "uww_lineup", "opp_lineup"]).agg(
    end_uww_score=("uww_score", "last"), end_opp_score=("opp_score", "last"),
    start_prev_uww_score=("prev_uww_score", "first"), start_prev_opp_score=("prev_opp_score", "first"),
    stint_seconds=("seconds_elapsed", "sum"),
).reset_index()
stints_for_box["uww_margin_change"] = (
    (stints_for_box["end_uww_score"] - stints_for_box["start_prev_uww_score"]) - (stints_for_box["end_opp_score"] - stints_for_box["start_prev_opp_score"])
)
stints_for_box["stint_minutes"] = (stints_for_box["stint_seconds"] / 60).round(2)

uww_minutes_margin = stints_for_box.groupby(GAME_KEYS + ["uww_lineup"]).agg(
    MIN=("stint_minutes", "sum"), **{"+/-": ("uww_margin_change", "sum")}
).reset_index().rename(columns={"uww_lineup": "lineup"})
uww_minutes_margin["team"] = "UW-Whitewater"

opp_minutes_margin = stints_for_box.groupby(GAME_KEYS + ["opp_lineup"]).agg(
    MIN=("stint_minutes", "sum"), **{"+/-": ("uww_margin_change", lambda s: -s.sum())}
).reset_index().rename(columns={"opp_lineup": "lineup"})
opp_minutes_margin["team"] = opp_minutes_margin["opponent"]

minutes_margin = pd.concat([uww_minutes_margin, opp_minutes_margin], ignore_index=True)
lineup_box_score = lineup_box_score.merge(minutes_margin, on=GAME_KEYS + ["team", "lineup"], how="left")

lineup_team_totals = lineup_box_score.groupby(GAME_KEYS + ["team"])["PTS"].sum().reset_index()
print("Validating lineup-level PTS totals against the schedule:\n")
for (opponent, game_date) in pbp_events[GAME_KEYS].dropna(subset=["opponent"]).drop_duplicates().itertuples(index=False):
    cand = schedule[schedule["opponent"].str.contains(re.escape(opponent), case=False, na=False)].copy()
    cand = cand[cand["date"].apply(parse_schedule_date) == game_date] if game_date is not None else cand
    if cand.empty:
        continue
    sched_row = cand.iloc[0]
    sel = (lineup_team_totals["opponent"] == opponent) & (lineup_team_totals["game_date"] == game_date)
    uww_pts = lineup_team_totals[sel & (lineup_team_totals["team"] == "UW-Whitewater")]["PTS"]
    opp_pts = lineup_team_totals[sel & (lineup_team_totals["team"] == opponent)]["PTS"]
    uww_val = int(uww_pts.iloc[0]) if not uww_pts.empty else None
    opp_val = int(opp_pts.iloc[0]) if not opp_pts.empty else None
    status = "OK" if (uww_val, opp_val) == (sched_row["team_score"], sched_row["opponent_score"]) else "MISMATCH -- check lineup attribution for this game"
    print(f"  {opponent} {game_date}: lineup box score {uww_val}-{opp_val} vs. schedule {sched_row['team_score']}-{sched_row['opponent_score']} [{status}]")

print(lineup_box_score.sort_values(["opponent", "team", "PTS"], ascending=[True, True, False])[[
    "opponent", "game_date", "team", "lineup", "MIN", "+/-", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "FG%", "3P%", "FT%",
]])

Validating lineup-level PTS totals against the schedule:

  St. Thomas (TX) Celts 2025-11-14: lineup box score 73-81 vs. schedule 73.0-81.0 [OK]
  Eureka Red Devils 2025-11-15: lineup box score 116-73 vs. schedule 116.0-73.0 [OK]
  Aurora Spartans 2025-11-19: lineup box score 82-73 vs. schedule 82.0-73.0 [OK]
  Simpson Storm 2025-11-25: lineup box score 100-78 vs. schedule 100.0-78.0 [OK]
  Ripon Red Hawks 2025-11-07: lineup box score 76-58 vs. schedule 76.0-58.0 [OK]
  Lawrence Vikings 2025-12-10: lineup box score 88-58 vs. schedule 88.0-58.0 [OK]
  Carroll (WI) Pioneers 2025-12-13: lineup box score 64-60 vs. schedule 64.0-60.0 [OK]
  Hope Flying Dutchmen 2025-12-19: lineup box score 95-92 vs. schedule 95.0-92.0 [OK]
  Alma Scots 2025-12-20: lineup box score 90-61 vs. schedule 90.0-61.0 [OK]
  Elmhurst Bluejays 2025-12-02: lineup box score 88-72 vs. schedule 88.0-72.0 [OK]
  Coe Kohawks 2025-12-30: lineup box score 73-86 vs. schedule 73.0-86.0 [OK]
  UW-River Falls Falcons 2026-01-10:


### Aggregate the upcoming opponent's 5-man lineup season box scores

Uses `pbp_events_upcoming` (the opponent's own games, before facing Whitewater) to build their season-aggregated 5-man lineup box scores -- their body of work reconstructed the same way as the cells above, just on the opponent's own games instead of UWW's.

In [101]:
# --- Season-aggregated 5-man lineup box scores for the UPCOMING OPPONENT -----------------------------------
# Uses pbp_events_upcoming to build the upcoming opponent's 5-man lineup season box scores -- their body of
# work BEFORE they face Whitewater, reconstructed the same way as the cells above but on the opponent's own games.
TEAM_LEVEL_UPCOMING = {"team_deadball_rebound_offensive", "team_deadball_rebound_defensive", "timeout"}

if pbp_events_upcoming.empty:
    upcoming_lineup_season = pd.DataFrame()
    print(f"No PBP data available for {upcoming_opponent_short} -- "
          "lineup season box scores will populate once PBP files are uploaded.")
else:
    real_up = pbp_events_upcoming[
        pbp_events_upcoming["player"].notna() & (~pbp_events_upcoming["event_type"].isin(TEAM_LEVEL_UPCOMING))
    ].sort_values("event_order")

    def _starting_lineup(team_events):
        first_seen = team_events.groupby("player").first()
        return set(first_seen[first_seen["event_type"] != "sub_in"].index)

    self_lineup_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)
    their_lineup_col = pd.Series(index=pbp_events_upcoming.index, dtype=object)

    # The upcoming opponent has the same problem on their own schedule -- e.g. three meetings with
    # one conference rival all carry the same `opponent` value.
    for (opp_name, opp_game_date), opp_rows in pbp_events_upcoming.groupby(GAME_KEYS, dropna=False):
        opp_real = real_up[(real_up["opponent"] == opp_name) & (real_up["game_date"] == opp_game_date)]
        period_starts = opp_real.dropna(subset=["period"]).groupby("period")["event_order"].min().sort_values()
        periods_in_order = period_starts.index.tolist()

        for team_label, target_col in [(upcoming_opponent_short, self_lineup_col), (opp_name, their_lineup_col)]:
            team_events = opp_real[opp_real["team"] == team_label]
            changes = []
            for period in periods_in_order:
                period_events = team_events[team_events["period"] == period].sort_values("event_order")
                starters = _starting_lineup(period_events)
                current = set(starters)
                changes.append((period_starts[period] - 1, frozenset(current)))
                sub_events = period_events[period_events["event_type"].isin(["sub_in", "sub_out"])].sort_values("event_order")
                for _, row in sub_events.iterrows():
                    if row["event_type"] == "sub_in":
                        current.add(row["player"])
                    else:
                        current.discard(row["player"])
                    if len(current) == 5:
                        changes.append((row["event_order"], frozenset(current)))

            if changes:
                changes_df = pd.DataFrame(changes, columns=["event_order", "lineup"]).sort_values("event_order")
                target_rows = opp_rows.sort_values("event_order")
                merged = pd.merge_asof(target_rows[["event_order"]], changes_df, on="event_order", direction="backward")
                lineup_strings = merged["lineup"].apply(lambda s: ", ".join(sorted(s)) if isinstance(s, frozenset) else None)
                lineup_strings.index = target_rows.index
                target_col.loc[target_rows.index] = lineup_strings

    pbp_up = pbp_events_upcoming.copy()
    pbp_up["self_lineup"] = self_lineup_col
    pbp_up["their_lineup"] = their_lineup_col

    stint_src = pbp_up[pbp_up["event_type"] != "period_marker"].sort_values(GAME_KEYS + ["event_order"]).copy()
    prev_self = stint_src.groupby(GAME_KEYS, dropna=False)["self_lineup"].shift(1)
    prev_their = stint_src.groupby(GAME_KEYS, dropna=False)["their_lineup"].shift(1)
    stint_changed = (stint_src["self_lineup"] != prev_self) | (stint_src["their_lineup"] != prev_their)
    stint_src["stint_num"] = stint_changed.fillna(True).groupby([stint_src[k] for k in GAME_KEYS]).cumsum()
    stint_src["prev_self_score"] = stint_src.groupby(GAME_KEYS, dropna=False)["uww_score"].shift(1).fillna(0)
    stint_src["prev_their_score"] = stint_src.groupby(GAME_KEYS, dropna=False)["opp_score"].shift(1).fillna(0)
    stint_src["clock"] = stint_src.groupby(GAME_KEYS + ["period"], dropna=False)["time_remaining_seconds"].cummin()
    stint_src["prev_time_remaining_seconds"] = stint_src.groupby(GAME_KEYS + ["period"], dropna=False)["clock"].shift(1)
    stint_src["seconds_elapsed"] = (stint_src["prev_time_remaining_seconds"] - stint_src["clock"]).clip(lower=0).fillna(0)

    stints = stint_src.groupby(GAME_KEYS + ["stint_num", "self_lineup"]).agg(
        end_self_score=("uww_score", "last"), end_their_score=("opp_score", "last"),
        start_prev_self_score=("prev_self_score", "first"), start_prev_their_score=("prev_their_score", "first"),
        stint_seconds=("seconds_elapsed", "sum"),
    ).reset_index()
    stints["margin_change"] = (
        (stints["end_self_score"] - stints["start_prev_self_score"]) -
        (stints["end_their_score"] - stints["start_prev_their_score"])
    )
    stints["stint_minutes"] = (stints["stint_seconds"] / 60).round(2)

    minutes_margin = stints.groupby(GAME_KEYS + ["self_lineup"]).agg(
        MIN=("stint_minutes", "sum"), **{"+/-": ("margin_change", "sum")}
    ).reset_index().rename(columns={"self_lineup": "lineup"})

    lu_events = pbp_up[
        (pbp_up["team"] == upcoming_opponent_short) & pbp_up["self_lineup"].notna()
    ].copy()
    lu_events["lineup"] = lu_events["self_lineup"]
    lu_events["points"] = lu_events.apply(
        lambda r: int(r["shot_type"]) if r["event_type"] == "made_shot" else (1 if r["event_type"] == "free_throw_made" else 0), axis=1)
    lu_events["is_fgm"] = lu_events["event_type"] == "made_shot"
    lu_events["is_fga"] = lu_events["event_type"].isin(["made_shot", "missed_shot"])
    lu_events["is_3pm"] = lu_events["is_fgm"] & (lu_events["shot_type"] == "3")
    lu_events["is_3pa"] = lu_events["is_fga"] & (lu_events["shot_type"] == "3")
    lu_events["is_ftm"] = lu_events["event_type"] == "free_throw_made"
    lu_events["is_fta"] = lu_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
    lu_events["is_oreb"] = lu_events["event_type"].isin(["rebound_offensive", "team_deadball_rebound_offensive"])
    lu_events["is_dreb"] = lu_events["event_type"].isin(["rebound_defensive", "team_deadball_rebound_defensive"])
    lu_events["is_ast"] = lu_events["event_type"] == "assist"
    lu_events["is_stl"] = lu_events["event_type"] == "steal"
    lu_events["is_blk"] = lu_events["event_type"] == "block"
    lu_events["is_to"] = lu_events["event_type"] == "turnover"
    lu_events["is_pf"] = lu_events["event_type"] == "foul"

    per_game_lineup = lu_events.groupby(GAME_KEYS + ["lineup"], as_index=False).agg(
        PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
        FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
        OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
        BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
    )
    per_game_lineup["REB"] = per_game_lineup["OREB"] + per_game_lineup["DREB"]
    per_game_lineup = per_game_lineup.merge(minutes_margin, on=GAME_KEYS + ["lineup"], how="left")

    upcoming_lineup_season = per_game_lineup.groupby("lineup").agg(
        GP=("game_date", "nunique"),
        MIN=("MIN", "sum"),
        **{"+/-": ("+/-", "sum")},
        PTS=("PTS", "sum"), FGM=("FGM", "sum"), FGA=("FGA", "sum"),
        FG3M=("FG3M", "sum"), FG3A=("FG3A", "sum"), FTM=("FTM", "sum"), FTA=("FTA", "sum"),
        OREB=("OREB", "sum"), DREB=("DREB", "sum"), REB=("REB", "sum"),
        AST=("AST", "sum"), STL=("STL", "sum"), BLK=("BLK", "sum"), TO=("TO", "sum"), PF=("PF", "sum"),
    ).reset_index()
    upcoming_lineup_season["FG%"] = (100 * upcoming_lineup_season["FGM"] / upcoming_lineup_season["FGA"]).round(1)
    upcoming_lineup_season["3P%"] = (100 * upcoming_lineup_season["FG3M"] / upcoming_lineup_season["FG3A"]).round(1)
    upcoming_lineup_season["FT%"] = (100 * upcoming_lineup_season["FTM"] / upcoming_lineup_season["FTA"]).round(1)
    upcoming_lineup_season["MIN"] = upcoming_lineup_season["MIN"].round(1)
    upcoming_lineup_season = upcoming_lineup_season.sort_values("MIN", ascending=False).reset_index(drop=True)

    print(f"{upcoming_opponent_short} season 5-man lineup box scores "
          f"({upcoming_lineup_season['GP'].max()} game(s) of PBP data, "
          f"{len(upcoming_lineup_season)} distinct units):")
    print(upcoming_lineup_season[[
        "lineup", "GP", "MIN", "+/-", "PTS", "FGM", "FGA", "FG%",
        "FG3M", "FG3A", "3P%", "FTM", "FTA", "FT%",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF",
    ]])

Loras Duhawks season 5-man lineup box scores (14 game(s) of PBP data, 344 distinct units):
                                                                              lineup  \
0           Damyen Jackson, Gavin Sarvis, Jack Haynes, Johnny Semany, Nolan Berendes   
1               Gavin Sarvis, Jack Haynes, Johnny Semany, Kyle Kober, Nolan Berendes   
2           Connor Mosele, Damyen Jackson, Gavin Sarvis, Jack Haynes, Nolan Berendes   
3                 Emmett Loew, Gavin Sarvis, Jack Haynes, Kyle Kober, Nolan Berendes   
4              Damyen Jackson, Gavin Sarvis, Jack Haynes, Kyle Kober, Nolan Berendes   
..                                                                               ...   
339  Brock Massey, Damyen Jackson, Johnny Semany, Nolan Berendes, Patrick Quarnstrom   
340             Athan Berchos, Dylan Kurt, Ethan Meyer, Juvon Crawford, Patrick Coen   
341      Athan Berchos, Ethan Meyer, Jim Navarrete, Patrick Coen, Patrick Quarnstrom   
342              Dylan Kurt, 


### Compute biggest scoring runs and largest lead/deficit, with lineups

Finds each game's biggest scoring run and largest lead/deficit for both teams, along with which 5-man lineup was on the floor for each.

In [103]:
# --- Scoring runs and largest lead/deficit per game, with the 5-man lineups on the floor for each ------------
scoring_events = pbp_events[pbp_events["event_type"].isin(["made_shot", "free_throw_made"])].copy()
scoring_events["points"] = scoring_events.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else 1, axis=1
)

def detailed_runs(group):
    """Same run-detection as before, but keeps each run's start/end event_order so the lineup on the floor can be looked up."""
    runs, cur_team, cur_pts, cur_start, cur_end = [], None, 0, None, None
    for _, row in group.sort_values("event_order").iterrows():
        if row["team"] == cur_team:
            cur_pts += row["points"]
            cur_end = row["event_order"]
        else:
            if cur_team is not None:
                runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
            cur_team, cur_pts, cur_start, cur_end = row["team"], row["points"], row["event_order"], row["event_order"]
    if cur_team is not None:
        runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
    return pd.DataFrame(runs)

def lineups_during(opponent, game_date, start_eo, end_eo):
    """Unique UWW/opponent lineups seen across [start_eo, end_eo] of ONE game -- normally just one of
    each, unless a sub happened mid-window. `game_date` is required: event_order restarts every game,
    so an opponent-only filter would sweep in the same event_order range from a rematch too."""
    window = pbp_events[
        (pbp_events["opponent"] == opponent) & (pbp_events["game_date"] == game_date)
        & (pbp_events["event_order"] >= start_eo) & (pbp_events["event_order"] <= end_eo)
    ]
    uww_l = window["uww_lineup"].dropna().unique().tolist()
    opp_l = window["opp_lineup"].dropna().unique().tolist()
    return (uww_l[0] if len(uww_l) == 1 else (" / ".join(uww_l) or None)), (opp_l[0] if len(opp_l) == 1 else (" / ".join(opp_l) or None))

run_rows = []
for (opponent, game_date), group in scoring_events.groupby(GAME_KEYS, dropna=False):
    runs_df = detailed_runs(group)
    biggest_by_team = runs_df.loc[runs_df.groupby("team")["run_points"].idxmax()].set_index("team") if not runs_df.empty else runs_df

    margins_full = pbp_events[(pbp_events["opponent"] == opponent) & (pbp_events["game_date"] == game_date)].dropna(subset=["uww_score"]).copy()
    margins_full["margin"] = margins_full["uww_score"] - margins_full["opp_score"]
    uww_lead_row = margins_full.loc[margins_full["margin"].idxmax()] if not margins_full.empty else None
    opp_lead_row = margins_full.loc[margins_full["margin"].idxmin()] if not margins_full.empty else None

    uww_run = biggest_by_team.loc["UW-Whitewater"] if "UW-Whitewater" in biggest_by_team.index else None
    opp_run = biggest_by_team.loc[opponent] if opponent in biggest_by_team.index else None
    uww_run_lineups = lineups_during(opponent, game_date, uww_run["start_event_order"], uww_run["end_event_order"]) if uww_run is not None else (None, None)
    opp_run_lineups = lineups_during(opponent, game_date, opp_run["start_event_order"], opp_run["end_event_order"]) if opp_run is not None else (None, None)

    run_rows.append({
        "opponent": opponent,
        "game_date": game_date,
        "uww_biggest_run": int(uww_run["run_points"]) if uww_run is not None else 0,
        "uww_run_uww_lineup": uww_run_lineups[0], "uww_run_opp_lineup": uww_run_lineups[1],
        "opponent_biggest_run": int(opp_run["run_points"]) if opp_run is not None else 0,
        "opp_run_uww_lineup": opp_run_lineups[0], "opp_run_opp_lineup": opp_run_lineups[1],
        "uww_largest_lead": int(uww_lead_row["margin"]) if uww_lead_row is not None else 0,
        "uww_lead_uww_lineup": uww_lead_row["uww_lineup"] if uww_lead_row is not None else None,
        "uww_lead_opp_lineup": uww_lead_row["opp_lineup"] if uww_lead_row is not None else None,
        "opponent_largest_lead": int(-opp_lead_row["margin"]) if opp_lead_row is not None else 0,
        "opp_lead_uww_lineup": opp_lead_row["uww_lineup"] if opp_lead_row is not None else None,
        "opp_lead_opp_lineup": opp_lead_row["opp_lineup"] if opp_lead_row is not None else None,
    })
scoring_runs = pd.DataFrame(run_rows)
print(scoring_runs)

print("Scoring run & lead summary, with the 5-man lineups on the floor for each:\n")
for _, row in scoring_runs.iterrows():
    print(f"{row['opponent']}: UWW's biggest run = {row['uww_biggest_run']} pts, {row['opponent']}'s biggest run = {row['opponent_biggest_run']} pts")
    print(f"    During UWW's run -- UWW: {row['uww_run_uww_lineup']} | {row['opponent']}: {row['uww_run_opp_lineup']}")
    print(f"    During {row['opponent']}'s run -- UWW: {row['opp_run_uww_lineup']} | {row['opponent']}: {row['opp_run_opp_lineup']}")
    print(f"    Largest UWW lead: {row['uww_largest_lead']} pts (UWW: {row['uww_lead_uww_lineup']} | {row['opponent']}: {row['uww_lead_opp_lineup']})")
    print(f"    Largest {row['opponent']} lead: {row['opponent_largest_lead']} pts (UWW: {row['opp_lead_uww_lineup']} | {row['opponent']}: {row['opp_lead_opp_lineup']})\n")

                     opponent   game_date  uww_biggest_run  \
0                  Alma Scots  2025-12-20                8   
1             Aurora Spartans  2025-11-19               13   
2       Carroll (WI) Pioneers  2025-12-13                7   
3                 Coe Kohawks  2025-12-30                7   
4           Elmhurst Bluejays  2025-12-02               10   
5           Eureka Red Devils  2025-11-15               12   
6        Hope Flying Dutchmen  2025-12-19                8   
7            Lawrence Vikings  2025-12-10                9   
8             Ripon Red Hawks  2025-11-07                7   
9               Simpson Storm  2025-11-25               15   
10      St. Thomas (TX) Celts  2025-11-14                6   
11     UW-Eau Claire Blugolds  2026-01-24                7   
12     UW-Eau Claire Blugolds  2026-02-14                8   
13        UW-La Crosse Eagles  2026-01-14                9   
14        UW-La Crosse Eagles  2026-02-04                9   
15      


### Surface clutch-time events with lineup context

Filters `pbp_events` down to the last 5 minutes of the 2nd half or any overtime with the score within 8 points, alongside the lineups on the floor for each event.

In [105]:
# --- Clutch-time event log: last 5 minutes of the 2nd half or any overtime, with the score within 8 points ----
clutch_mask = (
    (pbp_events["period"] != "H1")
    & (pbp_events["time_remaining_seconds"] <= 300)
    & ((pbp_events["uww_score"] - pbp_events["opp_score"]).abs() <= 8)
)
clutch_events = pbp_events[clutch_mask & pbp_events["event_type"].isin([
    "made_shot", "missed_shot", "free_throw_made", "free_throw_missed", "turnover", "steal", "foul", "block", "assist",
])].copy()

if clutch_events.empty:
    print("No clutch-time stretches yet -- no game so far has been within 8 points in the last 5 minutes of the "
          "2nd half or later. This will populate automatically as closer games are uploaded.")
else:
    print(clutch_events[[
        "opponent", "period", "time_remaining", "team", "player", "event_type", "raw_text", "uww_score", "opp_score",
        "uww_lineup", "opp_lineup",
    ]])
    clutch_points = clutch_events[clutch_events["event_type"].isin(["made_shot", "free_throw_made"])].copy()
    clutch_points["points"] = clutch_points.apply(
        lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else 1, axis=1
    )
    print("\nClutch-time points scored, by team:")
    print(clutch_points.groupby(GAME_KEYS + ["team"])["points"].sum().reset_index())

    print("\nUWW's 5-man lineup(s) used in clutch time, by game:")
    for (opponent, game_date), group in clutch_events.groupby(GAME_KEYS, dropna=False):
        print(f"  {opponent} {game_date}: {sorted(group['uww_lineup'].dropna().unique().tolist())}")

                    opponent period time_remaining                  team  \
442    St. Thomas (TX) Celts     H2     00:19 (H2)         UW-Whitewater   
1356         Aurora Spartans     H2     04:53 (H2)       Aurora Spartans   
1357         Aurora Spartans     H2     04:53 (H2)       Aurora Spartans   
1358         Aurora Spartans     H2     04:32 (H2)         UW-Whitewater   
1360         Aurora Spartans     H2     04:24 (H2)       Aurora Spartans   
...                      ...    ...            ...                   ...   
13664   UW-Stout Blue Devils     H2     00:10 (H2)         UW-Whitewater   
13665   UW-Stout Blue Devils     H2     00:10 (H2)  UW-Stout Blue Devils   
13666   UW-Stout Blue Devils     H2     00:10 (H2)  UW-Stout Blue Devils   
13674   UW-Stout Blue Devils     H2     00:00 (H2)         UW-Whitewater   
13675   UW-Stout Blue Devils     H2     00:00 (H2)         UW-Whitewater   

                player         event_type  \
442          Luke Bara          made_shot 


### Cross-reference actual PBP turnovers against the Keys-to-Victory splits, by lineup

Compares each game's actual PBP-derived turnover counts (and margin) against UWW's season averages and against whether turnovers were called out as a pre-game key, then breaks the win/loss record down by who actually won the turnover battle.

In [107]:
# --- Cross-reference ACTUAL PBP turnover counts against the Keys-to-Victory splits -----------------------
pbp_turnovers = (
    pbp_events[pbp_events["event_type"] == "turnover"]
    .groupby(["opponent", "team"])
    .size()
    .reset_index(name="turnovers")
)

uww_season_avg_to = float(uww_team_totals["TO"]) if uww_team_totals is not None else None
uww_season_avg_to_forced = float(uww_opp_totals["TO"]) if uww_opp_totals is not None else None

turnover_crossref_rows = []
for opponent in pbp_events["opponent"].dropna().unique():
    uww_to = pbp_turnovers[(pbp_turnovers["opponent"] == opponent) & (pbp_turnovers["team"] == "UW-Whitewater")]["turnovers"]
    opp_to = pbp_turnovers[(pbp_turnovers["opponent"] == opponent) & (pbp_turnovers["team"] == opponent)]["turnovers"]
    uww_to_val = int(uww_to.iloc[0]) if not uww_to.empty else 0
    opp_to_val = int(opp_to.iloc[0]) if not opp_to.empty else 0

    game_row = scouted_game_comparison[scouted_game_comparison["opponent"] == opponent]
    outcome = game_row["outcome"].iloc[0] if not game_row.empty else None
    was_key = "Ball Security / Turnovers" in game_categories.loc[game_categories["opponent"] == opponent, "category"].tolist()

    turnover_crossref_rows.append({
        "opponent": opponent, "outcome": outcome, "ball_security_was_a_key": was_key,
        "uww_actual_turnovers": uww_to_val, "uww_season_avg_turnovers": uww_season_avg_to,
        "uww_vs_season_avg": round(uww_to_val - uww_season_avg_to, 1) if uww_season_avg_to is not None else None,
        "opponent_actual_turnovers": opp_to_val, "uww_season_avg_turnovers_forced": uww_season_avg_to_forced,
        "turnover_margin_uww_favor": opp_to_val - uww_to_val,
    })

turnover_crossref = pd.DataFrame(turnover_crossref_rows)
print(turnover_crossref)

print("Actual PBP turnovers vs. the Keys-to-Victory 'Ball Security / Turnovers' emphasis:\n")
for _, row in turnover_crossref.iterrows():
    key_note = "WAS" if row["ball_security_was_a_key"] else "was NOT"
    print(f"{row['opponent']} ({row['outcome']}): ball security {key_note} called out as a pre-game key.")
    avg_to = row['uww_season_avg_turnovers']
    vs_avg = row['uww_vs_season_avg']
    avg_forced = row['uww_season_avg_turnovers_forced']
    avg_to_str = f"{avg_to:.1f}" if avg_to is not None else "N/A"
    vs_avg_str = f"{vs_avg:+.1f}" if vs_avg is not None else "N/A"
    avg_forced_str = f"{avg_forced:.1f}" if avg_forced is not None else "N/A"
    print(f"    UWW committed {row['uww_actual_turnovers']} turnovers this game vs. their {avg_to_str} season "
          f"average ({vs_avg_str}).")
    print(f"    {row['opponent']} committed {row['opponent_actual_turnovers']} turnovers (UWW forces "
          f"{avg_forced_str}/gm on average).")
    margin_desc = "in UWW's favor" if row["turnover_margin_uww_favor"] > 0 else ("against UWW" if row["turnover_margin_uww_favor"] < 0 else "even")
    print(f"    Turnover margin (opponent TOs minus UWW TOs): {row['turnover_margin_uww_favor']:+d} ({margin_desc})\n")

if len(turnover_crossref) > 1:
    pbp_turnover_split = (
        turnover_crossref.assign(won_turnover_battle=lambda d: d["turnover_margin_uww_favor"] > 0)
        .groupby("won_turnover_battle")["outcome"]
        .agg(games="count", wins=lambda s: (s == "W").sum(), losses=lambda s: (s == "L").sum())
        .reset_index()
    )
    pbp_turnover_split["win_pct"] = (pbp_turnover_split["wins"] / pbp_turnover_split["games"]).round(3)
    print("Win/loss record by who actually won the turnover battle (PBP-verified, once enough games have PBP data):")
    print(pbp_turnover_split)
else:
    print("Only 1 game has play-by-play data so far -- a real win/loss split by turnover-battle outcome will "
          "populate automatically once more games are uploaded.")

to_events = pbp_events[pbp_events["event_type"] == "turnover"]
uww_to_by_lineup = (
    to_events[to_events["team"] == "UW-Whitewater"]
    .groupby(["opponent", "uww_lineup"]).size().reset_index(name="turnovers")
    .sort_values("turnovers", ascending=False)
)
opp_to_by_lineup = (
    to_events[to_events["team"] != "UW-Whitewater"]
    .groupby(["opponent", "opp_lineup"]).size().reset_index(name="turnovers")
    .sort_values("turnovers", ascending=False)
)
print("\nUWW turnovers by 5-man lineup on the floor:")
print(uww_to_by_lineup)
print("\nOpponent turnovers by their 5-man lineup on the floor:")
print(opp_to_by_lineup)

                     opponent outcome  ball_security_was_a_key  \
0       St. Thomas (TX) Celts       L                     True   
1           Eureka Red Devils       W                    False   
2             Aurora Spartans       W                    False   
3               Simpson Storm       W                    False   
4             Ripon Red Hawks       W                    False   
5            Lawrence Vikings       W                    False   
6       Carroll (WI) Pioneers       W                    False   
7        Hope Flying Dutchmen    None                    False   
8                  Alma Scots       W                    False   
9           Elmhurst Bluejays       W                    False   
10                Coe Kohawks       L                    False   
11     UW-River Falls Falcons       W                    False   
12        UW-La Crosse Eagles       W                    False   
13       UW-Stout Blue Devils       W                    False   
14    UW-P


### Cross-reference actual PBP rebounding against the Keys-to-Victory splits, by lineup

Same cross-reference as the turnover cell above, but for rebounding -- actual PBP rebound totals/margin vs. season averages and the pre-game "Rebounding" key, plus the win/loss split by who won the rebound battle, and rebounds broken down by 5-man lineup.

In [109]:
# --- Cross-reference ACTUAL PBP rebounding totals against the Keys-to-Victory splits ----------------------
rebound_team_totals = pbp_box_score.groupby(["opponent", "team"])[["REB", "OREB", "DREB"]].sum().reset_index()

uww_season_avg_reb = {c: float(uww_team_totals[c]) for c in ["REB", "ORB", "DRB"]} if uww_team_totals is not None else None
uww_season_avg_reb_allowed = {c: float(uww_opp_totals[c]) for c in ["REB", "ORB", "DRB"]} if uww_opp_totals is not None else None

rebound_crossref_rows = []
for opponent in pbp_events["opponent"].dropna().unique():
    uww_row = rebound_team_totals[(rebound_team_totals["opponent"] == opponent) & (rebound_team_totals["team"] == "UW-Whitewater")]
    opp_row = rebound_team_totals[(rebound_team_totals["opponent"] == opponent) & (rebound_team_totals["team"] == opponent)]
    uww_reb = int(uww_row["REB"].iloc[0]) if not uww_row.empty else 0
    uww_oreb = int(uww_row["OREB"].iloc[0]) if not uww_row.empty else 0
    uww_dreb = int(uww_row["DREB"].iloc[0]) if not uww_row.empty else 0
    opp_reb = int(opp_row["REB"].iloc[0]) if not opp_row.empty else 0
    opp_oreb = int(opp_row["OREB"].iloc[0]) if not opp_row.empty else 0
    opp_dreb = int(opp_row["DREB"].iloc[0]) if not opp_row.empty else 0

    game_row = scouted_game_comparison[scouted_game_comparison["opponent"] == opponent]
    outcome = game_row["outcome"].iloc[0] if not game_row.empty else None
    was_key = "Rebounding" in game_categories.loc[game_categories["opponent"] == opponent, "category"].tolist()

    rebound_crossref_rows.append({
        "opponent": opponent, "outcome": outcome, "rebounding_was_a_key": was_key,
        "uww_actual_reb": uww_reb, "uww_season_avg_reb": uww_season_avg_reb["REB"] if uww_season_avg_reb else None,
        "uww_vs_season_avg_reb": round(uww_reb - uww_season_avg_reb["REB"], 1) if uww_season_avg_reb else None,
        "uww_actual_oreb": uww_oreb, "uww_actual_dreb": uww_dreb,
        "opponent_actual_reb": opp_reb, "uww_season_avg_reb_allowed": uww_season_avg_reb_allowed["REB"] if uww_season_avg_reb_allowed else None,
        "opponent_actual_oreb": opp_oreb, "opponent_actual_dreb": opp_dreb,
        "rebound_margin_uww_favor": uww_reb - opp_reb,
    })

rebound_crossref = pd.DataFrame(rebound_crossref_rows)
print(rebound_crossref)

print("Actual PBP rebounding vs. the Keys-to-Victory 'Rebounding' emphasis:\n")
for _, row in rebound_crossref.iterrows():
    key_note = "WAS" if row["rebounding_was_a_key"] else "was NOT"
    print(f"{row['opponent']} ({row['outcome']}): rebounding {key_note} called out as a pre-game key.")
    avg_reb = row['uww_season_avg_reb']
    vs_avg_reb = row['uww_vs_season_avg_reb']
    avg_allowed = row['uww_season_avg_reb_allowed']
    avg_reb_str = f"{avg_reb:.1f}" if avg_reb is not None else "N/A"
    vs_avg_reb_str = f"{vs_avg_reb:+.1f}" if vs_avg_reb is not None else "N/A"
    avg_allowed_str = f"{avg_allowed:.1f}" if avg_allowed is not None else "N/A"
    print(f"    UWW grabbed {row['uww_actual_reb']} rebounds ({row['uww_actual_oreb']} off. / {row['uww_actual_dreb']} def.) this game vs. "
          f"their {avg_reb_str} season average ({vs_avg_reb_str}).")
    print(f"    {row['opponent']} grabbed {row['opponent_actual_reb']} rebounds ({row['opponent_actual_oreb']} off. / "
          f"{row['opponent_actual_dreb']} def.) -- UWW allows {avg_allowed_str}/gm on average.")
    margin_desc = "in UWW's favor" if row["rebound_margin_uww_favor"] > 0 else ("against UWW" if row["rebound_margin_uww_favor"] < 0 else "even")
    print(f"    Rebound margin (UWW minus opponent): {row['rebound_margin_uww_favor']:+d} ({margin_desc})\n")

if len(rebound_crossref) > 1:
    pbp_rebound_split = (
        rebound_crossref.assign(won_rebound_battle=lambda d: d["rebound_margin_uww_favor"] > 0)
        .groupby("won_rebound_battle")["outcome"]
        .agg(games="count", wins=lambda s: (s == "W").sum(), losses=lambda s: (s == "L").sum())
        .reset_index()
    )
    pbp_rebound_split["win_pct"] = (pbp_rebound_split["wins"] / pbp_rebound_split["games"]).round(3)
    print("Win/loss record by who actually won the rebound battle (PBP-verified, once enough games have PBP data):")
    print(pbp_rebound_split)
else:
    print("Only 1 game has play-by-play data so far -- a real win/loss split by rebound-battle outcome will "
          "populate automatically once more games are uploaded.")

reb_events = pbp_events[pbp_events["event_type"].isin(["rebound_offensive", "rebound_defensive"])]
uww_reb_by_lineup = (
    reb_events[reb_events["team"] == "UW-Whitewater"]
    .groupby(["opponent", "uww_lineup"]).size().reset_index(name="rebounds")
    .sort_values("rebounds", ascending=False)
)
opp_reb_by_lineup = (
    reb_events[reb_events["team"] != "UW-Whitewater"]
    .groupby(["opponent", "opp_lineup"]).size().reset_index(name="rebounds")
    .sort_values("rebounds", ascending=False)
)
print("\nUWW rebounds by 5-man lineup on the floor:")
print(uww_reb_by_lineup)
print("\nOpponent rebounds by their 5-man lineup on the floor:")
print(opp_reb_by_lineup)

                     opponent outcome  rebounding_was_a_key  uww_actual_reb  \
0       St. Thomas (TX) Celts       L                  True              27   
1           Eureka Red Devils       W                  True              38   
2             Aurora Spartans       W                 False              30   
3               Simpson Storm       W                  True              32   
4             Ripon Red Hawks       W                  True              36   
5            Lawrence Vikings       W                 False              36   
6       Carroll (WI) Pioneers       W                 False              37   
7        Hope Flying Dutchmen    None                 False              34   
8                  Alma Scots       W                 False              27   
9           Elmhurst Bluejays       W                 False              29   
10                Coe Kohawks       L                 False              27   
11     UW-River Falls Falcons       W               


### Compute scoring margin and minutes played by 5-man lineup combination

Detects lineup-stint boundaries (any substitution for either team) across all of UWW's games, then aggregates minutes played and net scoring margin for each distinct lineup combination.

In [111]:
# --- Scoring margin and minutes played by 5-man lineup combination -----------------------------------------
pbp_scoreable = pbp_events[pbp_events["event_type"] != "period_marker"].sort_values(GAME_KEYS + ["event_order"]).copy()

# Every grouping here is on GAME_KEYS, not "opponent" -- see GAME_KEYS. `event_order` restarts at 0
# for each game, so sorting/grouping by opponent alone interleaves a home-and-home's events and the
# clock diff below re-counts the same seconds over and over (observed: 9x-33x inflated minutes).
prev_uww_lineup = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["uww_lineup"].shift(1)
prev_opp_lineup = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["opp_lineup"].shift(1)
lineup_changed = (pbp_scoreable["uww_lineup"] != prev_uww_lineup) | (pbp_scoreable["opp_lineup"] != prev_opp_lineup)
pbp_scoreable["stint_num"] = lineup_changed.fillna(True).groupby([pbp_scoreable[k] for k in GAME_KEYS]).cumsum()

pbp_scoreable["prev_uww_score"] = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["uww_score"].shift(1).fillna(0)
pbp_scoreable["prev_opp_score"] = pbp_scoreable.groupby(GAME_KEYS, dropna=False)["opp_score"].shift(1).fillna(0)

# Monotone clock, then diff -- see the matching comment in the lineup-box-score cell above.
pbp_scoreable["clock"] = pbp_scoreable.groupby(GAME_KEYS + ["period"], dropna=False)["time_remaining_seconds"].cummin()
pbp_scoreable["prev_time_remaining_seconds"] = pbp_scoreable.groupby(GAME_KEYS + ["period"], dropna=False)["clock"].shift(1)
pbp_scoreable["seconds_elapsed"] = (pbp_scoreable["prev_time_remaining_seconds"] - pbp_scoreable["clock"]).clip(lower=0).fillna(0)

lineup_stints = pbp_scoreable.groupby(GAME_KEYS + ["stint_num", "uww_lineup", "opp_lineup"]).agg(
    start_event_order=("event_order", "min"), end_event_order=("event_order", "max"),
    end_uww_score=("uww_score", "last"), end_opp_score=("opp_score", "last"),
    start_prev_uww_score=("prev_uww_score", "first"), start_prev_opp_score=("prev_opp_score", "first"),
    stint_seconds=("seconds_elapsed", "sum"), n_events=("event_order", "count"),
).reset_index()
lineup_stints["uww_margin_change"] = (
    (lineup_stints["end_uww_score"] - lineup_stints["start_prev_uww_score"])
    - (lineup_stints["end_opp_score"] - lineup_stints["start_prev_opp_score"])
)
lineup_stints["stint_minutes"] = (lineup_stints["stint_seconds"] / 60).round(2)
print(lineup_stints[[
    "opponent", "stint_num", "uww_lineup", "opp_lineup", "stint_minutes", "uww_margin_change", "n_events",
]].sort_values(["opponent", "stint_num"]))

# Per-game sanity check: five players x 40 minutes means each game must total about 200 minutes.
_clock_check = lineup_stints.groupby(GAME_KEYS)["stint_minutes"].sum().round(1)
_clock_bad = _clock_check[(_clock_check < 35) | (_clock_check > 65)]
if len(_clock_bad):
    print(f"WARNING: {len(_clock_bad)} game(s) have an implausible total elapsed clock "
          f"(expected ~40 min per game):\n{_clock_bad.to_string()}\n")
else:
    print(f"Clock check: all {len(_clock_check)} game(s) between 35 and 65 elapsed minutes.\n")

uww_lineup_summary = (
    lineup_stints.groupby(["opponent", "uww_lineup"])
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin=("uww_margin_change", "sum"))
    .reset_index()
)
uww_lineup_summary["margin_per_min"] = (uww_lineup_summary["net_margin"] / uww_lineup_summary["total_minutes"]).round(2)
uww_lineup_summary = uww_lineup_summary.sort_values("net_margin", ascending=False)

opp_lineup_summary = (
    lineup_stints.groupby(["opponent", "opp_lineup"])
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin_for_uww=("uww_margin_change", "sum"))
    .reset_index()
)
opp_lineup_summary["margin_per_min_for_uww"] = (opp_lineup_summary["net_margin_for_uww"] / opp_lineup_summary["total_minutes"]).round(2)
opp_lineup_summary = opp_lineup_summary.sort_values("net_margin_for_uww", ascending=True)

print("UWW's 5-man lineups, ranked by net scoring margin while on the floor:\n")
for _, row in uww_lineup_summary.iterrows():
    print(f"[{row['opponent']}] {row['uww_lineup']}")
    print(f"    {row['total_minutes']:.1f} min over {row['stints']} stint(s), net margin {row['net_margin']:+.0f} "
          f"({row['margin_per_min']:+.2f}/min)\n")
print(uww_lineup_summary)

print(f"\n{pbp_events['opponent'].dropna().unique().tolist()} lineups, ranked by how they fared AGAINST UWW (most negative net_margin_for_uww = best for them):\n")
for _, row in opp_lineup_summary.iterrows():
    print(f"[{row['opponent']}] {row['opp_lineup']}")
    print(f"    {row['total_minutes']:.1f} min over {row['stints']} stint(s), UWW's net margin vs. this lineup "
          f"{row['net_margin_for_uww']:+.0f} ({row['margin_per_min_for_uww']:+.2f}/min)\n")
print(opp_lineup_summary)

                  opponent  stint_num  \
0               Alma Scots          1   
1               Alma Scots          2   
2               Alma Scots          3   
3               Alma Scots          4   
4               Alma Scots          5   
...                    ...        ...   
1303  UW-Stout Blue Devils         44   
1258  UW-Stout Blue Devils         45   
1304  UW-Stout Blue Devils         45   
1259  UW-Stout Blue Devils         46   
1305  UW-Stout Blue Devils         46   

                                                                     uww_lineup  \
0           Brock Marino, Collin Madson, Isaac Verges, Luke Bara, Richie Warren   
1        Brock Marino, Collin Madson, Joey Berezowitz, Luke Bara, Richie Warren   
2        Brock Marino, Collin Madson, Joey Berezowitz, Luke Bara, Richie Warren   
3       Brock Marino, Collin Madson, Jake Quast, Joey Berezowitz, Kelton McEwen   
4       Brock Marino, Collin Madson, Jake Quast, Joey Berezowitz, Kelton McEwen   
...      


### Add minutes played onto the reconstructed box score

pbp_box_score (built above, before lineup_stints existed) aggregates scoring/rebounding/etc. events, none
of which map to a player's own on-court time by themselves -- so it never had a MIN column. lineup_stints
already reconstructs exactly that (via substitution tracking, the same 5-man units the Lineup Simulator and
Lineup Scouting features use), so minutes played per player per game is just those stint minutes summed up
for every stint that player's name appears in.

In [113]:
# --- Derive per-player minutes played from lineup_stints and merge onto pbp_box_score ------------------
_minutes_rows = []
for _, _stint in lineup_stints.iterrows():
    if pd.notna(_stint["uww_lineup"]):
        for _player in str(_stint["uww_lineup"]).split(", "):
            _minutes_rows.append({"opponent": _stint["opponent"], "game_date": _stint["game_date"], "team": "UW-Whitewater", "player": _player, "minutes": _stint["stint_minutes"]})
    if pd.notna(_stint["opp_lineup"]):
        for _player in str(_stint["opp_lineup"]).split(", "):
            _minutes_rows.append({"opponent": _stint["opponent"], "game_date": _stint["game_date"], "team": _stint["opponent"], "player": _player, "minutes": _stint["stint_minutes"]})

if _minutes_rows:
    player_minutes = pd.DataFrame(_minutes_rows).groupby(GAME_KEYS + ["team", "player"])["minutes"].sum().reset_index()
    player_minutes["MIN"] = player_minutes["minutes"].round(1)
    if "MIN" in pbp_box_score.columns:
        pbp_box_score = pbp_box_score.drop(columns=["MIN"])
    # Merged on the game, not just the opponent. Merging on (opponent, team, player) attached ONE
    # combined minutes figure to EVERY meeting's row against that opponent.
    pbp_box_score = pbp_box_score.merge(player_minutes[GAME_KEYS + ["team", "player", "MIN"]], on=GAME_KEYS + ["team", "player"], how="left")
    _n_matched = int(pbp_box_score["MIN"].notna().sum())
    print(f"Matched minutes played for {_n_matched} of {len(pbp_box_score)} pbp_box_score row(s).")
else:
    pbp_box_score["MIN"] = None
    print("No lineup stints available yet -- pbp_box_score.MIN left empty.")


Matched minutes played for 637 of 657 pbp_box_score row(s).



### Identify the lineups on court during each team's biggest scoring run

For each game's biggest scoring run (computed earlier), lists every lineup substitution either team made mid-run.

In [115]:
# --- Which lineup was on the floor for each team's biggest scoring run -------------------------------------
scoring_events_detail = pbp_events[pbp_events["event_type"].isin(["made_shot", "free_throw_made"])].copy()
scoring_events_detail["points"] = scoring_events_detail.apply(
    lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else 1, axis=1
)

def detailed_runs(group):
    runs, cur_team, cur_pts, cur_start, cur_end = [], None, 0, None, None
    for _, row in group.sort_values("event_order").iterrows():
        if row["team"] == cur_team:
            cur_pts += row["points"]
            cur_end = row["event_order"]
        else:
            if cur_team is not None:
                runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
            cur_team, cur_pts, cur_start, cur_end = row["team"], row["points"], row["event_order"], row["event_order"]
    if cur_team is not None:
        runs.append({"team": cur_team, "run_points": cur_pts, "start_event_order": cur_start, "end_event_order": cur_end})
    return pd.DataFrame(runs)

all_runs_detail = []
for (opponent, game_date), group in scoring_events_detail.groupby(GAME_KEYS, dropna=False):
    rd = detailed_runs(group)
    rd["opponent"] = opponent
    rd["game_date"] = game_date
    all_runs_detail.append(rd)
all_runs_detail = pd.concat(all_runs_detail, ignore_index=True)

biggest_runs_detail = (
    all_runs_detail.sort_values("run_points", ascending=False)
    .groupby(GAME_KEYS + ["team"], as_index=False)
    .first()
)

print("Lineups on the floor during each team's biggest scoring run:\n")
for _, run in biggest_runs_detail.iterrows():
    window = pbp_events[
        (pbp_events["opponent"] == run["opponent"])
        & (pbp_events["game_date"] == run["game_date"])
        & (pbp_events["event_order"] >= run["start_event_order"])
        & (pbp_events["event_order"] <= run["end_event_order"])
    ].sort_values("event_order")
    uww_lineups_during = window["uww_lineup"].dropna().unique().tolist()
    opp_lineups_during = window["opp_lineup"].dropna().unique().tolist()
    start_row, end_row = window.iloc[0], window.iloc[-1]

    print(f"[{run['opponent']}] {run['team']}'s biggest run: {run['run_points']} points "
          f"({start_row['period']} {start_row['time_remaining']} -> {end_row['period']} {end_row['time_remaining']})")
    if len(uww_lineups_during) == 1:
        print(f"    UWW lineup on the floor the whole run: {uww_lineups_during[0]}")
    else:
        print(f"    UWW made a substitution mid-run -- lineups on the floor: {uww_lineups_during}")
    if len(opp_lineups_during) == 1:
        print(f"    {run['opponent']} lineup on the floor the whole run: {opp_lineups_during[0]}")
    else:
        print(f"    {run['opponent']} made a substitution mid-run -- lineups on the floor: {opp_lineups_during}")
    print()

print(biggest_runs_detail)

Lineups on the floor during each team's biggest scoring run:

[Alma Scots] Alma Scots's biggest run: 8 points (H1 09:39 (H1) -> H1 07:28 (H1))
    UWW made a substitution mid-run -- lineups on the floor: ['Jake Quast, Joey Berezowitz, Kelton McEwen, Rashad Rogers, Tyshawn Teague-Johnson', 'Isaac Verges, Jake Quast, Joey Berezowitz, Rashad Rogers, Tyshawn Teague-Johnson', 'Collin Madson, Joey Berezowitz, Luke Bara, Richie Warren, Tyshawn Teague-Johnson']
    Alma Scots lineup on the floor the whole run: Donovan Collins, Elijah Sykes, Luciano Guerrazzi, Preston Malpass, Raine Rodich

[Alma Scots] UW-Whitewater's biggest run: 8 points (H1 18:44 (H1) -> H1 14:56 (H1))
    UWW made a substitution mid-run -- lineups on the floor: ['Brock Marino, Collin Madson, Isaac Verges, Luke Bara, Richie Warren', 'Brock Marino, Collin Madson, Joey Berezowitz, Luke Bara, Richie Warren']
    Alma Scots lineup on the floor the whole run: Donovan Collins, Josh Elliott, Korbin Heitzman, Logan St. Martin, Rain


### Season-wide 5-man lineup analysis across all games

Combines the lineup-stint data (minutes, margin) across every game played so far into one season-wide view of UWW's most-used and most-effective 5-man combinations.

In [117]:
# --- Season-wide UWW 5-man lineup analysis, combining minutes/margin across ALL games played so far -----------
season_uww_lineups = (
    lineup_stints.groupby("uww_lineup")
    .agg(
        games=("game_date", "nunique"),   # distinct GAMES, not distinct opponents -- a home-and-home is 2
        opponents=("opponent", lambda s: sorted(s.unique())),
        stints=("stint_num", "count"),
        total_minutes=("stint_minutes", "sum"),
        net_margin=("uww_margin_change", "sum"),
    )
    .reset_index()
)
season_uww_lineups["margin_per_min"] = (season_uww_lineups["net_margin"] / season_uww_lineups["total_minutes"]).round(2)
season_uww_lineups = season_uww_lineups.sort_values("total_minutes", ascending=False)

n_games_with_pbp = pbp_events.dropna(subset=["opponent"])[GAME_KEYS].drop_duplicates().shape[0]
print(f"Season-wide UWW 5-man lineups across {n_games_with_pbp} game(s) with play-by-play data "
      f"({season_uww_lineups.shape[0]} distinct lineups used):\n")
print("Most-used lineups overall (by total minutes on the floor):")
print(season_uww_lineups.head(10))

MEANINGFUL_MIN_MINUTES = 2.0
meaningful = season_uww_lineups[season_uww_lineups["total_minutes"] >= MEANINGFUL_MIN_MINUTES]
print(f"\nBest net-margin-per-minute UWW lineups season-wide (min {MEANINGFUL_MIN_MINUTES} minutes played):")
print(meaningful.sort_values("margin_per_min", ascending=False).head(10))
print(f"\nWorst net-margin-per-minute UWW lineups season-wide (min {MEANINGFUL_MIN_MINUTES} minutes played):")
print(meaningful.sort_values("margin_per_min", ascending=True).head(10))

recurring = season_uww_lineups[season_uww_lineups["games"] > 1].sort_values("games", ascending=False)
if recurring.empty:
    print("\nNo single 5-man lineup has repeated across multiple games yet -- each game so far has drawn from a "
          "distinct set of on-court combinations. This will populate as more games are added.")
else:
    print(f"\nLineups that have appeared in MORE than one game ({len(recurring)}):")
    print(recurring)

game_starting_lineups = (
    pbp_scoreable.sort_values(GAME_KEYS + ["event_order"])
    .groupby(GAME_KEYS, dropna=False)["uww_lineup"].first()
    .reset_index(name="starting_lineup")
)
print("\nUWW's starting (tip-off) lineup, by game:")
print(game_starting_lineups)
if game_starting_lineups["starting_lineup"].nunique() == 1:
    print("\nSame starting five has been used in EVERY game so far.")
else:
    print(f"\nStarting five has changed across games -- {game_starting_lineups['starting_lineup'].nunique()} "
          f"different starting combinations used over {n_games_with_pbp} game(s).")

lineups_per_game = lineup_stints.groupby(GAME_KEYS, dropna=False)["uww_lineup"].nunique().reset_index(name="distinct_lineups_used")
print("\nDistinct UWW 5-man combinations used, by game:")
print(lineups_per_game)

Season-wide UWW 5-man lineups across 27 game(s) with play-by-play data (281 distinct lineups used):

Most-used lineups overall (by total minutes on the floor):
                                                                  uww_lineup  \
74       Brock Marino, Collin Madson, Isaac Verges, Luke Bara, Richie Warren   
61        Brock Marino, Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara   
195     Collin Madson, Isaac Verges, Luke Bara, Rashad Rogers, Richie Warren   
184        Collin Madson, Isaac Verges, Jake Quast, Luke Bara, Richie Warren   
99    Brock Marino, Collin Madson, Joey Berezowitz, Luke Bara, Richie Warren   
168         Collin Madson, Isaac Verges, JR Lukenbill, Jake Quast, Luke Bara   
88       Brock Marino, Collin Madson, JR Lukenbill, Luke Bara, Richie Warren   
173      Collin Madson, Isaac Verges, JR Lukenbill, Luke Bara, Richie Warren   
64          Brock Marino, Collin Madson, Isaac Verges, Jake Quast, Luke Bara   
90   Brock Marino, Collin Madson, Jake Q


### UWW performance vs. opponent lineup profile and video-tagged play types, season-wide

Merges lineup stints with each opponent lineup's scouting-tag classification (Starter/Bench role mix) to see how UWW's net margin varies against different opponent lineup profiles, season-wide.

In [119]:
# --- UWW performance vs. OPPONENT lineup profile, and video-tagged PLAY TYPES -- both season-wide -------------
opp_lineup_class = pbp_events[GAME_KEYS + ["opp_lineup", "opp_lineup_summary"]].drop_duplicates()
stints_with_opp_class = lineup_stints.merge(opp_lineup_class, on=GAME_KEYS + ["opp_lineup"], how="left")

def opp_role_bucket(summary):
    if pd.isna(summary):
        return "Unknown (no scouting match)"
    m = re.search(r"(\d+) Starter", summary)
    if not m:
        return "Unknown (no scouting match)"
    starters = int(m.group(1))
    if starters >= 4:
        return "Starter-heavy (4-5 starters)"
    if starters <= 1:
        return "Bench-heavy (0-1 starters)"
    return "Mixed (2-3 starters)"

stints_with_opp_class["opp_role_bucket"] = stints_with_opp_class["opp_lineup_summary"].apply(opp_role_bucket)
role_bucket_summary = (
    stints_with_opp_class.groupby("opp_role_bucket")
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin_for_uww=("uww_margin_change", "sum"))
    .reset_index()
)
role_bucket_summary["margin_per_min"] = (role_bucket_summary["net_margin_for_uww"] / role_bucket_summary["total_minutes"]).round(2)
role_bucket_summary = role_bucket_summary.sort_values("total_minutes", ascending=False)
print("UWW's season-wide net margin by OPPONENT lineup role composition (Starter/Bench mix on the floor):\n")
print(role_bucket_summary)

def extract_style_tags(summary):
    if pd.isna(summary):
        return []
    m = re.search(r"Style: ([^|]+)", summary)
    if not m or "no tagged traits" in m.group(1):
        return []
    return [t.split(" x")[0].strip() for t in m.group(1).split(",")]

stints_with_opp_class["opp_style_tags"] = stints_with_opp_class["opp_lineup_summary"].apply(extract_style_tags)
style_summary = (
    stints_with_opp_class.explode("opp_style_tags").dropna(subset=["opp_style_tags"])
    .groupby("opp_style_tags")
    .agg(stints=("stint_num", "count"), total_minutes=("stint_minutes", "sum"), net_margin_for_uww=("uww_margin_change", "sum"))
    .reset_index()
)
style_summary["margin_per_min"] = (style_summary["net_margin_for_uww"] / style_summary["total_minutes"]).round(2)
style_summary = style_summary.sort_values("total_minutes", ascending=False)
print("\nUWW's season-wide net margin by the OPPONENT lineup's dominant scouted playing style (a lineup can "
      "count toward more than one tag):\n")
print(style_summary)

def extract_play_type(description, player):
    if pd.isna(description) or pd.isna(player):
        return None
    segments = [s.strip() for s in description.split(" > ")]
    player_norm = normalize_player_name(player)
    last_player_idx = None
    for idx, seg in enumerate(segments):
        m = re.match(r"^\d+\s+(.+)$", seg)
        if m and normalize_player_name(m.group(1)) == player_norm:
            last_player_idx = idx
    if last_player_idx is not None and last_player_idx + 1 < len(segments):
        return segments[last_player_idx + 1]
    return segments[1] if len(segments) > 1 else None

uww_shot_rows = pbp_events[
    (pbp_events["team"] == "UW-Whitewater")
    & pbp_events["event_type"].isin(["made_shot", "missed_shot"])
    & pbp_events["video_description"].notna()
].copy()
uww_shot_rows["play_type"] = uww_shot_rows.apply(lambda r: extract_play_type(r["video_description"], r["player"]), axis=1)
uww_shot_rows["made"] = uww_shot_rows["event_type"] == "made_shot"

play_type_summary = (
    uww_shot_rows.groupby("play_type")
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
play_type_summary["fg_pct"] = (100 * play_type_summary["makes"] / play_type_summary["attempts"]).round(1)
play_type_summary = play_type_summary.sort_values("attempts", ascending=False)
print(f"\nUWW's video-tagged shot-attempt play types, season-wide ({len(uww_shot_rows)} video-matched attempts "
      f"across {uww_shot_rows['opponent'].nunique()} game(s)):\n")
print(play_type_summary)

play_type_by_lineup = (
    uww_shot_rows.groupby(["play_type", "uww_lineup"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
play_type_by_lineup["fg_pct"] = (100 * play_type_by_lineup["makes"] / play_type_by_lineup["attempts"]).round(1)
top_play_types = play_type_summary.head(5)["play_type"].tolist()
print(f"\nFor the top {len(top_play_types)} most-attempted play types, which UWW lineup ran them most:\n")
print(
    play_type_by_lineup[play_type_by_lineup["play_type"].isin(top_play_types)]
    .sort_values(["play_type", "attempts"], ascending=[True, False])
)

play_type_by_player = (
    uww_shot_rows.groupby(["player", "play_type"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
play_type_by_player["fg_pct"] = (100 * play_type_by_player["makes"] / play_type_by_player["attempts"]).round(1)
print("\nUWW's video-tagged play-type efficiency by individual player, season-wide (every player/play-type "
      "combination with at least one attempt):\n")
print(play_type_by_player.sort_values(["player", "attempts"], ascending=[True, False]))

player_top_play_type = (
    play_type_by_player.sort_values("attempts", ascending=False)
    .groupby("player", as_index=False)
    .first()
    .rename(columns={"play_type": "most_used_play_type", "attempts": "attempts_of_that_type", "makes": "makes_of_that_type"})
)
player_overall = (
    uww_shot_rows.groupby("player")
    .agg(total_attempts=("made", "count"), total_makes=("made", "sum"))
    .reset_index()
)
player_overall["overall_fg_pct"] = (100 * player_overall["total_makes"] / player_overall["total_attempts"]).round(1)
player_summary = player_overall.merge(player_top_play_type, on="player", how="left").sort_values(
    "total_attempts", ascending=False
)
print("\nPer-player summary -- overall video-tagged volume/efficiency plus each player's single most-used play "
      "type, season-wide:\n")
print(player_summary[[
    "player", "total_attempts", "total_makes", "overall_fg_pct",
    "most_used_play_type", "attempts_of_that_type", "makes_of_that_type",
]])

starter_flags = pbp_box_score[pbp_box_score["team"] == "UW-Whitewater"][["opponent", "player", "started"]].drop_duplicates()
uww_shot_rows_roles = uww_shot_rows.merge(starter_flags, on=["opponent", "player"], how="left")
uww_shot_rows_roles["role"] = uww_shot_rows_roles["started"].map({True: "Starter", False: "Bench"}).fillna("Unknown")
n_unknown_role = (uww_shot_rows_roles["role"] == "Unknown").sum()
if n_unknown_role:
    print(f"NOTE: {n_unknown_role} attempt(s) could not be matched to a started/bench flag in pbp_box_score "
          f"and are grouped as 'Unknown' below.")

role_overall = (
    uww_shot_rows_roles.groupby("role")
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
role_overall["fg_pct"] = (100 * role_overall["makes"] / role_overall["attempts"]).round(1)
print("\nStarters vs. Bench: overall video-tagged shooting, season-wide:\n")
print(role_overall)

role_by_play_type = (
    uww_shot_rows_roles.groupby(["role", "play_type"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
role_by_play_type["fg_pct"] = (100 * role_by_play_type["makes"] / role_by_play_type["attempts"]).round(1)
role_by_play_type = role_by_play_type.sort_values(["role", "attempts"], ascending=[True, False])
print("\nStarters vs. Bench shooting, broken down by play type, season-wide:\n")
print(role_by_play_type)

UWW's season-wide net margin by OPPONENT lineup role composition (Starter/Bench mix on the floor):

                opp_role_bucket  stints  total_minutes  net_margin_for_uww  \
2  Starter-heavy (4-5 starters)     492         522.13                67.0   
1          Mixed (2-3 starters)     598         413.32                82.0   
3   Unknown (no scouting match)     152          99.08                36.0   
0    Bench-heavy (0-1 starters)      64          39.06                29.0   

   margin_per_min  
2            0.13  
1            0.20  
3            0.36  
0            0.74  

UWW's season-wide net margin by the OPPONENT lineup's dominant scouted playing style (a lineup can count toward more than one tag):

        opp_style_tags  stints  total_minutes  net_margin_for_uww  \
8  three_point_shooter    1166         959.24               213.0   
7       slasher_driver    1102         919.34               172.0   
4          post_scorer     470         413.32                79.0   


### Diagnose Spot-Up struggles: shot quality vs. shooter-specific

Breaks UWW's season-wide Spot-Up shooting down by shot mechanic (catch-and-shoot vs. pull-up vs. drive) and by contest level (guarded vs. open), to see whether struggles are about shot quality/shot selection rather than any one shooter.

In [121]:
# --- Diagnosing UWW's Spot-Up struggles: is it shot QUALITY (contested/long attempts) or SHOOTER-specific? ----
# The tagger's own vocabulary doesn't name every shot; these two labels mark where it runs out. Kept as
# named constants so the "best shot type" logic can recognise a residual bucket instead of presenting it as
# a real shot type.
UNCLASSIFIED_SHOT_MECHANIC = "Unclassified (no mechanic tag)"
NO_CONTEST_TAG = "Not tagged (contest recorded only on catch-and-shoot)"


def extract_shot_mechanic(description):
    """Which kind of shot this was, from the video tagger's own chained description.

    The first three tests are the tagger's SHOT MECHANIC vocabulary. Everything else used to fall through to
    a bucket called "Other" -- which was 19% of all tagged shots and, at 58.9%, the most efficient bucket on
    the board, so it kept winning "best shot type" while telling a coach nothing. Reading the raw tags, it was
    cuts to the rim, putbacks off the offensive glass, and post-ups.

    Those three are tested AFTER the mechanic tests, not before: they describe how a shot was CREATED rather
    than how it was released, and a post-up that finishes as a jumper should still count as a jumper. Checked
    against real tagged data -- "Cut" and "Offensive Rebound" appear in zero already-classified shots, and
    "Post-Up" in 169, all of which keep their existing (more specific) label under this ordering. Adding the
    tier shrinks the residual from 586 shots to 14.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "No Dribble Jumper" in d:
        return "Catch-and-shoot"
    if "Dribble Jumper" in d:
        return "Pull-up off the dribble"
    if "To Basket" in d:
        return "Drive to the basket"
    # --- fallback tier: shot ORIGIN, for tags carrying no mechanic keyword at all ---
    if "Offensive Rebound" in d:
        return "Putback off the offensive glass"
    if "Cut" in d:
        return "Cut to the basket"
    if "Post-Up" in d:
        return "Post-up"
    return UNCLASSIFIED_SHOT_MECHANIC


def extract_contest(description):
    """Defender contest, which the tagger records ONLY on catch-and-shoot jumpers.

    Verified across 3,039 tagged shots: every one of the 1,069 catch-and-shoot attempts carries Guarded or
    Open, and not one of the other 1,970 does. So a missing contest tag does not mean the shot was a drive --
    the previous label said "(drive, no contest tag)", which mislabelled every cut, putback and post-up as a
    drive. It means the contest dimension simply does not apply to this shot type.
    """
    if pd.isna(description):
        return None
    d = str(description)
    if "Guarded" in d:
        return "Guarded"
    if "Open" in d:
        return "Open"
    return NO_CONTEST_TAG

def extract_distance(description):
    if pd.isna(description):
        return None
    for tag in ["Long/3pt", "Medium/17' to <3p", "Short to < 17'"]:
        if tag in description:
            return tag
    return "N/A"

spotup = uww_shot_rows[uww_shot_rows["play_type"] == "Spot-Up"].copy()
spotup["shot_mechanic"] = spotup["video_description"].apply(extract_shot_mechanic)
spotup["contest"] = spotup["video_description"].apply(extract_contest)
spotup["distance"] = spotup["video_description"].apply(extract_distance)

print(f"Spot-Up shot-quality breakdown, season-wide ({len(spotup)} video-matched attempts):\n")

mechanic_summary = spotup.groupby("shot_mechanic").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
mechanic_summary["fg_pct"] = (100 * mechanic_summary["makes"] / mechanic_summary["attempts"]).round(1)
print("By shot mechanic:")
print(mechanic_summary.sort_values("attempts", ascending=False))

contest_summary = spotup.groupby("contest").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
contest_summary["fg_pct"] = (100 * contest_summary["makes"] / contest_summary["attempts"]).round(1)
print("\nBy contest level:")
print(contest_summary.sort_values("attempts", ascending=False))

distance_summary = spotup.groupby("distance").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
distance_summary["fg_pct"] = (100 * distance_summary["makes"] / distance_summary["attempts"]).round(1)
print("\nBy shot distance:")
print(distance_summary.sort_values("attempts", ascending=False))

catch_shoot = spotup[spotup["shot_mechanic"] == "Catch-and-shoot"]
cs_combo = catch_shoot.groupby(["contest", "distance"]).agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
cs_combo["fg_pct"] = (100 * cs_combo["makes"] / cs_combo["attempts"]).round(1)
print(f"\nCatch-and-shoot Spot-Up jumpers only ({len(catch_shoot)} attempts) -- contest x distance:")
print(cs_combo.sort_values("attempts", ascending=False))

player_spotup = spotup.groupby("player").agg(
    attempts=("made", "count"), makes=("made", "sum"),
    pct_catch_and_shoot=("shot_mechanic", lambda s: round(100 * (s == "Catch-and-shoot").mean(), 1)),
    pct_guarded=("contest", lambda s: round(100 * (s == "Guarded").mean(), 1)),
).reset_index()
player_spotup["fg_pct"] = (100 * player_spotup["makes"] / player_spotup["attempts"]).round(1)
player_spotup = player_spotup.sort_values("attempts", ascending=False)
print("\nPer-player Spot-Up volume/efficiency, with their catch-and-shoot% and guarded% of those attempts:")
print(player_spotup[["player", "attempts", "makes", "fg_pct", "pct_catch_and_shoot", "pct_guarded"]])

Spot-Up shot-quality breakdown, season-wide (411 video-matched attempts):

By shot mechanic:
             shot_mechanic  attempts  makes  fg_pct
0          Catch-and-shoot       253     94    37.2
1      Drive to the basket       106     48    45.3
2  Pull-up off the dribble        52     16    30.8

By contest level:
                                                 contest  attempts  makes  \
1  Not tagged (contest recorded only on catch-and-shoot)       158     64   
0                                                Guarded       147     54   
2                                                   Open       106     40   

   fg_pct  
1    40.5  
0    36.7  
2    37.7  

By shot distance:
            distance  attempts  makes  fg_pct
0           Long/3pt       265     99    37.4
2                N/A       106     48    45.3
3     Short to < 17'        23      5    21.7
1  Medium/17' to <3p        17      6    35.3

Catch-and-shoot Spot-Up jumpers only (253 attempts) -- contest x distance


### Check contest/mechanic patterns across ALL play types, not just Spot-Up

Extends the same shot-mechanic/contest-level breakdown from the cell above beyond Spot-Up, to every play type -- checking whether the same pattern holds more broadly or is specific to Spot-Up situations.

In [123]:
# --- Does the guarded/open contest pattern hold for catch-and-shoot jumpers on NON-SPOT-UP play types too? ----
uww_shot_rows["shot_mechanic"] = uww_shot_rows["video_description"].apply(extract_shot_mechanic)
uww_shot_rows["contest"] = uww_shot_rows["video_description"].apply(extract_contest)
uww_shot_rows["distance"] = uww_shot_rows["video_description"].apply(extract_distance)

catch_and_shoot_all = uww_shot_rows[uww_shot_rows["shot_mechanic"] == "Catch-and-shoot"]
print(f"ALL catch-and-shoot jumpers, every play type, season-wide ({len(catch_and_shoot_all)} attempts):\n")
by_play_type_mechanic = (
    catch_and_shoot_all.groupby("play_type")
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
by_play_type_mechanic["fg_pct"] = (100 * by_play_type_mechanic["makes"] / by_play_type_mechanic["attempts"]).round(1)
print("Which play types produce catch-and-shoot jumpers, and their efficiency:")
print(by_play_type_mechanic.sort_values("attempts", ascending=False))

contest_all = catch_and_shoot_all.groupby("contest").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
contest_all["fg_pct"] = (100 * contest_all["makes"] / contest_all["attempts"]).round(1)
print("\nGuarded vs. Open, ALL catch-and-shoot jumpers (every play type combined), season-wide:")
print(contest_all.sort_values("attempts", ascending=False))

guarded_all = (
    catch_and_shoot_all[catch_and_shoot_all["contest"] == "Guarded"].groupby("player")
    .agg(guarded_attempts=("made", "count"), guarded_makes=("made", "sum"))
    .reset_index()
)
guarded_all["guarded_fg_pct"] = (100 * guarded_all["guarded_makes"] / guarded_all["guarded_attempts"]).round(1)
open_all = (
    catch_and_shoot_all[catch_and_shoot_all["contest"] == "Open"].groupby("player")
    .agg(open_attempts=("made", "count"), open_makes=("made", "sum"))
    .reset_index()
)
open_all["open_fg_pct"] = (100 * open_all["open_makes"] / open_all["open_attempts"]).round(1)
compare_all = guarded_all.merge(open_all, on="player", how="outer")
compare_all["total_attempts"] = compare_all[["guarded_attempts", "open_attempts"]].sum(axis=1, skipna=True)
compare_all = compare_all.sort_values("total_attempts", ascending=False)
print("\nGuarded vs. Open catch-and-shoot jumpers, by player, ALL play types combined, season-wide:")
print(compare_all[[
    "player", "guarded_attempts", "guarded_makes", "guarded_fg_pct",
    "open_attempts", "open_makes", "open_fg_pct", "total_attempts",
]])

non_jumper = uww_shot_rows[uww_shot_rows["shot_mechanic"] != "Catch-and-shoot"]
non_jumper_summary = (
    non_jumper.groupby(["play_type", "shot_mechanic"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
non_jumper_summary["fg_pct"] = (100 * non_jumper_summary["makes"] / non_jumper_summary["attempts"]).round(1)
print("\nNon-catch-and-shoot attempts (drives, pull-ups, post moves, etc.), by play type and mechanic:")
print(non_jumper_summary.sort_values("attempts", ascending=False))

ALL catch-and-shoot jumpers, every play type, season-wide (415 attempts):

Which play types produce catch-and-shoot jumpers, and their efficiency:
          play_type  attempts  makes  fg_pct
7           Spot-Up       253     94    37.2
3        Off Screen        53     20    37.7
8        Transition        49     20    40.8
5      P&R Roll Man        21      7    33.3
4  P&R Ball Handler        18      4    22.2
0          Hand Off        10      2    20.0
1               ISO         5      2    40.0
2      No Play Type         3      1    33.3
6           Post-Up         3      1    33.3

Guarded vs. Open, ALL catch-and-shoot jumpers (every play type combined), season-wide:
   contest  attempts  makes  fg_pct
0  Guarded       236     79    33.5
1     Open       179     72    40.2

Guarded vs. Open catch-and-shoot jumpers, by player, ALL play types combined, season-wide:
                    player  guarded_attempts  guarded_makes  guarded_fg_pct  \
11               Luke Bara          


### Contest-level (guarded vs. open) Spot-Up breakdown for all shooters

Same guarded-vs-open contest breakdown as the flagged-player diagnosis, but run across every shooter on the roster rather than just the one player originally flagged.

In [125]:
# --- Same contest-level breakdown as the flagged-player note above, generalized to EVERY shooter with Spot-Up
# volume -- so the pattern (missing mostly on guarded/contested looks vs. mostly on open looks) can be checked
# player by player, not just for one flagged player.
player_contest = (
    spotup.groupby(["player", "contest"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
player_contest["fg_pct"] = (100 * player_contest["makes"] / player_contest["attempts"]).round(1)
player_contest = player_contest.sort_values(["player", "attempts"], ascending=[True, False])
print("Spot-Up shooting by player and contest level, season-wide (every player/contest combination with at "
      "least one attempt):\n")
print(player_contest)

guarded_summary = (
    spotup[spotup["contest"] == "Guarded"].groupby("player")
    .agg(guarded_attempts=("made", "count"), guarded_makes=("made", "sum"))
    .reset_index()
)
guarded_summary["guarded_fg_pct"] = (100 * guarded_summary["guarded_makes"] / guarded_summary["guarded_attempts"]).round(1)

open_summary = (
    spotup[spotup["contest"] == "Open"].groupby("player")
    .agg(open_attempts=("made", "count"), open_makes=("made", "sum"))
    .reset_index()
)
open_summary["open_fg_pct"] = (100 * open_summary["open_makes"] / open_summary["open_attempts"]).round(1)

contest_compare = guarded_summary.merge(open_summary, on="player", how="outer")
contest_compare["total_attempts"] = contest_compare[["guarded_attempts", "open_attempts"]].sum(axis=1, skipna=True)
contest_compare = contest_compare.sort_values("total_attempts", ascending=False)
print("\nGuarded vs. Open Spot-Up shooting side-by-side, by player (season-wide):\n")
print(contest_compare[[
    "player", "guarded_attempts", "guarded_makes", "guarded_fg_pct",
    "open_attempts", "open_makes", "open_fg_pct", "total_attempts",
]])

Spot-Up shooting by player and contest level, season-wide (every player/contest combination with at least one attempt):

                    player  \
0          Agape Keyes Jr.   
1          Agape Keyes Jr.   
3           Austin Ambrose   
2           Austin Ambrose   
5             Brock Marino   
6             Brock Marino   
4             Brock Marino   
8            Collin Madson   
7            Collin Madson   
9            Collin Madson   
12         Darius Chestnut   
11         Darius Chestnut   
10         Darius Chestnut   
14            Isaac Verges   
15            Isaac Verges   
13            Isaac Verges   
18         Isaiah Robinson   
16         Isaiah Robinson   
17         Isaiah Robinson   
20            JR Lukenbill   
19            JR Lukenbill   
21            JR Lukenbill   
22              Jake Quast   
23              Jake Quast   
24              Jake Quast   
25         Joey Berezowitz   
27         Joey Berezowitz   
26         Joey Berezowitz   
28       

### Coaching Flag: contested Spot-Up 3PT rate

**Season-wide (video-tagged data):** review the `player_contest` and `contest_compare` output above for any
player whose Spot-Up attempts skew heavily toward guarded catch-and-shoot looks with little or no open volume.
A skew like this may not reflect poor shooting on the contested look itself (compare the guarded FG% to that
player's season 3P%) -- the flag is more about shot-diet allocation: has he logged any OPEN catch-and-shoot
looks at all this season, or has every attempt come standstill against a set defense.

**Recommendation for coaching staff:** where this pattern shows up, review off-ball actions (relocation,
screens, drive-and-kick reads) to spring that player for easier looks. Treat as a small-sample signal -- 
re-check as more games are logged. (See the structured `coaching_flags_df` a few cells below for the
rule-based, automatically-updating version of this check across the full roster.)


### Investigate why guarded catch-and-shoot jumpers outperform open ones

Digs into the counter-intuitive pattern flagged in the previous note -- guarded catch-and-shoot attempts showing a higher make rate than open ones -- to determine whether it's a real effect or a small-sample artifact.

In [128]:
# --- Why might GUARDED catch-and-shoot jumpers outperform OPEN ones, season-wide? ---------------------------
# Three possible explanations to test: (1) it's just small-sample noise, (2) it's driven by one game/opponent,
# or (3) it's a SHOOTER-QUALITY CONFOUND -- the defense keys on UWW's best shooters (more Guarded attempts from
# good shooters), while lesser shooters get left Open more often, so the two buckets aren't comparing the same
# shooters to begin with.
import math

guarded_n = int(contest_all.loc[contest_all["contest"] == "Guarded", "attempts"].iloc[0])
guarded_makes_n = int(contest_all.loc[contest_all["contest"] == "Guarded", "makes"].iloc[0])
open_n = int(contest_all.loc[contest_all["contest"] == "Open", "attempts"].iloc[0])
open_makes_n = int(contest_all.loc[contest_all["contest"] == "Open", "makes"].iloc[0])

p1, p2 = guarded_makes_n / guarded_n, open_makes_n / open_n
p_pool = (guarded_makes_n + open_makes_n) / (guarded_n + open_n)
se = (p_pool * (1 - p_pool) * (1 / guarded_n + 1 / open_n)) ** 0.5
z = (p1 - p2) / se
p_value = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2))))
print(f"Two-proportion z-test, Guarded ({p1:.1%}, n={guarded_n}) vs. Open ({p2:.1%}, n={open_n}): "
      f"z={z:.2f}, p-value={p_value:.3f}")
print("(p > 0.05 means this gap is NOT statistically distinguishable from random chance at this sample size)\n")

by_game = (
    catch_and_shoot_all.groupby(GAME_KEYS + ["contest"])
    .agg(attempts=("made", "count"), makes=("made", "sum"))
    .reset_index()
)
by_game["fg_pct"] = (100 * by_game["makes"] / by_game["attempts"]).round(1)
print("Guarded vs. Open catch-and-shoot FG%, broken out by game:\n")
print(by_game.sort_values(["opponent", "contest"]))

season_3pt = stats[["PLAYER", "3P%"]].copy()
season_3pt["season_3p_pct"] = pd.to_numeric(season_3pt["3P%"].str.rstrip("%"), errors="coerce")
season_3pt["player_norm"] = season_3pt["PLAYER"].apply(normalize_player_name)

cs_long3 = catch_and_shoot_all[catch_and_shoot_all["distance"] == "Long/3pt"].copy()
cs_long3["player_norm"] = cs_long3["player"].apply(normalize_player_name)
cs_long3 = cs_long3.merge(season_3pt[["player_norm", "season_3p_pct"]], on="player_norm", how="left")

quality_by_contest = (
    cs_long3.groupby("contest")
    .agg(attempts=("made", "count"), avg_shooter_season_3p_pct=("season_3p_pct", "mean"))
    .reset_index()
)
quality_by_contest["avg_shooter_season_3p_pct"] = quality_by_contest["avg_shooter_season_3p_pct"].round(1)
print("\nAverage SEASON-LONG 3P% of the shooter taking the shot, by contest level (catch-and-shoot 3s only) -- "
      "tests whether the defense keys on UWW's better shooters, inflating the Guarded bucket's shooter-quality "
      "mix relative to Open:\n")
print(quality_by_contest)

Two-proportion z-test, Guarded (33.5%, n=236) vs. Open (40.2%, n=179): z=-1.42, p-value=0.157
(p > 0.05 means this gap is NOT statistically distinguishable from random chance at this sample size)

Guarded vs. Open catch-and-shoot FG%, broken out by game:

                     opponent   game_date  contest  attempts  makes  fg_pct
0                  Alma Scots  2025-12-20  Guarded         3      1    33.3
1                  Alma Scots  2025-12-20     Open        13      8    61.5
2             Aurora Spartans  2025-11-19  Guarded         4      1    25.0
3             Aurora Spartans  2025-11-19     Open        12      5    41.7
4       Carroll (WI) Pioneers  2025-12-13  Guarded        13      3    23.1
5       Carroll (WI) Pioneers  2025-12-13     Open         5      0     0.0
6                 Coe Kohawks  2025-12-30  Guarded        12      3    25.0
7                 Coe Kohawks  2025-12-30     Open         2      0     0.0
8           Elmhurst Bluejays  2025-12-02  Guarded         3

### Correction: "Guarded beats Open" is NOT a real team-wide pattern

An earlier pass through this notebook noted that UWW's catch-and-shoot jumpers were hitting at a higher rate
when **guarded** than when **open**, across every play type combined. The investigation above shows this does
**not** reliably hold up as a real tactical signal:

* **Check statistical significance** -- the two-proportion z-test above will show whether a gap this size is
  distinguishable from random chance at the current sample size (re-run as more games are logged; with only a
  few dozen total attempts, most gaps will not be significant).
* **Check whether it's driven by one specific game** -- the `by_game` breakdown above shows whether the pattern
  holds consistently across every logged game, or is really just one cold/hot shooting night skewing the total.
* **Check for a shooter-quality confound** -- the `quality_by_contest` breakdown compares the season-long 3P%
  of the shooters taking Guarded vs. Open looks; if Open shooters actually have a *higher* average season 3P%,
  that's evidence AGAINST guarded shots being intrinsically "better", not for it.

**Takeaway:** treat any aggregate "guarded > open" figure with suspicion until it clears all three checks above.
It should not be cited as evidence that contested shots are preferable to open ones -- individual player
patterns stand on their own volume/allocation evidence, not on this kind of small-sample team-wide claim.
Re-check once more games are logged and the sample grows.


### Structured coaching flags & recommendations database, per player

Consolidates every diagnosis surfaced above (Spot-Up quality, contest-level patterns, and the corrected guarded/open finding) into one structured, per-player table of coaching flags and recommendations.

In [131]:
# --- Structured coaching-flags DATABASE: one row per (player, flag) -- replaces one-off markdown notes and
# inline comments with a rule-based, queryable table that recomputes automatically as more games get logged.
# Every rostered player gets at least one row (falls back to season box-score context when a player's
# video-tagged shot volume is too thin for a play-type diagnosis).
# KNOWN_NAME_ALIASES reconciles known name-spelling mismatches BETWEEN data sources -- e.g. the PBP/video-
# tagging pipeline may spell a player differently than the official schedule-page season stats.
# Loaded from data/name_aliases.json (shared with streamlit_app.py) instead of a hardcoded dict here, so a
# newly-discovered mismatch only needs to be added in ONE place rather than kept in sync across the app and
# this notebook. Falls back to the one known mismatch if the file isn't found (e.g. first run before the app
# repo's data/ folder exists locally).
def _load_known_name_aliases():
    alias_path = os.path.join(OUTPUT_DIR, "name_aliases.json")
    if os.path.exists(alias_path):
        try:
            with open(alias_path) as _af:
                raw = json.load(_af)
            return {k.lower(): v.lower() for k, v in raw.items() if not k.startswith("_")}
        except Exception as _e:
            print(f"WARNING: could not read {alias_path} ({_e}); using inline fallback alias.")
    return {"mauryon turner": "maurquis turner"}

KNOWN_NAME_ALIASES = _load_known_name_aliases()

def resolve_player_key(name):
    # normalize_player_name() only folds CASING/spelling to pbp_events' own canonical form (and preserves the
    # original string untouched when no case-insensitive match exists there) -- it does NOT lowercase, so a
    # dict lookup against it needs an explicit .lower() to be reliably case-insensitive across sources whose
    # canonical spelling itself differs.
    norm = normalize_player_name(name).lower()
    return KNOWN_NAME_ALIASES.get(norm, norm)

coaching_flags = []

def add_flag(player, player_key, category, flag, evidence, recommendation, confidence, sentiment):
    coaching_flags.append({
        "player": player, "player_key": player_key, "category": category, "flag": flag,
        "evidence": evidence, "recommendation": recommendation, "confidence": confidence,
        "sentiment": sentiment,
    })

# Fresh, self-contained recomputation of UWW's video-tagged shot attempts with play type / mechanic / contest /
# distance tags (mirrors the logic in the cells above; kept self-contained so this cell doesn't depend on the
# execution order of earlier ones).
uww_shots = pbp_events[
    (pbp_events["team"] == "UW-Whitewater")
    & pbp_events["event_type"].isin(["made_shot", "missed_shot"])
    & pbp_events["video_description"].notna()
].copy()
uww_shots["play_type"] = uww_shots.apply(lambda r: extract_play_type(r["video_description"], r["player"]), axis=1)
uww_shots["made"] = uww_shots["event_type"] == "made_shot"
uww_shots["shot_mechanic"] = uww_shots["video_description"].apply(extract_shot_mechanic)
uww_shots["contest"] = uww_shots["video_description"].apply(extract_contest)
uww_shots["distance"] = uww_shots["video_description"].apply(extract_distance)
uww_shots["player_key"] = uww_shots["player"].apply(resolve_player_key)

# Season box score (full roster, including deep-bench players never captured in the video-tagged sample).
season = stats[~stats["PLAYER"].str.contains("Team|Opponent", case=False, na=False)].copy()
season["player_key"] = season["PLAYER"].apply(resolve_player_key)
for col, new_col in [("FG%", "fg_pct_season"), ("3P%", "three_pt_pct_season"), ("FT%", "ft_pct_season")]:
    season[new_col] = pd.to_numeric(season[col].astype(str).str.rstrip("%"), errors="coerce")
season["mpg_season"] = pd.to_numeric(season["MIN"], errors="coerce")
# The season-stats source table itself has a duplicate-row quirk for at least one jersey number (a placeholder
# row with all "-" stats under one name spelling, alongside a real-stats row under the corrected spelling).
# When the alias/key resolution above merges such rows onto the same player_key, prefer whichever row actually
# has real stats over an all-placeholder one.
season = season.sort_values("fg_pct_season", ascending=False, na_position="last").drop_duplicates("player_key", keep="first")

# Season-wide Starter/Bench role, from the per-game `started` flag already reconstructed from the PBP.
starter_flags = pbp_box_score[pbp_box_score["team"] == "UW-Whitewater"][["player", "started"]].drop_duplicates()
starter_flags["player_key"] = starter_flags["player"].apply(resolve_player_key)
player_role = starter_flags.groupby("player_key")["started"].any().map({True: "Starter", False: "Bench"})

# Canonical DISPLAY name per resolved key -- prefer the official season-stats spelling when available (the
# athletic department's own roster page), otherwise whatever spelling shows up in the video-tagged PBP data.
canonical_name_by_key = {}
for _, row in season.iterrows():
    canonical_name_by_key.setdefault(row["player_key"], row["PLAYER"])
for name in uww_shots["player"].unique():
    canonical_name_by_key.setdefault(resolve_player_key(name), name)

roster_keys = sorted(set(season["player_key"]) | set(uww_shots["player_key"]))

for player_key in roster_keys:
    player = canonical_name_by_key[player_key]
    p_shots = uww_shots[uww_shots["player_key"] == player_key]
    season_row = season[season["player_key"] == player_key]
    total_attempts = len(p_shots)
    has_positive_flag = False

    cs = p_shots[p_shots["shot_mechanic"] == "Catch-and-shoot"]
    guarded_cs = cs[cs["contest"] == "Guarded"]
    open_cs = cs[cs["contest"] == "Open"]

    # Rule 1: shot diet is skewed heavily toward contested catch-and-shoot looks. Requires guarded to be at
    # least 80% of catch-and-shoot volume, with at most 1 stray open attempt, so a single outlier doesn't mask
    # an otherwise heavily-contested shot diet, while still requiring a real majority skew.
    cs_total = len(guarded_cs) + len(open_cs)
    if len(guarded_cs) >= 3 and len(open_cs) <= 1 and cs_total and (len(open_cs) / cs_total) <= 0.2:
        g_makes, g_att, o_att = int(guarded_cs["made"].sum()), len(guarded_cs), len(open_cs)
        add_flag(
            player, player_key, "Shot selection",
            "Shot diet skewed heavily toward contested catch-and-shoot looks",
            f"{g_makes}-for-{g_att} ({100 * g_makes / g_att:.0f}%) on guarded catch-and-shoot attempts this "
            f"season, vs. only {o_att} open catch-and-shoot attempt(s) logged.",
            "Design more actions to create separation before the catch (relocation, screens, drive-and-kick "
            "reads) rather than relying on standstill catches against a set defense.",
            "Medium (small sample -- re-check as more games are logged)", "Negative",
        )

    # Rule 2: misses concentrate on OPEN catch-and-shoot looks specifically -- a shooter-specific issue, not a
    # shot-quality one.
    if len(open_cs) >= 3:
        o_makes, o_att = int(open_cs["made"].sum()), len(open_cs)
        o_pct = 100 * o_makes / o_att
        if o_pct <= 35:
            season_3p = None
            if not season_row.empty and pd.notna(season_row["three_pt_pct_season"].iloc[0]):
                season_3p = season_row["three_pt_pct_season"].iloc[0]
            evidence = f"{o_makes}-for-{o_att} ({o_pct:.0f}%) on open catch-and-shoot attempts this season"
            evidence += f" -- below his season 3P% of {season_3p:.1f}%." if season_3p is not None else "."
            add_flag(
                player, player_key, "Shooting efficiency",
                "Missing predominantly OPEN catch-and-shoot looks",
                evidence,
                "Since these are UNCONTESTED misses, treat as a shooting-mechanics/rhythm issue -- prioritize "
                "catch-and-shoot reps in practice rather than trying to generate better shot quality in-game.",
                "Medium (small sample -- re-check as more games are logged)", "Negative",
            )

    # Rule 3 / 4: efficiency on the player's single most-attempted play type (their "go-to" action).
    if total_attempts:
        top_pt = p_shots["play_type"].value_counts().idxmax()
        top_rows = p_shots[p_shots["play_type"] == top_pt]
        top_att, top_makes = len(top_rows), int(top_rows["made"].sum())
        top_pct = 100 * top_makes / top_att if top_att else None
        if top_att >= 4 and top_pct is not None:
            if top_pct <= 30:
                add_flag(
                    player, player_key, "Play-type efficiency",
                    f"Struggles specifically in his most-used action ({top_pt})",
                    f"{top_makes}-for-{top_att} ({top_pct:.0f}%) on {top_pt} -- his single most-attempted "
                    f"video-tagged action this season, well below his overall shooting split.",
                    f"Reduce reliance on {top_pt} as his primary look, or work on the specific mechanics/reads "
                    f"for that action in practice.",
                    "Medium (small sample -- re-check as more games are logged)", "Negative",
                )
            elif top_pct >= 70:
                add_flag(
                    player, player_key, "Play-type efficiency",
                    f"Highly efficient in his most-used action ({top_pt}) -- underused upside",
                    f"{top_makes}-for-{top_att} ({top_pct:.0f}%) on {top_pt}, his most-attempted video-tagged "
                    f"action this season.",
                    f"Consider increasing his usage/touches in {top_pt} sets -- the efficiency supports more "
                    f"volume there.",
                    "Medium (small sample -- re-check as more games are logged)", "Positive",
                )
                has_positive_flag = True

    # Rule 5: overall video-tagged efficiency, for a broader positive signal independent of a single play type.
    if total_attempts >= 8:
        overall_pct = 100 * p_shots["made"].sum() / total_attempts
        if overall_pct >= 70:
            add_flag(
                player, player_key, "Overall efficiency",
                "Highly efficient scorer on video-tagged attempts, season-wide",
                f"{int(p_shots['made'].sum())}-for-{total_attempts} ({overall_pct:.0f}%) across all video-"
                f"matched shot attempts this season.",
                "A clear, efficient scoring option -- consider featuring him more prominently in the "
                "half-court offense.",
                "Medium (small sample -- re-check as more games are logged)", "Positive",
            )
            has_positive_flag = True

    # Rule 6: season free-throw shooting (independent of video tagging -- covers every rostered player).
    if not season_row.empty:
        ft_pct = season_row["ft_pct_season"].iloc[0]
        ftm_a = str(season_row["FTM-A"].iloc[0])
        try:
            ft_attempts_per_game = float(ftm_a.split("-")[1]) if "-" in ftm_a else 0.0
        except ValueError:
            ft_attempts_per_game = 0.0
        if pd.notna(ft_pct) and ft_pct <= 60 and ft_attempts_per_game >= 0.5:
            add_flag(
                player, player_key, "Free-throw shooting",
                "Below-average free-throw shooter with meaningful attempt volume",
                f"{ft_pct:.1f}% FT this season ({ftm_a} per game).",
                "Target free-throw shooting in individual workouts -- meaningful attempt volume means this is "
                "costing points.",
                "High (season-long sample)", "Negative",
            )

    # --- POSITIVE-signal cascade: guarantees at least one Positive flag per player wherever the data supports
    # one, independent of whether the rules above already found a negative issue for him. Evaluated in
    # priority order; the first candidate with real supporting data is used.
    if not has_positive_flag:
        pt_stats = p_shots.groupby("play_type").agg(attempts=("made", "count"), makes=("made", "sum")).reset_index()
        pt_stats["pct"] = 100 * pt_stats["makes"] / pt_stats["attempts"]
        best_pt = pt_stats[pt_stats["attempts"] >= 3].sort_values("pct", ascending=False).head(1)
        if not best_pt.empty and best_pt["pct"].iloc[0] >= 65:
            r = best_pt.iloc[0]
            add_flag(
                player, player_key, "Play-type efficiency",
                f"Efficient secondary action: {r['play_type']}",
                f"{int(r['makes'])}-for-{int(r['attempts'])} ({r['pct']:.0f}%) on {r['play_type']} this season.",
                "A reliable look worth calling more often, even if not his primary action.",
                "Medium (small sample -- re-check as more games are logged)", "Positive",
            )
            has_positive_flag = True

    if not has_positive_flag and total_attempts >= 4:
        overall_pct = 100 * p_shots["made"].sum() / total_attempts
        if overall_pct >= 55:
            add_flag(
                player, player_key, "Overall efficiency",
                "Solid overall shooting on video-tagged attempts",
                f"{int(p_shots['made'].sum())}-for-{total_attempts} ({overall_pct:.0f}%) across all video-"
                f"matched shot attempts this season.",
                "A dependable scoring option in the offense.",
                "Medium (small sample -- re-check as more games are logged)", "Positive",
            )
            has_positive_flag = True

    if not has_positive_flag and not season_row.empty:
        r = season_row.iloc[0]
        if pd.notna(r["fg_pct_season"]) and r["fg_pct_season"] >= 45 and pd.notna(r["mpg_season"]) and r["mpg_season"] >= 3:
            add_flag(
                player, player_key, "Season shooting",
                "Solid season field-goal percentage",
                f"{r['FG%']} FG this season ({r['FGM-A']} per game, {r['MIN']} MPG).",
                "A reasonably efficient finisher for his role -- keep him involved in the offense.",
                "High (season-long sample)", "Positive",
            )
            has_positive_flag = True
        elif pd.notna(r["ft_pct_season"]) and r["ft_pct_season"] >= 70:
            add_flag(
                player, player_key, "Free-throw shooting",
                "Reliable free-throw shooter",
                f"{r['FT%']} FT this season ({r['FTM-A']} per game).",
                "A safe option to have on the floor in late-game free-throw situations.",
                "High (season-long sample)", "Positive",
            )
            has_positive_flag = True
        elif pd.notna(r["three_pt_pct_season"]) and r["three_pt_pct_season"] >= 33:
            add_flag(
                player, player_key, "Season shooting",
                "Respectable 3-point shooter this season",
                f"{r['3P%']} 3PT this season ({r['3PM-A']} per game).",
                "Worth designing catch-and-shoot looks for him specifically.",
                "High (season-long sample)", "Positive",
            )
            has_positive_flag = True
        elif pd.notna(r["mpg_season"]) and r["mpg_season"] > 0:
            add_flag(
                player, player_key, "General",
                "Has earned real game minutes this season",
                f"Averaging {r['MIN']} MPG over {r['GP-GS']} (GP-GS) -- no standout shooting number yet, but "
                f"he's seeing the floor.",
                "Keep tracking as more games/video are logged for a clearer efficiency signal.",
                "Low (no standout stat yet)", "Positive",
            )
            has_positive_flag = True

    if not has_positive_flag:
        add_flag(
            player, player_key, "General / limited data",
            "No performance data yet to flag positively",
            "No recorded minutes, shot attempts, or season stats found for him this season.",
            "Re-evaluate once he sees game action and stats are recorded.",
            "Low (no data)", "Neutral",
        )

coaching_flags_df = pd.DataFrame(coaching_flags)
coaching_flags_df["role"] = coaching_flags_df["player_key"].map(player_role).fillna("Unknown")
coaching_flags_df = coaching_flags_df[
    ["player", "player_key", "role", "sentiment", "category", "flag", "evidence", "recommendation", "confidence"]
]
_sentiment_order = {"Positive": 0, "Negative": 1, "Neutral": 2}
coaching_flags_df["_sentiment_order"] = coaching_flags_df["sentiment"].map(_sentiment_order)
coaching_flags_df = coaching_flags_df.sort_values(
    ["role", "player", "_sentiment_order", "category"], ascending=[False, True, True, True]
).drop(columns="_sentiment_order")

n_players_with_positive = coaching_flags_df.loc[coaching_flags_df["sentiment"] == "Positive", "player"].nunique()
print(f"Coaching flags database: {len(coaching_flags_df)} flag(s) across {coaching_flags_df['player'].nunique()} "
      f"player(s) (full roster: {len(roster_keys)}).")
print(f"{n_players_with_positive} of {len(roster_keys)} players have at least one Positive flag.\n")
print("Name reconciliation applied via KNOWN_NAME_ALIASES where the PBP/video-tagging spelling and the "
      "official season-stats spelling of a player's name differ.\n")
print(coaching_flags_df)

Coaching flags database: 38 flag(s) across 22 player(s) (full roster: 22).
21 of 22 players have at least one Positive flag.

Name reconciliation applied via KNOWN_NAME_ALIASES where the PBP/video-tagging spelling and the official season-stats spelling of a player's name differ.

                    player              player_key     role sentiment  \
31          Spencer Cullum          spencer cullum  Unknown   Neutral   
9             Brock Marino            brock marino  Starter  Positive   
8             Brock Marino            brock marino  Starter  Negative   
7             Brock Marino            brock marino  Starter  Negative   
10           Collin Madson           collin madson  Starter  Positive   
12            Isaac Verges            isaac verges  Starter  Positive   
20            JR Lukenbill            jr lukenbill  Starter  Positive   
19            JR Lukenbill            jr lukenbill  Starter  Negative   
18         Joey Berezowitz         joey berezowitz  Starter  P


### Expected box score for the upcoming game

Projects an expected box score for the next scouted opponent, combining UWW's season performance, the opponent's own lineup/tendency profile, and the coaching flags database above.

In [133]:
# --- Expected box score for the upcoming game (Elmhurst) ------------------------------------------------------
# Combines every signal already computed elsewhere in this notebook rather than inventing a new data source:
#   1. TEAM-LEVEL SCORE: a log5-style blend of each team's own scoring average with the OTHER side's actual
#      defensive output this season -- using ACTUAL results from completed games (not just season averages)
#      as the best available evidence of how UWW performs against this tier of competition.
#   2. OPPONENT PLAYER LINES: each player's own season per-game average, blended with the ACTUAL box-score
#      line their tag+stat-similarity comparable player (from the `best_matches` player-comparison cell) put
#      up in their real game against this same UWW defense -- a matchup-specific signal a season average
#      alone can't capture. Individual point projections are then scaled so they sum to the team-level
#      projection above.
#   3. UWW PLAYER LINES: each player's own season per-game average, scaled by the same team-level pace/quality
#      adjustment, preserving each player's share of the offense.
# NOTE: this cell is written for the specific upcoming opponent ("Elmhurst") named below -- update the
# opponent name if/when the upcoming matchup changes again. Elmhurst's PDF boxscore has no "Opponent" row (no
# points-allowed figure for them), so there's no way to log5 that side directly -- the model leans on their
# season scoring average instead.
UPCOMING_MATCHUP_OPPONENT = upcoming_opponent_short

def pct_to_float(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    return None if s in ("", "-") else float(s.rstrip("%"))

# --- 1. Team-level projected score --------------------------------------------------------------------------
uww_team_pts_season = float(stats.loc[stats["PLAYER"] == "Team Total", "PTS"].iloc[0])

played = schedule[schedule["outcome"].notna()]
uww_actual_pts_avg = played["team_score"].mean()
opp_actual_pts_avg = played["opponent_score"].mean()  # what UWW's actual scouted opponents have scored on them

_opp_ppg_rows = team_totals.loc[team_totals["opponent"] == UPCOMING_MATCHUP_OPPONENT, "team_ppg"]
if _opp_ppg_rows.empty:
    raise ValueError(
        f"'{UPCOMING_MATCHUP_OPPONENT}' not found in team_totals['opponent']. "
        f"Available opponents: {team_totals['opponent'].tolist()}. "
        f"Re-run the 'Cross-reference each scouted opponent' cell (cell 30) to rebuild team_totals."
    )
opponent_team_ppg = float(_opp_ppg_rows.iloc[0])
# Exclude any known data-entry-error team_ppg outliers from the "typical scouted opponent" scoring tier used to
# normalize the upcoming opponent's own average below.
tier_avg_ppg = team_totals.loc[team_totals["team_ppg"] < 200, "team_ppg"].mean()

# UWW's own scoring average, blended 50/50 with their ACTUAL scoring average against comparable competition
# this season (mean of the completed games) -- real matchup evidence, not just a season-long number.
expected_uww_pts = round(0.5 * uww_team_pts_season + 0.5 * uww_actual_pts_avg, 1)
# log5-style: the opponent's own scoring rate combined with how much UWW's real opponents have scored on them,
# normalized by the scoring tier those opponents represent.
expected_opponent_pts = round((opponent_team_ppg * opp_actual_pts_avg) / tier_avg_ppg, 1)

print(f"Projected final score: UW-Whitewater {expected_uww_pts:.0f} - {UPCOMING_MATCHUP_OPPONENT} {expected_opponent_pts:.0f} "
      f"(margin {expected_uww_pts - expected_opponent_pts:+.0f})")
print(f"  UWW inputs: season PTS/gm={uww_team_pts_season:.1f}, actual PTS/gm vs scouted opponents={uww_actual_pts_avg:.1f}")
print(f"  {UPCOMING_MATCHUP_OPPONENT} inputs: season PTS/gm={opponent_team_ppg:.1f}, UWW's actual PTS/gm allowed to scouted "
      f"opponents={opp_actual_pts_avg:.1f}, scouted-opponent scoring tier avg={tier_avg_ppg:.1f}")

# --- 2. Opponent projected player box score -------------------------------------------------------------------
opponent_players = player_profiles[player_profiles["opponent"] == UPCOMING_MATCHUP_OPPONENT].copy()
for col in ["PTS", "REB", "AST", "MIN"]:
    opponent_players[col] = pd.to_numeric(opponent_players[col], errors="coerce")

opponent_matches = best_matches[best_matches["target_opponent"] == UPCOMING_MATCHUP_OPPONENT]
opp_actuals = pbp_box_score[pbp_box_score["team"] != "UW-Whitewater"][
    ["opponent", "player", "PTS", "REB", "AST"]
].rename(columns={"opponent": "compared_opponent", "player": "compared_player",
                   "PTS": "comp_actual_PTS", "REB": "comp_actual_REB", "AST": "comp_actual_AST"})
comp_actuals = opponent_matches.merge(opp_actuals, on=["compared_opponent", "compared_player"], how="left")

opponent_players = opponent_players.merge(
    comp_actuals[["target_player", "compared_player", "compared_opponent", "similarity_score",
                  "comp_actual_PTS", "comp_actual_REB", "comp_actual_AST"]],
    left_on="name", right_on="target_player", how="left",
)

OWN_WEIGHT = 0.6  # own season average is the more direct predictor; the comp's actual game is a secondary signal

def blend_stat(row, own_col, comp_col):
    own, comp = row[own_col], row[comp_col]
    if pd.isna(comp):
        return own
    if pd.isna(own):
        return comp
    return OWN_WEIGHT * own + (1 - OWN_WEIGHT) * comp

for stat in ["PTS", "REB", "AST"]:
    opponent_players[f"blended_{stat}"] = opponent_players.apply(lambda r, s=stat: blend_stat(r, s, f"comp_actual_{s}"), axis=1)

blended_total_pts = opponent_players["blended_PTS"].sum()
opponent_scale = expected_opponent_pts / blended_total_pts if blended_total_pts else 1.0
opponent_players["projected_PTS"] = (opponent_players["blended_PTS"] * opponent_scale).round(1)
opponent_players["projected_REB"] = opponent_players["blended_REB"].round(1)
opponent_players["projected_AST"] = opponent_players["blended_AST"].round(1)

# Per-player projection basis -- a plain-text explanation of exactly how each line was derived. Intended to be
# surfaced as a hover tooltip/bubble wherever this table is displayed (e.g. an info icon next to each row in
# the app), rather than as its own always-visible column.
def opponent_projection_basis(row):
    if pd.isna(row["PTS"]):
        return "No season stats recorded for this player yet -- insufficient data to project."
    basis = f"Own season avg: {row['PTS']:.1f} PTS, {row['REB']:.1f} REB, {row['AST']:.1f} AST/gm"
    if pd.notna(row["comp_actual_PTS"]):
        basis += (
            f" | Blended {OWN_WEIGHT:.0%} own avg / {1 - OWN_WEIGHT:.0%} actual game -- comp match: "
            f"{row['compared_player']} ({row['compared_opponent']}, similarity {row['similarity_score']:.1f}) "
            f"actually posted {row['comp_actual_PTS']:.0f} PTS, {row['comp_actual_REB']:.0f} REB, "
            f"{row['comp_actual_AST']:.0f} AST vs this same UWW defense"
        )
    else:
        basis += " | No PBP data available for this player's comp match -- used own season average only"
    basis += f" | Scaled x{opponent_scale:.2f} so the roster sums to the projected team total ({expected_opponent_pts:.0f} pts)"
    return basis

opponent_players["projection_basis"] = opponent_players.apply(opponent_projection_basis, axis=1)

# MIN was previously left out of this selection even though opponent_players already carries the opponent's
# own season-average minutes (same source uww_player_profiles.MIN the app already reads elsewhere) -- added
# so the opponent's side of Projected Box Score isn't missing minutes played while UWW's own side has it.
projected_opponent_box = opponent_players[
    [c for c in ["name", "jersey_number", "role", "position", "MIN", "projected_PTS", "projected_REB", "projected_AST",
     "compared_player", "compared_opponent", "similarity_score", "projection_basis"] if c in opponent_players.columns]
].rename(columns={"compared_player": "comp_used", "compared_opponent": "comp_from_game"}).sort_values("projected_PTS", ascending=False).reset_index(drop=True)

print(f"\n{UPCOMING_MATCHUP_OPPONENT} projected team total: {projected_opponent_box['projected_PTS'].sum():.1f} pts "
      f"(scaled from a {OWN_WEIGHT:.0%} own-season-average / {1 - OWN_WEIGHT:.0%} comp-actual-game blend)")
print(projected_opponent_box)

# --- 3. UWW projected player box score -----------------------------------------------------------------------
# Scale against the SUM of individual player PTS averages, not the season "Team Total" row -- per-player
# averages are each computed over that player's OWN games played (GP-GS varies by player), so they don't sum
# exactly to the team's average PPG (a pre-existing quirk of the scraped season stats, not introduced here).
# Scaling to the raw individual sum keeps this table's total consistent with the printed team-level projection.
uww_player_rows = stats[~stats["PLAYER"].isin(["Team Total", "Opponent"])].copy()
for col in ["PTS", "REB", "AST", "MIN"]:
    uww_player_rows[col] = pd.to_numeric(uww_player_rows[col], errors="coerce")
raw_individual_pts_sum = uww_player_rows["PTS"].sum()
uww_scale = expected_uww_pts / raw_individual_pts_sum if raw_individual_pts_sum else 1.0
uww_player_rows["projected_PTS"] = (uww_player_rows["PTS"] * uww_scale).round(1)
uww_player_rows["projected_REB"] = uww_player_rows["REB"]
uww_player_rows["projected_AST"] = uww_player_rows["AST"]

# Per-player projection basis -- same idea as the opponent's: a plain-text explanation meant for a hover tooltip/bubble.
def uww_projection_basis(row):
    if pd.isna(row["PTS"]):
        return "No season stats recorded for this player yet -- insufficient data to project."
    min_str = f"{row['MIN']:.0f}" if pd.notna(row["MIN"]) else "?"
    return (
        f"Own season avg: {row['PTS']:.1f} PTS, {row['REB']:.1f} REB, {row['AST']:.1f} AST/gm over {min_str} min "
        f"| Scaled x{uww_scale:.2f} for this matchup's projected pace (UWW projected team total {expected_uww_pts:.0f} pts: "
        f"50% season PPG {uww_team_pts_season:.1f} + 50% actual PPG vs scouted opponents this season {uww_actual_pts_avg:.1f})"
    )

uww_player_rows["projection_basis"] = uww_player_rows.apply(uww_projection_basis, axis=1)

projected_uww_box = uww_player_rows[
    ["PLAYER", "MIN", "projected_PTS", "projected_REB", "projected_AST", "FG%", "3P%", "FT%", "projection_basis"]
].sort_values("projected_PTS", ascending=False).reset_index(drop=True)

print(f"\nUW-Whitewater projected team total: {projected_uww_box['projected_PTS'].sum():.1f} pts "
      f"(season averages scaled x{uww_scale:.2f} for pace/matchup)")
print(projected_uww_box)

# --- Narrative context from the opponent's game plan + relevant coaching flags on UWW's top projected scorers -
opponent_ktv = all_game_plans.loc[
    (all_game_plans["opponent"] == UPCOMING_MATCHUP_OPPONENT) & (all_game_plans["topic"] == "KEYS TO VICTORY"), "notes"
]
opponent_strengths = all_game_plans.loc[
    (all_game_plans["opponent"] == UPCOMING_MATCHUP_OPPONENT) & (all_game_plans["topic"] == "TEAM STRENGTHS"), "notes"
]
print(f"\n{UPCOMING_MATCHUP_OPPONENT}'s pre-game keys to victory:", opponent_ktv.iloc[0] if not opponent_ktv.empty else "n/a")
print(f"{UPCOMING_MATCHUP_OPPONENT}'s team strengths:", opponent_strengths.iloc[0] if not opponent_strengths.empty else "n/a")

top_scorers = set(projected_uww_box.head(5)["PLAYER"].str.lower())
relevant_flags = coaching_flags_df[coaching_flags_df["player_key"].isin(top_scorers)]
if not relevant_flags.empty:
    print("\nCoaching flags on UWW's top-5 projected scorers:")
    for _, f in relevant_flags.iterrows():
        print(f"  [{f['sentiment']}] {f['player']} -- {f['flag']} ({f['evidence']})")

Projected final score: UW-Whitewater 77 - Loras Duhawks 79 (margin -2)
  UWW inputs: season PTS/gm=78.4, actual PTS/gm vs scouted opponents=75.0
  Loras Duhawks inputs: season PTS/gm=80.6, UWW's actual PTS/gm allowed to scouted opponents=73.6, scouted-opponent scoring tier avg=75.0

Loras Duhawks projected team total: 79.0 pts (scaled from a 60% own-season-average / 40% comp-actual-game blend)
                  name jersey_number     role position   MIN  projected_PTS  \
0          Jack Haynes           #23  Starter        F  29.3            9.5   
1         Gavin Sarvis           #11  Starter        G  26.1            9.2   
2         Gavin Sarvis           #11  Starter        G  26.1            8.2   
3        Johnny Semany           #32  Starter        G  19.7            6.5   
4           Kyle Kober           #24    Bench        G  21.6            5.8   
5       Nolan Berendes           #12  Starter        F  30.7            5.8   
6        Johnny Semany           #32  Starter     


### Export tables as CSVs into the app's bundled data directory

Writes every table this notebook produces (schedule, box scores, scouting reports, PBP/lineup analytics, player comparisons, coaching flags, etc.) out as CSVs into `OUTPUT_DIR`, for the Streamlit app to read.


### Reconstruct a box score for the opponent's games before facing UWW

The shot-by-shot video-tagged data for these games already exists (pbp_events_upcoming, built above, and
already exported as uww_opponent_prior_games_pbp for the shot-selection analysis) -- this just aggregates
it into a per-player box score, the exact same way uww_pbp_box_score already does for UWW's own games
(same event_type -> stat mapping, same groupby). This is what lets the app's Last Five Games page show a
full box score when a coach clicks one of the opponent's own results, not just UWW's.

In [136]:
# --- Reconstruct a box score from pbp_events_upcoming, same event_type -> stat mapping as pbp_box_score ---
if not pbp_events_upcoming.empty:
    _pu_player_events = pbp_events_upcoming[
        pbp_events_upcoming["player"].notna() & (~pbp_events_upcoming["event_type"].isin(TEAM_LEVEL_EVENT_TYPES_BOX))
    ].copy()

    _pu_player_events["points"] = _pu_player_events.apply(
        lambda row: int(row["shot_type"]) if row["event_type"] == "made_shot" else (1 if row["event_type"] == "free_throw_made" else 0),
        axis=1,
    )
    _pu_player_events["is_fgm"] = _pu_player_events["event_type"] == "made_shot"
    _pu_player_events["is_fga"] = _pu_player_events["event_type"].isin(["made_shot", "missed_shot"])
    _pu_player_events["is_3pm"] = _pu_player_events["is_fgm"] & (_pu_player_events["shot_type"] == "3")
    _pu_player_events["is_3pa"] = _pu_player_events["is_fga"] & (_pu_player_events["shot_type"] == "3")
    _pu_player_events["is_ftm"] = _pu_player_events["event_type"] == "free_throw_made"
    _pu_player_events["is_fta"] = _pu_player_events["event_type"].isin(["free_throw_made", "free_throw_missed"])
    _pu_player_events["is_oreb"] = _pu_player_events["event_type"] == "rebound_offensive"
    _pu_player_events["is_dreb"] = _pu_player_events["event_type"] == "rebound_defensive"
    _pu_player_events["is_ast"] = _pu_player_events["event_type"] == "assist"
    _pu_player_events["is_stl"] = _pu_player_events["event_type"] == "steal"
    _pu_player_events["is_blk"] = _pu_player_events["event_type"] == "block"
    _pu_player_events["is_to"] = _pu_player_events["event_type"] == "turnover"
    _pu_player_events["is_pf"] = _pu_player_events["event_type"] == "foul"

    pbp_box_score_upcoming = _pu_player_events.groupby(["opponent", "game_date", "team", "player"]).agg(
        PTS=("points", "sum"), FGM=("is_fgm", "sum"), FGA=("is_fga", "sum"),
        FG3M=("is_3pm", "sum"), FG3A=("is_3pa", "sum"), FTM=("is_ftm", "sum"), FTA=("is_fta", "sum"),
        OREB=("is_oreb", "sum"), DREB=("is_dreb", "sum"), AST=("is_ast", "sum"), STL=("is_stl", "sum"),
        BLK=("is_blk", "sum"), TO=("is_to", "sum"), PF=("is_pf", "sum"),
    ).reset_index()

    # Same gap, same fix as pbp_box_score above: a bare team-level turnover (no player attached) is correctly
    # excluded from _pu_player_events via the player.notna() filter, but that means it never appeared anywhere
    # in this box score either. Surfaced as a synthetic "TEAM" row instead of silently dropped.
    _pu_team_level_turnovers = pbp_events_upcoming[pbp_events_upcoming["player"].isna() & (pbp_events_upcoming["event_type"] == "turnover")]
    if not _pu_team_level_turnovers.empty:
        _pu_team_to_rows = _pu_team_level_turnovers.groupby(["opponent", "game_date", "team"]).size().reset_index(name="TO")
        _pu_team_to_rows["player"] = "TEAM"
        for _stat_col in ["PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "OREB", "DREB", "AST", "STL", "BLK", "PF"]:
            _pu_team_to_rows[_stat_col] = 0
        pbp_box_score_upcoming = pd.concat([pbp_box_score_upcoming, _pu_team_to_rows], ignore_index=True)
        print(f"Added {len(_pu_team_to_rows)} synthetic TEAM row(s) for {int(_pu_team_to_rows['TO'].sum())} bare team-level turnover(s) in the opponent's prior games.")

    pbp_box_score_upcoming["REB"] = pbp_box_score_upcoming["OREB"] + pbp_box_score_upcoming["DREB"]
    pbp_box_score_upcoming["FG%"] = (100 * pbp_box_score_upcoming["FGM"] / pbp_box_score_upcoming["FGA"]).round(1)
    pbp_box_score_upcoming["3P%"] = (100 * pbp_box_score_upcoming["FG3M"] / pbp_box_score_upcoming["FG3A"]).round(1)
    pbp_box_score_upcoming["FT%"] = (100 * pbp_box_score_upcoming["FTM"] / pbp_box_score_upcoming["FTA"]).round(1)
    print(f"Reconstructed box score for {pbp_box_score_upcoming['opponent'].nunique()} of the upcoming opponent's prior game(s), {len(pbp_box_score_upcoming)} player-game row(s) total.")
else:
    pbp_box_score_upcoming = pd.DataFrame(columns=[
        "opponent", "game_date", "team", "player", "PTS", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
        "OREB", "DREB", "AST", "STL", "BLK", "TO", "PF", "REB", "FG%", "3P%", "FT%",
    ])
    print("No prior-game PBP data available yet -- pbp_box_score_upcoming is empty but well-formed.")


Added 17 synthetic TEAM row(s) for 23 bare team-level turnover(s) in the opponent's prior games.
Reconstructed box score for 18 of the upcoming opponent's prior game(s), 659 player-game row(s) total.



### Minutes played for the upcoming opponent's prior games

Reuses the lineup reconstruction and stints already built two cells up (`self_lineup`/`their_lineup`,
`stint_src`, `stints`, `minutes_margin`) instead of redoing that same substitution-tracking pass a second
time -- an earlier version of this cell duplicated that whole algorithm from scratch under different
column names, which was pure waste and a real risk of the two copies quietly drifting apart over time.
`stints`/`minutes_margin` above only cover the upcoming opponent's OWN lineups (that's all the season-box
cell needed); the one genuinely new piece here is doing the same for `their_lineup` (whichever third party
they played each game), so both sides of `pbp_box_score_upcoming` get a MIN column, not just one.

In [138]:
# --- Minutes played for BOTH sides of the upcoming opponent's prior games, reusing cell above's work --------
if pbp_events_upcoming.empty or "stint_src" not in globals():
    pbp_box_score_upcoming["MIN"] = None
    lineup_stints_upcoming = pd.DataFrame(columns=["opponent", "lineup", "MIN", "+/-"])
    print("No prior-game lineup data available yet -- MIN and lineup_stints_upcoming left empty.")
else:
    # Third-party side: same stint_seconds already computed in stint_src above, just grouped by their_lineup
    # instead of self_lineup (which is all the season-box cell needed and kept).
    third_party_minutes_by_lineup = stint_src.groupby(GAME_KEYS + ["their_lineup"])["seconds_elapsed"].sum().reset_index()
    third_party_minutes_by_lineup["MIN"] = (third_party_minutes_by_lineup["seconds_elapsed"] / 60).round(2)

    def _lineup_minutes_to_player_minutes(df, lineup_col, team_col_value_fn):
        rows = []
        for _, r in df.iterrows():
            if pd.isna(r[lineup_col]):
                continue
            for player in str(r[lineup_col]).split(", "):
                rows.append({"opponent": r["opponent"], "game_date": r["game_date"], "team": team_col_value_fn(r), "player": player, "minutes": r["MIN"]})
        return rows

    _self_rows = _lineup_minutes_to_player_minutes(
        minutes_margin.rename(columns={"lineup": "self_lineup"}), "self_lineup", lambda r: upcoming_opponent_short,
    )
    _third_party_rows = _lineup_minutes_to_player_minutes(
        third_party_minutes_by_lineup, "their_lineup", lambda r: r["opponent"],
    )

    _pu_minutes_rows = _self_rows + _third_party_rows
    if _pu_minutes_rows:
        _pu_player_minutes = pd.DataFrame(_pu_minutes_rows).groupby(GAME_KEYS + ["team", "player"])["minutes"].sum().reset_index()
        _pu_player_minutes["MIN"] = _pu_player_minutes["minutes"].round(1)
        if "MIN" in pbp_box_score_upcoming.columns:
            pbp_box_score_upcoming = pbp_box_score_upcoming.drop(columns=["MIN"])
        pbp_box_score_upcoming = pbp_box_score_upcoming.merge(
            _pu_player_minutes[GAME_KEYS + ["team", "player", "MIN"]], on=GAME_KEYS + ["team", "player"], how="left",
        )
        _pu_n_matched = int(pbp_box_score_upcoming["MIN"].notna().sum())
        print(f"Matched minutes played for {_pu_n_matched} of {len(pbp_box_score_upcoming)} pbp_box_score_upcoming row(s).")
    else:
        pbp_box_score_upcoming["MIN"] = None
        print("No lineup stints available yet -- pbp_box_score_upcoming.MIN left empty.")

    # Per-game lineup stints for the upcoming opponent's OWN lineups, already reconstructed above (`stints`) --
    # exported directly rather than rebuilt, matching upcoming_lineup_season's own scope (their own lineups
    # only; third-party lineup stints aren't exported standalone since nothing in the app needs that yet).
    lineup_stints_upcoming = stints.rename(columns={"self_lineup": "lineup"})[GAME_KEYS + ["lineup", "stint_minutes", "margin_change"]]


Matched minutes played for 642 of 659 pbp_box_score_upcoming row(s).



### Replace the upcoming opponent's PDF-scouting-report stats with PBP-derived ones

player_profiles' PTS/REB/AST/etc for a scouted opponent normally come from that opponent's own scouting-
report PDF (a static document, captured once). For the CURRENT upcoming opponent specifically, `pbp_box_
score_upcoming` now holds a fully reconstructed box score for every one of their games before facing UWW --
the exact same games `reference_date` scopes everything else in this notebook to. Overriding their
player_profiles stats with an aggregate of THAT instead makes those numbers correctly reference-date-aware,
the same way everything else already had to be fixed to be.

**Scope, explicitly**: this only replaces the CURRENT upcoming opponent's stats. For an opponent UWW has
ALREADY PLAYED, the only PBP data this notebook has for them is their single game against UWW (from
pbp_box_score) -- not a season's worth of games. Using that as a season-stat substitute would trade one
real problem (a stale PDF snapshot) for a different one (a single noisy game standing in for a season
average), silently. Already-played opponents keep their PDF-sourced stats for now -- a genuine, open
limitation, not something this cell quietly papers over.

In [140]:
# --- Override the upcoming opponent's player_profiles stats with a PBP-derived aggregate ---------------------
if pbp_box_score_upcoming.empty or upcoming_opponent_short is None:
    print("No prior-game box score available yet -- upcoming opponent's player_profiles stats left as-is (PDF-sourced, if any).")
else:
    _pu_own_box = pbp_box_score_upcoming[pbp_box_score_upcoming["team"] == upcoming_opponent_short].copy()
    _pu_own_box = _pu_own_box[_pu_own_box["player"] != "TEAM"]  # exclude the synthetic bare-turnover row
    if _pu_own_box.empty:
        print(f"No player-level PBP box score data available yet for {upcoming_opponent_short} -- player_profiles stats left as-is.")
    else:
        _pu_agg = _pu_own_box.groupby("player").agg(
            games=("game_date", "nunique"),
            PTS_total=("PTS", "sum"), REB_total=("REB", "sum"), MIN_total=("MIN", "sum"),
            OREB_total=("OREB", "sum"), DREB_total=("DREB", "sum"),
            FGM=("FGM", "sum"), FGA=("FGA", "sum"), FG3M=("FG3M", "sum"), FG3A=("FG3A", "sum"),
            FTM=("FTM", "sum"), FTA=("FTA", "sum"),
            AST=("AST", "sum"), STL=("STL", "sum"), BLK=("BLK", "sum"), TO=("TO", "sum"),
        ).reset_index()

        # PTS/REB/MIN/OREB/DREB: per-game averages (matches player_profiles' existing convention for PTS/REB/
        # MIN; OREB/DREB are new here -- the PDF-sourced player_profiles table never had a rebound split at
        # all, only combined REB, so these two columns didn't exist on this table before this override).
        # AST/STL/BLK/TO: left as SEASON TOTALS on purpose -- get_opponent_games_played() divides these
        # downstream throughout the app; pre-dividing here would double-divide them.
        _pu_agg["PTS"] = (_pu_agg["PTS_total"] / _pu_agg["games"]).round(1)
        _pu_agg["REB"] = (_pu_agg["REB_total"] / _pu_agg["games"]).round(1)
        _pu_agg["MIN"] = (_pu_agg["MIN_total"] / _pu_agg["games"]).round(1)
        _pu_agg["OREB"] = (_pu_agg["OREB_total"] / _pu_agg["games"]).round(1)
        _pu_agg["DREB"] = (_pu_agg["DREB_total"] / _pu_agg["games"]).round(1)
        # CONFIRMED BUG (fixed here): the original `.astype(str).replace("nan", "")` approach for a
        # zero-attempts player (FGA/FG3A/FTA == 0, a real case -- e.g. a player who saw the floor but never
        # attempted a 3) doesn't actually work in this pandas version: .astype(str) on a float NaN keeps it
        # as an actual null rather than stringifying it to the literal text "nan", so .replace("nan", "") has
        # nothing to match and the NaN survives all the way through, then poisons the "+ \'%\'" concatenation
        # into another NaN. Verified directly (not assumed) before shipping this fix -- the old code would
        # have left a real, silent NaN in this column for exactly the players this was meant to handle
        # gracefully. Using an explicit per-value formatter instead of a dtype-fragile string trick.
        def _pct_str(v):
            return f"{v}%" if pd.notna(v) else "-"
        _pu_agg["FG%"] = (100 * _pu_agg["FGM"] / _pu_agg["FGA"]).round(1).apply(_pct_str)
        _pu_agg["3P%"] = (100 * _pu_agg["FG3M"] / _pu_agg["FG3A"]).round(1).apply(_pct_str)
        _pu_agg["FT%"] = (100 * _pu_agg["FTM"] / _pu_agg["FTA"]).round(1).apply(_pct_str)
        _pu_agg["3PM-A"] = _pu_agg["FG3M"].astype(int).astype(str) + "-" + _pu_agg["FG3A"].astype(int).astype(str)
        _pu_agg["FTM-A"] = _pu_agg["FTM"].astype(int).astype(str) + "-" + _pu_agg["FTA"].astype(int).astype(str)

        # AST/STL/BLK/TO stay SEASON TOTALS (the app divides them) -- but the app was dividing by the
        # TEAM's games played, which understates anyone who missed time: Gavin Sarvis' 73 assists in the 24
        # games he actually played rendered as 2.6/gm instead of 3.0 because the divisor was Loras' 28.
        # Ship each player's own games-played alongside the totals so the app can divide correctly.
        _pu_agg["games_played"] = _pu_agg["games"]
        _pu_stat_cols = ["MIN", "FG%", "3PM-A", "3P%", "FTM-A", "FT%", "REB", "OREB", "DREB", "AST", "TO", "STL", "BLK", "PTS", "games_played"]
        if "games_played" not in player_profiles.columns:
            player_profiles["games_played"] = pd.NA
        _pu_replacement = _pu_agg[["player"] + _pu_stat_cols].rename(columns={"player": "name"})

        # Match by player name (pbp_box_score_upcoming has no jersey_number) -- case-insensitive, since roster
        # names elsewhere in this notebook are sometimes cased slightly differently between sources.
        _pu_upcoming_mask = player_profiles["opponent"] == upcoming_opponent_short
        _pu_name_to_row = {str(n).strip().casefold(): row for n, row in zip(_pu_replacement["name"], _pu_replacement.to_dict("records"))}
        _pu_n_matched = 0
        for _pu_idx in player_profiles[_pu_upcoming_mask].index:
            _pu_key = str(player_profiles.at[_pu_idx, "name"]).strip().casefold()
            if _pu_key in _pu_name_to_row:
                for _pu_col in _pu_stat_cols:
                    player_profiles.at[_pu_idx, _pu_col] = _pu_name_to_row[_pu_key][_pu_col]
                _pu_n_matched += 1
        print(f"Replaced PDF-sourced stats with PBP-derived stats for {_pu_n_matched} of {_pu_upcoming_mask.sum()} "
              f"{upcoming_opponent_short} player_profiles row(s), from {_pu_agg['games'].max()} prior game(s) of PBP data.")

        # --- Add PBP-derived players who have NO scout-report entry at all ---------------------------------
        # CONFIRMED BUG (fixed here): the loop above only ever UPDATES rows that already exist in
        # player_profiles, so a player who shows up in the opponent's prior-game PBP but was never in the
        # scout report was dropped on the floor entirely -- and with him, his contribution to every
        # team-level sum the app builds off this table.
        #
        # Confirmed case: Andrew Wells had 2 blocks for Eureka vs Dominican (IL), but "Wells" appears nowhere
        # in Eureka's scout report -- not in the roster AND not in its season BOXSCORE table, so the
        # `box_only` backfill in the player_profiles cell can't recover him either (that one only rescues
        # players the scout report's own boxscore lists). The app therefore summed 3 of Eureka's 5 blocks and
        # rendered 1.0 BPG (3/3) instead of 1.67 (5/3). Note the failing number was the NUMERATOR -- the
        # games-played denominator was correct all along.
        #
        # These additions get the same treatment `box_only` gives its own: role="Bench",
        # has_scouting_report=False, null demographics. jersey_number stays null because
        # pbp_box_score_upcoming has no jersey column (that's why the update loop above matches on name).
        _pu_existing = {str(n).strip().casefold() for n in player_profiles.loc[_pu_upcoming_mask, "name"].dropna()}
        _pu_add = _pu_replacement[
            ~_pu_replacement["name"].astype(str).str.strip().str.casefold().isin(_pu_existing)
        ].copy()
        if not _pu_add.empty:
            _pu_add["opponent"] = upcoming_opponent_short
            _pu_add["game_date"] = game_date_for(upcoming_opponent_short)
            _pu_add["jersey_number"] = None
            _pu_add["role"] = "Bench"
            _pu_add["has_scouting_report"] = False
            _pu_add["position_group"] = "Unknown"
            _pu_add["player_notes"] = ""
            _pu_add["keys_to_defending"] = ""
            _pu_add["notes_tags_display"] = ""
            _pu_add["keys_tags_display"] = ""
            for _pu_missing_col in player_profiles.columns:
                if _pu_missing_col not in _pu_add.columns:
                    _pu_add[_pu_missing_col] = None
            # Assigned AFTER the None-fill loop above on purpose: these two columns hold real sets (other
            # cells call set operations on them directly), and a column of shared Nones would break that.
            _pu_add["notes_tags"] = [set() for _ in range(len(_pu_add))]
            _pu_add["keys_tags"] = [set() for _ in range(len(_pu_add))]
            player_profiles = pd.concat(
                [player_profiles, _pu_add[player_profiles.columns.tolist()]], ignore_index=True
            )
            # _pu_upcoming_mask was built against the PRE-concat frame; reusing a now-too-short boolean mask
            # against the grown frame is a pandas IndexingError, so rebuild it before the diagnostics below.
            _pu_upcoming_mask = player_profiles["opponent"] == upcoming_opponent_short
            print(f"  Added {len(_pu_add)} PBP-only player(s) with no scout-report entry at all "
                  f"(role=Bench): {sorted(_pu_add['name'].astype(str))}")

            # Mirror the same additions into all_rosters, which is exported as uww_opponent_rosters and is
            # what the app's ROSTER panel reads -- without this a PBP-only player shows up in the team-stat
            # sums (player_profiles) but is invisible in the roster list, which reads as a data bug to whoever
            # is checking the numbers by hand. Kept to roster_cols only: all_rosters is the pre-enrichment
            # table (no stat columns, no tag columns).
            _pu_roster_add = _pu_add[[c for c in roster_cols if c in _pu_add.columns]].copy()
            for _pu_missing_roster_col in roster_cols:
                if _pu_missing_roster_col not in _pu_roster_add.columns:
                    _pu_roster_add[_pu_missing_roster_col] = None
            # Independent dedupe rather than relying on the enclosing `if not _pu_add.empty` guard: that guard
            # is driven by player_profiles, so re-running the player-profiles cell WITHOUT re-running the
            # roster cell would otherwise append a second copy of the same player here.
            _pu_roster_have = {
                (str(o).strip().casefold(), str(n).strip().casefold())
                for o, n in zip(all_rosters.get("opponent", []), all_rosters.get("name", []))
            }
            _pu_roster_add = _pu_roster_add[[
                (str(r["opponent"]).strip().casefold(), str(r["name"]).strip().casefold()) not in _pu_roster_have
                for _, r in _pu_roster_add.iterrows()
            ]]
            if not _pu_roster_add.empty:
                all_rosters = pd.concat([all_rosters, _pu_roster_add[roster_cols]], ignore_index=True)
                print(f"  Also added {len(_pu_roster_add)} of them to all_rosters (uww_opponent_rosters) so "
                      f"they appear in the app's ROSTER panel, not just the team-stat sums.")
        else:
            print("  No PBP-only players to add -- every player in the prior-game PBP already had a "
                  "player_profiles row.")
        _pu_roster_names = set(player_profiles.loc[_pu_upcoming_mask, "name"].astype(str))
        _pu_pbp_names = set(_pu_replacement["name"].astype(str))
        _pu_unmatched_names = _pu_roster_names - _pu_pbp_names
        if _pu_unmatched_names:
            print(f"  Roster player(s) with no matching PBP box-score row yet (kept their PDF-sourced stats, if any): {sorted(_pu_unmatched_names)}")
        # STRENGTHENED DIAGNOSTIC -- this exact override has now been reported as "still showing the old
        # number" once already; rather than guess a third time, print everything needed to tell apart the
        # three real possibilities in one look: (a) a genuine name-spelling mismatch between the roster and
        # what got parsed off the raw PBP text (b) the override ran but the app is reading a stale/un-
        # redeployed CSV (c) this cell genuinely hasn't been re-run since the fix went in.
        print(f"\n  Roster names for {upcoming_opponent_short} ({len(_pu_roster_names)}): {sorted(_pu_roster_names)}")
        print(f"  PBP-derived names found ({len(_pu_pbp_names)}): {sorted(_pu_pbp_names)}")
        print("  Per-player PTS after this override (spot-check against what the app is showing):")
        print(player_profiles.loc[_pu_upcoming_mask, ["name", "PTS"]].sort_values("PTS", ascending=False).to_string(index=False))


Replaced PDF-sourced stats with PBP-derived stats for 12 of 12 Loras Duhawks player_profiles row(s), from 28 prior game(s) of PBP data.
  Added 7 PBP-only player(s) with no scout-report entry at all (role=Bench): ['Athan Berchos', 'Cooper Grams', 'Emmett Loew', 'Ethan Meyer', 'Jim Navarrete', 'Juvon Crawford', 'Lukas Balling']
  Also added 7 of them to all_rosters (uww_opponent_rosters) so they appear in the app's ROSTER panel, not just the team-stat sums.

  Roster names for Loras Duhawks (19): ['Athan Berchos', 'Brock Massey', 'Connor Mosele', 'Cooper Grams', 'Damyen Jackson', 'Deng Makeer', 'Dylan Kurt', 'Emmett Loew', 'Ethan Meyer', 'Gavin Sarvis', 'Jack Haynes', 'Jim Navarrete', 'Johnny Semany', 'Juvon Crawford', 'Kyle Kober', 'Lukas Balling', 'Nolan Berendes', 'Patrick Coen', 'Patrick Quarnstrom']
  PBP-derived names found (19): ['Athan Berchos', 'Brock Massey', 'Connor Mosele', 'Cooper Grams', 'Damyen Jackson', 'Deng Makeer', 'Dylan Kurt', 'Emmett Loew', 'Ethan Meyer', 'Gavin Sa


### Replace the upcoming opponent's TEAM totals with PBP-derived ones too

`team_totals` (team_ppg / opp_ppg_allowed, exported as uww_opponent_team_totals) has the exact same problem
the player-level override two cells up already fixed: `extract_team_totals_from_pdf` reads a "Team Total"
row straight off the same static scouting-report PDF, with no reference_date awareness at all. Same fix,
same scope limit: only the CURRENT upcoming opponent gets overridden, using `pbp_box_score_upcoming`'s own
"TEAM"/per-player rows for their games before facing UWW.

In [142]:
# --- Override the upcoming opponent's team_totals (PPG / opponent-PPG-allowed) with a PBP-derived aggregate --
# NOTE: no team-level BLK here on purpose. Checked before building it: the app's Team Stats panel already
# computes team Blocks by summing player_profiles' own BLK column and dividing by get_opponent_games_played()
# -- which are both already fixed (the player-level PBP override two cells up, and the games-played scoping
# fix from earlier this project). Sum of player blocks IS the team total, by definition of what a box score
# is, so a separate team-level BLK column here would just be unused dead weight -- confirmed nothing in the
# app reads one before adding it.
if pbp_box_score_upcoming.empty or upcoming_opponent_short is None:
    print("No prior-game box score available yet -- upcoming opponent's team_totals left as-is (PDF-sourced, if any).")
else:
    _tt_own = pbp_box_score_upcoming[pbp_box_score_upcoming["team"] == upcoming_opponent_short]
    _tt_opp = pbp_box_score_upcoming[pbp_box_score_upcoming["team"] != upcoming_opponent_short]
    _tt_n_games = pbp_box_score_upcoming["game_date"].nunique()
    if _tt_own.empty or _tt_n_games == 0:
        print(f"No PBP box score data available yet for {upcoming_opponent_short} -- team_totals left as-is.")
    else:
        _tt_team_ppg = round(_tt_own["PTS"].sum() / _tt_n_games, 2)
        _tt_opp_ppg_allowed = round(_tt_opp["PTS"].sum() / _tt_n_games, 2) if not _tt_opp.empty else None

        if team_totals.empty or upcoming_opponent_short not in set(team_totals["opponent"]):
            team_totals = pd.concat([team_totals, pd.DataFrame([{
                "opponent": upcoming_opponent_short, "team_ppg": _tt_team_ppg, "opp_ppg_allowed": _tt_opp_ppg_allowed,
            }])], ignore_index=True)
            print(f"Added a new PBP-derived team_totals row for {upcoming_opponent_short} (none existed from a PDF).")
        else:
            _tt_mask = team_totals["opponent"] == upcoming_opponent_short
            team_totals.loc[_tt_mask, "team_ppg"] = _tt_team_ppg
            if _tt_opp_ppg_allowed is not None:
                team_totals.loc[_tt_mask, "opp_ppg_allowed"] = _tt_opp_ppg_allowed
            print(f"Replaced PDF-sourced team_totals for {upcoming_opponent_short}: team_ppg={_tt_team_ppg}, "
                  f"opp_ppg_allowed={_tt_opp_ppg_allowed} (from {_tt_n_games} prior game(s) of PBP data).")


Replaced PDF-sourced team_totals for Loras Duhawks: team_ppg=80.79, opp_ppg_allowed=72.89 (from 28 prior game(s) of PBP data).


In [143]:
# --- Export the same tables as CSV files directly into the Streamlit app's source directory, so the app can
# read them as static bundled data (pandas.read_csv) instead of querying a SQL warehouse / Unity Catalog at
# runtime. (The Databricks-only Delta-table export used inside the workspace is skipped in this portable
# version -- there's no Spark session outside Databricks, and this CSV export is the app's actual data source.)
APP_DATA_DIR = OUTPUT_DIR
os.makedirs(APP_DATA_DIR, exist_ok=True)

# opponent_schedules is derived straight from the combined "schedule" frame (which already loops over every
# team's MHTML in schedules_dir -- UWW's own plus any opponent's -- and tags each row with "team").
opponent_schedules = (
    schedule[schedule["team"] != "UW-Whitewater Warhawks"]
    .rename(columns={"team": "opponent", "date": "game_date", "opponent": "vs_opponent"})
    [["opponent", "game_date", "vs_opponent", "location", "outcome", "team_score", "opponent_score", "point_margin"]]
    .reset_index(drop=True)
)

# uww_pbp_events is trimmed to the columns the app actually renders -- the full ~25-column table is unnecessarily
# large to bundle as a static CSV inside the app package.
PBP_EVENTS_EXPORT_COLS = [
    "opponent", "game_date", "event_order", "period", "time_remaining",
    "team", "player", "event_type", "raw_text", "shot_type", "uww_score", "opp_score", "uww_lineup",
    "video_description", "coach_note", "play_call",
]
# shot_type/uww_lineup were previously only exported on the clutch-events slice (CLUTCH_EVENTS_EXPORT_COLS
# below) even though both are already computed on every row of the full pbp_events DataFrame, not just
# clutch-time ones -- added here so the app can answer questions that need which 5-man UWW lineup was on the
# floor for a given shot on ANY possession, not just crunch-time ones (e.g. "which lineup gets our best shot
# type most often").

# uww_clutch_events / uww_scoring_runs -- both already computed earlier in this notebook (the "Clutch-time
# event log" and "Scoring runs and largest lead/deficit" cells) but were previously never added to csv_tables,
# so they never reached the app despite the analysis already existing. clutch_events is a filtered slice of
# pbp_events (same shape), so it needs the same column trim plus shot_type/uww_lineup/opp_lineup so the app can
# show point values and which 5-man units were on the floor during clutch stretches.
CLUTCH_EVENTS_EXPORT_COLS = [
    "opponent", "game_date", "event_order", "period", "time_remaining", "team", "player", "event_type",
    "raw_text", "shot_type", "uww_score", "opp_score", "uww_lineup", "opp_lineup", "video_description",
]
clutch_events_export = clutch_events[[c for c in CLUTCH_EVENTS_EXPORT_COLS if c in clutch_events.columns]]

# opponent column = whoever the upcoming opponent ACTUALLY played in that game (not always Whitewater --
# these are their games BEFORE facing Whitewater), team column = which side that specific row's event
# belongs to (the upcoming opponent's own play, or their opponent-in-that-game's play). Filtering
# team != upcoming_opponent_short gives exactly "what other teams did against this opponent's defense" --
# previously computed inline for print/diagnostic output only (see the "opponent_events" split a few cells
# up) and never exported, so the app had no way to use it at all.
OPPONENT_PRIOR_PBP_EXPORT_COLS = [
    "opponent", "game_date", "event_order", "period", "time_remaining", "time_remaining_seconds",
    "team", "player", "event_type", "raw_text", "shot_type", "video_description",
]
# NOTE: no per-event lineup columns here (unlike PBP_EVENTS_EXPORT_COLS' "uww_lineup" for UWW's own games) --
# self_lineup/their_lineup only exist on pbp_up, a local copy made a couple cells up for the season-lineup-box
# computation, never attached back onto pbp_events_upcoming itself. Nothing currently needs them at the
# per-event level for the opponent's prior games (uww_opp_lineup_season_box already covers the season-
# aggregate use case, and the minutes-played cell re-derives what it needs from stint_src directly).

csv_tables = {
    "uww_schedule": schedule,
    "uww_season_stats": stats,
    "uww_pbp_events": pbp_events[PBP_EVENTS_EXPORT_COLS],
    "uww_pbp_box_score": pbp_box_score,
    "uww_lineup_stints": lineup_stints,
    "uww_coaching_flags": coaching_flags_df,
    "uww_opponent_rosters": all_rosters,
    "uww_player_profiles": player_profiles,
    "uww_opponent_game_plans": all_game_plans,
    "uww_ktv_splits": splits,
    "uww_ktv_game_categories": game_categories,
    "uww_pbp_derived_keys": pbp_derived_keys,
    "uww_opponent_team_totals": team_totals,
    "uww_projected_box_score": projected_uww_box,
    "uww_opponent_projected_box_score": projected_opponent_box,
    "uww_player_comparisons": best_matches,
    "uww_opp_lineup_season_box": upcoming_lineup_season,
    "uww_opponent_schedules": opponent_schedules,
    "uww_clutch_events": clutch_events_export,
    "uww_scoring_runs": scoring_runs,
    "uww_coach_notes": coach_notes,
    "uww_opponent_prior_games_pbp": pbp_events_upcoming[[c for c in OPPONENT_PRIOR_PBP_EXPORT_COLS if c in pbp_events_upcoming.columns]],
    "uww_opponent_prior_games_box_score": pbp_box_score_upcoming,
    "uww_opponent_prior_games_lineup_stints": lineup_stints_upcoming,
}

csv_export_status = []
for name, df in csv_tables.items():
    path = os.path.join(APP_DATA_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    csv_export_status.append((name, len(df), os.path.getsize(path)))

print(pd.DataFrame(csv_export_status, columns=["table", "rows", "csv_bytes"]))

                                     table   rows  csv_bytes
0                             uww_schedule    541     252796
1                         uww_season_stats     25       2707
2                           uww_pbp_events  13677    3278773
3                        uww_pbp_box_score    657      74495
4                        uww_lineup_stints   1306     298389
5                       uww_coaching_flags     38      12477
6                     uww_opponent_rosters    169      20267
7                      uww_player_profiles    233      55532
8                  uww_opponent_game_plans    261      54420
9                           uww_ktv_splits      9        598
10                 uww_ktv_game_categories     33       2302
11                    uww_pbp_derived_keys      3        903
12                uww_opponent_team_totals     19        608
13                 uww_projected_box_score     23       5843
14        uww_opponent_projected_box_score     18       6788
15                  uww_


### Extract and save team logos from PBP MHTML files

Pulls each team's logo image out of the play-by-play MHTML snapshots and saves it into the app's `data/logo` directory, for the Streamlit app's UI.

In [145]:
# --- Extract team logos embedded in the PBP/schedule MHTML files and save them into the Streamlit app's
# data/logo/ directory. Each MHTML archive (Chrome "Save as Webpage, Single File") bundles referenced images
# as binary MIME parts. Team logos appear under two URL patterns:
#   1. https://download.fastmodeltechnologies.com/FSimages/logos/NCAAB-III/<Team>.png
#   2. https://stats-assets.fastmodelsports.com/images/teams/<Team>  (alternate, e.g. Simpson)
# The filename becomes "<Team>.png" -- matching the short_opponent names used elsewhere in the app.
from urllib.parse import unquote as _logo_unquote

_LOGO_DIR = os.path.join(APP_DATA_DIR, "logo")
os.makedirs(_LOGO_DIR, exist_ok=True)

_LOGO_URL_PATTERN_1 = "https://download.fastmodeltechnologies.com/FSimages/logos/NCAAB-III/"
_LOGO_URL_PATTERN_2 = "https://stats-assets.fastmodelsports.com/images/teams/"

_logo_mhtml_files = sorted(glob.glob(f"{volume_dir}/*.mhtml"))

_logos_saved = {}
for _mf in _logo_mhtml_files:
    with open(_mf, "rb") as _f:
        _raw = _f.read()
    _msg = email.message_from_bytes(_raw, policy=policy.default)
    for _part in _msg.walk():
        _loc = _part.get("Content-Location", "")
        _ct = _part.get_content_type()
        if not _ct.startswith("image/"):
            continue
        _team_name = None
        if _LOGO_URL_PATTERN_1 in _loc:
            _team_name = _logo_unquote(_loc.replace(_LOGO_URL_PATTERN_1, "").replace(".png", ""))
        elif _LOGO_URL_PATTERN_2 in _loc:
            _team_name = _logo_unquote(_loc.split("/")[-1])
        if _team_name and _team_name not in _logos_saved:
            _payload = _part.get_payload(decode=True)
            if _payload and len(_payload) > 100:
                _filepath = os.path.join(_LOGO_DIR, f"{_team_name}.png")
                with open(_filepath, "wb") as _out:
                    _out.write(_payload)
                _logos_saved[_team_name] = len(_payload)

print(f"Saved {len(_logos_saved)} team logos to {_LOGO_DIR}:")
for _name in sorted(_logos_saved):
    print(f"  - {_name} ({_logos_saved[_name]:,} bytes)")

Saved 21 team logos to ../data\logo:
  - Alma (226,793 bytes)
  - Aurora (136,971 bytes)
  - Carroll (WI) (109,313 bytes)
  - Coe (115,946 bytes)
  - Elmhurst (118,533 bytes)
  - Eureka (150,170 bytes)
  - Hope (66,665 bytes)
  - Lawrence (84,599 bytes)
  - Loras (95,745 bytes)
  - Ripon (87,371 bytes)
  - Simpson (76,108 bytes)
  - St. Thomas (TX) (188,379 bytes)
  - UW-Eau Claire (70,774 bytes)
  - UW-La Crosse (26,203 bytes)
  - UW-Oshkosh (166,949 bytes)
  - UW-Platteville (71,166 bytes)
  - UW-River Falls (89,021 bytes)
  - UW-Stevens Point (139,555 bytes)
  - UW-Stout (194,873 bytes)
  - UW-Whitewater (145,562 bytes)
  - Washington-St. Louis (87,899 bytes)



### Extract player headshots from scouting report PDFs

Pulls each scouted player's headshot image out of their FastScout scouting-report PDF, for the Streamlit app's player-comparison UI.

In [147]:
# --- Extract player headshot images from scouting reports (PDF or HTML) and save them to the Streamlit
# app's data/player_images/ directory.
# For PDFs: identify headshots by eliminating repeated header images, keeping portrait-oriented images on
#   roster pages, and matching to player names by vertical position order.
# For HTMLs: find <img> elements within playerGroup tiles and download from their URLs, matching to the
#   player name parsed from the same tile's player-info-line spans.
import re as _pi_re
import urllib.request
from urllib.parse import urlparse, quote, urlunparse
from collections import Counter as _PICounter


try:
    import fitz  # pymupdf -- only needed for PDF reports
except ModuleNotFoundError:
    fitz = None  # no PDFs to process if pymupdf isn't installed


_PI_OUTPUT_DIR = os.path.join(APP_DATA_DIR, "player_images")
os.makedirs(_PI_OUTPUT_DIR, exist_ok=True)


_PI_PLAYER_PATTERN = _pi_re.compile(r"#\d+\s*[\u2022\u00b7]\s*(.+?)\s*[\u2022\u00b7]\s*[GCFPG/]+\s*[\u2022\u00b7]")




# --- HTML headshot extraction: find <img> in each playerGroup tile, download from URL ---
def _extract_headshots_from_html(html_path, opponent_name, output_dir):
    """Extract player headshots from an HTML scout report by finding <img> elements in playerGroup tiles."""
    with open(html_path, "r", encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html, "html.parser")
    printable = soup.find(class_="PrintableNode")
    if printable is None:
        return {}, {}, []


    extracted = {}
    skipped = {}
    _pending_downloads = []  # URLs that need authenticated download via Playwright


    # Diagnostic: count playerGroup tiles found
    _pg_tiles = [t for t in printable.find_all("div") if "Tile" in (t.get("class") or []) and "playerGroup" in (t.get("class") or [])]
    print(f"    [DIAG] Found {len(_pg_tiles)} playerGroup tiles in PrintableNode")
    if not _pg_tiles:
        # Try alternate: find any div with 'player' in class name
        _alt_tiles = [t for t in printable.find_all("div") if any("player" in c.lower() for c in (t.get("class") or []))]
        print(f"    [DIAG] Alternate: {len(_alt_tiles)} divs with 'player' in class")
        if _alt_tiles:
            print(f"    [DIAG] First alt classes: {_alt_tiles[0].get('class')}")
        # Also show all unique Tile classes
        _all_tiles = [t for t in printable.find_all("div") if "Tile" in (t.get("class") or [])]
        _tile_class_sets = set()
        for _t in _all_tiles[:20]:
            _tile_class_sets.add(tuple(sorted(_t.get("class", []))))
        print(f"    [DIAG] Unique Tile class combos (first 20): {list(_tile_class_sets)[:10]}")
    else:
        # Show first tile's structure in detail
        _first = _pg_tiles[0]
        _has_info = _first.find("span", class_="player-info-line")
        _has_img = _first.find("img")
        print(f"    [DIAG] First playerGroup: has player-info-line={_has_info is not None}, has img={_has_img is not None}")
        if _has_img:
            print(f"    [DIAG] First img src: {_has_img.get('src', '')[:100]}")
        if _has_info:
            _info_div = _has_info.find("div", class_=lambda c: c and "display-flex" in c)
            print(f"    [DIAG] info_div found: {_info_div is not None}")
            if _info_div is None:
                # Show what divs exist inside player-info-line
                _inner_divs = _has_info.find_all("div", limit=5)
                print(f"    [DIAG] Divs inside player-info-line: {[(d.get('class'), d.get_text()[:50]) for d in _inner_divs]}")
            else:
                _fspans = _info_div.find_all("span", recursive=False)
                _flds = [s.get_text(" ", strip=True) for s in _fspans]
                _itext = " \u2022 ".join(f for f in _flds if f)
                print(f"    [DIAG] info_text = {repr(_itext[:120])}")
                _nmatch = _PI_PLAYER_PATTERN.search(_itext)
                print(f"    [DIAG] regex match: {_nmatch is not None}")
                if _nmatch:
                    print(f"    [DIAG] captured name: {repr(_nmatch.group(1))}")
        else:
            _spans = _first.find_all("span", limit=5)
            print(f"    [DIAG] First tile spans: {[(s.get('class'), s.get_text()[:40]) for s in _spans]}")


    for tile in printable.find_all("div"):
        classes = tile.get("class", [])
        if "Tile" not in classes or "playerGroup" not in classes:
            continue


        # Extract player name from player-info-line
        info_span = tile.find("span", class_="player-info-line")
        if info_span is None:
            continue
        info_div = info_span.find("div", class_=lambda c: c and "display-flex" in c)
        if info_div is None:
            continue
        field_spans = info_div.find_all("span", recursive=False)
        fields = [s.get_text(" ", strip=True) for s in field_spans]
        info_text = " \u2022 ".join(f for f in fields if f)
        name_match = _PI_PLAYER_PATTERN.search(info_text)
        if not name_match:
            continue
        player_name = name_match.group(1).strip()
        safe_name = _pi_re.sub(r'[^\w\s\-]', '', player_name).strip()


        # Find the headshot <img> in this tile -- prefer the real HTTP headshot URL over
        # inline data: placeholders. FastScout headshot URLs typically contain "personnel",
        # "images/personnel/", "media-attachments", "headshot", "player", or "FSimages".
        img_tag = None
        for img in tile.find_all("img"):
            src = img.get("src", "")
            if not src or src.endswith(".svg") or "logo" in src.lower():
                continue
            # Never let a data: URI overwrite an already-found HTTP img
            if src.startswith("data:") and img_tag is not None:
                continue
            img_tag = img
            if "headshot" in src.lower() or "player" in src.lower() or "personnel" in src.lower() or "FSimages" in src or "media-attachments" in src:
                break  # best candidate found


        if img_tag is None:
            continue
        img_src = img_tag.get("src", "")


        filepath = os.path.join(output_dir, f"{safe_name}.png")
        if os.path.exists(filepath):
            skipped[safe_name] = opponent_name
            continue


        try:
            if img_src.startswith("data:"):
                import base64
                header, b64data = img_src.split(",", 1)
                img_bytes = base64.b64decode(b64data)
            else:
                # URL-encode path segments (spaces, parens, etc.) while keeping scheme/host intact
                _parsed = urlparse(img_src)
                _encoded_url = urlunparse(_parsed._replace(path=quote(_parsed.path, safe="/")))
                req = urllib.request.Request(_encoded_url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=15) as resp:
                    img_bytes = resp.read()


            if len(img_bytes) > 500:  # skip tiny placeholders
                with open(filepath, "wb") as out_f:
                    out_f.write(img_bytes)
                extracted[safe_name] = opponent_name
            elif img_src.startswith(("http://", "https://")):
                # CDN returned auth-wall placeholder -- collect for Playwright batch download
                # (only HTTP(S) URLs can be re-fetched with auth; data: URIs are genuinely empty)
                _pending_downloads.append((player_name, safe_name, img_src, filepath, opponent_name))
        except Exception as dl_err:
            print(f"    [WARN] Could not download headshot for {player_name}: {type(dl_err).__name__}: {dl_err}")


    # --- Collect pending downloads (do NOT attempt async here -- the caller batches them) ---
    if _pending_downloads:
        print(f"    [AUTH] {len(_pending_downloads)} headshot(s) need authenticated download (deferred to batch)")


    return extracted, skipped, _pending_downloads




# --- PDF headshot extraction (original PyMuPDF approach) ---
def _extract_headshots_from_pdf(pdf_path, opponent_name, output_dir):
    """Extract player headshots from a PDF scout report using PyMuPDF."""
    if fitz is None:
        print(f"    [SKIP] pymupdf not installed -- cannot process PDF: {os.path.basename(pdf_path)}")
        return {}, {}, {}


    doc = fitz.open(pdf_path)
    extracted = {}
    skipped = {}
    diag = {"file": os.path.basename(pdf_path), "format": "pdf", "pages": doc.page_count,
            "total_images": 0, "candidate_headshots": 0, "names_matched": 0}


    xref_count = _PICounter()
    for pn in range(doc.page_count):
        for img in doc[pn].get_images(full=True):
            xref_count[img[0]] += 1
    header_xrefs = {xref for xref, cnt in xref_count.items() if cnt >= 3}


    for pn in range(doc.page_count):
        page = doc[pn]
        images = page.get_images(full=True)
        diag["total_images"] += len(images)


        headshots = []
        for img in images:
            xref = img[0]
            if xref in header_xrefs:
                continue
            base = doc.extract_image(xref)
            w, h = base['width'], base['height']
            if h >= w * 0.8 and len(base['image']) > 5000:
                rects = page.get_image_rects(xref)
                if rects:
                    headshots.append({'y': rects[0].y0, 'image_data': base['image'], 'ext': base['ext']})


        if not headshots:
            continue
        diag["candidate_headshots"] += len(headshots)
        headshots.sort(key=lambda x: x['y'])


        text = page.get_text("text")
        names = _PI_PLAYER_PATTERN.findall(text)
        diag["names_matched"] += len(names)


        for i in range(min(len(headshots), len(names))):
            name = names[i].strip()
            safe = _pi_re.sub(r'[^\w\s\-]', '', name).strip()
            filename = f"{safe}.{headshots[i]['ext']}"
            filepath = os.path.join(output_dir, filename)
            if os.path.exists(filepath):
                skipped[safe] = opponent_name
            else:
                with open(filepath, 'wb') as out_f:
                    out_f.write(headshots[i]['image_data'])
                extracted[safe] = opponent_name


    doc.close()
    return extracted, skipped, diag




# --- Main loop: process all scout reports (HTML and PDF) ---
_pi_all_extracted = {}
_pi_skipped = {}
_pi_all_pending = []  # (player_name, safe_name, url, filepath, opponent_name) tuples deferred for batch auth download
_pi_diag = []


print(f"Processing {len(scout_report_files)} scout report(s) from scout_report_files...")
for _pi_path in sorted(scout_report_files):
    _pi_file = os.path.basename(_pi_path)
    _pi_opponent = opponent_from_scout_filename(_pi_path)


    if _pi_path.lower().endswith(".html"):
        _ext, _skip, _pending = _extract_headshots_from_html(_pi_path, _pi_opponent, _PI_OUTPUT_DIR)
        _pi_all_extracted.update(_ext)
        _pi_skipped.update(_skip)
        _pi_all_pending.extend(_pending)
        _pi_diag.append({"file": _pi_file, "format": "html", "extracted": len(_ext), "skipped": len(_skip)})
    else:
        _ext, _skip, _d = _extract_headshots_from_pdf(_pi_path, _pi_opponent, _PI_OUTPUT_DIR)
        _pi_all_extracted.update(_ext)
        _pi_skipped.update(_skip)
        if _d:
            _pi_diag.append(_d)


# --- Batch authenticated download of ALL pending headshots in a single Playwright session ---
# run_in_fastscout_session is SYNCHRONOUS (dispatches fn(page) on a dedicated worker thread using
# Playwright's sync API) -- no async/await/event-loop needed here.
if _pi_all_pending:
    print(f"\n[AUTH] Batch downloading {len(_pi_all_pending)} headshot(s) via authenticated Playwright session...")
    try:
        def _batch_download_headshots(page):
            downloaded = 0
            for _pname, _sname, _url, _fpath, _opp in _pi_all_pending:
                try:
                    _resp = page.request.get(_url)
                    if _resp.ok:
                        _body = _resp.body()
                        if len(_body) > 500:
                            with open(_fpath, "wb") as _f:
                                _f.write(_body)
                            _pi_all_extracted[_sname] = _opp
                            downloaded += 1
                        else:
                            print(f"  [SKIP] {_pname}: authenticated resp still only {len(_body)} bytes")
                    else:
                        print(f"  [SKIP] {_pname}: HTTP {_resp.status}")
                except Exception as _e:
                    print(f"  [WARN] {_pname}: {type(_e).__name__}: {_e}")
            return downloaded


        _pi_pw_downloaded = run_in_fastscout_session(_batch_download_headshots)
        if _pi_pw_downloaded:
            print(f"[OK] Downloaded {_pi_pw_downloaded}/{len(_pi_all_pending)} headshot(s) via authenticated session")
        else:
            print(f"[INFO] Playwright session connected but no images exceeded 500 bytes -- "
                  f"headshots may not be uploaded on FastScout for these teams.")
    except Exception as _sess_err:
        print(f"[INFO] Cannot authenticate for headshot download: {type(_sess_err).__name__}: {_sess_err}")
        print(f"[INFO] Set FASTSCOUT_USERNAME/FASTSCOUT_PASSWORD env vars and re-run to download headshots.")
else:
    print("\n[INFO] No headshots required authenticated download.")


print(f"\nExtracted {len(_pi_all_extracted)} new player headshot(s) to {_PI_OUTPUT_DIR}")
if _pi_skipped:
    print(f"Skipped {len(_pi_skipped)} already-on-disk headshot(s)")
for _pi_name, _pi_opp in sorted(_pi_all_extracted.items(), key=lambda x: x[1]):
    print(f"  [NEW] [{_pi_opp}] {_pi_name}")


if not _pi_all_extracted and not _pi_skipped:
    print("\n--- DIAGNOSTIC: 0 headshots found. Per-report breakdown: ---")
    if not _pi_diag:
        print("  scout_pdf_files was EMPTY -- no reports to process.")
    for _d in _pi_diag:
        if _d.get("format") == "html":
            print(f"  {_d['file']}: HTML format, {_d['extracted']} extracted, {_d['skipped']} skipped")
        else:
            print(f"  {_d['file']}: {_d.get('pages', '?')} pages, {_d.get('total_images', 0)} images, "
                  f"{_d.get('candidate_headshots', 0)} candidates, {_d.get('names_matched', 0)} names matched")



Processing 24 scout report(s) from scout_report_files...
    [DIAG] Found 9 playerGroup tiles in PrintableNode
    [DIAG] First playerGroup: has player-info-line=True, has img=True
    [DIAG] First img src: https://stats-assets.fastmodelsports.com/images/personnel/St. Thomas (TX)/2025-2026/Angel Johnson/17
    [DIAG] info_div found: True
    [DIAG] info_text = '#10 • Angel Johnson • G • 6\'2" • SR'
    [DIAG] regex match: True
    [DIAG] captured name: 'Angel Johnson'
    [DIAG] Found 7 playerGroup tiles in PrintableNode
    [DIAG] First playerGroup: has player-info-line=True, has img=True
    [DIAG] First img src: https://stats-assets.fastmodelsports.com/images/personnel/Eureka/2025-2026/Jaxson Provost/1762669410
    [DIAG] info_div found: True
    [DIAG] info_text = '#12 • Jaxson Provost • G • 5\'10" • 170 lbs • JR'
    [DIAG] regex match: True
    [DIAG] captured name: 'Jaxson Provost'
    [DIAG] Found 9 playerGroup tiles in PrintableNode
    [DIAG] First playerGroup: has player-inf